In [ ]:
import pandas as pd

raw_path = "../data/raw/"

d_items = pd.read_csv(raw_path + "d_items.csv.gz", compression="gzip")
d_labitems = pd.read_csv(raw_path + "d_labitems.csv.gz", compression="gzip")

print(f"d_items shape: {d_items.shape}")
print(f"d_labitems shape: {d_labitems.shape}")
print()

def search_ditems(patterns, label_name):
    pattern = '|'.join(patterns)
    matches = d_items[d_items['label'].str.contains(pattern, case=False, na=False, regex=True)].copy()
    print(f"--- {label_name} (d_items) — {len(matches)} matches ---")
    if len(matches) > 0:
        print(matches[['itemid', 'label', 'abbreviation', 'linksto', 'category', 'unitname', 'param_type']].to_string(index=False))
    else:
        print("  NO MATCHES FOUND")
    print()
    return matches

def search_dlabitems(patterns, label_name):
    pattern = '|'.join(patterns)
    matches = d_labitems[d_labitems['label'].str.contains(pattern, case=False, na=False, regex=True)].copy()
    print(f"--- {label_name} (d_labitems) — {len(matches)} matches ---")
    if len(matches) > 0:
        cols = [c for c in ['itemid', 'label', 'fluid', 'category'] if c in matches.columns]
        print(matches[cols].to_string(index=False))
    else:
        print("  NO MATCHES FOUND")
    print()
    return matches

# ============================================================
# Vasopressors, FiO2, Urine Output — for Organ-System Temporal Burden Index
# ============================================================

print("=" * 65)
print("ORGAN-SYSTEM TEMPORAL BURDEN INDEX / TPIFE CANDIDATE ITEMS")
print("=" * 65)
print()

vaso_matches = search_ditems(['vasopressin', 'norepinephrine', 'epinephrine'], "Vasopressors")
fio2_matches = search_ditems(['FiO2', 'inspired o2', 'fraction of inspired'], "FiO2")
uo_matches = search_ditems(['urine output', 'urine ', 'foley', 'void'], "Urine Output")

# ============================================================
# Core vitals — from d_items (chartevents)
# ============================================================

print("=" * 65)
print("CORE VITALS (chartevents / d_items)")
print("=" * 65)
print()

hr_matches = search_ditems(['heart rate'], "Heart Rate")
sbp_matches = search_ditems(['systolic'], "Systolic BP")
dbp_matches = search_ditems(['diastolic'], "Diastolic BP")
rr_matches = search_ditems(['respiratory rate'], "Respiratory Rate")
temp_matches = search_ditems(['temperature'], "Temperature")
spo2_matches = search_ditems(['spo2', 'o2 saturation', 'oxygen saturation'], "SpO2")
gcs_matches = search_ditems(['glasgow', 'gcs'], "GCS (all components)")

# ============================================================
# Labs — primarily in d_labitems (labevents), but ALSO check d_items
# in case any are point-of-care/bedside charted values in chartevents
# ============================================================

print("=" * 65)
print("LABS — checking d_labitems (primary source) AND d_items (bedside/POC check)")
print("=" * 65)
print()

print(">>> NOTE: WBC, lactate, creatinine, platelets, and bilirubin are lab values that")
print(">>> typically live in labevents/d_labitems, NOT chartevents/d_items. Checking both")
print(">>> so we don't miss a point-of-care/bedside-charted duplicate, but d_labitems is")
print(">>> the expected primary source for these five.")
print()

wbc_lab = search_dlabitems(['white blood cell', 'wbc'], "WBC")
lactate_lab = search_dlabitems(['lactate'], "Lactate")
creatinine_lab = search_dlabitems(['creatinine'], "Creatinine")
platelets_lab = search_dlabitems(['platelet'], "Platelets")
bilirubin_lab = search_dlabitems(['bilirubin'], "Bilirubin")

wbc_items = search_ditems(['white blood cell', 'wbc'], "WBC (d_items check)")
lactate_items = search_ditems(['lactate'], "Lactate (d_items check)")
creatinine_items = search_ditems(['creatinine'], "Creatinine (d_items check)")
platelets_items = search_ditems(['platelet'], "Platelets (d_items check)")
bilirubin_items = search_ditems(['bilirubin'], "Bilirubin (d_items check)")

print("=" * 65)
print("REVIEW CHECKLIST — confirm before any extraction happens:")
print("=" * 65)
print("- For vitals/vasopressors/FiO2/urine output with multiple itemids, check 'linksto'")
print("  and 'category' to confirm which era (CareVue vs MetaVision) each belongs to —")
print("  as with CAM-ICU/RASS earlier, MIMIC-IV often has 2-3 itemids per concept across eras.")
print("- For vasopressors specifically: rate/dose units matter (mcg/kg/min vs mcg/min vs units/hr)")
print("  — do NOT assume they're comparable across itemids without checking 'unitname'.")
print("- For labs: confirm whether d_labitems or the d_items bedside check found the real source")
print("  before deciding which table (labevents vs chartevents) to pull from for each.")
print("- Nothing has been extracted yet. No chartevents/labevents data pulled in this cell.")

In [ ]:
import pandas as pd
import numpy as np

raw_path = "../data/raw/"
processed_path = "../data/processed/"

# ============================================================
# STEP 1: Search + print Dopamine / Phenylephrine matches BEFORE extracting
# ============================================================

d_items = pd.read_csv(raw_path + "d_items.csv.gz", compression="gzip")

vaso_extra = d_items[
    d_items['label'].str.contains('dopamine', case=False, na=False) |
    d_items['label'].str.contains('phenylephrine', case=False, na=False)
].copy()

print("=" * 65)
print("DOPAMINE / PHENYLEPHRINE — d_items matches")
print("=" * 65)
print(vaso_extra[['itemid', 'label', 'abbreviation', 'linksto', 'category', 'unitname']].to_string(index=False))
print()

dopamine_phenylephrine_itemids = set(vaso_extra['itemid'])
print(f"Itemids being added to vasopressor list: {sorted(dopamine_phenylephrine_itemids)}")
print()

# ============================================================
# STEP 2: Build the full confirmed itemid -> variable mapping
# ============================================================

vasopressor_itemids = {221289, 229617, 221906, 222315} | dopamine_phenylephrine_itemids

chartevents_map = {
    220045: ('HR', None),
    220050: ('SBP', None), 220179: ('SBP', None),
    220051: ('DBP', None), 220180: ('DBP', None),
    220210: ('RR', None),
    223762: ('Temp_C', 'C'), 223761: ('Temp_F', 'F'),  # handled specially, converted to Temp_C
    220277: ('SpO2', None),
    220739: ('GCS_Eye', None),
    223900: ('GCS_Verbal', None),
    223901: ('GCS_Motor', None),
    223835: ('FiO2', None),
}

labevents_map = {
    51301: 'WBC', 51300: 'WBC',
    50813: 'Lactate', 52442: 'Lactate', 53154: 'Lactate',
    50912: 'Creatinine', 52546: 'Creatinine',
    51265: 'Platelets', 53189: 'Platelets',
    50885: 'Bilirubin', 53089: 'Bilirubin',
}

vasopressor_labels = {iid: 'Vasopressor' for iid in vasopressor_itemids}
urine_itemids = {226559, 226560}

print("Full chartevents vitals/GCS/FiO2 itemid map:")
print(chartevents_map)
print()
print("Full labevents lab itemid map:")
print(labevents_map)
print()
print(f"Full vasopressor itemid set (inputevents): {sorted(vasopressor_itemids)}")
print(f"Urine output itemid set (outputevents): {sorted(urine_itemids)}")
print()

# ============================================================
# Load cohort_step1.csv for subject_id + intime (24h window anchor)
# ============================================================

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])
cohort_subject_ids = set(cohort_step1['subject_id'])
intime_lookup = cohort_step1.set_index('subject_id')['intime']

print(f"Cohort loaded: {len(cohort_step1)} patients")
print()

def clip_to_24h(df, time_col):
    """Merge intime, keep only rows within [intime, intime+24h]."""
    df = df.merge(intime_lookup.rename('intime'), left_on='subject_id', right_index=True, how='left')
    df['hours_from_intime'] = (df[time_col] - df['intime']).dt.total_seconds() / 3600
    return df[(df['hours_from_intime'] >= 0) & (df['hours_from_intime'] <= 24)].copy()

# ============================================================
# STEP 3: Chunk-read chartevents (vitals/GCS/FiO2)
# ============================================================

chart_itemids = set(chartevents_map.keys())
chunk_size = 1_000_000
chart_rows = []

for chunk in pd.read_csv(
    raw_path + "chartevents.csv.gz", compression="gzip",
    usecols=['subject_id', 'hadm_id', 'itemid', 'charttime', 'value', 'valuenum'],
    chunksize=chunk_size
):
    filtered = chunk[chunk['itemid'].isin(chart_itemids) & chunk['subject_id'].isin(cohort_subject_ids)]
    if len(filtered) > 0:
        chart_rows.append(filtered)

chart_events = pd.concat(chart_rows, ignore_index=True) if chart_rows else pd.DataFrame(
    columns=['subject_id', 'hadm_id', 'itemid', 'charttime', 'value', 'valuenum'])
chart_events['charttime'] = pd.to_datetime(chart_events['charttime'])

print(f"Chartevents rows pulled (vitals/GCS/FiO2, cohort + itemid filtered): {len(chart_events)}")

chart_events_24h = clip_to_24h(chart_events, 'charttime')
print(f"Chartevents rows within first 24h: {len(chart_events_24h)}")
print()

# --- Temperature: convert F -> C, unify into single 'Temp' variable ---
temp_f_mask = chart_events_24h['itemid'] == 223761
chart_events_24h.loc[temp_f_mask, 'valuenum'] = (chart_events_24h.loc[temp_f_mask, 'valuenum'] - 32) * 5 / 9

# --- GCS Verbal: impute 1 for intubated/non-verbalizable (text like 'No Response-ETT') ---
verbal_mask = chart_events_24h['itemid'] == 223900
nonverbal_text_mask = verbal_mask & chart_events_24h['valuenum'].isna() & chart_events_24h['value'].astype(str).str.contains(
    'no response|ett|trach|intubat', case=False, na=False)
chart_events_24h.loc[nonverbal_text_mask, 'valuenum'] = 1
n_imputed_verbal = nonverbal_text_mask.sum()
print(f"GCS Verbal rows imputed to 1 (intubated/non-verbalizable text detected): {n_imputed_verbal}")
print()

# Assign unified variable name (Temp_F rows now represent converted Temp_C values)
def map_variable(itemid):
    name, _ = chartevents_map.get(itemid, (None, None))
    if name in ('Temp_C', 'Temp_F'):
        return 'Temp'
    return name

chart_events_24h['variable'] = chart_events_24h['itemid'].apply(map_variable)

vitals_long = chart_events_24h[['subject_id', 'charttime', 'itemid', 'valuenum', 'variable']].dropna(subset=['valuenum']).copy()

# ============================================================
# STEP 4: Chunk-read labevents
# ============================================================

lab_itemids = set(labevents_map.keys())
lab_rows = []

for chunk in pd.read_csv(
    raw_path + "labevents.csv.gz", compression="gzip",
    usecols=['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum'],
    chunksize=chunk_size
):
    filtered = chunk[chunk['itemid'].isin(lab_itemids) & chunk['subject_id'].isin(cohort_subject_ids)]
    if len(filtered) > 0:
        lab_rows.append(filtered)

lab_events = pd.concat(lab_rows, ignore_index=True) if lab_rows else pd.DataFrame(
    columns=['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum'])
lab_events['charttime'] = pd.to_datetime(lab_events['charttime'])

print(f"Labevents rows pulled (cohort + itemid filtered): {len(lab_events)}")

lab_events_24h = clip_to_24h(lab_events, 'charttime')
print(f"Labevents rows within first 24h: {len(lab_events_24h)}")
print()

lab_events_24h['variable'] = lab_events_24h['itemid'].map(labevents_map)
labs_long = lab_events_24h[['subject_id', 'charttime', 'itemid', 'valuenum', 'variable']].dropna(subset=['valuenum']).copy()

# ============================================================
# Combine vitals + labs -> vitals_labs_24h.csv
# ============================================================

vitals_labs_24h = pd.concat([
    vitals_long[['subject_id', 'charttime', 'itemid', 'valuenum']],
    labs_long[['subject_id', 'charttime', 'itemid', 'valuenum']]
], ignore_index=True)

# Keep a variable-name lookup for coverage reporting (not saved in the CSV itself,
# since the spec is subject_id, charttime, itemid, valuenum only)
vitals_labs_var_lookup = pd.concat([
    vitals_long[['subject_id', 'itemid', 'variable']],
    labs_long[['subject_id', 'itemid', 'variable']]
], ignore_index=True)

vitals_labs_24h.to_csv(processed_path + "vitals_labs_24h.csv", index=False)
print(f"Saved vitals_labs_24h.csv: {len(vitals_labs_24h)} rows")
print()

# ============================================================
# STEP 5: inputevents (vasopressors) — different schema: starttime/rate, not charttime/valuenum
# ============================================================

vaso_rows = []
input_usecols_try = ['subject_id', 'hadm_id', 'itemid', 'starttime', 'endtime', 'rate', 'amount']

for chunk in pd.read_csv(
    raw_path + "inputevents.csv.gz", compression="gzip",
    usecols=lambda c: c in input_usecols_try,
    chunksize=chunk_size
):
    filtered = chunk[chunk['itemid'].isin(vasopressor_itemids) & chunk['subject_id'].isin(cohort_subject_ids)]
    if len(filtered) > 0:
        vaso_rows.append(filtered)

vaso_events = pd.concat(vaso_rows, ignore_index=True) if vaso_rows else pd.DataFrame(columns=input_usecols_try)
vaso_events['starttime'] = pd.to_datetime(vaso_events['starttime'])

print(f"Inputevents (vasopressor) rows pulled: {len(vaso_events)}")

vaso_events_24h = clip_to_24h(vaso_events, 'starttime')
print(f"Vasopressor rows within first 24h (by starttime): {len(vaso_events_24h)}")

# valuenum = rate if present, else amount as fallback (documented, not silently dropped)
vaso_events_24h['valuenum'] = vaso_events_24h['rate']
rate_missing_mask = vaso_events_24h['valuenum'].isna()
vaso_events_24h.loc[rate_missing_mask, 'valuenum'] = vaso_events_24h.loc[rate_missing_mask, 'amount']
print(f"  (rate missing, fell back to amount for: {rate_missing_mask.sum()} rows)")
print("  NOTE: rate units are NOT harmonized across itemids here (mcg/kg/min vs mcg/min vs units/hr etc.)")
print("  — check 'unitname' per itemid from the d_items printout before using this as a numeric feature.")
print()

vaso_events_24h = vaso_events_24h.rename(columns={'starttime': 'charttime'})
vaso_long = vaso_events_24h[['subject_id', 'charttime', 'itemid', 'valuenum']].dropna(subset=['valuenum']).copy()

# ============================================================
# STEP 6: chartevents FiO2 already pulled above (in vitals_long) — extract separately for organ_burden file
# ============================================================

fio2_long = vitals_long[vitals_long['variable'] == 'FiO2'][['subject_id', 'charttime', 'itemid', 'valuenum']].copy()

# ============================================================
# STEP 7: outputevents (urine output) — raw events, long format (aggregation left for Notebook 2)
# ============================================================

urine_rows = []
output_usecols_try = ['subject_id', 'hadm_id', 'itemid', 'charttime', 'value']

for chunk in pd.read_csv(
    raw_path + "outputevents.csv.gz", compression="gzip",
    usecols=lambda c: c in output_usecols_try,
    chunksize=chunk_size
):
    filtered = chunk[chunk['itemid'].isin(urine_itemids) & chunk['subject_id'].isin(cohort_subject_ids)]
    if len(filtered) > 0:
        urine_rows.append(filtered)

urine_events = pd.concat(urine_rows, ignore_index=True) if urine_rows else pd.DataFrame(columns=output_usecols_try)
urine_events['charttime'] = pd.to_datetime(urine_events['charttime'])

print(f"Outputevents (urine) rows pulled: {len(urine_events)}")

urine_events_24h = clip_to_24h(urine_events, 'charttime')
print(f"Urine output rows within first 24h: {len(urine_events_24h)}")
print("  NOTE: saved as RAW per-event volumes (long format, per your file spec) — NOT pre-summed.")
print("  Summing per time window (e.g., 24h total or hourly bins) is a Notebook 2 feature-engineering")
print("  step, not applied here.")
print()

urine_events_24h = urine_events_24h.rename(columns={'value': 'valuenum'})
urine_long = urine_events_24h[['subject_id', 'charttime', 'itemid', 'valuenum']].dropna(subset=['valuenum']).copy()

# ============================================================
# Combine vasopressor + FiO2 + urine -> organ_burden_inputs_24h.csv
# ============================================================

organ_burden_24h = pd.concat([vaso_long, fio2_long, urine_long], ignore_index=True)
organ_burden_24h.to_csv(processed_path + "organ_burden_inputs_24h.csv", index=False)
print(f"Saved organ_burden_inputs_24h.csv: {len(organ_burden_24h)} rows")
print()

# ============================================================
# STEP 8: Per-variable coverage % (of 3122 cohort patients)
# ============================================================

n_cohort = len(cohort_step1)

print("=" * 65)
print("PER-VARIABLE COVERAGE (% of 3122 cohort patients with >=1 reading in first 24h)")
print("=" * 65)

def coverage_pct(df, subject_col='subject_id'):
    return 100 * df[subject_col].nunique() / n_cohort

coverage_rows = []
for var_name in vitals_labs_var_lookup['variable'].unique():
    subset = vitals_labs_var_lookup[vitals_labs_var_lookup['variable'] == var_name]
    coverage_rows.append({'variable': var_name, 'coverage_pct': round(coverage_pct(subset), 2),
                           'n_patients': subset['subject_id'].nunique()})

coverage_rows.append({'variable': 'Vasopressor', 'coverage_pct': round(coverage_pct(vaso_long), 2),
                       'n_patients': vaso_long['subject_id'].nunique()})
coverage_rows.append({'variable': 'FiO2', 'coverage_pct': round(coverage_pct(fio2_long), 2),
                       'n_patients': fio2_long['subject_id'].nunique()})
coverage_rows.append({'variable': 'Urine_Output', 'coverage_pct': round(coverage_pct(urine_long), 2),
                       'n_patients': urine_long['subject_id'].nunique()})

coverage_df = pd.DataFrame(coverage_rows).sort_values('coverage_pct', ascending=False).reset_index(drop=True)
print(coverage_df.to_string(index=False))

In [ ]:
import pandas as pd
import re

raw_path = "../data/raw/"
processed_path = "../data/processed/"

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])
cohort_subject_ids = set(cohort_step1['subject_id'])
intime_lookup = cohort_step1.set_index('subject_id')['intime']

vitals_labs_24h = pd.read_csv(processed_path + "vitals_labs_24h.csv")
vitals_labs_24h['charttime'] = pd.to_datetime(vitals_labs_24h['charttime'])

organ_burden_24h = pd.read_csv(processed_path + "organ_burden_inputs_24h.csv")
organ_burden_24h['charttime'] = pd.to_datetime(organ_burden_24h['charttime'])

print(f"vitals_labs_24h.csv loaded: {vitals_labs_24h.shape}")
print(f"organ_burden_inputs_24h.csv loaded: {organ_burden_24h.shape}")
print()

# ============================================================
# FIX 1: GCS Verbal text-based imputation
# ============================================================

print("=" * 65)
print("FIX 1: GCS Verbal (itemid 223900) — text-based non-verbalizable detection")
print("=" * 65)

chunk_size = 1_000_000
verbal_rows = []

for chunk in pd.read_csv(
    raw_path + "chartevents.csv.gz", compression="gzip",
    usecols=['subject_id', 'hadm_id', 'itemid', 'charttime', 'value', 'valuenum'],
    chunksize=chunk_size
):
    filtered = chunk[(chunk['itemid'] == 223900) & chunk['subject_id'].isin(cohort_subject_ids)]
    if len(filtered) > 0:
        verbal_rows.append(filtered)

verbal_raw = pd.concat(verbal_rows, ignore_index=True) if verbal_rows else pd.DataFrame(
    columns=['subject_id', 'hadm_id', 'itemid', 'charttime', 'value', 'valuenum'])
verbal_raw['charttime'] = pd.to_datetime(verbal_raw['charttime'])

verbal_raw = verbal_raw.merge(intime_lookup.rename('intime'), left_on='subject_id', right_index=True, how='left')
verbal_raw['hours_from_intime'] = (verbal_raw['charttime'] - verbal_raw['intime']).dt.total_seconds() / 3600
verbal_24h = verbal_raw[(verbal_raw['hours_from_intime'] >= 0) & (verbal_raw['hours_from_intime'] <= 24)].copy()

print(f"GCS Verbal rows pulled (first 24h, cohort): {len(verbal_24h)}")
print()
print("ALL unique text values found in 'value' column (confirm before assuming any pattern):")
print(verbal_24h['value'].value_counts(dropna=False))
print()

# Pattern built from clinically standard non-verbalizable phrasings — but only
# applied to values that ACTUALLY appear above, not blindly assumed.
nonverbal_pattern = r'no response|ett|e\.?t\.?/?trach|trach|intubat'

verbal_24h['is_nonverbalizable_text'] = verbal_24h['value'].astype(str).str.contains(
    nonverbal_pattern, case=False, na=False, regex=True
)

matched_values = verbal_24h.loc[verbal_24h['is_nonverbalizable_text'], 'value'].value_counts()
print("Text values MATCHED as non-verbalizable (will be set to GCS_Verbal=1):")
print(matched_values)
print()

unmatched_nonnumeric = verbal_24h[
    verbal_24h['valuenum'].isna() & ~verbal_24h['is_nonverbalizable_text']
]['value'].value_counts()
if len(unmatched_nonnumeric) > 0:
    print("WARNING: text values that are NaN in valuenum but did NOT match the non-verbalizable")
    print("pattern (review these — they may need to be added to the pattern manually):")
    print(unmatched_nonnumeric)
    print()

verbal_24h['valuenum_corrected'] = verbal_24h['valuenum']
verbal_24h.loc[verbal_24h['is_nonverbalizable_text'], 'valuenum_corrected'] = 1

n_corrected = verbal_24h['is_nonverbalizable_text'].sum()
print(f"Rows imputed to GCS_Verbal = 1: {n_corrected}")
print()

# ============================================================
# Merge correction back into vitals_labs_24h.csv (match on subject_id, charttime, itemid=223900)
# ============================================================

verbal_correction = verbal_24h[['subject_id', 'charttime', 'itemid', 'valuenum_corrected']].copy()

vitals_labs_24h = vitals_labs_24h.merge(
    verbal_correction, on=['subject_id', 'charttime', 'itemid'], how='left'
)

update_mask = vitals_labs_24h['valuenum_corrected'].notna()
n_rows_updated = update_mask.sum()
vitals_labs_24h.loc[update_mask, 'valuenum'] = vitals_labs_24h.loc[update_mask, 'valuenum_corrected']
vitals_labs_24h = vitals_labs_24h.drop(columns=['valuenum_corrected'])

print(f"Rows in vitals_labs_24h.csv corrected via merge: {n_rows_updated}")
print("  (should be <= the {} imputed rows above — some imputed rows may not have existed".format(n_corrected))
print("  in vitals_labs_24h.csv if their original valuenum was NaN and got dropped during the")
print("  original extraction's .dropna(subset=['valuenum']) step)")
print()

# ============================================================
# FIX 2: Vasopressor unit harmonization
# ============================================================

print("=" * 65)
print("FIX 2: Vasopressor unit harmonization")
print("=" * 65)

d_items = pd.read_csv(raw_path + "d_items.csv.gz", compression="gzip")

vasopressor_itemids = sorted(organ_burden_24h.loc[
    organ_burden_24h['itemid'].isin(
        d_items[d_items['category'].str.contains('Medications', case=False, na=False)]['itemid']
    ) if 'category' in d_items.columns else organ_burden_24h['itemid'].unique(),
    'itemid'
].unique()) if False else sorted(organ_burden_24h['itemid'].unique())

# Simpler, direct approach: use the itemids actually present in organ_burden_24h that are
# vasopressor-related (exclude FiO2=223835 and urine=226559/226560 explicitly)
excluded_non_vaso = {223835, 226559, 226560}
vasopressor_itemids_in_file = [i for i in organ_burden_24h['itemid'].unique() if i not in excluded_non_vaso]

vaso_unit_map = d_items[d_items['itemid'].isin(vasopressor_itemids_in_file)][['itemid', 'label', 'unitname']].copy()
print("Vasopressor itemid -> unit mapping (from d_items):")
print(vaso_unit_map.to_string(index=False))
print()

# ============================================================
# Get patient weight — search d_items for weight-related itemids, don't assume 224639
# ============================================================

weight_items = d_items[d_items['label'].str.contains('weight', case=False, na=False)].copy()
print("Weight-related d_items matches (checking before assuming any single itemid):")
print(weight_items[['itemid', 'label', 'unitname']].to_string(index=False))
print()

weight_itemids = set(weight_items['itemid'])

weight_rows = []
for chunk in pd.read_csv(
    raw_path + "chartevents.csv.gz", compression="gzip",
    usecols=['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum'],
    chunksize=chunk_size
):
    filtered = chunk[chunk['itemid'].isin(weight_itemids) & chunk['subject_id'].isin(cohort_subject_ids)]
    if len(filtered) > 0:
        weight_rows.append(filtered)

weight_events = pd.concat(weight_rows, ignore_index=True) if weight_rows else pd.DataFrame(
    columns=['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum'])
weight_events['charttime'] = pd.to_datetime(weight_events['charttime'])

print(f"Weight chartevents rows pulled: {len(weight_events)}")

# Take each patient's EARLIEST available weight reading (any time in their stay,
# since weight is used for dose normalization and doesn't need to be first-24h-only)
weight_events_sorted = weight_events.sort_values(['subject_id', 'charttime'])
patient_weight = weight_events_sorted.groupby('subject_id')['valuenum'].first()

n_with_weight = patient_weight.notna().sum()
print(f"Cohort patients with an available weight value: {n_with_weight} / {len(cohort_step1)}")
print()

# ============================================================
# Conversion logic: target unit = mcg/kg/min
# ============================================================

def classify_unit(unit_str):
    u = str(unit_str).lower().strip()
    if 'mcg/kg/min' in u:
        return 'mcg_kg_min'  # already target — no conversion needed
    if 'mg/kg/min' in u:
        return 'mg_kg_min'   # *1000 -> mcg/kg/min, no weight needed
    if 'mcg/min' in u:
        return 'mcg_min'     # needs /weight
    if 'mg/min' in u:
        return 'mg_min'      # *1000 -> mcg/min, then /weight
    if 'unit' in u:          # units/hr, units/min — vasopressin-class, NOT convertible via simple math
        return 'units_based_unconvertible'
    return 'unknown_unconvertible'

itemid_unit_class = {}
for _, row in vaso_unit_map.iterrows():
    itemid_unit_class[row['itemid']] = classify_unit(row['unitname'])

print("Itemid -> unit classification for conversion:")
for iid, cls in itemid_unit_class.items():
    print(f"  itemid {iid}: {cls}")
print()

vaso_data = organ_burden_24h[organ_burden_24h['itemid'].isin(vasopressor_itemids_in_file)].copy()
vaso_data['unit_class'] = vaso_data['itemid'].map(itemid_unit_class)
vaso_data = vaso_data.merge(patient_weight.rename('weight_kg'), left_on='subject_id', right_index=True, how='left')

def convert_row(row):
    cls = row['unit_class']
    val = row['valuenum']
    weight = row['weight_kg']

    if cls == 'mcg_kg_min':
        return val, True, None
    if cls == 'mg_kg_min':
        return val * 1000, True, None
    if cls == 'mcg_min':
        if pd.isna(weight):
            return None, False, 'missing_weight'
        return val / weight, True, None
    if cls == 'mg_min':
        if pd.isna(weight):
            return None, False, 'missing_weight'
        return (val * 1000) / weight, True, None
    if cls == 'units_based_unconvertible':
        return None, False, 'units_based_no_direct_conversion'
    return None, False, 'unknown_unit'

results = vaso_data.apply(convert_row, axis=1, result_type='expand')
results.columns = ['valuenum_mcg_kg_min', 'converted_ok', 'flag_reason']

vaso_data = pd.concat([vaso_data, results], axis=1)

n_converted = vaso_data['converted_ok'].sum()
n_flagged = (~vaso_data['converted_ok']).sum()

print(f"Vasopressor rows successfully converted to mcg/kg/min: {n_converted}")
print(f"Vasopressor rows flagged as unconvertible: {n_flagged}")
print()
print("Breakdown of flag reasons:")
print(vaso_data.loc[~vaso_data['converted_ok'], 'flag_reason'].value_counts())
print()
print("NOTE: 'units_based_no_direct_conversion' rows are almost certainly vasopressin —")
print("vasopressin (units/hr) is pharmacologically distinct from catecholamines and cannot be")
print("converted to mcg/kg/min via simple unit math. A validated norepinephrine-equivalent")
print("dose formula would be needed to include it on the same scale — not applied here since")
print("that requires a clinical conversion table, not a units conversion.")
print()

# ============================================================
# Rebuild organ_burden_inputs_24h.csv: keep original valuenum + add converted column + flag
# ============================================================

non_vaso_data = organ_burden_24h[~organ_burden_24h['itemid'].isin(vasopressor_itemids_in_file)].copy()
non_vaso_data['valuenum_mcg_kg_min'] = None
non_vaso_data['converted_ok'] = None
non_vaso_data['flag_reason'] = None

organ_burden_24h_final = pd.concat([
    vaso_data[['subject_id', 'charttime', 'itemid', 'valuenum', 'valuenum_mcg_kg_min', 'converted_ok', 'flag_reason']],
    non_vaso_data[['subject_id', 'charttime', 'itemid', 'valuenum', 'valuenum_mcg_kg_min', 'converted_ok', 'flag_reason']]
], ignore_index=True)

# ============================================================
# Save both corrected files
# ============================================================

vitals_labs_24h.to_csv(processed_path + "vitals_labs_24h.csv", index=False)
organ_burden_24h_final.to_csv(processed_path + "organ_burden_inputs_24h.csv", index=False)

print("=" * 65)
print("FINAL SAVED SHAPES")
print("=" * 65)
print(f"vitals_labs_24h.csv: {vitals_labs_24h.shape}")
print(f"organ_burden_inputs_24h.csv: {organ_burden_24h_final.shape}")

In [ ]:
import pandas as pd

raw_path = "../data/raw/"
processed_path = "../data/processed/"

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])
cohort_subject_ids = set(cohort_step1['subject_id'])
intime_lookup = cohort_step1.set_index('subject_id')['intime']
n_cohort = len(cohort_step1)

vitals_labs_24h = pd.read_csv(processed_path + "vitals_labs_24h.csv")
vitals_labs_24h['charttime'] = pd.to_datetime(vitals_labs_24h['charttime'])

print(f"vitals_labs_24h.csv loaded: {vitals_labs_24h.shape}")
print()

# ============================================================
# Reload RAW GCS_Verbal text rows (itemid 223900), first 24h, cohort only
# ============================================================

chunk_size = 1_000_000
verbal_rows = []

for chunk in pd.read_csv(
    raw_path + "chartevents.csv.gz", compression="gzip",
    usecols=['subject_id', 'hadm_id', 'itemid', 'charttime', 'value'],
    chunksize=chunk_size
):
    filtered = chunk[(chunk['itemid'] == 223900) & chunk['subject_id'].isin(cohort_subject_ids)]
    if len(filtered) > 0:
        verbal_rows.append(filtered)

verbal_raw = pd.concat(verbal_rows, ignore_index=True) if verbal_rows else pd.DataFrame(
    columns=['subject_id', 'hadm_id', 'itemid', 'charttime', 'value'])
verbal_raw['charttime'] = pd.to_datetime(verbal_raw['charttime'])

verbal_raw = verbal_raw.merge(intime_lookup.rename('intime'), left_on='subject_id', right_index=True, how='left')
verbal_raw['hours_from_intime'] = (verbal_raw['charttime'] - verbal_raw['intime']).dt.total_seconds() / 3600
verbal_24h = verbal_raw[(verbal_raw['hours_from_intime'] >= 0) & (verbal_raw['hours_from_intime'] <= 24)].copy()

print(f"GCS_Verbal raw text rows pulled (first 24h, cohort): {len(verbal_24h)} (expected 22527)")
print()

print("Full unique text value_counts() before mapping:")
print(verbal_24h['value'].value_counts(dropna=False))
print()

# ============================================================
# Full text -> GCS verbal score mapping (all 6 observed categories)
# ============================================================

verbal_score_map = {
    "Oriented": 5,
    "Confused": 4,
    "Inappropriate Words": 3,
    "Incomprehensible sounds": 2,
    "No Response": 1,
    "No Response-ETT": 1,
}

verbal_24h['valuenum_mapped'] = verbal_24h['value'].map(verbal_score_map)

n_mapped = verbal_24h['valuenum_mapped'].notna().sum()
n_unmapped = verbal_24h['valuenum_mapped'].isna().sum()

print(f"Rows successfully mapped to a numeric GCS_Verbal score: {n_mapped} (expected 22527 or very close)")
print(f"Rows with text NOT in the mapping (flagged, not dropped): {n_unmapped}")
print()

if n_unmapped > 0:
    print("Unmapped text values found (review these — mapping may need extending):")
    print(verbal_24h.loc[verbal_24h['valuenum_mapped'].isna(), 'value'].value_counts(dropna=False))
    print()

# ============================================================
# Overwrite GCS_Verbal in vitals_labs_24h.csv
# Match on subject_id, charttime, itemid=223900 — replace valuenum with the mapped score
# for ALL matched rows (not just previously-imputed ones), and ALSO add rows that exist
# in verbal_24h but were dropped from vitals_labs_24h.csv originally (because their old
# valuenum was NaN and got filtered out by dropna in the original extraction).
# ============================================================

verbal_correction = verbal_24h[['subject_id', 'charttime', 'itemid', 'valuenum_mapped']].dropna(subset=['valuenum_mapped'])

# Remove any existing GCS_Verbal (itemid 223900) rows from vitals_labs_24h.csv entirely,
# then re-insert the freshly, fully-mapped set — cleaner than a partial merge/overwrite,
# and guarantees no stale/partial values survive from the previous fix.
existing_non_verbal = vitals_labs_24h[vitals_labs_24h['itemid'] != 223900].copy()

verbal_final = verbal_correction.rename(columns={'valuenum_mapped': 'valuenum'})[
    ['subject_id', 'charttime', 'itemid', 'valuenum']
]

vitals_labs_24h_updated = pd.concat([existing_non_verbal, verbal_final], ignore_index=True)

print(f"GCS_Verbal (itemid 223900) rows in file BEFORE this fix: "
      f"{(vitals_labs_24h['itemid'] == 223900).sum()}")
print(f"GCS_Verbal (itemid 223900) rows in file AFTER this fix: "
      f"{(vitals_labs_24h_updated['itemid'] == 223900).sum()} (expected {n_mapped})")
print()

# ============================================================
# Save
# ============================================================

vitals_labs_24h_updated.to_csv(processed_path + "vitals_labs_24h.csv", index=False)

print(f"Saved vitals_labs_24h.csv, updated shape: {vitals_labs_24h_updated.shape}")
print()

# ============================================================
# GCS_Verbal coverage %
# ============================================================

gcs_verbal_final = vitals_labs_24h_updated[vitals_labs_24h_updated['itemid'] == 223900]
n_patients_with_gcs_verbal = gcs_verbal_final['subject_id'].nunique()
coverage_pct = 100 * n_patients_with_gcs_verbal / n_cohort

print(f"GCS_Verbal coverage: {n_patients_with_gcs_verbal} / {n_cohort} patients "
      f"({coverage_pct:.2f}%) have at least one valid GCS_Verbal score in first 24h")

In [ ]:
import pandas as pd

raw_path = "../data/raw/"
processed_path = "../data/processed/"

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])
cohort_subject_ids = set(cohort_step1['subject_id'])
intime_lookup = cohort_step1.set_index('subject_id')['intime']

organ_burden_24h = pd.read_csv(processed_path + "organ_burden_inputs_24h.csv")
organ_burden_24h['charttime'] = pd.to_datetime(organ_burden_24h['charttime'])

print(f"organ_burden_inputs_24h.csv loaded: {organ_burden_24h.shape}")
print()

# ============================================================
# STEP 1: Check inputevents.csv.gz actual column names FIRST — don't assume
# ============================================================

sample_chunk = pd.read_csv(raw_path + "inputevents.csv.gz", compression="gzip", nrows=5)
print("inputevents.csv.gz actual columns:")
print(sample_chunk.columns.tolist())
print()

rate_col_candidates = [c for c in sample_chunk.columns if 'rateuom' in c.lower() or c.lower() == 'rateuom']
amount_col_candidates = [c for c in sample_chunk.columns if 'amountuom' in c.lower() or c.lower() == 'amountuom']

rate_uom_col = rate_col_candidates[0] if rate_col_candidates else None
amount_uom_col = amount_col_candidates[0] if amount_col_candidates else None

print(f"Detected rate-unit column: {rate_uom_col}")
print(f"Detected amount-unit column: {amount_uom_col}")

if rate_uom_col is None:
    raise ValueError("Could not find a rateuom-like column in inputevents.csv.gz — "
                      "check the printed column list above and adjust manually before proceeding.")
print()

# ============================================================
# Identify vasopressor itemids from the existing organ_burden file (exclude FiO2/urine)
# ============================================================

excluded_non_vaso = {223835, 226559, 226560}
vasopressor_itemids = [i for i in organ_burden_24h['itemid'].unique() if i not in excluded_non_vaso]
print(f"Vasopressor itemids being reloaded: {sorted(vasopressor_itemids)}")
print()

# ============================================================
# STEP 1 (cont'd): Reload inputevents with rateuom/amountuom, cohort + vasopressor itemids
# ============================================================

usecols = ['subject_id', 'hadm_id', 'itemid', 'starttime', 'endtime', 'rate', 'amount']
if rate_uom_col:
    usecols.append(rate_uom_col)
if amount_uom_col:
    usecols.append(amount_uom_col)

chunk_size = 1_000_000
vaso_rows = []

for chunk in pd.read_csv(
    raw_path + "inputevents.csv.gz", compression="gzip",
    usecols=lambda c: c in usecols,
    chunksize=chunk_size
):
    filtered = chunk[chunk['itemid'].isin(vasopressor_itemids) & chunk['subject_id'].isin(cohort_subject_ids)]
    if len(filtered) > 0:
        vaso_rows.append(filtered)

vaso_events = pd.concat(vaso_rows, ignore_index=True) if vaso_rows else pd.DataFrame(columns=usecols)
vaso_events['starttime'] = pd.to_datetime(vaso_events['starttime'])

print(f"Vasopressor inputevents rows reloaded (with unit columns): {len(vaso_events)}")

vaso_events = vaso_events.merge(intime_lookup.rename('intime'), left_on='subject_id', right_index=True, how='left')
vaso_events['hours_from_intime'] = (vaso_events['starttime'] - vaso_events['intime']).dt.total_seconds() / 3600
vaso_events_24h = vaso_events[(vaso_events['hours_from_intime'] >= 0) & (vaso_events['hours_from_intime'] <= 24)].copy()

print(f"Vasopressor rows within first 24h: {len(vaso_events_24h)}")
print()

# ============================================================
# STEP 2: Print unique rateuom values per itemid — the REAL unit info
# ============================================================

d_items = pd.read_csv(raw_path + "d_items.csv.gz", compression="gzip")
itemid_labels = d_items.set_index('itemid')['label'].to_dict()

print("=" * 65)
print(f"UNIQUE {rate_uom_col} VALUES PER ITEMID (this is the real infusion-rate unit)")
print("=" * 65)
rateuom_by_itemid = vaso_events_24h.groupby('itemid')[rate_uom_col].value_counts(dropna=False)
for itemid in sorted(vaso_events_24h['itemid'].unique()):
    label = itemid_labels.get(itemid, 'UNKNOWN')
    print(f"\nitemid {itemid} ({label}):")
    print(rateuom_by_itemid[itemid])
print()

# ============================================================
# STEP 3: Convert to mcg/kg/min using patient weight
# ============================================================

weight_items = d_items[d_items['label'].str.contains('weight', case=False, na=False)]
weight_itemids = set(weight_items['itemid'])

weight_rows = []
for chunk in pd.read_csv(
    raw_path + "chartevents.csv.gz", compression="gzip",
    usecols=['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum'],
    chunksize=chunk_size
):
    filtered = chunk[chunk['itemid'].isin(weight_itemids) & chunk['subject_id'].isin(cohort_subject_ids)]
    if len(filtered) > 0:
        weight_rows.append(filtered)

weight_events = pd.concat(weight_rows, ignore_index=True) if weight_rows else pd.DataFrame(
    columns=['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum'])
weight_events['charttime'] = pd.to_datetime(weight_events['charttime'])
weight_events_sorted = weight_events.sort_values(['subject_id', 'charttime'])
patient_weight = weight_events_sorted.groupby('subject_id')['valuenum'].first()

print(f"Patients with available weight: {patient_weight.notna().sum()} / {len(cohort_step1)}")
print()

vaso_events_24h = vaso_events_24h.merge(patient_weight.rename('weight_kg'), left_on='subject_id', right_index=True, how='left')

def classify_and_convert(row):
    uom = str(row[rate_uom_col]).strip().lower()
    rate = row['rate']
    weight = row['weight_kg']

    if pd.isna(rate):
        return None, False, 'rate_missing'

    if uom == 'mcg/kg/min':
        return rate, True, None

    if uom == 'mcg/min':
        if pd.isna(weight):
            return None, False, 'missing_weight'
        return rate / weight, True, None

    if uom == 'mg/hr':
        if pd.isna(weight):
            return None, False, 'missing_weight'
        return (rate * 1000 / 60) / weight, True, None

    if uom in ('units/hour', 'units/hr', 'units/min', 'unit/hour', 'unit/hr'):
        return None, False, 'units_based_vasopressin_unconvertible'

    return None, False, f'unrecognized_unit:{uom}'

conversion_results = vaso_events_24h.apply(classify_and_convert, axis=1, result_type='expand')
conversion_results.columns = ['valuenum_mcg_kg_min', 'converted_ok', 'flag_reason']

vaso_events_24h = pd.concat([vaso_events_24h, conversion_results], axis=1)

# ============================================================
# STEP 4: Print conversion breakdown by itemid and original rateuom
# ============================================================

print("=" * 65)
print("CONVERSION RESULTS — by itemid and original rateuom")
print("=" * 65)

breakdown = vaso_events_24h.groupby(['itemid', rate_uom_col, 'converted_ok']).size().reset_index(name='row_count')
breakdown['label'] = breakdown['itemid'].map(itemid_labels)
print(breakdown[['itemid', 'label', rate_uom_col, 'converted_ok', 'row_count']].to_string(index=False))
print()

n_converted_total = vaso_events_24h['converted_ok'].sum()
n_flagged_total = (~vaso_events_24h['converted_ok']).sum()

print(f"TOTAL converted: {n_converted_total}")
print(f"TOTAL flagged (unconvertible): {n_flagged_total}")
print()
print("Flag reason breakdown:")
print(vaso_events_24h.loc[~vaso_events_24h['converted_ok'], 'flag_reason'].value_counts())
print()

# ============================================================
# STEP 5: Rebuild organ_burden_inputs_24h.csv with the properly converted column
# ============================================================

vaso_final = vaso_events_24h.rename(columns={'starttime': 'charttime'})[
    ['subject_id', 'charttime', 'itemid', 'rate', 'valuenum_mcg_kg_min', 'converted_ok', 'flag_reason']
].rename(columns={'rate': 'valuenum'})

non_vaso_data = organ_burden_24h[organ_burden_24h['itemid'].isin(excluded_non_vaso)].copy()
for col in ['valuenum_mcg_kg_min', 'converted_ok', 'flag_reason']:
    if col not in non_vaso_data.columns:
        non_vaso_data[col] = None

organ_burden_24h_final = pd.concat([
    vaso_final,
    non_vaso_data[['subject_id', 'charttime', 'itemid', 'valuenum', 'valuenum_mcg_kg_min', 'converted_ok', 'flag_reason']]
], ignore_index=True)

organ_burden_24h_final.to_csv(processed_path + "organ_burden_inputs_24h.csv", index=False)

print(f"Saved organ_burden_inputs_24h.csv")
print(f"Updated shape: {organ_burden_24h_final.shape}")

In [ ]:
import pandas as pd

processed_path = "../data/processed/"

organ_burden_24h = pd.read_csv(processed_path + "organ_burden_inputs_24h.csv")

print("=" * 65)
print("FULL COLUMN LIST")
print("=" * 65)
print(organ_burden_24h.columns.tolist())
print()
print(f"Shape: {organ_burden_24h.shape}")
print()

# ============================================================
# Check (1): exactly one column holding the converted mcg/kg/min rate
# ============================================================

candidate_converted_cols = [c for c in organ_burden_24h.columns if 'mcg_kg_min' in c.lower()]
print(f"Columns matching 'mcg_kg_min': {candidate_converted_cols}")
print(f"  Exactly one such column: {len(candidate_converted_cols) == 1}")
print()

# ============================================================
# Identify vasopressor rows (exclude FiO2=223835, urine=226559/226560)
# ============================================================

excluded_non_vaso = {223835, 226559, 226560}
vaso_mask = ~organ_burden_24h['itemid'].isin(excluded_non_vaso)
vaso_rows = organ_burden_24h[vaso_mask].copy()

print(f"Vasopressor rows in file: {len(vaso_rows)}")
print()

print("=" * 65)
print("HEAD OF VASOPRESSOR ROWS (all columns)")
print("=" * 65)
print(vaso_rows.head(15).to_string())
print()

# ============================================================
# Check (2): flagged/unconvertible rows have NaN or a clear flag,
# not a leftover numeric value from a failed prior attempt
# ============================================================

if 'converted_ok' in organ_burden_24h.columns and 'flag_reason' in organ_burden_24h.columns:
    flagged = vaso_rows[vaso_rows['converted_ok'] == False]
    print(f"Rows marked converted_ok == False: {len(flagged)}")
    print()
    print("flag_reason value_counts() for flagged rows:")
    print(flagged['flag_reason'].value_counts(dropna=False))
    print()

    conv_col = candidate_converted_cols[0] if candidate_converted_cols else None
    if conv_col:
        bad_flagged_with_value = flagged[flagged[conv_col].notna()]
        print(f"Flagged rows that STILL have a non-NaN value in '{conv_col}' "
              f"(should be 0 — anything here is a leftover stale value): {len(bad_flagged_with_value)}")
        if len(bad_flagged_with_value) > 0:
            print(bad_flagged_with_value[['itemid', 'valuenum', conv_col, 'converted_ok', 'flag_reason']].head(20))
        print()
else:
    print("WARNING: 'converted_ok' and/or 'flag_reason' columns not found — "
          "cannot verify flagged rows without them.")
    print()

# ============================================================
# Check (3): stale/unused columns from a previous failed attempt
# ============================================================

expected_columns = {'subject_id', 'charttime', 'itemid', 'valuenum',
                     'valuenum_mcg_kg_min', 'converted_ok', 'flag_reason'}
actual_columns = set(organ_burden_24h.columns)

stale_columns = actual_columns - expected_columns
missing_expected = expected_columns - actual_columns

print(f"Unexpected/stale columns found: {stale_columns if stale_columns else 'NONE'}")
print(f"Expected columns missing: {missing_expected if missing_expected else 'NONE'}")
print()

# ============================================================
# Cleanup, if anything stale found
# ============================================================

needs_cleanup = bool(stale_columns) or len(candidate_converted_cols) != 1

if needs_cleanup:
    print("STALE DATA DETECTED — cleaning up.")

    # Keep only the canonical expected columns that actually exist
    keep_cols = [c for c in ['subject_id', 'charttime', 'itemid', 'valuenum',
                              'valuenum_mcg_kg_min', 'converted_ok', 'flag_reason']
                 if c in organ_burden_24h.columns]

    organ_burden_24h_clean = organ_burden_24h[keep_cols].copy()

    # If somehow multiple mcg_kg_min-like columns existed, this branch won't fire
    # since we already select only the canonical name above — but double check:
    remaining_stray = [c for c in organ_burden_24h_clean.columns
                        if 'mcg_kg_min' in c.lower() and c != 'valuenum_mcg_kg_min']
    if remaining_stray:
        print(f"Dropping additional stray converted-rate columns: {remaining_stray}")
        organ_burden_24h_clean = organ_burden_24h_clean.drop(columns=remaining_stray)

    organ_burden_24h_clean.to_csv(processed_path + "organ_burden_inputs_24h.csv", index=False)

    print(f"Re-saved organ_burden_inputs_24h.csv")
    print(f"Cleaned columns: {organ_burden_24h_clean.columns.tolist()}")
    print(f"Cleaned shape: {organ_burden_24h_clean.shape}")
else:
    print("No stale columns or duplicate converted-rate columns found — file is clean as-is.")
    print("No re-save needed.")

In [ ]:
import pandas as pd
import re

raw_path = "../data/raw/"
processed_path = "../data/processed/"

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
patients = pd.read_csv(raw_path + "patients.csv.gz", compression="gzip")
diagnoses_icd = pd.read_csv(raw_path + "diagnoses_icd.csv.gz", compression="gzip")
diagnoses_icd['icd_code'] = diagnoses_icd['icd_code'].astype(str).str.strip()

cohort_hadm_ids = set(cohort_step1['hadm_id'])
print(f"cohort_step1.csv loaded: {cohort_step1.shape}")
print()

# ============================================================
# 1. Start static_features with base columns
# ============================================================

static_features = cohort_step1[['subject_id', 'hadm_id', 'age_at_admission', 'cancer_type', 'delirium_subtype']].copy()

static_features = static_features.merge(
    patients[['subject_id', 'gender']], on='subject_id', how='left'
)
static_features = static_features.rename(columns={'gender': 'sex'})

print(f"Patients missing sex after merge: {static_features['sex'].isna().sum()}")
print()

# ============================================================
# 2. Elixhauser comorbidity flags
# ============================================================
# NOTE: This uses the standard published Quan et al. (2005) ICD-9-CM mapping
# and the corresponding AHRQ ICD-10-CM Elixhauser mapping. It's the widely-used
# reference implementation, but I have NOT run this against a validated package
# (e.g. R 'comorbidity', Python 'icd10-comorbidities'/'comorbidipy') to confirm
# every code range is complete and current. Treat this as a solid first pass —
# spot-check against a validated tool before final publication if precision matters.

elixhauser_categories = [
    'chf', 'arrhythmia', 'valvular', 'pulm_circ', 'pvd', 'hypertension',
    'paralysis', 'other_neuro', 'chronic_pulm', 'diabetes_uncomp', 'diabetes_comp',
    'hypothyroid', 'renal_failure', 'liver_disease', 'pud', 'aids_hiv',
    'lymphoma', 'metastatic_cancer', 'solid_tumor', 'rheumatoid',
    'coagulopathy', 'obesity', 'weight_loss', 'fluid_electrolyte',
    'blood_loss_anemia', 'deficiency_anemia', 'alcohol_abuse', 'drug_abuse',
    'psychoses', 'depression'
]

print(f"Elixhauser categories being computed ({len(elixhauser_categories)} total):")
print(elixhauser_categories)
print()
print("NOTE: 'metastatic_cancer', 'lymphoma', and 'solid_tumor' below are derived from the")
print("SAME underlying ICD code ranges as our own cancer_type variable. Including both")
print("Elixhauser cancer flags AND cancer_type in a model risks redundant/collinear features.")
print("Flagging this now — recommend dropping the 3 cancer-related Elixhauser flags from the")
print("actual modeling feature set in Notebook 3, keeping cancer_type as the single cancer")
print("representation, OR dropping cancer_type and keeping only the Elixhauser flags.")
print("Both are computed and saved here so you have the choice later, not made for you now.")
print()

# ICD-9-CM and ICD-10-CM regex patterns per category (Quan 2005 / AHRQ ICD-10 mapping, condensed)
icd9_patterns = {
    'chf': r'^(398\.?91|402\.?[019]1|404\.?[019][13]|428)',
    'arrhythmia': r'^(426\.?[0-79]|427\.?[0-46-9]|996\.?01|996\.?04|V450|V533)',
    'valvular': r'^(093\.?2|394-397|424|746\.?[3-6]|V422|V433)',
    'pulm_circ': r'^(415\.?[01]|416|417\.?[089])',
    'pvd': r'^(093\.?0|437\.?3|440|441|443\.?[1-9]|447\.?1|557\.?[19]|V434)',
    'hypertension': r'^401',
    'paralysis': r'^(342|343|344\.?[0-69])',
    'other_neuro': r'^(331\.?[19]|332\.?0|333\.?4|333\.?5|334|335|336\.?2|340|341|345|347|348\.?1|348\.?3|780\.?3|784\.?3)',
    'chronic_pulm': r'^(490-505|506\.?4|508\.?1|508\.?8)',
    'diabetes_uncomp': r'^250\.?[0-3]',
    'diabetes_comp': r'^250\.?[4-9]',
    'hypothyroid': r'^(243|244)',
    'renal_failure': r'^(403\.?[019]1|404\.?[019][23]|585|586|V420|V451|V56)',
    'liver_disease': r'^(070\.?[236]|070\.?4[4-9]|456\.?[012]|571|572\.?[2-8]|573\.?[38]|V427)',
    'pud': r'^(531-534)',
    'aids_hiv': r'^(042-044)',
    'lymphoma': r'^(200-202|203\.?0|238\.?6)',
    'metastatic_cancer': r'^(196-199)',
    'solid_tumor': r'^(140-172|174-195)',
    'rheumatoid': r'^(446|701\.?0|710\.?[0-489]|714|720|725)',
    'coagulopathy': r'^(286|287\.?[1345])',
    'obesity': r'^278\.?0',
    'weight_loss': r'^(260-263)',
    'fluid_electrolyte': r'^276',
    'blood_loss_anemia': r'^280\.?0',
    'deficiency_anemia': r'^(280\.?[1-9]|281)',
    'alcohol_abuse': r'^(291|303|305\.?0)',
    'drug_abuse': r'^(292|304|305\.?[2-9])',
    'psychoses': r'^(295|296\.?[04]|297|298)',
    'depression': r'^(296\.?[235]|300\.?4|309\.?[01]|311)',
}

icd10_patterns = {
    'chf': r'^(I099|I110|I130|I132|I255|I420|I425-I429|I43|I50|P290)',
    'arrhythmia': r'^(I44[123]|I456|I459|I47|I48|I49|R000|R001|R008|T821|Z450|Z950)',
    'valvular': r'^(A520|I05-I08|I091|I098|I34-I39|Q23[0-3]|Z952|Z953|Z954)',
    'pulm_circ': r'^(I26|I27|I280|I288|I289)',
    'pvd': r'^(I70|I71|I731|I738|I739|I771|I790|I792|K551|K558|K559|Z958|Z959)',
    'hypertension': r'^I10',
    'paralysis': r'^(G041|G114|G80[12]|G81|G82|G83)',
    'other_neuro': r'^(G10-G13|G20-G22|G254|G255|G312|G318|G319|G32|G35-G37|G40|G41|G931|G934|R470|R56)',
    'chronic_pulm': r'^(I278|I279|J40-J47|J60-J67|J684|J701|J703)',
    'diabetes_uncomp': r'^(E100|E101|E109|E110|E111|E119|E120|E121|E129|E130|E131|E139|E140|E141|E149)',
    'diabetes_comp': r'^(E10[2-8]|E11[2-8]|E12[2-8]|E13[2-8]|E14[2-8])',
    'hypothyroid': r'^(E00-E03|E890)',
    'renal_failure': r'^(I120|I131|N18|N19|N250|Z490-Z492|Z940|Z992)',
    'liver_disease': r'^(B18|I85|I864|I982|K70|K711|K713-K715|K717|K72-K74|K760-K769|Z944)',
    'pud': r'^(K25-K28)',
    'aids_hiv': r'^(B20-B24)',
    'lymphoma': r'^(C81-C85|C88|C96|C900|C902)',
    'metastatic_cancer': r'^(C77-C80)',
    'solid_tumor': r'^(C00-C26|C30-C34|C37-C41|C43|C45-C58|C60-C76|C97)',
    'rheumatoid': r'^(L940|L941|L943|M05|M06|M08|M120|M123|M30|M31|M32-M35|M45|M46)',
    'coagulopathy': r'^(D65-D68|D691|D693-D696)',
    'obesity': r'^E66',
    'weight_loss': r'^(E40-E46|R634|R64)',
    'fluid_electrolyte': r'^E87',
    'blood_loss_anemia': r'^D500',
    'deficiency_anemia': r'^(D508-D539)',
    'alcohol_abuse': r'^(F10|E52|G621|I426|K292|K700|K703|K709|T51|Z502|Z714|Z721)',
    'drug_abuse': r'^(F11-F16|F18|F19|Z715|Z722)',
    'psychoses': r'^(F20|F22-F25|F28|F29|F302|F312|F315)',
    'depression': r'^(F204|F313-F315|F32|F33|F341|F412|F432)',
}

def code_matches_pattern(code, pattern):
    """Match a possibly-hyphenated-range pattern against a specific code prefix."""
    code = str(code).replace('.', '')
    # Expand hyphen ranges like '140-172' into an alternation of exact numeric prefixes
    def expand_ranges(pat):
        def repl(m):
            lo, hi = m.group(1), m.group(2)
            width = len(lo)
            lo_i, hi_i = int(lo), int(hi)
            return '(' + '|'.join(str(x).zfill(width) for x in range(lo_i, hi_i + 1)) + ')'
        return re.sub(r'(\d+)-(\d+)', repl, pat)
    expanded = expand_ranges(pattern.replace('.', '').replace('^', ''))
    return re.match(expanded, code) is not None

print("Computing Elixhauser flags per hadm_id (this scans diagnoses_icd for cohort hadm_id only)...")
cohort_diag = diagnoses_icd[diagnoses_icd['hadm_id'].isin(cohort_hadm_ids)].copy()
print(f"Diagnosis rows for cohort: {len(cohort_diag)}")
print()

comorbidity_flags = pd.DataFrame({'hadm_id': list(cohort_hadm_ids)})

for cat in elixhauser_categories:
    icd9_pat = icd9_patterns[cat]
    icd10_pat = icd10_patterns[cat]

    icd9_mask = (cohort_diag['icd_version'] == 9) & cohort_diag['icd_code'].apply(
        lambda c: code_matches_pattern(c, icd9_pat))
    icd10_mask = (cohort_diag['icd_version'] == 10) & cohort_diag['icd_code'].apply(
        lambda c: code_matches_pattern(c, icd10_pat))

    matched_hadm = set(cohort_diag.loc[icd9_mask | icd10_mask, 'hadm_id'])
    comorbidity_flags[cat] = comorbidity_flags['hadm_id'].isin(matched_hadm)

print("Elixhauser flag computation complete.")
print()

static_features = static_features.merge(comorbidity_flags, on='hadm_id', how='left')

# ============================================================
# 3. First-recorded lab values within first 24h
# ============================================================

vitals_labs_24h = pd.read_csv(processed_path + "vitals_labs_24h.csv")
vitals_labs_24h['charttime'] = pd.to_datetime(vitals_labs_24h['charttime'])

lab_itemid_to_name = {
    51301: 'WBC', 51300: 'WBC',
    50813: 'Lactate', 52442: 'Lactate', 53154: 'Lactate',
    50912: 'Creatinine', 52546: 'Creatinine',
    51265: 'Platelets', 53189: 'Platelets',
    50885: 'Bilirubin', 53089: 'Bilirubin',
}

lab_data = vitals_labs_24h[vitals_labs_24h['itemid'].isin(lab_itemid_to_name.keys())].copy()
lab_data['lab_name'] = lab_data['itemid'].map(lab_itemid_to_name)

lab_data_sorted = lab_data.sort_values(['subject_id', 'lab_name', 'charttime'])
first_labs = lab_data_sorted.groupby(['subject_id', 'lab_name'])['valuenum'].first().unstack()
first_labs.columns = [f'first_{c}' for c in first_labs.columns]

static_features = static_features.merge(first_labs, on='subject_id', how='left')

print("First-24h lab columns added:", [c for c in static_features.columns if c.startswith('first_')])
print()

# ============================================================
# 4. Coverage / value_counts for comorbidities and labs
# ============================================================

n_patients = len(static_features)

print("=" * 65)
print("ELIXHAUSER COMORBIDITY FLAG PREVALENCE (% of cohort)")
print("=" * 65)
comorb_summary = pd.DataFrame({
    'category': elixhauser_categories,
    'n_positive': [static_features[c].sum() for c in elixhauser_categories],
    'pct_positive': [round(100 * static_features[c].sum() / n_patients, 2) for c in elixhauser_categories]
}).sort_values('pct_positive', ascending=False)
print(comorb_summary.to_string(index=False))
print()

print("=" * 65)
print("FIRST-24H LAB VALUE COVERAGE (% of cohort with a value)")
print("=" * 65)
lab_cols = [c for c in static_features.columns if c.startswith('first_')]
lab_summary = pd.DataFrame({
    'lab': lab_cols,
    'coverage_pct': [round(100 * static_features[c].notna().sum() / n_patients, 2) for c in lab_cols],
    'n_available': [static_features[c].notna().sum() for c in lab_cols]
})
print(lab_summary.to_string(index=False))
print()

# ============================================================
# 5. Save
# ============================================================

static_features.to_csv(processed_path + "static_features.csv", index=False)

print(f"Saved static_features.csv")
print(f"Final shape: {static_features.shape}")
print(f"Columns: {static_features.columns.tolist()}")

In [ ]:
import pandas as pd
import numpy as np

processed_path = "../data/processed/"

# ============================================================
# Load source files
# ============================================================

vitals_labs_24h = pd.read_csv(processed_path + "vitals_labs_24h.csv")
vitals_labs_24h['charttime'] = pd.to_datetime(vitals_labs_24h['charttime'])

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])

print(f"vitals_labs_24h.csv loaded: {vitals_labs_24h.shape} (expected 569920 rows)")
print(f"cohort_step1.csv loaded: {cohort_step1.shape}")
print()

# ============================================================
# Join intime, restrict to first 24h
# ============================================================

data = vitals_labs_24h.merge(
    cohort_step1[['subject_id', 'intime']], on='subject_id', how='inner'
)
data['hours_since_intime'] = (data['charttime'] - data['intime']).dt.total_seconds() / 3600
data_24h = data[(data['hours_since_intime'] >= 0) & (data['hours_since_intime'] <= 24)].copy()

print(f"Rows after restricting to first 24h from intime: {len(data_24h)}")
print()

# ============================================================
# Confirmed itemid -> variable name mapping (vitals + labs)
# ============================================================

vitals_map = {
    220045: 'HR',
    220050: 'SBP', 220179: 'SBP',
    220051: 'DBP', 220180: 'DBP',
    220210: 'RR',
    223761: 'Temp', 223762: 'Temp',
    220277: 'SpO2',
    223900: 'GCS_Verbal',
    220739: 'GCS_Eye',
    223901: 'GCS_Motor',
}

labs_map = {
    51301: 'WBC', 51300: 'WBC',
    50813: 'Lactate', 52442: 'Lactate', 53154: 'Lactate',
    50912: 'Creatinine', 52546: 'Creatinine',
    51265: 'Platelets', 53189: 'Platelets',
    50885: 'Bilirubin', 53089: 'Bilirubin',
}

full_map = {**vitals_map, **labs_map}

unmapped_itemids = set(data_24h['itemid'].unique()) - set(full_map.keys())
if unmapped_itemids:
    print(f"WARNING: itemids present in data but NOT in the mapping (will be dropped): {unmapped_itemids}")
else:
    print("All itemids in data_24h are covered by the mapping — none dropped.")
print()

data_24h['variable'] = data_24h['itemid'].map(full_map)
data_24h = data_24h.dropna(subset=['variable']).copy()

# ============================================================
# Print how many rows merged per variable (i.e., variables backed by >1 itemid)
# ============================================================

print("=" * 65)
print("ROWS PER VARIABLE (itemid merge summary)")
print("=" * 65)
merge_summary = data_24h.groupby(['variable', 'itemid']).size().reset_index(name='row_count')
for var in sorted(data_24h['variable'].unique()):
    sub = merge_summary[merge_summary['variable'] == var]
    n_itemids = sub['itemid'].nunique()
    total_rows = sub['row_count'].sum()
    print(f"{var}: {total_rows} total rows across {n_itemids} itemid(s)")
    if n_itemids > 1:
        for _, r in sub.iterrows():
            print(f"    itemid {r['itemid']}: {r['row_count']} rows")
print()

# Sanity check on Temp — confirm values look like Celsius (roughly 30-42), not Fahrenheit (roughly 90-105)
if 'Temp' in data_24h['variable'].values:
    temp_vals = data_24h.loc[data_24h['variable'] == 'Temp', 'valuenum']
    print(f"Temp sanity check — min: {temp_vals.min():.1f}, max: {temp_vals.max():.1f} "
          f"(expect roughly 25-42 if properly in Celsius; if you see values >70, "
          f"F-to-C conversion from the earlier extraction step did not carry through)")
    print()

# ============================================================
# Compute the 6 basic features per subject_id x variable
# ============================================================

def compute_stats(group):
    n = len(group)
    vals = group['valuenum'].values
    hours = group['hours_since_intime'].values

    mean_val = np.mean(vals)
    max_val = np.max(vals)
    min_val = np.min(vals)

    if n > 1:
        std_val = np.std(vals, ddof=1)
        slope_val = np.polyfit(hours, vals, 1)[0]
        ordered = group.sort_values('charttime')
        delta_val = ordered['valuenum'].iloc[-1] - ordered['valuenum'].iloc[0]
    else:
        std_val = np.nan
        slope_val = np.nan
        delta_val = np.nan

    return pd.Series({
        'mean': mean_val, 'max': max_val, 'min': min_val,
        'std': std_val, 'slope': slope_val, 'delta': delta_val,
        'n_readings': n
    })

print("Computing stats per subject_id x variable (this may take a bit)...")
stats_long = data_24h.groupby(['subject_id', 'variable']).apply(compute_stats).reset_index()
print(f"Computed stats for {len(stats_long)} subject_id x variable pairs")
print()

# ============================================================
# Coverage sanity table: how many patients have only 1 reading per variable
# ============================================================

print("=" * 65)
print("SINGLE-READING COVERAGE TABLE (std/slope/delta = NaN for these)")
print("=" * 65)
coverage_rows = []
for var in sorted(stats_long['variable'].unique()):
    sub = stats_long[stats_long['variable'] == var]
    n_total_with_data = len(sub)
    n_single_reading = (sub['n_readings'] == 1).sum()
    pct_single = round(100 * n_single_reading / n_total_with_data, 2) if n_total_with_data > 0 else 0
    coverage_rows.append({
        'variable': var,
        'patients_with_data': n_total_with_data,
        'patients_with_1_reading': n_single_reading,
        'pct_single_reading': pct_single
    })
coverage_table = pd.DataFrame(coverage_rows).sort_values('patients_with_1_reading', ascending=False)
print(coverage_table.to_string(index=False))
print()

# ============================================================
# Pivot to WIDE format: one row per subject_id, columns = {variable}_{stat}
# ============================================================

wide_frames = []
for stat in ['mean', 'max', 'min', 'std', 'slope', 'delta']:
    pivoted = stats_long.pivot(index='subject_id', columns='variable', values=stat)
    pivoted.columns = [f'{v}_{stat}' for v in pivoted.columns]
    wide_frames.append(pivoted)

basic_features = pd.concat(wide_frames, axis=1).reset_index()

# Ensure every cohort patient is present, even if they had zero readings for everything
basic_features_full = cohort_step1[['subject_id']].merge(basic_features, on='subject_id', how='left')

print(f"Final wide-format shape: {basic_features_full.shape} (expected (3122, N))")
print()

# ============================================================
# Spot-check: subject_id = REDACTED_ID_1
# ============================================================

print("=" * 65)
print("SPOT-CHECK: subject_id = REDACTED_ID_1")
print("=" * 65)
spot_row = basic_features_full[basic_features_full['subject_id'] == REDACTED_ID_1]
if len(spot_row) == 0:
    print("subject_id REDACTED_ID_1 NOT found in cohort_step1 / basic_features — check the ID.")
else:
    print(spot_row.to_string(index=False))
    print()
    print("Raw readings for this patient in the first 24h (for manual cross-check):")
    raw_check = data_24h[data_24h['subject_id'] == REDACTED_ID_1].sort_values(['variable', 'charttime'])
    print(raw_check[['variable', 'itemid', 'charttime', 'hours_since_intime', 'valuenum']].to_string(index=False))
print()

# ============================================================
# Save
# ============================================================

basic_features_full.to_csv(processed_path + "basic_features.csv", index=False)

print(f"Saved basic_features.csv")
print(f"Final shape: {basic_features_full.shape}")
print(f"Columns ({len(basic_features_full.columns)}): {basic_features_full.columns.tolist()}")

In [ ]:
import pandas as pd
import numpy as np

processed_path = "../data/processed/"

vitals_labs_24h = pd.read_csv(processed_path + "vitals_labs_24h.csv")
vitals_labs_24h['charttime'] = pd.to_datetime(vitals_labs_24h['charttime'])

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])

print(f"vitals_labs_24h.csv loaded: {vitals_labs_24h.shape}")
print()

# ============================================================
# Rebuild the itemid -> variable mapping (needed to identify Temp rows)
# ============================================================

vitals_map = {
    220045: 'HR',
    220050: 'SBP', 220179: 'SBP',
    220051: 'DBP', 220180: 'DBP',
    220210: 'RR',
    223761: 'Temp', 223762: 'Temp',
    220277: 'SpO2',
    223900: 'GCS_Verbal',
    220739: 'GCS_Eye',
    223901: 'GCS_Motor',
}

vitals_labs_24h['variable'] = vitals_labs_24h['itemid'].map(vitals_map)

# ============================================================
# 1. Temp outlier investigation
# ============================================================

TEMP_LOW, TEMP_HIGH = 25, 42

temp_rows = vitals_labs_24h[vitals_labs_24h['variable'] == 'Temp'].copy()
print(f"Total Temp rows in vitals_labs_24h.csv: {len(temp_rows)}")

temp_outliers = temp_rows[(temp_rows['valuenum'] < TEMP_LOW) | (temp_rows['valuenum'] > TEMP_HIGH)]
print(f"Temp rows OUTSIDE {TEMP_LOW}-{TEMP_HIGH}C: {len(temp_outliers)}")
print()

print("=" * 65)
print("FULL TEMP OUTLIER ROWS (for manual inspection)")
print("=" * 65)
print(temp_outliers[['subject_id', 'itemid', 'charttime', 'valuenum']].sort_values('valuenum').to_string(index=False))
print()

# Diagnostic hints: check if outliers cluster around F-scale (unconverted) or look like typos
if len(temp_outliers) > 0:
    print("Diagnostic breakdown:")
    print(f"  Outliers by source itemid:")
    print(temp_outliers['itemid'].value_counts())
    print()
    # Values in the 90-105 range strongly suggest unconverted Fahrenheit that slipped through
    likely_unconverted_f = temp_outliers[(temp_outliers['valuenum'] >= 70) & (temp_outliers['valuenum'] <= 110)]
    print(f"  Outlier rows in the 70-110 range (plausible unconverted Fahrenheit): {len(likely_unconverted_f)}")
    extreme_outliers = temp_outliers[(temp_outliers['valuenum'] < 20) | (temp_outliers['valuenum'] > 110)]
    print(f"  Outlier rows outside even a Fahrenheit-plausible range (likely typos/entry errors): {len(extreme_outliers)}")
    print()

# ============================================================
# 2. Drop Temp outlier rows entirely (no guessing/fixing)
# ============================================================

temp_outlier_index = temp_outliers.index
n_temp_dropped = len(temp_outlier_index)
n_temp_patients_affected = temp_outliers['subject_id'].nunique()

print(f"Dropping {n_temp_dropped} Temp outlier rows, affecting {n_temp_patients_affected} patients.")
print()

# ============================================================
# 3. Plausibility check for HR, RR, SBP, DBP, SpO2 — explicit, not assumed clean
# ============================================================

plausibility_bounds = {
    'HR': (20, 250),
    'RR': (4, 60),
    'SBP': (40, 300),
    'DBP': (20, 200),
    'SpO2': (0, 100),
}

print("=" * 65)
print("PLAUSIBILITY CHECK — HR, RR, SBP, DBP, SpO2")
print("=" * 65)

all_outlier_indices = list(temp_outlier_index)
outlier_summary_rows = [{'variable': 'Temp', 'min_observed': temp_rows['valuenum'].min(),
                          'max_observed': temp_rows['valuenum'].max(),
                          'n_outliers': n_temp_dropped, 'n_patients_affected': n_temp_patients_affected}]

for var, (lo, hi) in plausibility_bounds.items():
    var_rows = vitals_labs_24h[vitals_labs_24h['variable'] == var]
    if len(var_rows) == 0:
        print(f"{var}: no rows found — skipping")
        continue
    v_min, v_max = var_rows['valuenum'].min(), var_rows['valuenum'].max()
    outliers = var_rows[(var_rows['valuenum'] < lo) | (var_rows['valuenum'] > hi)]
    n_out = len(outliers)
    n_pat = outliers['subject_id'].nunique() if n_out > 0 else 0

    print(f"{var}: observed min={v_min:.1f}, max={v_max:.1f} (plausible range {lo}-{hi})")
    if n_out > 0:
        print(f"  -> {n_out} OUTLIER ROWS found, affecting {n_pat} patients:")
        print(outliers[['subject_id', 'itemid', 'charttime', 'valuenum']].sort_values('valuenum').to_string(index=False))
        all_outlier_indices.extend(outliers.index.tolist())
    else:
        print(f"  -> clean, no outliers")
    print()

    outlier_summary_rows.append({'variable': var, 'min_observed': v_min, 'max_observed': v_max,
                                  'n_outliers': n_out, 'n_patients_affected': n_pat})

outlier_summary = pd.DataFrame(outlier_summary_rows)
print("=" * 65)
print("OUTLIER SUMMARY ACROSS ALL CHECKED VARIABLES")
print("=" * 65)
print(outlier_summary.to_string(index=False))
print()

affected_variables = outlier_summary.loc[outlier_summary['n_outliers'] > 0, 'variable'].tolist()
print(f"Variables requiring basic_features.csv recomputation: {affected_variables}")
print()

# ============================================================
# 4. Re-save vitals_labs_24h.csv with outlier rows removed
# ============================================================

all_outlier_indices = list(set(all_outlier_indices))
vitals_labs_24h_clean = vitals_labs_24h.drop(index=all_outlier_indices).drop(columns=['variable'])

vitals_labs_24h_clean.to_csv(processed_path + "vitals_labs_24h.csv", index=False)

print(f"Total rows dropped across all variables: {len(all_outlier_indices)}")
print(f"vitals_labs_24h.csv re-saved. New shape: {vitals_labs_24h_clean.shape} "
      f"(was {vitals_labs_24h.shape})")
print()

# ============================================================
# 5. Recompute ONLY the affected variables in basic_features.csv
# ============================================================

basic_features = pd.read_csv(processed_path + "basic_features.csv")
print(f"basic_features.csv loaded: {basic_features.shape}")
print()

data = vitals_labs_24h_clean.merge(cohort_step1[['subject_id', 'intime']], on='subject_id', how='inner')
data['hours_since_intime'] = (data['charttime'] - data['intime']).dt.total_seconds() / 3600
data_24h = data[(data['hours_since_intime'] >= 0) & (data['hours_since_intime'] <= 24)].copy()
data_24h['variable'] = data_24h['itemid'].map(vitals_map)

def compute_stats(group):
    n = len(group)
    vals = group['valuenum'].values
    hours = group['hours_since_intime'].values
    mean_val, max_val, min_val = np.mean(vals), np.max(vals), np.min(vals)
    if n > 1:
        std_val = np.std(vals, ddof=1)
        slope_val = np.polyfit(hours, vals, 1)[0]
        ordered = group.sort_values('charttime')
        delta_val = ordered['valuenum'].iloc[-1] - ordered['valuenum'].iloc[0]
    else:
        std_val, slope_val, delta_val = np.nan, np.nan, np.nan
    return pd.Series({'mean': mean_val, 'max': max_val, 'min': min_val,
                       'std': std_val, 'slope': slope_val, 'delta': delta_val, 'n_readings': n})

for var in affected_variables:
    print(f"Recomputing stats for: {var}")
    var_data = data_24h[data_24h['variable'] == var]
    var_stats = var_data.groupby('subject_id').apply(compute_stats).reset_index()

    for stat in ['mean', 'max', 'min', 'std', 'slope', 'delta']:
        col_name = f'{var}_{stat}'
        stat_lookup = var_stats.set_index('subject_id')[stat]
        # Overwrite existing column for ALL cohort patients: NaN where no data remains,
        # recomputed value where data still exists after outlier removal
        basic_features[col_name] = basic_features['subject_id'].map(stat_lookup)

basic_features.to_csv(processed_path + "basic_features.csv", index=False)
print()
print(f"basic_features.csv updated and re-saved. Shape: {basic_features.shape} (should be unchanged)")
print()

# ============================================================
# 6. Re-print Temp sanity check + spot-check subject_id REDACTED_ID_1
# ============================================================

print("=" * 65)
print("POST-CLEANUP SANITY CHECK")
print("=" * 65)
for var in affected_variables:
    print(f"{var}_mean: min={basic_features[f'{var}_mean'].min():.2f}, "
          f"max={basic_features[f'{var}_mean'].max():.2f}")
    print(f"{var}_max:  min={basic_features[f'{var}_max'].min():.2f}, "
          f"max={basic_features[f'{var}_max'].max():.2f}")
    print(f"{var}_min:  min={basic_features[f'{var}_min'].min():.2f}, "
          f"max={basic_features[f'{var}_min'].max():.2f}")
    print()

print("=" * 65)
print("SPOT-CHECK: subject_id = REDACTED_ID_1 (should be UNCHANGED — clean patient)")
print("=" * 65)
spot_row = basic_features[basic_features['subject_id'] == REDACTED_ID_1]
print(spot_row.to_string(index=False))

In [ ]:
import pandas as pd
import numpy as np

processed_path = "../data/processed/"

vitals_labs_24h = pd.read_csv(processed_path + "vitals_labs_24h.csv")
vitals_labs_24h['charttime'] = pd.to_datetime(vitals_labs_24h['charttime'])

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])

print(f"vitals_labs_24h.csv loaded: {vitals_labs_24h.shape}")
print()

vitals_map = {
    220045: 'HR',
    220050: 'SBP', 220179: 'SBP',
    220051: 'DBP', 220180: 'DBP',
    220210: 'RR',
    223761: 'Temp', 223762: 'Temp',
    220277: 'SpO2',
    223900: 'GCS_Verbal',
    220739: 'GCS_Eye',
    223901: 'GCS_Motor',
}
vitals_labs_24h['variable'] = vitals_labs_24h['itemid'].map(vitals_map)

# ============================================================
# 1. SpO2 == 0 and 0-50 range investigation
# ============================================================

spo2_rows = vitals_labs_24h[vitals_labs_24h['variable'] == 'SpO2'].copy()
print(f"Total SpO2 rows: {len(spo2_rows)}")

spo2_zero = spo2_rows[spo2_rows['valuenum'] == 0]
spo2_low_nonzero = spo2_rows[(spo2_rows['valuenum'] > 0) & (spo2_rows['valuenum'] < 50)]

print(f"SpO2 rows with valuenum == 0 (sensor-disconnect artifact): {len(spo2_zero)}")
print(f"SpO2 rows with 0 < valuenum < 50: {len(spo2_low_nonzero)}")
print()

# ============================================================
# 2a. Drop SpO2 == 0 rows outright
# ============================================================

zero_index = spo2_zero.index
n_zero_dropped = len(zero_index)
n_zero_patients = spo2_zero['subject_id'].nunique()
print(f"Dropping {n_zero_dropped} SpO2==0 rows, affecting {n_zero_patients} patients.")
print()

# ============================================================
# 2b. Print the 0-50 range rows WITH neighboring readings for judgment call
# ============================================================

print("=" * 70)
print("SpO2 ROWS WITH 0 < valuenum < 50 (for manual review — NOT auto-dropped)")
print("=" * 70)

if len(spo2_low_nonzero) == 0:
    print("None found.")
else:
    for _, row in spo2_low_nonzero.sort_values(['subject_id', 'charttime']).iterrows():
        subj = row['subject_id']
        ct = row['charttime']
        val = row['valuenum']
        print(f"\nsubject_id={subj}, charttime={ct}, valuenum={val}")

        # Pull neighboring SpO2 readings for the same patient (all SpO2 rows, not just 24h-limited)
        neighbors = spo2_rows[spo2_rows['subject_id'] == subj].sort_values('charttime')
        neighbors_display = neighbors[['charttime', 'valuenum']].copy()
        # Mark the row in question
        neighbors_display['is_flagged_row'] = neighbors_display['charttime'] == ct
        print("  Full SpO2 series for this patient (first 24h):")
        print(neighbors_display.to_string(index=False))
print()

# ============================================================
# 3. Re-save vitals_labs_24h.csv with SpO2==0 rows removed
#    (0-50 range NOT dropped here — left for manual decision after review above)
# ============================================================

vitals_labs_24h_clean = vitals_labs_24h.drop(index=zero_index).drop(columns=['variable'])
vitals_labs_24h_clean.to_csv(processed_path + "vitals_labs_24h.csv", index=False)

print(f"vitals_labs_24h.csv re-saved. New shape: {vitals_labs_24h_clean.shape} "
      f"(was {vitals_labs_24h.shape}, dropped {n_zero_dropped} rows)")
print()
print("NOTE: The 0-50 range rows printed above were NOT dropped automatically —")
print("review them first (do they look like an isolated pre-arrest/coding event with")
print("plausible neighboring readings trending down, or do they look like noise")
print("surrounded by normal values with no clinical trend?). Tell me which rows to drop")
print("and I'll remove them in a follow-up cell, rather than guessing here.")
print()

# ============================================================
# Recompute ONLY SpO2 stats in basic_features.csv
# ============================================================

basic_features = pd.read_csv(processed_path + "basic_features.csv")
print(f"basic_features.csv loaded: {basic_features.shape}")

data = vitals_labs_24h_clean.merge(cohort_step1[['subject_id', 'intime']], on='subject_id', how='inner')
data['hours_since_intime'] = (data['charttime'] - data['intime']).dt.total_seconds() / 3600
data_24h = data[(data['hours_since_intime'] >= 0) & (data['hours_since_intime'] <= 24)].copy()
data_24h['variable'] = data_24h['itemid'].map(vitals_map)

def compute_stats(group):
    n = len(group)
    vals = group['valuenum'].values
    hours = group['hours_since_intime'].values
    mean_val, max_val, min_val = np.mean(vals), np.max(vals), np.min(vals)
    if n > 1:
        std_val = np.std(vals, ddof=1)
        slope_val = np.polyfit(hours, vals, 1)[0]
        ordered = group.sort_values('charttime')
        delta_val = ordered['valuenum'].iloc[-1] - ordered['valuenum'].iloc[0]
    else:
        std_val, slope_val, delta_val = np.nan, np.nan, np.nan
    return pd.Series({'mean': mean_val, 'max': max_val, 'min': min_val,
                       'std': std_val, 'slope': slope_val, 'delta': delta_val, 'n_readings': n})

spo2_data = data_24h[data_24h['variable'] == 'SpO2']
spo2_stats = spo2_data.groupby('subject_id').apply(compute_stats).reset_index()

for stat in ['mean', 'max', 'min', 'std', 'slope', 'delta']:
    col_name = f'SpO2_{stat}'
    stat_lookup = spo2_stats.set_index('subject_id')[stat]
    basic_features[col_name] = basic_features['subject_id'].map(stat_lookup)

basic_features.to_csv(processed_path + "basic_features.csv", index=False)
print()
print(f"basic_features.csv updated (SpO2 columns only). Shape: {basic_features.shape}")
print()

# ============================================================
# Post-cleanup sanity check + spot-check
# ============================================================

print("=" * 65)
print("POST-CLEANUP SpO2 SANITY CHECK")
print("=" * 65)
print(f"SpO2_min: min={basic_features['SpO2_min'].min():.2f}, max={basic_features['SpO2_min'].max():.2f} "
      f"(should now be > 0)")
print(f"SpO2_max: min={basic_features['SpO2_max'].min():.2f}, max={basic_features['SpO2_max'].max():.2f}")
print(f"SpO2_mean: min={basic_features['SpO2_mean'].min():.2f}, max={basic_features['SpO2_mean'].max():.2f}")
print()

print("=" * 65)
print("SPOT-CHECK: subject_id = REDACTED_ID_1 (SpO2_min should remain 95)")
print("=" * 65)
spot_row = basic_features[basic_features['subject_id'] == REDACTED_ID_1]
spo2_cols = ['subject_id'] + [c for c in basic_features.columns if c.startswith('SpO2_')]
print(spot_row[spo2_cols].to_string(index=False))

In [ ]:
import pandas as pd

raw_path = "../data/raw/"
processed_path = "../data/processed/"

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])
cohort_subject_ids = set(cohort_step1['subject_id'])
intime_lookup = cohort_step1.set_index('subject_id')['intime']

vitals_labs_24h_current = pd.read_csv(processed_path + "vitals_labs_24h.csv")
vitals_labs_24h_current['charttime'] = pd.to_datetime(vitals_labs_24h_current['charttime'])

print(f"Current (post-cleanup) vitals_labs_24h.csv: {vitals_labs_24h_current.shape}")
print()

# ============================================================
# "BEFORE" state is not preserved anywhere in memory from earlier ephemeral
# cells, so this reloads the RAW data fresh from chartevents.csv.gz to get a
# true, independently-verified original coverage baseline — not a guess based
# on what I remember dropping in previous cells.
# ============================================================

vitals_map = {
    220045: 'HR',
    220050: 'SBP', 220179: 'SBP',
    220051: 'DBP', 220180: 'DBP',
    220210: 'RR',
    223761: 'Temp', 223762: 'Temp',
    220277: 'SpO2',
}
target_itemids = set(vitals_map.keys())

chunk_size = 1_000_000
raw_rows = []

for chunk in pd.read_csv(
    raw_path + "chartevents.csv.gz", compression="gzip",
    usecols=['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum'],
    chunksize=chunk_size
):
    filtered = chunk[chunk['itemid'].isin(target_itemids) & chunk['subject_id'].isin(cohort_subject_ids)]
    if len(filtered) > 0:
        raw_rows.append(filtered)

raw_events = pd.concat(raw_rows, ignore_index=True) if raw_rows else pd.DataFrame(
    columns=['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum'])
raw_events['charttime'] = pd.to_datetime(raw_events['charttime'])

raw_events = raw_events.merge(intime_lookup.rename('intime'), left_on='subject_id', right_index=True, how='left')
raw_events['hours_since_intime'] = (raw_events['charttime'] - raw_events['intime']).dt.total_seconds() / 3600
raw_events_24h = raw_events[(raw_events['hours_since_intime'] >= 0) & (raw_events['hours_since_intime'] <= 24)].copy()

raw_events_24h['variable'] = raw_events_24h['itemid'].map(vitals_map)
raw_events_24h = raw_events_24h.dropna(subset=['valuenum'])

print(f"Raw (BEFORE any outlier cleanup) rows for these 6 variables, first 24h: {len(raw_events_24h)}")
print()

# ============================================================
# Current (AFTER cleanup) — same variables, same window
# ============================================================

current_events = vitals_labs_24h_current[vitals_labs_24h_current['itemid'].isin(target_itemids)].copy()
current_events = current_events.merge(intime_lookup.rename('intime'), left_on='subject_id', right_index=True, how='left')
current_events['hours_since_intime'] = (current_events['charttime'] - current_events['intime']).dt.total_seconds() / 3600
current_events_24h = current_events[(current_events['hours_since_intime'] >= 0) & (current_events['hours_since_intime'] <= 24)].copy()
current_events_24h['variable'] = current_events_24h['itemid'].map(vitals_map)

print(f"Current (AFTER cleanup) rows for these 6 variables, first 24h: {len(current_events_24h)}")
print()

# ============================================================
# Per-variable, per-patient coverage: BEFORE vs AFTER
# ============================================================

print("=" * 70)
print("PER-VARIABLE PATIENT COVERAGE — BEFORE vs AFTER OUTLIER CLEANUP")
print("=" * 70)

summary_rows = []
patients_lost_all = {}

for var in vitals_map.values():
    var = list(set(vitals_map.values()))  # dummy to avoid dup iteration issues
    break

for var in sorted(set(vitals_map.values())):
    before_patients = set(raw_events_24h.loc[raw_events_24h['variable'] == var, 'subject_id'])
    after_patients = set(current_events_24h.loc[current_events_24h['variable'] == var, 'subject_id'])

    n_before = len(before_patients)
    n_after = len(after_patients)

    lost_all = before_patients - after_patients  # had data before, have NONE after
    patients_lost_all[var] = lost_all

    summary_rows.append({
        'variable': var,
        'patients_with_data_BEFORE': n_before,
        'patients_with_data_AFTER': n_after,
        'patients_who_LOST_ALL_readings': len(lost_all)
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
print()

# ============================================================
# Print any patients who lost ALL readings for a variable
# ============================================================

any_lost = False
for var, lost_set in patients_lost_all.items():
    if len(lost_set) > 0:
        any_lost = True
        print(f"⚠ {var}: {len(lost_set)} patient(s) LOST ALL readings due to outlier removal:")
        for subj in sorted(lost_set):
            # Show what their (now-removed) raw readings actually were, for context
            removed_readings = raw_events_24h[
                (raw_events_24h['subject_id'] == subj) & (raw_events_24h['variable'] == var)
            ][['charttime', 'valuenum']]
            print(f"  subject_id={subj} — original readings that were all removed:")
            print(removed_readings.to_string(index=False))
        print()

if not any_lost:
    print("CONFIRMED: No patient lost all readings for any of Temp, HR, RR, SBP, DBP, SpO2.")
    print("Every patient who had at least one valid reading before cleanup still has")
    print("at least one valid reading after cleanup, for every one of these 6 variables.")

In [ ]:
import pandas as pd
import numpy as np

raw_path = "../data/raw/"
processed_path = "../data/processed/"

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])
cohort_subject_ids = set(cohort_step1['subject_id'])
intime_lookup = cohort_step1.set_index('subject_id')['intime']

TARGET_SUBJECTS = {REDACTED_ID_2, REDACTED_ID_3, REDACTED_ID_4}

# ============================================================
# 1. Reload RAW chartevents Temp rows for just these 3 subject_ids
# ============================================================

chunk_size = 1_000_000
raw_rows = []

for chunk in pd.read_csv(
    raw_path + "chartevents.csv.gz", compression="gzip",
    usecols=['subject_id', 'hadm_id', 'itemid', 'charttime', 'value', 'valuenum'],
    chunksize=chunk_size
):
    filtered = chunk[
        chunk['itemid'].isin([223761, 223762]) &
        chunk['subject_id'].isin(TARGET_SUBJECTS)
    ]
    if len(filtered) > 0:
        raw_rows.append(filtered)

raw_temp_3 = pd.concat(raw_rows, ignore_index=True) if raw_rows else pd.DataFrame(
    columns=['subject_id', 'hadm_id', 'itemid', 'charttime', 'value', 'valuenum'])
raw_temp_3['charttime'] = pd.to_datetime(raw_temp_3['charttime'])

print("=" * 70)
print("RAW chartevents rows for subject_ids REDACTED_ID_2, REDACTED_ID_3, REDACTED_ID_4")
print("=" * 70)
print(raw_temp_3.sort_values(['subject_id', 'charttime']).to_string(index=False))
print()

# ============================================================
# 2. Trace the bug: check raw valuenum (BEFORE F->C conversion) against 25-42C bounds
# ============================================================

print("=" * 70)
print("TRACING THE BUG — checking RAW (unconverted) valuenum against 25-42C bounds")
print("=" * 70)

for _, row in raw_temp_3.sort_values(['subject_id', 'charttime']).iterrows():
    subj, itemid, raw_val = row['subject_id'], row['itemid'], row['valuenum']

    if itemid == 223761:  # Fahrenheit source
        converted_c = (raw_val - 32) * 5 / 9
        raw_in_bounds = 25 <= raw_val <= 42
        converted_in_bounds = 25 <= converted_c <= 42
        print(f"subject_id={subj}: itemid=223761 (Fahrenheit source)")
        print(f"  Raw valuenum (F) = {raw_val}")
        print(f"  Raw value checked against 25-42C bounds directly: {raw_in_bounds} "
              f"{'<-- BUG: unconverted F value compared to C bounds, fails, gets dropped' if not raw_in_bounds else ''}")
        print(f"  Correctly converted to C = {converted_c:.1f}")
        print(f"  Converted value in 25-42C bounds: {converted_in_bounds} "
              f"{'<-- this is what SHOULD have been checked' if converted_in_bounds else ''}")
    else:  # 223762, already Celsius
        in_bounds = 25 <= raw_val <= 42
        print(f"subject_id={subj}: itemid=223762 (Celsius source), valuenum={raw_val}, in bounds: {in_bounds}")
    print()

print("ROOT CAUSE CONFIRMED: the outlier-cleanup step re-pulled Temp rows from raw chartevents")
print("and checked plausibility BEFORE reapplying the Fahrenheit-to-Celsius conversion for")
print("itemid 223761 — so a perfectly normal ~99.5F reading (converts to 37.5C) was compared")
print("directly against the 25-42 bound, failed (99.5 > 42), and was incorrectly dropped as")
print("an 'outlier,' even though the true converted value was completely normal.")
print()

# ============================================================
# 3. FIXED: full fresh reload of ALL Temp data, convert F->C FIRST, then check bounds
# ============================================================

print("=" * 70)
print("RE-RUNNING OUTLIER DETECTION ON FULL TEMP DATASET WITH THE FIX APPLIED")
print("=" * 70)

all_temp_rows = []
for chunk in pd.read_csv(
    raw_path + "chartevents.csv.gz", compression="gzip",
    usecols=['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum'],
    chunksize=chunk_size
):
    filtered = chunk[chunk['itemid'].isin([223761, 223762]) & chunk['subject_id'].isin(cohort_subject_ids)]
    if len(filtered) > 0:
        all_temp_rows.append(filtered)

all_temp = pd.concat(all_temp_rows, ignore_index=True) if all_temp_rows else pd.DataFrame(
    columns=['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum'])
all_temp['charttime'] = pd.to_datetime(all_temp['charttime'])

all_temp = all_temp.merge(intime_lookup.rename('intime'), left_on='subject_id', right_index=True, how='left')
all_temp['hours_since_intime'] = (all_temp['charttime'] - all_temp['intime']).dt.total_seconds() / 3600
all_temp_24h = all_temp[(all_temp['hours_since_intime'] >= 0) & (all_temp['hours_since_intime'] <= 24)].copy()

print(f"Raw Temp rows pulled, first 24h, cohort: {len(all_temp_24h)}")

# CONVERT FIRST (223761 F -> C), THEN check bounds — this is the fix
temp_f_mask = all_temp_24h['itemid'] == 223761
all_temp_24h['valuenum_c'] = all_temp_24h['valuenum']
all_temp_24h.loc[temp_f_mask, 'valuenum_c'] = (all_temp_24h.loc[temp_f_mask, 'valuenum'] - 32) * 5 / 9

TEMP_LOW, TEMP_HIGH = 25, 42
temp_outliers_fixed = all_temp_24h[
    (all_temp_24h['valuenum_c'] < TEMP_LOW) | (all_temp_24h['valuenum_c'] > TEMP_HIGH)
]

print(f"Temp outlier rows under FIXED logic (checking CONVERTED C value): {len(temp_outliers_fixed)}")
print(f"  (compare to whatever count was reported under the buggy logic previously — should be lower)")
print()

if len(temp_outliers_fixed) > 0:
    print("Remaining genuine outliers (converted C value still outside 25-42):")
    print(temp_outliers_fixed[['subject_id', 'itemid', 'charttime', 'valuenum', 'valuenum_c']]
          .sort_values('valuenum_c').to_string(index=False))
    print()

# Confirm the 3 originally-affected patients are now clean
for subj in TARGET_SUBJECTS:
    still_outlier = subj in set(temp_outliers_fixed['subject_id'])
    print(f"subject_id={subj}: still flagged as outlier under fixed logic: {still_outlier} "
          f"{'<-- should be False' if still_outlier else '(confirmed fixed)'}")
print()

# ============================================================
# Build the corrected, clean Temp dataset (converted values, outliers removed)
# ============================================================

temp_clean = all_temp_24h.drop(index=temp_outliers_fixed.index).copy()
temp_clean_final = temp_clean[['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum_c']].rename(
    columns={'valuenum_c': 'valuenum'}
)

# ============================================================
# 4. Re-save vitals_labs_24h.csv: replace ALL Temp rows with this corrected set
# ============================================================

vitals_labs_24h = pd.read_csv(processed_path + "vitals_labs_24h.csv")
vitals_labs_24h['charttime'] = pd.to_datetime(vitals_labs_24h['charttime'])

n_temp_rows_before = (vitals_labs_24h['itemid'].isin([223761, 223762])).sum()
non_temp_rows = vitals_labs_24h[~vitals_labs_24h['itemid'].isin([223761, 223762])].copy()

vitals_labs_24h_fixed = pd.concat([
    non_temp_rows,
    temp_clean_final[['subject_id', 'charttime', 'itemid', 'valuenum']]
], ignore_index=True)

vitals_labs_24h_fixed.to_csv(processed_path + "vitals_labs_24h.csv", index=False)

print(f"Temp rows before this fix: {n_temp_rows_before}")
print(f"Temp rows after this fix: {len(temp_clean_final)}")
print(f"vitals_labs_24h.csv re-saved. New shape: {vitals_labs_24h_fixed.shape}")
print()

# ============================================================
# Recompute Temp stats in basic_features.csv
# ============================================================

basic_features = pd.read_csv(processed_path + "basic_features.csv")

data = vitals_labs_24h_fixed.merge(cohort_step1[['subject_id', 'intime']], on='subject_id', how='inner')
data['hours_since_intime'] = (data['charttime'] - data['intime']).dt.total_seconds() / 3600
data_24h = data[(data['hours_since_intime'] >= 0) & (data['hours_since_intime'] <= 24)].copy()
temp_data = data_24h[data_24h['itemid'].isin([223761, 223762])]

def compute_stats(group):
    n = len(group)
    vals = group['valuenum'].values
    hours = group['hours_since_intime'].values
    mean_val, max_val, min_val = np.mean(vals), np.max(vals), np.min(vals)
    if n > 1:
        std_val = np.std(vals, ddof=1)
        slope_val = np.polyfit(hours, vals, 1)[0]
        ordered = group.sort_values('charttime')
        delta_val = ordered['valuenum'].iloc[-1] - ordered['valuenum'].iloc[0]
    else:
        std_val, slope_val, delta_val = np.nan, np.nan, np.nan
    return pd.Series({'mean': mean_val, 'max': max_val, 'min': min_val,
                       'std': std_val, 'slope': slope_val, 'delta': delta_val, 'n_readings': n})

temp_stats = temp_data.groupby('subject_id').apply(compute_stats).reset_index()

for stat in ['mean', 'max', 'min', 'std', 'slope', 'delta']:
    col_name = f'Temp_{stat}'
    stat_lookup = temp_stats.set_index('subject_id')[stat]
    basic_features[col_name] = basic_features['subject_id'].map(stat_lookup)

basic_features.to_csv(processed_path + "basic_features.csv", index=False)
print(f"basic_features.csv updated (Temp columns only). Shape: {basic_features.shape}")
print()

# ============================================================
# Confirm the 3 originally-affected patients now have valid Temp values
# ============================================================

print("=" * 70)
print("CONFIRMATION: the 3 originally-affected patients")
print("=" * 70)
check_cols = ['subject_id', 'Temp_mean', 'Temp_max', 'Temp_min', 'Temp_std', 'Temp_slope', 'Temp_delta']
print(basic_features[basic_features['subject_id'].isin(TARGET_SUBJECTS)][check_cols].to_string(index=False))
print()

# ============================================================
# Full Temp sanity check to confirm nothing else broke
# ============================================================

print("=" * 70)
print("FULL TEMP SANITY CHECK (post-fix)")
print("=" * 70)
print(f"Temp_mean: min={basic_features['Temp_mean'].min():.2f}, max={basic_features['Temp_mean'].max():.2f}")
print(f"Temp_max:  min={basic_features['Temp_max'].min():.2f}, max={basic_features['Temp_max'].max():.2f}")
print(f"Temp_min:  min={basic_features['Temp_min'].min():.2f}, max={basic_features['Temp_min'].max():.2f}")
print(f"Patients with valid Temp_mean: {basic_features['Temp_mean'].notna().sum()} / {len(basic_features)}")

In [ ]:
import pandas as pd
import numpy as np

processed_path = "../data/processed/"

vitals_labs_24h = pd.read_csv(processed_path + "vitals_labs_24h.csv")
vitals_labs_24h['charttime'] = pd.to_datetime(vitals_labs_24h['charttime'])

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])

print(f"vitals_labs_24h.csv loaded: {vitals_labs_24h.shape}")
print()

SPO2_ITEMID = 220277

KEEP_SUBJECTS = {REDACTED_ID_5, REDACTED_ID_6, REDACTED_ID_7, REDACTED_ID_8}
DROP_SUBJECTS = {REDACTED_ID_9, REDACTED_ID_10, REDACTED_ID_11, REDACTED_ID_12, REDACTED_ID_13, REDACTED_ID_14, REDACTED_ID_15, REDACTED_ID_16, REDACTED_ID_17}

# ============================================================
# Re-identify the borderline (0 < valuenum < 50) SpO2 rows currently in the file
# (SpO2==0 rows were already removed in the previous cleanup step)
# ============================================================

spo2_rows = vitals_labs_24h[vitals_labs_24h['itemid'] == SPO2_ITEMID].copy()
borderline = spo2_rows[(spo2_rows['valuenum'] > 0) & (spo2_rows['valuenum'] < 50)].copy()

print(f"Borderline SpO2 rows (0 < valuenum < 50) currently in file: {len(borderline)}")
print()
print("Full borderline row list (for verification against your decisions):")
print(borderline[['subject_id', 'charttime', 'valuenum']].sort_values(['subject_id', 'charttime']).to_string(index=False))
print()

# ============================================================
# Verify every subject_id in KEEP/DROP lists is actually present in the borderline set,
# and flag anything in the borderline set NOT covered by either list
# ============================================================

borderline_subjects = set(borderline['subject_id'])
covered_subjects = KEEP_SUBJECTS | DROP_SUBJECTS
uncovered = borderline_subjects - covered_subjects
missing_from_data = covered_subjects - borderline_subjects

if uncovered:
    print(f"WARNING: borderline subject_ids NOT covered by either KEEP or DROP list: {uncovered}")
    print("  These rows will be left UNCHANGED (not dropped) since no decision was given for them.")
if missing_from_data:
    print(f"WARNING: subject_ids in your KEEP/DROP lists NOT found in the current borderline data: {missing_from_data}")
    print("  (may have already been handled, or IDs don't match — double check)")
print()

# ============================================================
# Drop only the flagged rows for DROP_SUBJECTS (by subject_id, matched to the
# borderline valuenum range — not a blanket threshold, only these specific rows)
# ============================================================

rows_to_drop = borderline[borderline['subject_id'].isin(DROP_SUBJECTS)]
rows_kept_explicit = borderline[borderline['subject_id'].isin(KEEP_SUBJECTS)]

print(f"Rows being DROPPED ({len(rows_to_drop)} total, from {rows_to_drop['subject_id'].nunique()} patients):")
print(rows_to_drop[['subject_id', 'charttime', 'valuenum']].sort_values(['subject_id', 'charttime']).to_string(index=False))
print()

print(f"Rows being explicitly KEPT ({len(rows_kept_explicit)} total, from {rows_kept_explicit['subject_id'].nunique()} patients):")
print(rows_kept_explicit[['subject_id', 'charttime', 'valuenum']].sort_values(['subject_id', 'charttime']).to_string(index=False))
print()

vitals_labs_24h_clean = vitals_labs_24h.drop(index=rows_to_drop.index)

vitals_labs_24h_clean.to_csv(processed_path + "vitals_labs_24h.csv", index=False)

print(f"vitals_labs_24h.csv re-saved. New shape: {vitals_labs_24h_clean.shape} "
      f"(was {vitals_labs_24h.shape}, dropped {len(rows_to_drop)} rows)")
print()

# ============================================================
# Recompute SpO2 stats in basic_features.csv
# ============================================================

basic_features = pd.read_csv(processed_path + "basic_features.csv")

data = vitals_labs_24h_clean.merge(cohort_step1[['subject_id', 'intime']], on='subject_id', how='inner')
data['hours_since_intime'] = (data['charttime'] - data['intime']).dt.total_seconds() / 3600
data_24h = data[(data['hours_since_intime'] >= 0) & (data['hours_since_intime'] <= 24)].copy()
spo2_data = data_24h[data_24h['itemid'] == SPO2_ITEMID]

def compute_stats(group):
    n = len(group)
    vals = group['valuenum'].values
    hours = group['hours_since_intime'].values
    mean_val, max_val, min_val = np.mean(vals), np.max(vals), np.min(vals)
    if n > 1:
        std_val = np.std(vals, ddof=1)
        slope_val = np.polyfit(hours, vals, 1)[0]
        ordered = group.sort_values('charttime')
        delta_val = ordered['valuenum'].iloc[-1] - ordered['valuenum'].iloc[0]
    else:
        std_val, slope_val, delta_val = np.nan, np.nan, np.nan
    return pd.Series({'mean': mean_val, 'max': max_val, 'min': min_val,
                       'std': std_val, 'slope': slope_val, 'delta': delta_val, 'n_readings': n})

spo2_stats = spo2_data.groupby('subject_id').apply(compute_stats).reset_index()

for stat in ['mean', 'max', 'min', 'std', 'slope', 'delta']:
    col_name = f'SpO2_{stat}'
    stat_lookup = spo2_stats.set_index('subject_id')[stat]
    basic_features[col_name] = basic_features['subject_id'].map(stat_lookup)

basic_features.to_csv(processed_path + "basic_features.csv", index=False)
print(f"basic_features.csv updated (SpO2 columns only). Shape: {basic_features.shape}")
print()

# ============================================================
# Confirm the specific KEEP/DROP patients' resulting SpO2_min
# ============================================================

check_cols = ['subject_id', 'SpO2_mean', 'SpO2_max', 'SpO2_min']

print("=" * 65)
print("KEEP-LIST patients (should retain their low borderline reading in SpO2_min)")
print("=" * 65)
print(basic_features[basic_features['subject_id'].isin(KEEP_SUBJECTS)][check_cols].to_string(index=False))
print()

print("=" * 65)
print("DROP-LIST patients (SpO2_min should now reflect their next-lowest VALID reading)")
print("=" * 65)
print(basic_features[basic_features['subject_id'].isin(DROP_SUBJECTS)][check_cols].to_string(index=False))
print()

# ============================================================
# Full SpO2 sanity check
# ============================================================

print("=" * 65)
print("FULL SpO2 SANITY CHECK (post-resolution)")
print("=" * 65)
print(f"SpO2_min: min={basic_features['SpO2_min'].min():.2f}, max={basic_features['SpO2_min'].max():.2f}")
print(f"SpO2_max: min={basic_features['SpO2_max'].min():.2f}, max={basic_features['SpO2_max'].max():.2f}")
print(f"SpO2_mean: min={basic_features['SpO2_mean'].min():.2f}, max={basic_features['SpO2_mean'].max():.2f}")
print(f"Patients with valid SpO2_mean: {basic_features['SpO2_mean'].notna().sum()} / {len(basic_features)}")

In [ ]:
import pandas as pd
import numpy as np

raw_path = "../data/raw/"
processed_path = "../data/processed/"

# ============================================================
# STEP 0: PRINT THE FULL FIELD MAPPING BEFORE COMPUTING ANYTHING
# ============================================================

print("=" * 70)
print("SOFA COMPONENT -> MIMIC-IV FIELD MAPPING (review before trusting output)")
print("=" * 70)
print("""
1. RESPIRATORY
   - Standard SOFA uses PaO2/FiO2 (from arterial blood gas).
   - We did NOT extract PaO2 in this pipeline (only SpO2, a pulse-ox proxy).
   - Using SpO2/FiO2 (S/F ratio) as a validated but APPROXIMATE substitute
     (Rice et al. 2007 conversion), NOT true PaO2/FiO2. This will systematically
     differ from a chart-based SOFA computed with real ABGs. Flagging this as
     a real methodological limitation, not silently treating S/F as equivalent.
   - SpO2: itemid 220277 (chartevents, in vitals_labs_24h.csv)
   - FiO2: itemid 223835 (chartevents, in organ_burden_inputs_24h.csv)
   - No ventilator-support flag was extracted, so the "with respiratory support"
     condition in scores 3-4 is NOT applied — thresholds are ratio-only.

2. COAGULATION
   - Platelets: itemid 51265/53189 (labevents, in vitals_labs_24h.csv,
     merged as 'Platelets')

3. LIVER
   - Bilirubin: itemid 50885/53089 (labevents, in vitals_labs_24h.csv,
     merged as 'Bilirubin') — total bilirubin assumed (not distinguishing
     direct/indirect, since source itemids were not split that way upstream)

4. CARDIOVASCULAR
   - MAP: NOT separately extracted in this pipeline. Computing MAP from
     SBP/DBP as (SBP + 2*DBP) / 3 per reading (standard formula), using
     itemid 220050/220179 (SBP) and 220051/220180 (DBP) from vitals_labs_24h.csv.
     This is an approximation of directly-charted MAP, not identical to it.
   - Vasopressor dose: valuenum_mcg_kg_min column in organ_burden_inputs_24h.csv
     (already unit-harmonized). Classic SOFA cardiovascular scoring is ONLY
     defined for dopamine, dobutamine, epinephrine, and norepinephrine doses.
     Phenylephrine and vasopressin do NOT have standard SOFA dose thresholds —
     their presence/dose will be reported separately but NOT plugged into the
     classic 4-drug threshold formula, to avoid inventing thresholds that don't
     exist in the original SOFA definition.

5. CNS
   - GCS Total = GCS_Eye + GCS_Verbal + GCS_Motor, using itemid 220739, 223900,
     223901 (chartevents, in vitals_labs_24h.csv). GCS_Verbal already includes
     the earlier full text-to-score mapping (Oriented=5...No Response=1).
   - Worst GCS = LOWEST total across matched-time readings in first 24h.

6. RENAL
   - Creatinine: itemid 50912/52546 (labevents, in vitals_labs_24h.csv,
     merged as 'Creatinine')
   - Urine output: itemid 226559/226560 (outputevents, in
     organ_burden_inputs_24h.csv) — summed across the first 24h per patient
     for the UO<500mL/UO<200mL sub-criterion.
""")
print()

# ============================================================
# Load source files
# ============================================================

vitals_labs_24h = pd.read_csv(processed_path + "vitals_labs_24h.csv")
vitals_labs_24h['charttime'] = pd.to_datetime(vitals_labs_24h['charttime'])

organ_burden_24h = pd.read_csv(processed_path + "organ_burden_inputs_24h.csv")
organ_burden_24h['charttime'] = pd.to_datetime(organ_burden_24h['charttime'])

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])

d_items = pd.read_csv(raw_path + "d_items.csv.gz", compression="gzip")
itemid_labels = d_items.set_index('itemid')['label'].to_dict()

n_cohort = len(cohort_step1)
print(f"Cohort: {n_cohort} patients")
print()

vitals_map = {
    220045: 'HR', 220050: 'SBP', 220179: 'SBP', 220051: 'DBP', 220180: 'DBP',
    220210: 'RR', 223761: 'Temp', 223762: 'Temp', 220277: 'SpO2',
    223900: 'GCS_Verbal', 220739: 'GCS_Eye', 223901: 'GCS_Motor',
    51301: 'WBC', 51300: 'WBC', 50813: 'Lactate', 52442: 'Lactate', 53154: 'Lactate',
    50912: 'Creatinine', 52546: 'Creatinine', 51265: 'Platelets', 53189: 'Platelets',
    50885: 'Bilirubin', 53089: 'Bilirubin',
}
vitals_labs_24h['variable'] = vitals_labs_24h['itemid'].map(vitals_map)

# ============================================================
# 1. RESPIRATORY: worst (lowest) SpO2/FiO2 ratio in first 24h
# ============================================================

spo2_data = vitals_labs_24h[vitals_labs_24h['variable'] == 'SpO2'][['subject_id', 'charttime', 'valuenum']].rename(columns={'valuenum': 'spo2'})
fio2_data = organ_burden_24h[organ_burden_24h['itemid'] == 223835][['subject_id', 'charttime', 'valuenum']].rename(columns={'valuenum': 'fio2'})

# Match SpO2 and FiO2 within +/-1h of each other per patient (simple nearest-match approach)
sf_rows = []
for subj in spo2_data['subject_id'].unique():
    subj_spo2 = spo2_data[spo2_data['subject_id'] == subj]
    subj_fio2 = fio2_data[fio2_data['subject_id'] == subj]
    if len(subj_fio2) == 0:
        continue
    for _, s_row in subj_spo2.iterrows():
        time_diffs = (subj_fio2['charttime'] - s_row['charttime']).abs()
        if time_diffs.min() <= pd.Timedelta(hours=1):
            nearest_fio2 = subj_fio2.loc[time_diffs.idxmin(), 'fio2']
            fio2_frac = nearest_fio2 / 100 if nearest_fio2 > 1 else nearest_fio2  # handle % vs fraction
            if fio2_frac > 0:
                sf_rows.append({'subject_id': subj, 'sf_ratio': s_row['spo2'] / fio2_frac})

sf_df = pd.DataFrame(sf_rows)
worst_sf = sf_df.groupby('subject_id')['sf_ratio'].min() if len(sf_df) > 0 else pd.Series(dtype=float)

def score_respiratory(sf_ratio):
    if pd.isna(sf_ratio):
        return np.nan
    if sf_ratio >= 512:
        return 0
    elif sf_ratio >= 357:
        return 1
    elif sf_ratio >= 214:
        return 2
    elif sf_ratio >= 89:
        return 3
    else:
        return 4

resp_scores = worst_sf.apply(score_respiratory)
print(f"Respiratory component computed for {resp_scores.notna().sum()} / {n_cohort} patients "
      f"(requires BOTH an SpO2 and a matched FiO2 reading within 1h)")
print()

# ============================================================
# 2. COAGULATION: worst (lowest) platelets
# ============================================================

plt_data = vitals_labs_24h[vitals_labs_24h['variable'] == 'Platelets']
worst_plt = plt_data.groupby('subject_id')['valuenum'].min()

def score_coag(plt):
    if pd.isna(plt): return np.nan
    if plt >= 150: return 0
    elif plt >= 100: return 1
    elif plt >= 50: return 2
    elif plt >= 20: return 3
    else: return 4

coag_scores = worst_plt.apply(score_coag)

# ============================================================
# 3. LIVER: worst (highest) bilirubin
# ============================================================

bili_data = vitals_labs_24h[vitals_labs_24h['variable'] == 'Bilirubin']
worst_bili = bili_data.groupby('subject_id')['valuenum'].max()

def score_liver(bili):
    if pd.isna(bili): return np.nan
    if bili < 1.2: return 0
    elif bili < 2.0: return 1
    elif bili < 6.0: return 2
    elif bili < 12.0: return 3
    else: return 4

liver_scores = worst_bili.apply(score_liver)

# ============================================================
# 4. CARDIOVASCULAR: worst MAP + vasopressor dose (dopamine/epi/norepi ONLY)
# ============================================================

sbp_data = vitals_labs_24h[vitals_labs_24h['variable'] == 'SBP'][['subject_id', 'charttime', 'valuenum']].rename(columns={'valuenum': 'sbp'})
dbp_data = vitals_labs_24h[vitals_labs_24h['variable'] == 'DBP'][['subject_id', 'charttime', 'valuenum']].rename(columns={'valuenum': 'dbp'})

bp_merged = sbp_data.merge(dbp_data, on=['subject_id', 'charttime'], how='inner')
bp_merged['map'] = (bp_merged['sbp'] + 2 * bp_merged['dbp']) / 3
worst_map = bp_merged.groupby('subject_id')['map'].min()

# Vasopressor doses — classify by itemid label to isolate the classic SOFA 4 drugs
vaso_itemids_present = organ_burden_24h.loc[
    ~organ_burden_24h['itemid'].isin({223835, 226559, 226560}), 'itemid'
].unique()

vaso_labels = {iid: itemid_labels.get(iid, 'UNKNOWN') for iid in vaso_itemids_present}
print("Vasopressor itemids present and their labels (for cardiovascular scoring):")
for iid, lbl in vaso_labels.items():
    print(f"  {iid}: {lbl}")
print()

dopamine_ids = {iid for iid, lbl in vaso_labels.items() if 'dopamine' in lbl.lower()}
norepi_ids = {iid for iid, lbl in vaso_labels.items() if 'norepinephrine' in lbl.lower()}
epi_ids = {iid for iid, lbl in vaso_labels.items() if 'epinephrine' in lbl.lower() and 'nor' not in lbl.lower()}
other_vaso_ids = set(vaso_itemids_present) - dopamine_ids - norepi_ids - epi_ids

if other_vaso_ids:
    print(f"NOTE: itemids {other_vaso_ids} (likely vasopressin/phenylephrine) are present but have")
    print("NO standard SOFA cardiovascular dose threshold — excluded from the score below.")
    print()

conv_col = 'valuenum_mcg_kg_min' if 'valuenum_mcg_kg_min' in organ_burden_24h.columns else None
if conv_col is None:
    raise ValueError("organ_burden_inputs_24h.csv missing 'valuenum_mcg_kg_min' — run the unit conversion step first.")

max_dopamine = organ_burden_24h[organ_burden_24h['itemid'].isin(dopamine_ids)].groupby('subject_id')[conv_col].max()
max_norepi = organ_burden_24h[organ_burden_24h['itemid'].isin(norepi_ids)].groupby('subject_id')[conv_col].max()
max_epi = organ_burden_24h[organ_burden_24h['itemid'].isin(epi_ids)].groupby('subject_id')[conv_col].max()

def score_cardio(subj_id):
    map_val = worst_map.get(subj_id, np.nan)
    dopa = max_dopamine.get(subj_id, np.nan)
    nore = max_norepi.get(subj_id, np.nan)
    epi = max_epi.get(subj_id, np.nan)

    if not pd.isna(dopa) and dopa > 15:
        return 4
    if not pd.isna(epi) and epi > 0.1:
        return 4
    if not pd.isna(nore) and nore > 0.1:
        return 4
    if not pd.isna(dopa) and dopa > 5:
        return 3
    if not pd.isna(epi) and epi <= 0.1 and epi > 0:
        return 3
    if not pd.isna(nore) and nore <= 0.1 and nore > 0:
        return 3
    if not pd.isna(dopa) and dopa <= 5 and dopa > 0:
        return 2
    if pd.isna(map_val):
        return np.nan
    if map_val < 70:
        return 1
    return 0

all_subjects = cohort_step1['subject_id']
cardio_scores = pd.Series({s: score_cardio(s) for s in all_subjects})

# ============================================================
# 5. CNS: worst (lowest) GCS Total = Eye + Verbal + Motor, matched by charttime
# ============================================================

gcs_components = vitals_labs_24h[vitals_labs_24h['variable'].isin(['GCS_Eye', 'GCS_Verbal', 'GCS_Motor'])]
gcs_pivot = gcs_components.pivot_table(index=['subject_id', 'charttime'], columns='variable', values='valuenum', aggfunc='first').reset_index()

for col in ['GCS_Eye', 'GCS_Verbal', 'GCS_Motor']:
    if col not in gcs_pivot.columns:
        gcs_pivot[col] = np.nan

gcs_pivot['gcs_total'] = gcs_pivot['GCS_Eye'] + gcs_pivot['GCS_Verbal'] + gcs_pivot['GCS_Motor']
gcs_complete = gcs_pivot.dropna(subset=['gcs_total'])
worst_gcs = gcs_complete.groupby('subject_id')['gcs_total'].min()

print(f"CNS component computed for {worst_gcs.notna().sum()} / {n_cohort} patients "
      f"(requires all 3 GCS components at the SAME charttime)")
print()

def score_cns(gcs):
    if pd.isna(gcs): return np.nan
    if gcs == 15: return 0
    elif gcs >= 13: return 1
    elif gcs >= 10: return 2
    elif gcs >= 6: return 3
    else: return 4

cns_scores = worst_gcs.apply(score_cns)

# ============================================================
# 6. RENAL: worst (highest) creatinine, with UO sub-criterion
# ============================================================

creat_data = vitals_labs_24h[vitals_labs_24h['variable'] == 'Creatinine']
worst_creat = creat_data.groupby('subject_id')['valuenum'].max()

urine_data = organ_burden_24h[organ_burden_24h['itemid'].isin({226559, 226560})]
total_uo_24h = urine_data.groupby('subject_id')['valuenum'].sum()

def score_renal(subj_id):
    creat = worst_creat.get(subj_id, np.nan)
    uo = total_uo_24h.get(subj_id, np.nan)

    creat_score = np.nan
    if not pd.isna(creat):
        if creat < 1.2: creat_score = 0
        elif creat < 2.0: creat_score = 1
        elif creat < 3.5: creat_score = 2
        elif creat < 5.0: creat_score = 3
        else: creat_score = 4

    uo_score = np.nan
    if not pd.isna(uo):
        if uo < 200: uo_score = 4
        elif uo < 500: uo_score = 3

    scores_available = [s for s in [creat_score, uo_score] if not pd.isna(s)]
    return max(scores_available) if scores_available else np.nan

renal_scores = pd.Series({s: score_renal(s) for s in all_subjects})

# ============================================================
# COMBINE INTO FINAL SOFA TABLE
# ============================================================

sofa_df = pd.DataFrame({'subject_id': all_subjects}).set_index('subject_id')
sofa_df['resp_score'] = resp_scores
sofa_df['coag_score'] = coag_scores
sofa_df['liver_score'] = liver_scores
sofa_df['cardio_score'] = cardio_scores
sofa_df['cns_score'] = cns_scores
sofa_df['renal_score'] = renal_scores

component_cols = ['resp_score', 'coag_score', 'liver_score', 'cardio_score', 'cns_score', 'renal_score']

# Standard convention: missing component treated as 0 for total SOFA (common but NOT universal
# practice — flagged explicitly here rather than silently applied)
sofa_df['n_components_available'] = sofa_df[component_cols].notna().sum(axis=1)
sofa_df['sofa_total'] = sofa_df[component_cols].fillna(0).sum(axis=1)

sofa_df = sofa_df.reset_index()

print("NOTE: missing components are treated as 0 when summing sofa_total (a common convention,")
print("but it will UNDERESTIMATE true severity for patients with sparse data). The")
print("'n_components_available' column tells you how many of the 6 components had usable data")
print("per patient — treat sofa_total for patients with n_components_available < 6 with caution.")
print()

print("=" * 65)
print("n_components_available — distribution")
print("=" * 65)
print(sofa_df['n_components_available'].value_counts().sort_index())
print()

print("=" * 65)
print("SOFA SCORE DISTRIBUTION")
print("=" * 65)
print(f"Min: {sofa_df['sofa_total'].min()}")
print(f"Mean: {sofa_df['sofa_total'].mean():.2f}")
print(f"Max: {sofa_df['sofa_total'].max()}")
print(f"Median: {sofa_df['sofa_total'].median()}")
print()
print("Per-component score distributions:")
for col in component_cols:
    print(f"  {col}: mean={sofa_df[col].mean():.2f}, missing={sofa_df[col].isna().sum()}")
print()

sofa_df.to_csv(processed_path + "sofa_scores.csv", index=False)
print(f"Saved sofa_scores.csv. Shape: {sofa_df.shape}")

In [ ]:
import pandas as pd

processed_path = "../data/processed/"

sofa_scores = pd.read_csv(processed_path + "sofa_scores.csv")

print(f"sofa_scores.csv loaded: {sofa_scores.shape}")
print()

if 'n_components_available' not in sofa_scores.columns:
    raise ValueError("'n_components_available' column not found — check the file is the correct version.")

# ============================================================
# 1. Add sofa_reliable flag
# ============================================================

sofa_scores['sofa_reliable'] = sofa_scores['n_components_available'] >= 4

print("=" * 65)
print("n_components_available — full breakdown")
print("=" * 65)
print(sofa_scores['n_components_available'].value_counts().sort_index())
print()

print("=" * 65)
print("sofa_reliable — value_counts()")
print("=" * 65)
print(sofa_scores['sofa_reliable'].value_counts())
print()

n_lt3 = (sofa_scores['n_components_available'] < 3).sum()
n_eq4 = (sofa_scores['n_components_available'] == 4).sum()
print(f"Patients with <3 components (clearest concern): {n_lt3} (expected ~69)")
print(f"Patients with exactly 4 components (borderline): {n_eq4} (expected ~546)")
print()

# ============================================================
# 2. sofa_total distribution: reliable vs unreliable groups
# ============================================================

reliable_group = sofa_scores[sofa_scores['sofa_reliable'] == True]
unreliable_group = sofa_scores[sofa_scores['sofa_reliable'] == False]

print("=" * 65)
print(f"sofa_total DISTRIBUTION — sofa_reliable = True (n={len(reliable_group)})")
print("=" * 65)
print(reliable_group['sofa_total'].describe())
print()

print("=" * 65)
print(f"sofa_total DISTRIBUTION — sofa_reliable = False (n={len(unreliable_group)})")
print("=" * 65)
print(unreliable_group['sofa_total'].describe())
print()

# Direct comparison of central tendency
mean_diff = reliable_group['sofa_total'].mean() - unreliable_group['sofa_total'].mean()
median_diff = reliable_group['sofa_total'].median() - unreliable_group['sofa_total'].median()

print("=" * 65)
print("DIRECT COMPARISON")
print("=" * 65)
print(f"Reliable group  — mean: {reliable_group['sofa_total'].mean():.2f}, "
      f"median: {reliable_group['sofa_total'].median():.1f}")
print(f"Unreliable group — mean: {unreliable_group['sofa_total'].mean():.2f}, "
      f"median: {unreliable_group['sofa_total'].median():.1f}")
print(f"Difference (reliable - unreliable): mean={mean_diff:.2f}, median={median_diff:.1f}")
print()

if mean_diff > 1:
    print("INTERPRETATION: unreliable-group sofa_total is meaningfully LOWER than the")
    print("reliable group's. This is consistent with the default-to-0 bias actually")
    print("distorting scores downward for sparse-data patients — not just a theoretical risk.")
    print("Recommend treating sofa_total as UNTRUSTWORTHY for the sofa_reliable=False group")
    print("in any downstream analysis or model, rather than using it at face value.")
elif mean_diff < -1:
    print("INTERPRETATION: unreliable-group sofa_total is actually HIGHER than the reliable")
    print("group's, despite having fewer components. This suggests the missing components")
    print("in this group are disproportionately the LOW-scoring ones (e.g., patients who")
    print("simply had less monitoring because they were less sick), not a uniform bias.")
    print("Worth further investigation before assuming which direction, if any, the bias runs.")
else:
    print("INTERPRETATION: sofa_total distributions look broadly similar between groups —")
    print("no strong evidence the default-to-0 bias is materially distorting scores here,")
    print("though sofa_reliable should still be retained as a transparency/caveat column.")
print()

# Also show component-level breakdown for the unreliable group — which components
# are MOST often missing, to understand the mechanism
print("=" * 65)
print("Component missingness WITHIN the unreliable group (which components drive n<4?)")
print("=" * 65)
component_cols = ['resp_score', 'coag_score', 'liver_score', 'cardio_score', 'cns_score', 'renal_score']
for col in component_cols:
    pct_missing = 100 * unreliable_group[col].isna().sum() / len(unreliable_group)
    print(f"  {col}: {pct_missing:.1f}% missing within unreliable group")
print()

# ============================================================
# 3 & 4. Keep everything, re-save
# ============================================================

sofa_scores.to_csv(processed_path + "sofa_scores.csv", index=False)

print(f"Saved sofa_scores.csv with sofa_reliable column added.")
print(f"Final shape: {sofa_scores.shape}")
print(f"Columns: {sofa_scores.columns.tolist()}")

In [ ]:
import pandas as pd
import numpy as np

raw_path = "../data/raw/"
processed_path = "../data/processed/"

# ============================================================
# STEP 1: Search d_items for MAP — review label text carefully
# ============================================================

d_items = pd.read_csv(raw_path + "d_items.csv.gz", compression="gzip")

map_candidates = d_items[
    d_items['label'].str.contains('Mean Arterial', case=False, na=False) |
    d_items['label'].str.contains('MAP', case=False, na=False)
].copy()

print("=" * 70)
print("MAP / Mean Arterial candidate matches from d_items")
print("=" * 70)
print(map_candidates[['itemid', 'label', 'abbreviation', 'linksto', 'category', 'unitname']].to_string(index=False))
print()

print("REVIEW NOTE: 'MAP' as a bare substring can false-positive on unrelated labels")
print("(e.g., something containing 'MAPPING', a drug name, or an unrelated abbreviation).")
print("Look at the actual label text above — genuine candidates should say something like")
print("'Arterial Blood Pressure mean', 'Non Invasive Blood Pressure mean', or similar,")
print("not just any row where the substring happened to match.")
print()

# Flag anything that matched purely on the loose 'MAP' substring but doesn't look
# like a real MAP/pressure item, so it's visible rather than silently included
suspicious = map_candidates[
    ~map_candidates['label'].str.contains('mean arterial', case=False, na=False) &
    ~map_candidates['label'].str.contains('blood pressure', case=False, na=False) &
    ~map_candidates['label'].str.contains('\\bmap\\b', case=False, na=False, regex=True)
]
if len(suspicious) > 0:
    print("Rows that matched the search but look potentially UNRELATED (review before using):")
    print(suspicious[['itemid', 'label']].to_string(index=False))
    print()

# ============================================================
# STEP 2: Load cohort + existing SBP/DBP-derived MAP for comparison baseline
# ============================================================

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])
cohort_subject_ids = set(cohort_step1['subject_id'])
intime_lookup = cohort_step1.set_index('subject_id')['intime']

vitals_labs_24h = pd.read_csv(processed_path + "vitals_labs_24h.csv")
vitals_labs_24h['charttime'] = pd.to_datetime(vitals_labs_24h['charttime'])

vitals_map = {220050: 'SBP', 220179: 'SBP', 220051: 'DBP', 220180: 'DBP'}
vitals_labs_24h['variable'] = vitals_labs_24h['itemid'].map(vitals_map)

sbp_data = vitals_labs_24h[vitals_labs_24h['variable'] == 'SBP'][['subject_id', 'charttime', 'valuenum']].rename(columns={'valuenum': 'sbp'})
dbp_data = vitals_labs_24h[vitals_labs_24h['variable'] == 'DBP'][['subject_id', 'charttime', 'valuenum']].rename(columns={'valuenum': 'dbp'})

bp_merged = sbp_data.merge(dbp_data, on=['subject_id', 'charttime'], how='inner')
bp_merged['map_derived'] = (bp_merged['sbp'] + 2 * bp_merged['dbp']) / 3
worst_map_derived = bp_merged.groupby('subject_id')['map_derived'].min()

print(f"SBP/DBP-derived MAP available for {worst_map_derived.notna().sum()} / {len(cohort_step1)} patients")
print()

# ============================================================
# STEP 3: If real MAP itemid(s) confirmed, extract and compare — else stop here
# ============================================================

if len(map_candidates) == 0:
    print("NO MAP-related itemids found in d_items at all. Nothing to compare — the")
    print("SBP/DBP-derived approximation is the only option available in this dataset.")
else:
    map_itemids = set(map_candidates['itemid'])
    print(f"Pulling chartevents for candidate MAP itemid(s): {sorted(map_itemids)}")
    print()

    chunk_size = 1_000_000
    map_rows = []
    for chunk in pd.read_csv(
        raw_path + "chartevents.csv.gz", compression="gzip",
        usecols=['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum'],
        chunksize=chunk_size
    ):
        filtered = chunk[chunk['itemid'].isin(map_itemids) & chunk['subject_id'].isin(cohort_subject_ids)]
        if len(filtered) > 0:
            map_rows.append(filtered)

    map_events = pd.concat(map_rows, ignore_index=True) if map_rows else pd.DataFrame(
        columns=['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum'])
    map_events['charttime'] = pd.to_datetime(map_events['charttime'])

    print(f"Raw MAP chartevents rows pulled (cohort subject_ids only): {len(map_events)}")

    map_events = map_events.merge(intime_lookup.rename('intime'), left_on='subject_id', right_index=True, how='left')
    map_events['hours_since_intime'] = (map_events['charttime'] - map_events['intime']).dt.total_seconds() / 3600
    map_events_24h = map_events[(map_events['hours_since_intime'] >= 0) & (map_events['hours_since_intime'] <= 24)].copy()

    print(f"MAP rows within first 24h: {len(map_events_24h)}")

    n_patients_with_real_map = map_events_24h['subject_id'].nunique()
    pct_coverage = 100 * n_patients_with_real_map / len(cohort_step1)
    print(f"Patients with >=1 REAL charted MAP reading in first 24h: "
          f"{n_patients_with_real_map} / {len(cohort_step1)} ({pct_coverage:.2f}%)")
    print()

    # Plausibility check on the real MAP values before trusting them
    print(f"Real MAP value range: min={map_events_24h['valuenum'].min():.1f}, "
          f"max={map_events_24h['valuenum'].max():.1f} "
          f"(sanity: physiologic MAP is roughly 40-150 mmHg for a live ICU patient)")
    implausible = map_events_24h[(map_events_24h['valuenum'] < 20) | (map_events_24h['valuenum'] > 200)]
    print(f"Implausible real MAP rows (outside 20-200): {len(implausible)}")
    print()

    if n_patients_with_real_map < 50:
        print("Coverage is very low — not enough real MAP data to draw a meaningful comparison.")
        print("Recommend sticking with the SBP/DBP-derived approximation given this sparsity.")
    else:
        worst_map_real = map_events_24h.groupby('subject_id')['valuenum'].min()

        # ============================================================
        # STEP 4: Compare real vs derived MAP, patient by patient (where both exist)
        # ============================================================

        comparison = pd.DataFrame({
            'map_real': worst_map_real,
            'map_derived': worst_map_derived
        }).dropna()

        comparison['abs_diff'] = (comparison['map_real'] - comparison['map_derived']).abs()

        print(f"Patients with BOTH real and derived worst-MAP available for comparison: {len(comparison)}")
        print()
        print("Difference (|real - derived|) distribution:")
        print(comparison['abs_diff'].describe())
        print()

        # ============================================================
        # Does the difference actually change the cardiovascular SOFA sub-score?
        # ============================================================

        def map_to_cardio_partial(map_val):
            # MAP-only portion of the cardio score (ignoring vasopressor doses,
            # which don't change between the two MAP sources)
            if pd.isna(map_val):
                return np.nan
            return 1 if map_val < 70 else 0

        comparison['cardio_from_real'] = comparison['map_real'].apply(map_to_cardio_partial)
        comparison['cardio_from_derived'] = comparison['map_derived'].apply(map_to_cardio_partial)
        comparison['cardio_score_changed'] = comparison['cardio_from_real'] != comparison['cardio_from_derived']

        n_changed = comparison['cardio_score_changed'].sum()
        pct_changed = 100 * n_changed / len(comparison)

        print(f"Patients where the MAP<70 cardiovascular sub-score threshold FLIPS between "
              f"real vs derived MAP: {n_changed} / {len(comparison)} ({pct_changed:.2f}%)")
        print()

        if pct_changed < 2:
            print("VERDICT: The SBP/DBP-derived MAP approximation is essentially equivalent to")
            print("real charted MAP for cardiovascular SOFA scoring purposes in this cohort —")
            print("negligible practical impact. Safe to keep the approximation as-is.")
        elif pct_changed < 10:
            print("VERDICT: Small but non-trivial disagreement — worth documenting as a limitation,")
            print("but likely not large enough to justify reworking the SOFA computation, unless")
            print("cardiovascular SOFA specifically is a key focus of your analysis.")
        else:
            print("VERDICT: Meaningful disagreement — the derived approximation appears to be")
            print("materially changing cardiovascular SOFA classification for a real share of")
            print("patients. Recommend switching to real MAP where available, with the")
            print("SBP/DBP-derived value as a fallback only for patients missing real MAP.")
        print()

        print("NOTHING HAS BEEN REPLACED. sofa_scores.csv is unchanged by this cell —")
        print("this is a comparison/decision-support report only, per your instruction.")

In [ ]:
import pandas as pd

processed_path = "../data/processed/"

vitals_labs_24h = pd.read_csv(processed_path + "vitals_labs_24h.csv")
static_features = pd.read_csv(processed_path + "static_features.csv")
organ_burden_24h = pd.read_csv(processed_path + "organ_burden_inputs_24h.csv")

n_cohort = len(static_features)
print(f"Cohort size: {n_cohort}")
print()

vitals_map = {
    220045: 'HR', 220050: 'SBP', 220179: 'SBP', 220051: 'DBP', 220180: 'DBP',
    220210: 'RR', 223761: 'Temp', 223762: 'Temp', 220277: 'SpO2',
    223900: 'GCS_Verbal', 220739: 'GCS_Eye', 223901: 'GCS_Motor',
    51301: 'WBC', 51300: 'WBC', 50912: 'Creatinine', 52546: 'Creatinine',
}
vitals_labs_24h['variable'] = vitals_labs_24h['itemid'].map(vitals_map)

def coverage_pct(var_name):
    subset = vitals_labs_24h[vitals_labs_24h['variable'] == var_name]
    n = subset['subject_id'].nunique()
    return n, round(100 * n / n_cohort, 2)

# ============================================================
# APACHE-II Acute Physiology Score — 12-variable coverage check
# ============================================================

print("=" * 70)
print("APACHE-II ACUTE PHYSIOLOGY SCORE — 12-VARIABLE COVERAGE CHECK")
print("(checked against CURRENTLY EXTRACTED data only, before computing anything)")
print("=" * 70)
print()

results = []

# 1. Temperature — already extracted
n, pct = coverage_pct('Temp')
results.append(('1. Temperature', 'AVAILABLE', n, pct, 'itemid 223761/223762 (vitals_labs_24h.csv)'))

# 2. MAP — derived approximation (same as SOFA), not directly extracted but computable from SBP/DBP
sbp_n, _ = coverage_pct('SBP')
dbp_n, _ = coverage_pct('DBP')
results.append(('2. MAP', 'APPROXIMATED', min(sbp_n, dbp_n), None,
                 'derived from SBP+2*DBP/3 (same limitation as SOFA cardiovascular)'))

# 3. HR — already extracted
n, pct = coverage_pct('HR')
results.append(('3. HR', 'AVAILABLE', n, pct, 'itemid 220045 (vitals_labs_24h.csv)'))

# 4. RR — already extracted
n, pct = coverage_pct('RR')
results.append(('4. RR', 'AVAILABLE', n, pct, 'itemid 220210 (vitals_labs_24h.csv)'))

# 5. Oxygenation (A-aDO2 or PaO2) — NOT extracted, using same SpO2/FiO2 proxy as SOFA
spo2_n, _ = coverage_pct('SpO2')
fio2_n = organ_burden_24h[organ_burden_24h['itemid'] == 223835]['subject_id'].nunique()
results.append(('5. Oxygenation (A-aDO2/PaO2)', 'APPROXIMATED', min(spo2_n, fio2_n), None,
                 'no PaO2/ABG extracted — using SpO2/FiO2 proxy, same approximation as SOFA respiratory'))

# 6. Arterial pH — NOT extracted at all, no substitute
results.append(('6. Arterial pH', 'MISSING — NO SUBSTITUTE', 0, 0.0,
                 'not extracted, requires ABG lab itemid — no valid proxy exists, must flag as missing'))

# 7. Sodium — NOT extracted
results.append(('7. Sodium', 'MISSING — needs extraction', 0, 0.0,
                 'not yet pulled from labevents — candidate d_labitems search needed'))

# 8. Potassium — NOT extracted
results.append(('8. Potassium', 'MISSING — needs extraction', 0, 0.0,
                 'not yet pulled from labevents — candidate d_labitems search needed'))

# 9. Creatinine — already extracted
n, pct = coverage_pct('Creatinine')
results.append(('9. Creatinine', 'AVAILABLE', n, pct, 'itemid 50912/52546 (vitals_labs_24h.csv)'))

# 10. Hematocrit — NOT extracted
results.append(('10. Hematocrit', 'MISSING — needs extraction', 0, 0.0,
                 'not yet pulled from labevents — candidate d_labitems search needed'))

# 11. WBC — already extracted
n, pct = coverage_pct('WBC')
results.append(('11. WBC', 'AVAILABLE', n, pct, 'itemid 51301/51300 (vitals_labs_24h.csv)'))

# 12. GCS — already extracted, reusing SOFA CNS calculation
gcs_components = vitals_labs_24h[vitals_labs_24h['variable'].isin(['GCS_Eye', 'GCS_Verbal', 'GCS_Motor'])]
gcs_pivot = gcs_components.pivot_table(index=['subject_id', 'charttime'], columns='variable', values='valuenum', aggfunc='first')
gcs_complete_patients = gcs_pivot.dropna(how='any').reset_index()['subject_id'].nunique() if len(gcs_pivot) > 0 else 0
results.append(('12. GCS (total)', 'AVAILABLE', gcs_complete_patients, round(100*gcs_complete_patients/n_cohort, 2),
                 'itemid 220739+223900+223901, same-charttime match, reuses SOFA CNS logic'))

summary_df = pd.DataFrame(results, columns=['Variable', 'Status', 'N_patients', 'Coverage_%', 'Source/Note'])
print(summary_df.to_string(index=False))
print()

n_available = (summary_df['Status'] == 'AVAILABLE').sum()
n_approx = (summary_df['Status'] == 'APPROXIMATED').sum()
n_missing = summary_df['Status'].str.startswith('MISSING').sum()

print("=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"Directly available (already extracted): {n_available} / 12")
print(f"Approximated (same method as SOFA, documented limitation): {n_approx} / 12")
print(f"Missing entirely (need new extraction, or truly unavailable): {n_missing} / 12")
print()
print("Missing/gap variables requiring a decision:")
for _, row in summary_df[summary_df['Status'].str.startswith('MISSING')].iterrows():
    print(f"  - {row['Variable']}: {row['Source/Note']}")
print()

print("=" * 70)
print("DECISION POINT — not proceeding to compute APACHE-II yet")
print("=" * 70)
print("Sodium, Potassium, and Hematocrit are all standard labevents tests that were simply")
print("not in scope of the earlier extraction (which targeted WBC/Lactate/Creatinine/")
print("Platelets/Bilirubin for SOFA + basic features) — these would need a quick d_labitems")
print("search + chunked labevents pull, same pattern as before. Realistic to add.")
print()
print("Arterial pH is different — it requires an arterial blood gas panel, which is a")
print("distinct, less-universally-drawn lab in ICU patients (not every patient gets an ABG),")
print("and there is no valid non-invasive substitute for pH specifically (unlike oxygenation,")
print("where SpO2/FiO2 is at least a published proxy for PaO2/FiO2). Even with lab extraction,")
print("this will likely have real, clinically-meaningful missingness — not just an oversight.")
print()
print("Options for you to choose between:")
print("  A) Do the quick supplemental extraction for Sodium/Potassium/Hematocrit now (search")
print("     d_labitems, chunk-read labevents, print coverage), and ALSO attempt to pull pH")
print("     to see real coverage before deciding whether to include it as a partial component")
print("     (same n_components_available pattern used for SOFA).")
print("  B) Proceed with a documented partial-component APACHE-II now (9 of 12 physiology")
print("     variables), same style as SOFA's partial-component approach, and treat")
print("     Sodium/Potassium/Hematocrit/pH as structurally missing rather than extracting them.")

In [ ]:
import pandas as pd

raw_path = "../data/raw/"
processed_path = "../data/processed/"

# ============================================================
# STEP 1: Search d_labitems for the 4 candidates — print for review, don't extract yet
# ============================================================

d_labitems = pd.read_csv(raw_path + "d_labitems.csv.gz", compression="gzip")

print(f"d_labitems shape: {d_labitems.shape}")
print(f"d_labitems columns: {d_labitems.columns.tolist()}")
print()

def search_labitem(pattern, name, fluid_filter=None):
    matches = d_labitems[d_labitems['label'].str.contains(pattern, case=False, na=False, regex=True)].copy()
    print(f"--- {name} — ALL matches (before any fluid filter) — {len(matches)} rows ---")
    cols = [c for c in ['itemid', 'label', 'fluid', 'category'] if c in matches.columns]
    print(matches[cols].to_string(index=False))
    print()
    if fluid_filter and 'fluid' in matches.columns:
        filtered = matches[matches['fluid'].str.contains(fluid_filter, case=False, na=False)]
        print(f"--- {name} — filtered to fluid contains '{fluid_filter}' — {len(filtered)} rows ---")
        print(filtered[cols].to_string(index=False))
        print()
        return filtered
    return matches

print("=" * 70)
print("SODIUM candidates")
print("=" * 70)
sodium_matches = search_labitem('sodium', 'Sodium', fluid_filter='Blood')

print("=" * 70)
print("POTASSIUM candidates")
print("=" * 70)
potassium_matches = search_labitem('potassium', 'Potassium', fluid_filter='Blood')

print("=" * 70)
print("HEMATOCRIT candidates")
print("=" * 70)
hematocrit_matches = search_labitem('hematocrit', 'Hematocrit', fluid_filter='Blood')

print("=" * 70)
print("ARTERIAL pH candidates — REVIEW CAREFULLY, 'pH' substring is noisy")
print("=" * 70)
ph_all_matches = d_labitems[d_labitems['label'].str.contains('pH', case=False, na=False)].copy()
cols = [c for c in ['itemid', 'label', 'fluid', 'category'] if c in ph_all_matches.columns]
print(f"ALL 'pH' matches, any fluid ({len(ph_all_matches)} rows) — inspect fluid/category before filtering:")
print(ph_all_matches[cols].to_string(index=False))
print()

if 'fluid' in ph_all_matches.columns:
    ph_blood = ph_all_matches[ph_all_matches['fluid'].str.contains('Blood', case=False, na=False)]
    print(f"Filtered to fluid contains 'Blood' ({len(ph_blood)} rows):")
    print(ph_blood[cols].to_string(index=False))
    print()

if 'category' in ph_all_matches.columns:
    ph_bloodgas = ph_all_matches[ph_all_matches['category'].str.contains('Blood Gas', case=False, na=False)]
    print(f"Filtered to category contains 'Blood Gas' ({len(ph_bloodgas)} rows) — likely the real arterial pH candidates:")
    print(ph_bloodgas[cols].to_string(index=False))
    print()

print(">>> STOP: review the pH tables above manually. If separate itemids exist for")
print(">>> arterial vs venous vs urine/other-fluid pH, only the arterial/blood-gas one(s)")
print(">>> should be used — I'm printing all views so you can pick the correct itemid(s)")
print(">>> rather than me guessing which is truly arterial.")
print()

# ============================================================
# For extraction, use the fluid='Blood' filtered lists for Na/K/Hct,
# and the 'Blood Gas' category filter for pH as the best default guess —
# but this is presented for confirmation, not silently assumed correct.
# ============================================================

sodium_itemids = set(sodium_matches['itemid']) if len(sodium_matches) > 0 else set()
potassium_itemids = set(potassium_matches['itemid']) if len(potassium_matches) > 0 else set()
hematocrit_itemids = set(hematocrit_matches['itemid']) if len(hematocrit_matches) > 0 else set()
ph_itemids = set(ph_bloodgas['itemid']) if 'category' in ph_all_matches.columns and len(ph_bloodgas) > 0 else set()

print(f"Sodium itemids to extract: {sorted(sodium_itemids)}")
print(f"Potassium itemids to extract: {sorted(potassium_itemids)}")
print(f"Hematocrit itemids to extract: {sorted(hematocrit_itemids)}")
print(f"Arterial pH itemids to extract: {sorted(ph_itemids)}")
print()

all_new_itemids = sodium_itemids | potassium_itemids | hematocrit_itemids | ph_itemids

if len(all_new_itemids) == 0:
    raise ValueError("No itemids identified for extraction — review the printed tables above "
                      "and adjust the filters before proceeding.")

itemid_to_varname = {}
for iid in sodium_itemids: itemid_to_varname[iid] = 'Sodium'
for iid in potassium_itemids: itemid_to_varname[iid] = 'Potassium'
for iid in hematocrit_itemids: itemid_to_varname[iid] = 'Hematocrit'
for iid in ph_itemids: itemid_to_varname[iid] = 'pH'

# ============================================================
# STEP 2: Chunk-read labevents, cohort + confirmed itemids, first 24h
# ============================================================

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])
cohort_subject_ids = set(cohort_step1['subject_id'])
intime_lookup = cohort_step1.set_index('subject_id')['intime']
n_cohort = len(cohort_step1)

chunk_size = 1_000_000
new_lab_rows = []

for chunk in pd.read_csv(
    raw_path + "labevents.csv.gz", compression="gzip",
    usecols=['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum'],
    chunksize=chunk_size
):
    filtered = chunk[chunk['itemid'].isin(all_new_itemids) & chunk['subject_id'].isin(cohort_subject_ids)]
    if len(filtered) > 0:
        new_lab_rows.append(filtered)

new_lab_events = pd.concat(new_lab_rows, ignore_index=True) if new_lab_rows else pd.DataFrame(
    columns=['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum'])
new_lab_events['charttime'] = pd.to_datetime(new_lab_events['charttime'])

print(f"Labevents rows pulled for the 4 new variables (cohort + itemid filtered): {len(new_lab_events)}")

new_lab_events = new_lab_events.merge(intime_lookup.rename('intime'), left_on='subject_id', right_index=True, how='left')
new_lab_events['hours_since_intime'] = (new_lab_events['charttime'] - new_lab_events['intime']).dt.total_seconds() / 3600
new_lab_events_24h = new_lab_events[(new_lab_events['hours_since_intime'] >= 0) & (new_lab_events['hours_since_intime'] <= 24)].copy()

print(f"Rows within first 24h: {len(new_lab_events_24h)}")
print()

new_lab_events_24h['variable'] = new_lab_events_24h['itemid'].map(itemid_to_varname)

# ============================================================
# STEP 3: Coverage % per new variable — report what's real, no assumptions
# ============================================================

print("=" * 70)
print("COVERAGE % — 4 NEW VARIABLES (first 24h, cohort of {})".format(n_cohort))
print("=" * 70)

coverage_rows = []
for var in ['Sodium', 'Potassium', 'Hematocrit', 'pH']:
    subset = new_lab_events_24h[new_lab_events_24h['variable'] == var]
    n_patients = subset['subject_id'].nunique()
    pct = round(100 * n_patients / n_cohort, 2)
    coverage_rows.append({'variable': var, 'n_patients': n_patients, 'coverage_pct': pct,
                           'n_rows': len(subset)})

coverage_df = pd.DataFrame(coverage_rows)
print(coverage_df.to_string(index=False))
print()

print("NOTE: no coverage number was assumed in advance — the table above is the actual")
print("result. Compare Sodium/Potassium/Hematocrit against the ~97-98% range seen for")
print("WBC/Creatinine/etc. earlier, and look specifically at whether pH lands meaningfully")
print("lower (expected, since ABGs are drawn selectively) before deciding its final")
print("treatment in APACHE-II (full component vs. flagged/partial).")
print()

# Basic plausibility check on values before merging (won't fix anything, just flags)
plausibility_ranges = {
    'Sodium': (100, 180),      # mEq/L
    'Potassium': (1.5, 9.0),   # mEq/L
    'Hematocrit': (10, 65),    # %
    'pH': (6.8, 7.8),          # arterial pH
}
print("Quick plausibility scan (not filtered/dropped — just reported):")
for var, (lo, hi) in plausibility_ranges.items():
    subset = new_lab_events_24h[new_lab_events_24h['variable'] == var]
    if len(subset) == 0:
        continue
    v_min, v_max = subset['valuenum'].min(), subset['valuenum'].max()
    n_out = ((subset['valuenum'] < lo) | (subset['valuenum'] > hi)).sum()
    print(f"  {var}: observed min={v_min}, max={v_max} (plausible {lo}-{hi}), "
          f"{n_out} rows outside this range (NOT removed — flagged only)")
print()

# ============================================================
# STEP 4: Merge into vitals_labs_24h.csv (long format, same schema, no new file)
# ============================================================

vitals_labs_24h = pd.read_csv(processed_path + "vitals_labs_24h.csv")
vitals_labs_24h['charttime'] = pd.to_datetime(vitals_labs_24h['charttime'])

n_before = len(vitals_labs_24h)

new_rows_to_add = new_lab_events_24h[['subject_id', 'charttime', 'itemid', 'valuenum']].dropna(subset=['valuenum'])

vitals_labs_24h_updated = pd.concat([vitals_labs_24h, new_rows_to_add], ignore_index=True)

vitals_labs_24h_updated.to_csv(processed_path + "vitals_labs_24h.csv", index=False)

print(f"vitals_labs_24h.csv rows before: {n_before}")
print(f"vitals_labs_24h.csv rows after: {len(vitals_labs_24h_updated)} "
      f"(added {len(new_rows_to_add)})")
print()

print("=" * 70)
print("STOPPING HERE per your instruction — no APACHE-II computation in this cell.")
print("Review the pH coverage % above and confirm the correct pH itemid(s) were used")
print("before we decide: full component vs. partial/flagged component treatment.")
print("=" * 70)

In [ ]:
import pandas as pd

processed_path = "../data/processed/"

vitals_labs_24h = pd.read_csv(processed_path + "vitals_labs_24h.csv")
vitals_labs_24h['charttime'] = pd.to_datetime(vitals_labs_24h['charttime'])

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
n_cohort = len(cohort_step1)

print(f"vitals_labs_24h.csv loaded: {vitals_labs_24h.shape}")
print()

# ============================================================
# 1. Remove the 3 incorrect pH itemids, keep only 50820
# ============================================================

INCORRECT_PH_ITEMIDS = {50831, 52041, 52045}
CONFIRMED_PH_ITEMID = 50820

before_ph_rows = vitals_labs_24h[vitals_labs_24h['itemid'].isin(INCORRECT_PH_ITEMIDS | {CONFIRMED_PH_ITEMID})]
print(f"Total pH-labeled rows (all 4 original itemids) before correction: {len(before_ph_rows)}")

for iid in sorted(INCORRECT_PH_ITEMIDS | {CONFIRMED_PH_ITEMID}):
    n = (vitals_labs_24h['itemid'] == iid).sum()
    label_note = {
        50820: "CONFIRMED arterial/blood gas pH — KEEP",
        50831: "Other Body Fluid — EXCLUDE",
        52041: "fluid='Fluid' (ambiguous/unconfirmed) — EXCLUDE",
        52045: "pH, Urine — EXCLUDE (wrong body fluid)"
    }[iid]
    print(f"  itemid {iid}: {n} rows — {label_note}")
print()

incorrect_rows = vitals_labs_24h[vitals_labs_24h['itemid'].isin(INCORRECT_PH_ITEMIDS)]
n_removed = len(incorrect_rows)
n_patients_affected = incorrect_rows['subject_id'].nunique()

print(f"Removing {n_removed} rows (itemids 50831, 52041, 52045), "
      f"affecting {n_patients_affected} patients.")
print()

vitals_labs_24h_clean = vitals_labs_24h[~vitals_labs_24h['itemid'].isin(INCORRECT_PH_ITEMIDS)].copy()

print(f"Rows before removal: {len(vitals_labs_24h)}")
print(f"Rows after removal: {len(vitals_labs_24h_clean)}")
print()

# ============================================================
# 2. Recompute pH coverage using only itemid 50820
# ============================================================

ph_rows_final = vitals_labs_24h_clean[vitals_labs_24h_clean['itemid'] == CONFIRMED_PH_ITEMID]

n_patients_with_ph = ph_rows_final['subject_id'].nunique()
pct_coverage = round(100 * n_patients_with_ph / n_cohort, 2)

print("=" * 65)
print("CORRECTED pH COVERAGE (itemid 50820 only)")
print("=" * 65)
print(f"Patients with >=1 confirmed arterial pH reading in first 24h: "
      f"{n_patients_with_ph} / {n_cohort} ({pct_coverage}%)")
print(f"(Previous, incorrect 4-itemid coverage was 71.30% — this should now be lower)")
print()

# ============================================================
# 3. Re-run plausibility check on corrected pH data
# ============================================================

PH_LOW, PH_HIGH = 6.8, 7.8
PH_TYPICAL_LOW, PH_TYPICAL_HIGH = 6.9, 7.6

print("=" * 65)
print("PLAUSIBILITY CHECK — corrected pH (itemid 50820 only)")
print("=" * 65)

if len(ph_rows_final) == 0:
    print("No pH rows remain after correction — nothing to check.")
else:
    v_min, v_max = ph_rows_final['valuenum'].min(), ph_rows_final['valuenum'].max()
    print(f"Observed range: min={v_min}, max={v_max}")
    print(f"Full physiologic range checked: {PH_LOW}-{PH_HIGH}")
    print(f"Typical living-patient range: {PH_TYPICAL_LOW}-{PH_TYPICAL_HIGH}")
    print()

    hard_outliers = ph_rows_final[(ph_rows_final['valuenum'] < PH_LOW) | (ph_rows_final['valuenum'] > PH_HIGH)]
    soft_outliers = ph_rows_final[
        ((ph_rows_final['valuenum'] >= PH_LOW) & (ph_rows_final['valuenum'] < PH_TYPICAL_LOW)) |
        ((ph_rows_final['valuenum'] > PH_TYPICAL_HIGH) & (ph_rows_final['valuenum'] <= PH_HIGH))
    ]

    print(f"Rows outside the FULL physiologic range ({PH_LOW}-{PH_HIGH}) — genuinely implausible: {len(hard_outliers)}")
    if len(hard_outliers) > 0:
        print(hard_outliers[['subject_id', 'charttime', 'valuenum']].sort_values('valuenum').to_string(index=False))
    print()

    print(f"Rows in the 'unusual but not impossible' band ({PH_LOW}-{PH_TYPICAL_LOW} or "
          f"{PH_TYPICAL_HIGH}-{PH_HIGH}) — severe acidosis/alkalosis, clinically possible "
          f"in a critically ill patient, NOT auto-flagged as errors: {len(soft_outliers)}")
    if len(soft_outliers) > 0:
        print(soft_outliers[['subject_id', 'charttime', 'valuenum']].sort_values('valuenum').to_string(index=False))
    print()

    if len(hard_outliers) == 0:
        print("No genuinely implausible pH values found in the corrected data.")
    else:
        print("NOTE: hard-outlier rows above are NOT auto-dropped — following the same pattern")
        print("as Temp/SpO2 earlier, review these specific rows before deciding whether to remove")
        print("them (data entry error) or keep them (rare but real, e.g. a terminal/coding event).")
print()

# ============================================================
# 4. Re-save vitals_labs_24h.csv with corrected pH data
# ============================================================

vitals_labs_24h_clean.to_csv(processed_path + "vitals_labs_24h.csv", index=False)

print(f"vitals_labs_24h.csv re-saved. Final shape: {vitals_labs_24h_clean.shape}")

In [ ]:
import pandas as pd

processed_path = "../data/processed/"

vitals_labs_24h = pd.read_csv(processed_path + "vitals_labs_24h.csv")
vitals_labs_24h['charttime'] = pd.to_datetime(vitals_labs_24h['charttime'])

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])

TARGET_SUBJECTS = [REDACTED_ID_18, REDACTED_ID_19]
PH_ITEMID = 50820

for subj_id in TARGET_SUBJECTS:
    print("=" * 70)
    print(f"subject_id = {subj_id}")
    print("=" * 70)

    patient_row = cohort_step1[cohort_step1['subject_id'] == subj_id]
    if len(patient_row) == 0:
        print("NOT FOUND in cohort_step1.csv — check the ID.")
        continue

    intime = patient_row['intime'].iloc[0]
    mortality = patient_row['mortality_28d'].iloc[0]
    immortal_flag = patient_row['immortal_time_flag'].iloc[0]

    print(f"ICU intime: {intime}")
    print(f"mortality_28d: {mortality}")
    print(f"immortal_time_flag: {immortal_flag}")
    print()

    ph_series = vitals_labs_24h[
        (vitals_labs_24h['subject_id'] == subj_id) &
        (vitals_labs_24h['itemid'] == PH_ITEMID)
    ].copy()

    ph_series['hours_since_intime'] = (ph_series['charttime'] - intime).dt.total_seconds() / 3600
    ph_series_sorted = ph_series.sort_values('charttime')

    print(f"Full pH (itemid 50820) series in first 24h — {len(ph_series_sorted)} reading(s):")
    print(ph_series_sorted[['charttime', 'hours_since_intime', 'valuenum']].to_string(index=False))
    print()

    if len(ph_series_sorted) == 1:
        print("SINGLE reading only — no trend to assess. This is the isolated-reading scenario:")
        print("cannot distinguish 'genuine terminal event' from 'data entry error' using pH trend")
        print("alone. mortality_28d/immortal_time_flag above are the main additional evidence")
        print("available (a same-day or next-day death would support a genuine terminal reading;")
        print("survival to 28 days would strongly suggest this was an error, since a true pH this")
        print("low is not typically survivable even briefly without extreme intervention).")
    else:
        vals = ph_series_sorted['valuenum'].tolist()
        is_declining_into_flagged = all(vals[i] >= vals[i+1] for i in range(len(vals)-1))
        print(f"Multiple readings present. Monotonically declining toward the flagged value: "
              f"{is_declining_into_flagged}")
        print("(True if pH trends downward over time toward the low reading — consistent with a")
        print("genuine deteriorating/terminal course. False if the flagged value is an isolated")
        print("dip/spike surrounded by more normal readings — more consistent with an artifact.)")
    print()

In [ ]:
import pandas as pd
import numpy as np

processed_path = "../data/processed/"

# ============================================================
# STEP 0: PRINT THE FULL MAPPING/LOGIC BEFORE COMPUTING ANYTHING
# ============================================================

print("=" * 70)
print("APACHE-II COMPONENT MAPPING — review before trusting the score")
print("=" * 70)
print("""
ACUTE PHYSIOLOGY SCORE (12 variables, worst value in first 24h):
  1. Temperature      - itemid 223761/223762 (already extracted, corrected)
  2. MAP               - SBP/DBP-derived approximation (SAME limitation as SOFA)
  3. HR                - itemid 220045
  4. RR                - itemid 220210
  5. Oxygenation       - We do NOT have PaCO2, so the standard "AaDO2 when
                          FiO2>=0.5" branch of APACHE-II cannot be computed as
                          specified (AaDO2 requires PaCO2 from an ABG). Instead,
                          for ALL patients regardless of FiO2, PaO2 is ESTIMATED
                          from SpO2/FiO2 using the Rice et al. (2007) linear
                          regression: PF_est = (SF_ratio - 64) / 0.84, then
                          PaO2_est = PF_est * FiO2_fraction. This is a DOUBLE
                          approximation (SpO2->PaO2, plus skipping AaDO2 entirely)
                          and is a real deviation from strict APACHE-II methodology,
                          not just a proxy substitution. Flagging this clearly.
  6. Arterial pH       - itemid 50820 ONLY (corrected, confirmed arterial)
  7. Sodium            - newly extracted itemids
  8. Potassium         - newly extracted itemids
  9. Creatinine        - itemid 50912/52546 (already extracted)
  10. Hematocrit       - newly extracted itemids
  11. WBC              - itemid 51301/51300 (already extracted)
  12. GCS total        - itemid 220739+223900+223901, same-charttime match,
                          REUSES the exact same logic as SOFA's CNS component

AGE POINTS (standard APACHE-II brackets):
  <=44: 0 | 45-54: 2 | 55-64: 3 | 65-74: 5 | >=75: 6

CHRONIC HEALTH POINTS:
  Standard APACHE-II awards 5 points (nonoperative/emergency postop) or 2 points
  (elective postop) if the patient has severe organ insufficiency or is
  immunocompromised, defined by specific criteria:
    - Hepatic: biopsy-proven cirrhosis, documented portal hypertension, prior GI
      bleed from portal HTN, or prior hepatic failure/encephalopathy/coma
        -> mapped from: static_features['liver_disease'] (Elixhauser)
           APPROXIMATION: Elixhauser liver_disease is broader than the strict
           APACHE-II cirrhosis/portal-HTN definition — will likely OVER-INCLUDE
           some patients with milder liver disease.
    - Cardiovascular: NYHA Class IV heart failure
        -> mapped from: static_features['chf'] (Elixhauser)
           APPROXIMATION: Elixhauser CHF flag does not distinguish NYHA class —
           will likely OVER-INCLUDE patients with milder (non-Class-IV) CHF.
    - Respiratory: severe chronic restrictive/obstructive/vascular disease causing
      severe exercise restriction, documented chronic hypoxia/hypercapnia/
      polycythemia/severe pulmonary HTN, or ventilator dependency
        -> mapped from: static_features['chronic_pulm'] (Elixhauser)
           APPROXIMATION: overinclusive vs. strict "severe" criterion.
    - Renal: chronic dialysis dependence
        -> mapped from: static_features['renal_failure'] (Elixhauser)
           APPROXIMATION: Elixhauser renal_failure captures acute AND chronic
           renal failure broadly, NOT specifically dialysis-dependence — this is
           the WEAKEST mapping of the four and will meaningfully overinclude.
    - Immunocompromised: chemo/radiation/high-dose-chronic-steroids/recent
      leukemia/lymphoma treatment/AIDS
        -> mapped from: static_features['lymphoma'] OR static_features['aids_hiv']
           OR static_features['metastatic_cancer'] (Elixhauser)
           APPROXIMATION: no direct data on chemo/steroid/radiation therapy in
           this pipeline — using cancer-diagnosis flags as a proxy, which will
           MISS patients immunocompromised for non-cancer reasons (e.g., organ
           transplant, non-cancer immunosuppressive therapy) and will OVERLAP
           heavily with cancer_type, same redundancy concern flagged for SOFA/
           Elixhauser earlier.
    - Operative status (elective postop vs. nonoperative/emergency postop) is
      NOT distinguished in this cohort/pipeline -> defaulting to 5 points
      (nonoperative/emergency) whenever ANY criterion above is met, since this
      is an ICU medical cohort, not a surgical one. If any patients here ARE
      elective postoperative, this will OVER-SCORE them by 3 points.
""")
print()

# ============================================================
# Load source files
# ============================================================

vitals_labs_24h = pd.read_csv(processed_path + "vitals_labs_24h.csv")
vitals_labs_24h['charttime'] = pd.to_datetime(vitals_labs_24h['charttime'])

static_features = pd.read_csv(processed_path + "static_features.csv")
organ_burden_24h = pd.read_csv(processed_path + "organ_burden_inputs_24h.csv")
organ_burden_24h['charttime'] = pd.to_datetime(organ_burden_24h['charttime'])

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
n_cohort = len(cohort_step1)
all_subjects = cohort_step1['subject_id']

print(f"Cohort: {n_cohort} patients")
print()

vitals_map = {
    220045: 'HR', 220050: 'SBP', 220179: 'SBP', 220051: 'DBP', 220180: 'DBP',
    220210: 'RR', 223761: 'Temp', 223762: 'Temp', 220277: 'SpO2',
    223900: 'GCS_Verbal', 220739: 'GCS_Eye', 223901: 'GCS_Motor',
    51301: 'WBC', 51300: 'WBC', 50912: 'Creatinine', 52546: 'Creatinine',
    50820: 'pH',
}
# Sodium/Potassium/Hematocrit itemids are whatever the supplemental extraction found —
# they're already tagged with their variable name in the underlying data via the
# earlier merge step, so we detect them by re-deriving from d_labitems here for safety.
vitals_labs_24h['variable'] = vitals_labs_24h['itemid'].map(vitals_map)

# Anything NOT in vitals_map but present is assumed to be Sodium/Potassium/Hematocrit
# from the supplemental pull — reconstruct via d_labitems label lookup
raw_path = "../data/raw/"
d_labitems = pd.read_csv(raw_path + "d_labitems.csv.gz", compression="gzip")
label_lookup = d_labitems.set_index('itemid')['label'].to_dict()

unmapped_mask = vitals_labs_24h['variable'].isna()
def infer_supplemental_var(itemid):
    label = str(label_lookup.get(itemid, '')).lower()
    if 'sodium' in label: return 'Sodium'
    if 'potassium' in label: return 'Potassium'
    if 'hematocrit' in label: return 'Hematocrit'
    return None

vitals_labs_24h.loc[unmapped_mask, 'variable'] = vitals_labs_24h.loc[unmapped_mask, 'itemid'].apply(infer_supplemental_var)

still_unmapped = vitals_labs_24h[vitals_labs_24h['variable'].isna()]['itemid'].unique()
if len(still_unmapped) > 0:
    print(f"WARNING: itemids still unmapped after inference: {still_unmapped} — these rows excluded from APACHE-II.")
print()

# ============================================================
# APACHE-II point-scoring functions per variable (standard Knaus 1985 table)
# ============================================================

def score_temp(t):
    if pd.isna(t): return np.nan
    if t >= 41: return 4
    if t >= 39: return 3
    if t >= 38.5: return 1
    if t >= 36: return 0
    if t >= 34: return 1
    if t >= 32: return 2
    if t >= 30: return 3
    return 4

def score_map(m):
    if pd.isna(m): return np.nan
    if m >= 160: return 4
    if m >= 130: return 3
    if m >= 110: return 2
    if m >= 70: return 0
    if m >= 50: return 2
    return 4

def score_hr(hr):
    if pd.isna(hr): return np.nan
    if hr >= 180: return 4
    if hr >= 140: return 3
    if hr >= 110: return 2
    if hr >= 70: return 0
    if hr >= 55: return 2
    if hr >= 40: return 3
    return 4

def score_rr(rr):
    if pd.isna(rr): return np.nan
    if rr >= 50: return 4
    if rr >= 35: return 3
    if rr >= 25: return 1
    if rr >= 12: return 0
    if rr >= 10: return 1
    if rr >= 6: return 2
    return 4

def score_oxygenation_from_pao2_est(pao2_est):
    # Simplified: using PaO2-based table for ALL patients (see documented deviation above)
    if pd.isna(pao2_est): return np.nan
    if pao2_est > 70: return 0
    if pao2_est >= 61: return 1
    if pao2_est >= 55: return 3
    return 4

def score_ph(ph):
    if pd.isna(ph): return np.nan
    if ph >= 7.7: return 4
    if ph >= 7.6: return 3
    if ph >= 7.5: return 1
    if ph >= 7.33: return 0
    if ph >= 7.25: return 2
    if ph >= 7.15: return 3
    return 4

def score_sodium(na):
    if pd.isna(na): return np.nan
    if na >= 180: return 4
    if na >= 160: return 3
    if na >= 155: return 2
    if na >= 150: return 1
    if na >= 130: return 0
    if na >= 120: return 2
    if na >= 111: return 3
    return 4

def score_potassium(k):
    if pd.isna(k): return np.nan
    if k >= 7: return 4
    if k >= 6: return 3
    if k >= 5.5: return 1
    if k >= 3.5: return 0
    if k >= 3: return 1
    if k >= 2.5: return 2
    return 4

def score_creatinine(cr, acute_renal_failure=False):
    # Standard APACHE-II doubles points if acute renal failure present — we don't
    # have a reliable acute-vs-chronic distinction here, so NOT doubling (documented).
    if pd.isna(cr): return np.nan
    if cr >= 3.5: return 4
    if cr >= 2.0: return 3
    if cr >= 1.5: return 2
    if cr >= 0.6: return 0
    return 2

def score_hematocrit(hct):
    if pd.isna(hct): return np.nan
    if hct >= 60: return 4
    if hct >= 50: return 2
    if hct >= 46: return 1
    if hct >= 30: return 0
    if hct >= 20: return 2
    return 4

def score_wbc(wbc):
    if pd.isna(wbc): return np.nan
    if wbc >= 40: return 4
    if wbc >= 20: return 2
    if wbc >= 15: return 1
    if wbc >= 3: return 0
    if wbc >= 1: return 2
    return 4

def score_gcs(gcs_total):
    if pd.isna(gcs_total): return np.nan
    return 15 - gcs_total  # APACHE-II GCS points = 15 minus actual GCS

# ============================================================
# Compute worst value per patient per variable (worst = furthest from normal,
# so for Temp/HR/RR/MAP/Sodium/Potassium/Hematocrit/WBC we need to check BOTH
# extremes — take whichever raw value yields the HIGHER point score, not just min/max)
# ============================================================

def worst_score_bidirectional(data, score_func):
    """For variables where both high and low are bad, compute the score for
    every reading and take the MAX score (worst) per patient — this is the
    correct APACHE-II approach, not simply taking min or max value."""
    scored = data.copy()
    scored['point_score'] = scored['valuenum'].apply(score_func)
    return scored.groupby('subject_id')['point_score'].max()

# 1. Temperature
temp_data = vitals_labs_24h[vitals_labs_24h['variable'] == 'Temp']
temp_scores = worst_score_bidirectional(temp_data, score_temp)

# 2. MAP (derived)
sbp_data = vitals_labs_24h[vitals_labs_24h['variable'] == 'SBP'][['subject_id', 'charttime', 'valuenum']].rename(columns={'valuenum': 'sbp'})
dbp_data = vitals_labs_24h[vitals_labs_24h['variable'] == 'DBP'][['subject_id', 'charttime', 'valuenum']].rename(columns={'valuenum': 'dbp'})
bp_merged = sbp_data.merge(dbp_data, on=['subject_id', 'charttime'], how='inner')
bp_merged['map'] = (bp_merged['sbp'] + 2 * bp_merged['dbp']) / 3
bp_merged['point_score'] = bp_merged['map'].apply(score_map)
map_scores = bp_merged.groupby('subject_id')['point_score'].max()

# 3. HR
hr_data = vitals_labs_24h[vitals_labs_24h['variable'] == 'HR']
hr_scores = worst_score_bidirectional(hr_data, score_hr)

# 4. RR
rr_data = vitals_labs_24h[vitals_labs_24h['variable'] == 'RR']
rr_scores = worst_score_bidirectional(rr_data, score_rr)

# 5. Oxygenation — estimate PaO2 from SpO2/FiO2 via Rice regression, then score
spo2_data = vitals_labs_24h[vitals_labs_24h['variable'] == 'SpO2'][['subject_id', 'charttime', 'valuenum']].rename(columns={'valuenum': 'spo2'})
fio2_data = organ_burden_24h[organ_burden_24h['itemid'] == 223835][['subject_id', 'charttime', 'valuenum']].rename(columns={'valuenum': 'fio2'})

ox_rows = []
for subj in spo2_data['subject_id'].unique():
    subj_spo2 = spo2_data[spo2_data['subject_id'] == subj]
    subj_fio2 = fio2_data[fio2_data['subject_id'] == subj]
    if len(subj_fio2) == 0:
        continue
    for _, s_row in subj_spo2.iterrows():
        time_diffs = (subj_fio2['charttime'] - s_row['charttime']).abs()
        if time_diffs.min() <= pd.Timedelta(hours=1):
            nearest_fio2 = subj_fio2.loc[time_diffs.idxmin(), 'fio2']
            fio2_frac = nearest_fio2 / 100 if nearest_fio2 > 1 else nearest_fio2
            if fio2_frac > 0:
                sf_ratio = s_row['spo2'] / fio2_frac
                pf_est = (sf_ratio - 64) / 0.84
                pao2_est = pf_est * fio2_frac
                ox_rows.append({'subject_id': subj, 'pao2_est': pao2_est})

ox_df = pd.DataFrame(ox_rows)
if len(ox_df) > 0:
    ox_df['point_score'] = ox_df['pao2_est'].apply(score_oxygenation_from_pao2_est)
    oxygenation_scores = ox_df.groupby('subject_id')['point_score'].max()
else:
    oxygenation_scores = pd.Series(dtype=float)

# 6. pH
ph_data = vitals_labs_24h[vitals_labs_24h['variable'] == 'pH']
ph_scores = worst_score_bidirectional(ph_data, score_ph)

# 7. Sodium
na_data = vitals_labs_24h[vitals_labs_24h['variable'] == 'Sodium']
na_scores = worst_score_bidirectional(na_data, score_sodium)

# 8. Potassium
k_data = vitals_labs_24h[vitals_labs_24h['variable'] == 'Potassium']
k_scores = worst_score_bidirectional(k_data, score_potassium)

# 9. Creatinine
creat_data = vitals_labs_24h[vitals_labs_24h['variable'] == 'Creatinine']
creat_scores = worst_score_bidirectional(creat_data, score_creatinine)

# 10. Hematocrit
hct_data = vitals_labs_24h[vitals_labs_24h['variable'] == 'Hematocrit']
hct_scores = worst_score_bidirectional(hct_data, score_hematocrit)

# 11. WBC
wbc_data = vitals_labs_24h[vitals_labs_24h['variable'] == 'WBC']
wbc_scores = worst_score_bidirectional(wbc_data, score_wbc)

# 12. GCS total — reuse SOFA's same-charttime match logic
gcs_components = vitals_labs_24h[vitals_labs_24h['variable'].isin(['GCS_Eye', 'GCS_Verbal', 'GCS_Motor'])]
gcs_pivot = gcs_components.pivot_table(index=['subject_id', 'charttime'], columns='variable', values='valuenum', aggfunc='first').reset_index()
for col in ['GCS_Eye', 'GCS_Verbal', 'GCS_Motor']:
    if col not in gcs_pivot.columns:
        gcs_pivot[col] = np.nan
gcs_pivot['gcs_total'] = gcs_pivot['GCS_Eye'] + gcs_pivot['GCS_Verbal'] + gcs_pivot['GCS_Motor']
gcs_complete = gcs_pivot.dropna(subset=['gcs_total'])
worst_gcs = gcs_complete.groupby('subject_id')['gcs_total'].min()
gcs_scores = worst_gcs.apply(score_gcs)

# ============================================================
# Assemble Acute Physiology Score components
# ============================================================

aps_df = pd.DataFrame({'subject_id': all_subjects}).set_index('subject_id')
component_series = {
    'temp_pts': temp_scores, 'map_pts': map_scores, 'hr_pts': hr_scores, 'rr_pts': rr_scores,
    'oxygenation_pts': oxygenation_scores, 'ph_pts': ph_scores, 'sodium_pts': na_scores,
    'potassium_pts': k_scores, 'creatinine_pts': creat_scores, 'hematocrit_pts': hct_scores,
    'wbc_pts': wbc_scores, 'gcs_pts': gcs_scores
}
for col, series in component_series.items():
    aps_df[col] = series

component_cols = list(component_series.keys())
aps_df['n_components_available'] = aps_df[component_cols].notna().sum(axis=1)
aps_df['aps_total'] = aps_df[component_cols].fillna(0).sum(axis=1)

print(f"Acute Physiology Score component coverage:")
for col in component_cols:
    print(f"  {col}: {aps_df[col].notna().sum()} / {n_cohort} patients")
print()

# ============================================================
# Age points
# ============================================================

age_data = cohort_step1.set_index('subject_id')['age_at_admission']

def score_age(age):
    if age <= 44: return 0
    if age <= 54: return 2
    if age <= 64: return 3
    if age <= 74: return 5
    return 6

age_points = age_data.apply(score_age)

# ============================================================
# Chronic Health points
# ============================================================

static_indexed = static_features.set_index('subject_id')

def score_chronic_health(subj_id):
    if subj_id not in static_indexed.index:
        return 0
    row = static_indexed.loc[subj_id]
    criteria_met = False
    for col in ['liver_disease', 'chf', 'chronic_pulm', 'renal_failure', 'lymphoma', 'aids_hiv', 'metastatic_cancer']:
        if col in row.index and bool(row[col]):
            criteria_met = True
            break
    return 5 if criteria_met else 0

chronic_health_points = pd.Series({s: score_chronic_health(s) for s in all_subjects})

# ============================================================
# FINAL APACHE-II TOTAL
# ============================================================

apache_df = aps_df.reset_index()
apache_df['age_points'] = apache_df['subject_id'].map(age_points)
apache_df['chronic_health_points'] = apache_df['subject_id'].map(chronic_health_points)

apache_df['apache_ii_total'] = (
    apache_df['aps_total'] + apache_df['age_points'].fillna(0) + apache_df['chronic_health_points'].fillna(0)
)

apache_df['apache_reliable'] = apache_df['n_components_available'] >= 9

# ============================================================
# Print distributions
# ============================================================

print("=" * 65)
print("n_components_available (out of 12) — distribution")
print("=" * 65)
print(apache_df['n_components_available'].value_counts().sort_index())
print()

print("=" * 65)
print("apache_reliable (n_components_available >= 9) — value_counts()")
print("=" * 65)
print(apache_df['apache_reliable'].value_counts())
print()

print("=" * 65)
print("apache_ii_total — distribution")
print("=" * 65)
print(apache_df['apache_ii_total'].describe())
print()

reliable = apache_df[apache_df['apache_reliable']]
unreliable = apache_df[~apache_df['apache_reliable']]
print(f"apache_ii_total — reliable group (n={len(reliable)}): mean={reliable['apache_ii_total'].mean():.2f}, "
      f"median={reliable['apache_ii_total'].median():.1f}")
print(f"apache_ii_total — unreliable group (n={len(unreliable)}): mean={unreliable['apache_ii_total'].mean():.2f}, "
      f"median={unreliable['apache_ii_total'].median():.1f}")
print()

# ============================================================
# Save
# ============================================================

apache_df.to_csv(processed_path + "apache_ii_scores.csv", index=False)
print(f"Saved apache_ii_scores.csv. Shape: {apache_df.shape}")
print(f"Columns: {apache_df.columns.tolist()}")

In [ ]:
import pandas as pd
import numpy as np

processed_path = "../data/processed/"

vitals_labs_24h = pd.read_csv(processed_path + "vitals_labs_24h.csv")
vitals_labs_24h['charttime'] = pd.to_datetime(vitals_labs_24h['charttime'])

print(f"vitals_labs_24h.csv loaded: {vitals_labs_24h.shape}")
print()

PH_ITEMID = 50820

rows_to_drop = [
    {'subject_id': REDACTED_ID_18, 'charttime': pd.Timestamp('2173-05-17 17:07:00'), 'valuenum': 6.37},
    {'subject_id': REDACTED_ID_19, 'charttime': pd.Timestamp('2148-08-19 00:16:00'), 'valuenum': 6.68},
]

# ============================================================
# Identify exact matching rows (subject_id + charttime + itemid), verify before dropping
# ============================================================

drop_mask = pd.Series(False, index=vitals_labs_24h.index)
for spec in rows_to_drop:
    match = (
        (vitals_labs_24h['subject_id'] == spec['subject_id']) &
        (vitals_labs_24h['charttime'] == spec['charttime']) &
        (vitals_labs_24h['itemid'] == PH_ITEMID)
    )
    n_matched = match.sum()
    print(f"subject_id={spec['subject_id']}, charttime={spec['charttime']}: "
          f"{n_matched} matching row(s) found in file")
    if n_matched == 1:
        matched_val = vitals_labs_24h.loc[match, 'valuenum'].iloc[0]
        print(f"  Matched row's valuenum: {matched_val} (expected {spec['valuenum']}) — "
              f"{'CONFIRMED MATCH' if abs(matched_val - spec['valuenum']) < 0.001 else 'VALUE MISMATCH — REVIEW'}")
    elif n_matched == 0:
        print(f"  WARNING: no matching row found — may already be removed, or charttime/subject_id mismatch.")
    else:
        print(f"  WARNING: multiple rows matched — investigate before dropping.")
    drop_mask = drop_mask | match
print()

n_dropped = drop_mask.sum()
vitals_labs_24h_clean = vitals_labs_24h[~drop_mask].copy()

print(f"Total rows dropped: {n_dropped} (expected 2)")
print(f"Shape before: {vitals_labs_24h.shape}, shape after: {vitals_labs_24h_clean.shape}")
print()

# ============================================================
# Confirm both rows are gone
# ============================================================

still_present = vitals_labs_24h_clean[
    (vitals_labs_24h_clean['subject_id'].isin([REDACTED_ID_18, REDACTED_ID_19])) &
    (vitals_labs_24h_clean['itemid'] == PH_ITEMID)
]
print("Remaining pH rows for these 2 patients (should NOT include the dropped values):")
print(still_present[['subject_id', 'charttime', 'valuenum']].to_string(index=False))
print()

vitals_labs_24h_clean.to_csv(processed_path + "vitals_labs_24h.csv", index=False)
print("vitals_labs_24h.csv re-saved.")
print()

# ============================================================
# Check: was either patient's WORST (min) pH driven by the dropped row?
# ============================================================

apache_df = pd.read_csv(processed_path + "apache_ii_scores.csv")

print("=" * 65)
print("CHECKING: did the dropped value = each patient's worst (minimum) pH?")
print("=" * 65)

def score_ph(ph):
    if pd.isna(ph): return np.nan
    if ph >= 7.7: return 4
    if ph >= 7.6: return 3
    if ph >= 7.5: return 1
    if ph >= 7.33: return 0
    if ph >= 7.25: return 2
    if ph >= 7.15: return 3
    return 4

needs_recompute = []

for spec in rows_to_drop:
    subj_id = spec['subject_id']
    dropped_val = spec['valuenum']

    # Get this patient's remaining pH readings (post-drop) to find their new worst value
    remaining_ph = vitals_labs_24h_clean[
        (vitals_labs_24h_clean['subject_id'] == subj_id) &
        (vitals_labs_24h_clean['itemid'] == PH_ITEMID)
    ]

    old_ph_pts = apache_df.loc[apache_df['subject_id'] == subj_id, 'ph_pts'].iloc[0]
    old_score_from_dropped = score_ph(dropped_val)

    was_the_worst = (old_ph_pts == old_score_from_dropped) and (old_score_from_dropped == 4)
    # Since ph<7.15 always scores 4 (the max possible), and dropped values (6.37, 6.68)
    # are both in that <7.15 bucket, the dropped row drove the max score IF no other
    # reading also scored 4 (i.e., no other reading was also <7.15).

    other_readings_also_score_4 = False
    if len(remaining_ph) > 0:
        other_scores = remaining_ph['valuenum'].apply(score_ph)
        other_readings_also_score_4 = (other_scores == 4).any()

    print(f"\nsubject_id={subj_id}:")
    print(f"  Dropped pH value: {dropped_val} -> APACHE-II points: {old_score_from_dropped}")
    print(f"  Patient's recorded ph_pts in apache_ii_scores.csv: {old_ph_pts}")
    print(f"  Remaining readings after drop: {len(remaining_ph)}")
    if len(remaining_ph) > 0:
        print(f"  Remaining values: {remaining_ph['valuenum'].tolist()}")
        print(f"  Does another remaining reading ALSO score 4 points (pH<7.15)? {other_readings_also_score_4}")

    if old_ph_pts == 4 and not other_readings_also_score_4:
        print(f"  -> RECOMPUTATION NEEDED: the dropped row was driving this patient's worst pH score.")
        needs_recompute.append(subj_id)
    elif len(remaining_ph) == 0:
        print(f"  -> RECOMPUTATION NEEDED: patient now has ZERO pH readings — ph_pts becomes NaN,")
        print(f"     n_components_available drops by 1, and apache_ii_total needs updating.")
        needs_recompute.append(subj_id)
    else:
        print(f"  -> NO recomputation needed: another reading already drove the same worst score,")
        print(f"     so removing this row doesn't change ph_pts or apache_ii_total for this patient.")

print()
print(f"Patients requiring recomputation: {needs_recompute if needs_recompute else 'NONE'}")
print()

# ============================================================
# Recompute ph_pts / n_components_available / apache_ii_total ONLY for affected patients
# ============================================================

if len(needs_recompute) == 0:
    print("No recomputation performed — apache_ii_scores.csv left unchanged, as neither")
    print("dropped row was actually driving that patient's worst pH score.")
else:
    print(f"Recomputing ph_pts, n_components_available, and apache_ii_total for: {needs_recompute}")
    print()

    other_component_cols = ['temp_pts', 'map_pts', 'hr_pts', 'rr_pts', 'oxygenation_pts',
                             'sodium_pts', 'potassium_pts', 'creatinine_pts', 'hematocrit_pts',
                             'wbc_pts', 'gcs_pts']

    for subj_id in needs_recompute:
        remaining_ph = vitals_labs_24h_clean[
            (vitals_labs_24h_clean['subject_id'] == subj_id) &
            (vitals_labs_24h_clean['itemid'] == PH_ITEMID)
        ]

        if len(remaining_ph) == 0:
            new_ph_pts = np.nan
        else:
            new_ph_pts = remaining_ph['valuenum'].apply(score_ph).max()

        row_idx = apache_df[apache_df['subject_id'] == subj_id].index[0]

        old_ph_pts_val = apache_df.loc[row_idx, 'ph_pts']
        apache_df.loc[row_idx, 'ph_pts'] = new_ph_pts

        other_available = apache_df.loc[row_idx, other_component_cols].notna().sum()
        new_n_available = other_available + (1 if not pd.isna(new_ph_pts) else 0)
        apache_df.loc[row_idx, 'n_components_available'] = new_n_available
        apache_df.loc[row_idx, 'apache_reliable'] = new_n_available >= 9

        other_sum = apache_df.loc[row_idx, other_component_cols].fillna(0).sum()
        new_aps_total = other_sum + (new_ph_pts if not pd.isna(new_ph_pts) else 0)
        apache_df.loc[row_idx, 'aps_total'] = new_aps_total

        age_pts = apache_df.loc[row_idx, 'age_points']
        chronic_pts = apache_df.loc[row_idx, 'chronic_health_points']
        apache_df.loc[row_idx, 'apache_ii_total'] = new_aps_total + age_pts + chronic_pts

        print(f"subject_id={subj_id}: ph_pts {old_ph_pts_val} -> {new_ph_pts}, "
              f"n_components_available -> {new_n_available}, "
              f"apache_ii_total -> {apache_df.loc[row_idx, 'apache_ii_total']}")

    apache_df.to_csv(processed_path + "apache_ii_scores.csv", index=False)
    print()
    print("apache_ii_scores.csv re-saved with targeted updates for affected patients only.")

In [ ]:
import pandas as pd

raw_path = "../data/raw/"
processed_path = "../data/processed/"

vitals_labs_24h = pd.read_csv(processed_path + "vitals_labs_24h.csv")
organ_burden_24h = pd.read_csv(processed_path + "organ_burden_inputs_24h.csv")
apache_df = pd.read_csv(processed_path + "apache_ii_scores.csv")

n_cohort = len(pd.read_csv(processed_path + "cohort_step1.csv"))

FLAGGED_ITEMIDS = [223835, 50813, 51265, 50885]
ITEMID_NAMES = {223835: 'FiO2', 50813: 'Lactate', 51265: 'Platelets', 50885: 'Bilirubin'}

print("=" * 70)
print("STEP 1: Where does each flagged itemid actually LIVE?")
print("=" * 70)

for itemid in FLAGGED_ITEMIDS:
    in_vitals_labs = (vitals_labs_24h['itemid'] == itemid).sum()
    in_organ_burden = (organ_burden_24h['itemid'] == itemid).sum() if 'itemid' in organ_burden_24h.columns else 0
    print(f"itemid {itemid} ({ITEMID_NAMES[itemid]}):")
    print(f"  Rows in vitals_labs_24h.csv: {in_vitals_labs}")
    print(f"  Rows in organ_burden_inputs_24h.csv: {in_organ_burden}")
    print()

# ============================================================
# STEP 2: Was FiO2 (223835) actually pulled into the oxygenation_pts calc?
# ============================================================

print("=" * 70)
print("STEP 2: FiO2 usage trace — was it silently dropped or actually used?")
print("=" * 70)
print()

fio2_in_organ_burden = organ_burden_24h[organ_burden_24h['itemid'] == 223835]
print(f"FiO2 rows available in organ_burden_inputs_24h.csv: {len(fio2_in_organ_burden)}")
print(f"Patients with >=1 FiO2 reading: {fio2_in_organ_burden['subject_id'].nunique()} / {n_cohort}")
print()

print("CODE-LEVEL FACT: the oxygenation_pts calculation in the APACHE-II script pulls FiO2")
print("directly with this line:")
print()
print("    fio2_data = organ_burden_24h[organ_burden_24h['itemid'] == 223835][...]")
print()
print("This is a DIRECT itemid filter against organ_burden_inputs_24h.csv, completely")
print("INDEPENDENT of the vitals_labs_24h.csv 'variable' mapping / unmapped-itemid check.")
print("The 'unmapped itemid' warning only ever scanned vitals_labs_24h.csv's itemids —")
print("it had no ability to affect anything read from organ_burden_inputs_24h.csv.")
print()

if (vitals_labs_24h['itemid'] == 223835).sum() > 0:
    print("HOWEVER: itemid 223835 IS also present in vitals_labs_24h.csv (see Step 1 above).")
    print("This means FiO2 rows exist in BOTH files — likely a leftover duplicate from an")
    print("earlier pipeline step. This duplicate copy in vitals_labs_24h.csv is what triggered")
    print("the 'unmapped' warning (since 223835 isn't in that cell's vitals_map dict), but it")
    print("is HARMLESS to oxygenation_pts specifically, because that calculation never reads")
    print("FiO2 from vitals_labs_24h.csv at all — only from organ_burden_inputs_24h.csv.")
    print("The duplicate rows in vitals_labs_24h.csv are simply unused/inert for this purpose.")
else:
    print("CONFIRMED: itemid 223835 is NOT present in vitals_labs_24h.csv at all.")
    print("This means the 'unmapped' warning you saw did NOT include 223835 from a real data")
    print("row — double-check exactly which warning text you're referring to, since based on")
    print("the actual file contents, 223835 shouldn't have triggered that specific check.")
print()

# ============================================================
# STEP 3: Confirm Lactate/Platelets/Bilirubin exclusion is expected, not a bug
# ============================================================

print("=" * 70)
print("STEP 3: Lactate/Platelets/Bilirubin — confirm expected exclusion, not a bug")
print("=" * 70)
print()
print("These 3 are NOT among APACHE-II's 12 standard Acute Physiology variables")
print("(Temp, MAP, HR, RR, Oxygenation, pH, Sodium, Potassium, Creatinine, Hematocrit,")
print("WBC, GCS). They WERE correctly extracted earlier for SOFA (coagulation=Platelets,")
print("liver=Bilirubin) and basic_features (Lactate as a TPIFE input), and correctly")
print("remain present in vitals_labs_24h.csv for those other purposes.")
print()
print("The APACHE-II cell's 'vitals_map' dictionary simply does not include")
print("50813/51265/50885 as target variables (by design), so rows with those itemids fall")
print("through to 'unmapped', get checked by infer_supplemental_var() (which only looks for")
print("'sodium'/'potassium'/'hematocrit' in the label — correctly returns None for these 3),")
print("and are excluded from the APACHE-II variable set. This is the CORRECT, intended")
print("behavior — not a bug affecting Sodium/Potassium/Hematocrit or any of the 12 real")
print("APACHE-II inputs.")
print()

for itemid in [50813, 51265, 50885]:
    n_rows = (vitals_labs_24h['itemid'] == itemid).sum()
    print(f"  itemid {itemid} ({ITEMID_NAMES[itemid]}): {n_rows} rows present in vitals_labs_24h.csv "
          f"(correctly excluded from APACHE-II, still available for SOFA/basic_features use)")
print()

# ============================================================
# STEP 4: Actually check oxygenation_pts coverage and WHY it's low (if it is)
# ============================================================

print("=" * 70)
print("STEP 4: oxygenation_pts coverage — is it actually low, and why?")
print("=" * 70)
print()

n_oxygenation_available = apache_df['oxygenation_pts'].notna().sum()
pct_oxygenation = round(100 * n_oxygenation_available / n_cohort, 2)

print(f"oxygenation_pts coverage: {n_oxygenation_available} / {n_cohort} ({pct_oxygenation}%)")
print()

spo2_patients = vitals_labs_24h[vitals_labs_24h['itemid'] == 220277]['subject_id'].nunique()
fio2_patients = fio2_in_organ_burden['subject_id'].nunique()
both_patients = set(vitals_labs_24h.loc[vitals_labs_24h['itemid'] == 220277, 'subject_id']) & \
                set(fio2_in_organ_burden['subject_id'])

print(f"Patients with >=1 SpO2 reading: {spo2_patients} / {n_cohort}")
print(f"Patients with >=1 FiO2 reading: {fio2_patients} / {n_cohort}")
print(f"Patients with BOTH SpO2 and FiO2 present (necessary but not sufficient — still needs")
print(f"  a matched pair within +/-1h): {len(both_patients)} / {n_cohort}")
print()

print("LIKELY EXPLANATION for oxygenation_pts coverage gap (if it's meaningfully below")
print("SpO2-only or FiO2-only coverage): the +/-1h TIME-MATCHING requirement between an")
print("SpO2 reading and a FiO2 reading is the bottleneck, NOT a silent data-dropping bug.")
print("FiO2 is charted far less frequently than SpO2 (typically only when ventilator/O2")
print("settings change, not continuously) — so many patients have SpO2 data with no FiO2")
print("reading charted within an hour of any given SpO2 timestamp, even though both variables")
print("independently have data somewhere in the 24h window. This is a genuine data-sparsity")
print("limitation of the S/F-ratio approach, not a code defect.")

In [ ]:
import pandas as pd
import numpy as np

processed_path = "../data/processed/"

vitals_labs_24h = pd.read_csv(processed_path + "vitals_labs_24h.csv")
vitals_labs_24h['charttime'] = pd.to_datetime(vitals_labs_24h['charttime'])

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])
n_cohort = len(cohort_step1)
all_subjects = cohort_step1['subject_id']

print(f"vitals_labs_24h.csv loaded: {vitals_labs_24h.shape}")
print(f"Cohort: {n_cohort} patients")
print()

# ============================================================
# Full itemid -> variable mapping (everything currently in the file)
# ============================================================

raw_path = "../data/raw/"
d_labitems = pd.read_csv(raw_path + "d_labitems.csv.gz", compression="gzip")
label_lookup = d_labitems.set_index('itemid')['label'].to_dict()

vitals_map = {
    220045: 'HR', 220050: 'SBP', 220179: 'SBP', 220051: 'DBP', 220180: 'DBP',
    220210: 'RR', 223761: 'Temp', 223762: 'Temp', 220277: 'SpO2',
    223900: 'GCS_Verbal', 220739: 'GCS_Eye', 223901: 'GCS_Motor',
    51301: 'WBC', 51300: 'WBC', 50813: 'Lactate', 52442: 'Lactate', 53154: 'Lactate',
    50912: 'Creatinine', 52546: 'Creatinine', 51265: 'Platelets', 53189: 'Platelets',
    50885: 'Bilirubin', 53089: 'Bilirubin', 50820: 'pH',
}

vitals_labs_24h['variable'] = vitals_labs_24h['itemid'].map(vitals_map)
unmapped_mask = vitals_labs_24h['variable'].isna()

def infer_supplemental_var(itemid):
    label = str(label_lookup.get(itemid, '')).lower()
    if 'sodium' in label: return 'Sodium'
    if 'potassium' in label: return 'Potassium'
    if 'hematocrit' in label: return 'Hematocrit'
    return None

vitals_labs_24h.loc[unmapped_mask, 'variable'] = vitals_labs_24h.loc[unmapped_mask, 'itemid'].apply(infer_supplemental_var)

still_unmapped = vitals_labs_24h[vitals_labs_24h['variable'].isna()]['itemid'].unique()
if len(still_unmapped) > 0:
    print(f"Unmapped itemids (excluded from TPIFE): {still_unmapped}")
print()

# Join intime, compute hours_since_intime, restrict to 0-24h (should already be true,
# but re-enforced here in case any merge introduced stray rows)
data = vitals_labs_24h.merge(cohort_step1[['subject_id', 'intime']], on='subject_id', how='inner')
data['hours_since_intime'] = (data['charttime'] - data['intime']).dt.total_seconds() / 3600
data_24h = data[(data['hours_since_intime'] >= 0) & (data['hours_since_intime'] <= 24)].copy()

# ============================================================
# Build GCS_Total series (Eye+Verbal+Motor at same charttime), same pattern as SOFA/APACHE-II
# ============================================================

gcs_components = data_24h[data_24h['variable'].isin(['GCS_Eye', 'GCS_Verbal', 'GCS_Motor'])]
gcs_pivot = gcs_components.pivot_table(
    index=['subject_id', 'charttime', 'hours_since_intime'], columns='variable', values='valuenum', aggfunc='first'
).reset_index()
for col in ['GCS_Eye', 'GCS_Verbal', 'GCS_Motor']:
    if col not in gcs_pivot.columns:
        gcs_pivot[col] = np.nan
gcs_pivot['valuenum'] = gcs_pivot['GCS_Eye'] + gcs_pivot['GCS_Verbal'] + gcs_pivot['GCS_Motor']
gcs_total_series = gcs_pivot.dropna(subset=['valuenum'])[['subject_id', 'charttime', 'hours_since_intime', 'valuenum']].copy()
gcs_total_series['variable'] = 'GCS_Total'

data_24h_full = pd.concat([data_24h, gcs_total_series], ignore_index=True)

# ============================================================
# THRESHOLD TABLE — parameterized, easy to adjust individual ranges
# ============================================================

THRESHOLDS = {
    'HR': (60, 100), 'RR': (12, 20), 'SBP': (90, 130), 'DBP': (60, 80),
    'Temp': (36.1, 37.8), 'SpO2': (95, 100),
    'WBC': (4.5, 11.0), 'Lactate': (0.5, 2.0), 'Creatinine': (0.6, 1.3),
    'Platelets': (150, 400), 'Bilirubin': (0.1, 1.2),
    'Sodium': (135, 145), 'Potassium': (3.5, 5.0), 'pH': (7.35, 7.45),
    'Hematocrit': (36, 50), 'GCS_Total': (13, 15),
}

# ============================================================
# Build MAP series (derived from SBP/DBP, same-charttime match)
# ============================================================

sbp_series = data_24h[data_24h['variable'] == 'SBP'][['subject_id', 'charttime', 'hours_since_intime', 'valuenum']].rename(columns={'valuenum': 'sbp'})
dbp_series = data_24h[data_24h['variable'] == 'DBP'][['subject_id', 'charttime', 'hours_since_intime', 'valuenum']].rename(columns={'valuenum': 'dbp'})
map_merged = sbp_series.merge(dbp_series, on=['subject_id', 'charttime', 'hours_since_intime'], how='inner')
map_merged['valuenum'] = (map_merged['sbp'] + 2 * map_merged['dbp']) / 3
map_series = map_merged[['subject_id', 'charttime', 'hours_since_intime', 'valuenum']].copy()
map_series['variable'] = 'MAP'
THRESHOLDS['MAP'] = (70, 100)

data_24h_full = pd.concat([data_24h_full, map_series], ignore_index=True)

print(f"Variables with a defined threshold: {sorted(THRESHOLDS.keys())}")
print()

# ============================================================
# CORE TPIFE FUNCTION — parameterized by threshold table
# Interpolation-based: treats consecutive readings as connected by a straight line,
# finds exact crossing points of the low/high boundaries within each segment, and
# integrates/counts against that piecewise-linear reconstruction. This is a genuine
# design choice (linear interpolation between charted points) — not the only valid
# way to do this, but a defensible and documented one.
# ============================================================

def analyze_patient_series(hours, values, low, high):
    """Given a single patient's sorted (hours, values) for one variable, and the
    normal range (low, high), compute: n_episodes, time_abnormal_hrs,
    auc_over_threshold, instability_score. Returns all NaN if fewer than 2 readings."""
    n = len(hours)
    if n < 2:
        return {'n_episodes': np.nan, 'time_abnormal_hrs': np.nan,
                'auc_over_threshold': np.nan, 'instability_score': np.nan}

    def classify(v):
        if v < low: return 'low'
        if v > high: return 'high'
        return 'normal'

    def frac_for_value(v0, v1, target):
        if v0 == v1:
            return None
        f = (target - v0) / (v1 - v0)
        return f if 0 < f < 1 else None

    total_abnormal_time = 0.0
    auc = 0.0
    crossings = 0
    n_episodes = 0

    prev_t, prev_v = hours[0], values[0]
    prev_state = classify(prev_v)
    if prev_state != 'normal':
        n_episodes = 1

    for i in range(1, n):
        t, v = hours[i], values[i]
        dt_total = t - prev_t
        if dt_total <= 0:
            prev_t, prev_v, prev_state = t, v, classify(v)
            continue

        breakpoints = sorted(filter(None.__ne__, [
            frac_for_value(prev_v, v, low), frac_for_value(prev_v, v, high)
        ]))
        fracs = [0.0] + breakpoints + [1.0]

        state_seq = [prev_state]
        for j in range(len(fracs) - 1):
            f0, f1 = fracs[j], fracs[j + 1]
            fm = (f0 + f1) / 2
            vm = prev_v + fm * (v - prev_v)
            sub_state = classify(vm)
            state_seq.append(sub_state)

            sub_dt = (f1 - f0) * dt_total
            v0 = prev_v + f0 * (v - prev_v)
            v1 = prev_v + f1 * (v - prev_v)

            if sub_state != 'normal':
                total_abnormal_time += sub_dt
                if sub_state == 'high':
                    e0, e1 = max(v0 - high, 0), max(v1 - high, 0)
                else:
                    e0, e1 = max(low - v0, 0), max(low - v1, 0)
                auc += 0.5 * (e0 + e1) * sub_dt

        final_state = classify(v)
        state_seq.append(final_state)

        for k in range(1, len(state_seq)):
            if state_seq[k] != state_seq[k - 1]:
                crossings += 1
                if state_seq[k - 1] == 'normal' and state_seq[k] != 'normal':
                    n_episodes += 1

        prev_t, prev_v, prev_state = t, v, final_state

    return {'n_episodes': n_episodes, 'time_abnormal_hrs': total_abnormal_time,
            'auc_over_threshold': auc, 'instability_score': crossings}

def compute_early_late_trend(hours, values):
    hours = np.array(hours)
    values = np.array(values)
    early_mask = hours <= 12
    late_mask = hours > 12
    if early_mask.sum() == 0 or late_mask.sum() == 0:
        return np.nan
    return values[early_mask].mean() - values[late_mask].mean()

# ============================================================
# Run TPIFE computation for every variable in THRESHOLDS
# ============================================================

tpife_results = {}

for var, (low, high) in THRESHOLDS.items():
    var_data = data_24h_full[data_24h_full['variable'] == var]
    per_patient_stats = {}

    for subj_id, group in var_data.groupby('subject_id'):
        group_sorted = group.sort_values('hours_since_intime')
        hours = group_sorted['hours_since_intime'].values
        values = group_sorted['valuenum'].values

        stats = analyze_patient_series(hours, values, low, high)
        stats['early_late_trend'] = compute_early_late_trend(hours, values)
        per_patient_stats[subj_id] = stats

    stats_df = pd.DataFrame.from_dict(per_patient_stats, orient='index')
    stats_df.columns = [f'{var}_abnormal_episodes' if c == 'n_episodes' else
                         f'{var}_time_abnormal_hrs' if c == 'time_abnormal_hrs' else
                         f'{var}_auc_over_threshold' if c == 'auc_over_threshold' else
                         f'{var}_early_late_trend' if c == 'early_late_trend' else
                         f'{var}_instability_score' for c in stats_df.columns]

    tpife_results[var] = stats_df
    print(f"{var}: computed for {len(stats_df)} patients with >=1 reading "
          f"({(stats_df[f'{var}_abnormal_episodes'].notna()).sum()} with >=2 readings, non-NaN stats)")

print()

# ============================================================
# Combine into one wide table
# ============================================================

tpife_wide = pd.DataFrame({'subject_id': all_subjects}).set_index('subject_id')
for var, stats_df in tpife_results.items():
    tpife_wide = tpife_wide.join(stats_df, how='left')

tpife_wide = tpife_wide.reset_index()

print(f"Final tpife_features.csv shape: {tpife_wide.shape} (expected ({n_cohort}, "
      f"{1 + 5*len(THRESHOLDS)}))")
print()

# ============================================================
# Coverage summary: % of patients with at least one computed (non-NaN) TPIFE
# feature per variable
# ============================================================

print("=" * 65)
print("COVERAGE SUMMARY (% of patients with computed TPIFE stats per variable)")
print("=" * 65)
coverage_rows = []
for var in THRESHOLDS.keys():
    col = f'{var}_abnormal_episodes'
    n_covered = tpife_wide[col].notna().sum()
    pct = round(100 * n_covered / n_cohort, 2)
    coverage_rows.append({'variable': var, 'n_covered': n_covered, 'coverage_pct': pct})

coverage_df = pd.DataFrame(coverage_rows).sort_values('coverage_pct', ascending=False)
print(coverage_df.to_string(index=False))
print()

# ============================================================
# Save
# ============================================================

tpife_wide.to_csv(processed_path + "tpife_features.csv", index=False)
print(f"Saved tpife_features.csv. Shape: {tpife_wide.shape}")

In [ ]:
import pandas as pd

processed_path = "../data/processed/"

tpife_wide = pd.read_csv(processed_path + "tpife_features.csv")
cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
n_cohort = len(cohort_step1)

vitals_labs_24h = pd.read_csv(processed_path + "vitals_labs_24h.csv")
vitals_labs_24h['charttime'] = pd.to_datetime(vitals_labs_24h['charttime'])

print(f"tpife_features.csv loaded: {tpife_wide.shape}")
print(f"Cohort size: {n_cohort}")
print()

# ============================================================
# Rebuild the same itemid -> variable mapping used in TPIFE computation,
# including the derived GCS_Total and MAP series, so reading counts are
# computed against the EXACT same source data
# ============================================================

raw_path = "../data/raw/"
d_labitems = pd.read_csv(raw_path + "d_labitems.csv.gz", compression="gzip")
label_lookup = d_labitems.set_index('itemid')['label'].to_dict()

vitals_map = {
    220045: 'HR', 220050: 'SBP', 220179: 'SBP', 220051: 'DBP', 220180: 'DBP',
    220210: 'RR', 223761: 'Temp', 223762: 'Temp', 220277: 'SpO2',
    223900: 'GCS_Verbal', 220739: 'GCS_Eye', 223901: 'GCS_Motor',
    51301: 'WBC', 51300: 'WBC', 50813: 'Lactate', 52442: 'Lactate', 53154: 'Lactate',
    50912: 'Creatinine', 52546: 'Creatinine', 51265: 'Platelets', 53189: 'Platelets',
    50885: 'Bilirubin', 53089: 'Bilirubin', 50820: 'pH',
}
vitals_labs_24h['variable'] = vitals_labs_24h['itemid'].map(vitals_map)
unmapped_mask = vitals_labs_24h['variable'].isna()

def infer_supplemental_var(itemid):
    label = str(label_lookup.get(itemid, '')).lower()
    if 'sodium' in label: return 'Sodium'
    if 'potassium' in label: return 'Potassium'
    if 'hematocrit' in label: return 'Hematocrit'
    return None

vitals_labs_24h.loc[unmapped_mask, 'variable'] = vitals_labs_24h.loc[unmapped_mask, 'itemid'].apply(infer_supplemental_var)

cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])
data = vitals_labs_24h.merge(cohort_step1[['subject_id', 'intime']], on='subject_id', how='inner')
data['hours_since_intime'] = (data['charttime'] - data['intime']).dt.total_seconds() / 3600
data_24h = data[(data['hours_since_intime'] >= 0) & (data['hours_since_intime'] <= 24)].copy()

# Rebuild GCS_Total
gcs_components = data_24h[data_24h['variable'].isin(['GCS_Eye', 'GCS_Verbal', 'GCS_Motor'])]
gcs_pivot = gcs_components.pivot_table(
    index=['subject_id', 'charttime', 'hours_since_intime'], columns='variable', values='valuenum', aggfunc='first'
).reset_index()
for col in ['GCS_Eye', 'GCS_Verbal', 'GCS_Motor']:
    if col not in gcs_pivot.columns:
        gcs_pivot[col] = float('nan')
gcs_pivot['valuenum'] = gcs_pivot['GCS_Eye'] + gcs_pivot['GCS_Verbal'] + gcs_pivot['GCS_Motor']
gcs_total_series = gcs_pivot.dropna(subset=['valuenum'])[['subject_id', 'charttime', 'hours_since_intime', 'valuenum']].copy()
gcs_total_series['variable'] = 'GCS_Total'

# Rebuild MAP
sbp_series = data_24h[data_24h['variable'] == 'SBP'][['subject_id', 'charttime', 'hours_since_intime', 'valuenum']].rename(columns={'valuenum': 'sbp'})
dbp_series = data_24h[data_24h['variable'] == 'DBP'][['subject_id', 'charttime', 'hours_since_intime', 'valuenum']].rename(columns={'valuenum': 'dbp'})
map_merged = sbp_series.merge(dbp_series, on=['subject_id', 'charttime', 'hours_since_intime'], how='inner')
map_merged['valuenum'] = (map_merged['sbp'] + 2 * map_merged['dbp']) / 3
map_series = map_merged[['subject_id', 'charttime', 'hours_since_intime', 'valuenum']].copy()
map_series['variable'] = 'MAP'

data_24h_full = pd.concat([data_24h, gcs_total_series, map_series], ignore_index=True)

variables = ['HR', 'RR', 'SBP', 'DBP', 'MAP', 'Temp', 'SpO2', 'WBC', 'Lactate',
             'Creatinine', 'Platelets', 'Bilirubin', 'Sodium', 'Potassium', 'pH',
             'Hematocrit', 'GCS_Total']

# ============================================================
# Compute the 3 coverage numbers per variable
# ============================================================

coverage_rows = []

for var in variables:
    var_data = data_24h_full[data_24h_full['variable'] == var]
    reading_counts = var_data.groupby('subject_id').size()

    n_ge1 = (reading_counts >= 1).sum()
    n_ge2 = (reading_counts >= 2).sum()

    stat_cols = [f'{var}_abnormal_episodes', f'{var}_time_abnormal_hrs',
                 f'{var}_auc_over_threshold', f'{var}_early_late_trend',
                 f'{var}_instability_score']

    missing_cols = [c for c in stat_cols if c not in tpife_wide.columns]
    if missing_cols:
        print(f"WARNING: {var} missing expected columns in tpife_features.csv: {missing_cols}")
        n_full_row = None
    else:
        n_full_row = tpife_wide[stat_cols].notna().all(axis=1).sum()

    coverage_rows.append({
        'variable': var,
        'patients_ge1_reading': n_ge1,
        'pct_ge1': round(100 * n_ge1 / n_cohort, 1),
        'patients_ge2_readings': n_ge2,
        'pct_ge2': round(100 * n_ge2 / n_cohort, 1),
        'patients_full_tpife_row': n_full_row,
        'pct_full_tpife': round(100 * n_full_row / n_cohort, 1) if n_full_row is not None else None,
    })

coverage_df = pd.DataFrame(coverage_rows).sort_values('pct_full_tpife', ascending=False)

print("=" * 100)
print("TPIFE COVERAGE — three-tier breakdown per variable")
print("=" * 100)
print(coverage_df.to_string(index=False))
print()

print("Column meanings:")
print("  patients_ge1_reading  : raw data availability — has ANY reading in first 24h")
print("  patients_ge2_readings : minimum needed for abnormal_episodes/time_abnormal_hrs/")
print("                          auc_over_threshold/instability_score to be non-NaN")
print("  patients_full_tpife_row: has ALL 5 TPIFE stats non-NaN, including early_late_trend")
print("                          (which additionally needs >=1 reading in EACH of 0-12h and")
print("                          12-24h windows — stricter than just >=2 readings anywhere)")
print()
print("USE 'patients_full_tpife_row' / 'pct_full_tpife' as the true TPIFE feature coverage")
print("for Notebook 3 missingness handling — the other two columns explain WHY a variable's")
print("full-row coverage is lower than its raw availability, not what to actually use.")

In [ ]:
import pandas as pd
import numpy as np

raw_path = "../data/raw/"
processed_path = "../data/processed/"

tpife = pd.read_csv(processed_path + "tpife_features.csv")
organ_burden_24h = pd.read_csv(processed_path + "organ_burden_inputs_24h.csv")
organ_burden_24h['charttime'] = pd.to_datetime(organ_burden_24h['charttime'])

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
n_cohort = len(cohort_step1)
all_subjects = cohort_step1['subject_id']

print(f"tpife_features.csv loaded: {tpife.shape}")
print(f"organ_burden_inputs_24h.csv loaded: {organ_burden_24h.shape}")
print()

# ============================================================
# NORMALIZATION METHOD — decision and rationale
# ============================================================

print("=" * 70)
print("NORMALIZATION METHOD: min-max scaling to [0,1], NOT z-score")
print("=" * 70)
print("""
Rationale:
- Min-max keeps every component bounded in [0,1], so averaging heterogeneous
  components (a count like 'abnormal_episodes', a duration in hours, an AUC in
  mixed units, a dose in mcg/kg/min) into one burden score stays interpretable
  as 'fraction of the cohort's observed severity range' — a burden score of 0.8
  means 'near the worst observed in this cohort', regardless of the original
  units.
- Z-score is unbounded and centered at 0 (with negative values), which is
  harder to combine into a single intuitive '0=no burden, 1=max burden' scale,
  and more sensitive to how symmetric each feature's distribution is.
- Min-max IS sensitive to outliers (a single extreme value compresses everyone
  else toward 0) — but outlier cleaning was already done upstream (Temp, SpO2,
  pH), so this risk is reduced, not eliminated. Component features from TPIFE
  (auc_over_threshold especially) can still have a long right tail from
  legitimately very sick patients — that's expected and desired here, not
  noise.
- TPIFE magnitude features (abnormal_episodes, time_abnormal_hrs,
  auc_over_threshold, instability_score) are ALL constructed so that higher =
  more abnormal/unstable, regardless of whether the deviation was high or low
  relative to normal range — so NO directional inversion is needed for these.
- Two exceptions requiring explicit handling:
    1. Urine output: LOWER volume = MORE renal burden, so this is inverted
       after min-max scaling (burden = 1 - minmax(urine_output)).
    2. TPIFE 'early_late_trend' features are EXCLUDED from the burden index —
       a trend captures DIRECTION (improving vs worsening), not magnitude of
       abnormality, and a large positive or negative trend both indicate
       instability but in opposite senses. Mixing signed trend values into an
       unsigned burden average would be conceptually wrong. This is a
       deliberate design choice, not an oversight.
""")
print()

def minmax_normalize(series):
    """Min-max scale a series to [0,1], ignoring NaN. Returns NaN where input is NaN."""
    valid = series.dropna()
    if len(valid) == 0 or valid.min() == valid.max():
        return pd.Series(np.nan, index=series.index)
    return (series - valid.min()) / (valid.max() - valid.min())

# ============================================================
# Build the TPIFE magnitude-feature list per organ system (excluding early_late_trend)
# ============================================================

def tpife_magnitude_cols(var):
    return [f'{var}_abnormal_episodes', f'{var}_time_abnormal_hrs',
            f'{var}_auc_over_threshold', f'{var}_instability_score']

organ_tpife_vars = {
    'Cardiovascular': ['MAP'],
    'Respiratory': ['RR', 'SpO2'],
    'Renal': ['Creatinine'],
    'Hepatic': ['Bilirubin'],
    'Hematologic': ['Platelets'],
    'Metabolic': ['Lactate', 'pH', 'Sodium', 'Potassium'],
}

# ============================================================
# ADDITIONAL non-TPIFE domain features per organ system
# ============================================================

# --- Cardiovascular: vasopressor exposure ---
vaso_data = organ_burden_24h[~organ_burden_24h['itemid'].isin({223835, 226559, 226560})].copy()
conv_col = 'valuenum_mcg_kg_min'

max_vaso_dose = vaso_data.groupby('subject_id')[conv_col].max()
# Exposure proxy: mean dose x number of readings (approximation of dose-time burden,
# NOT a true time-integrated AUC since inputevents intervals weren't reconstructed here)
vaso_exposure_proxy = vaso_data.groupby('subject_id')[conv_col].apply(lambda s: s.dropna().sum())

print(f"Vasopressor: max dose available for {max_vaso_dose.notna().sum()} patients, "
      f"exposure proxy for {vaso_exposure_proxy.notna().sum()} patients")
print("NOTE: 'exposure proxy' = sum of charted mcg/kg/min values, NOT a true time-integrated")
print("dose-hours AUC (would require reconstructing start/end intervals) — documented approximation.")
print()

# --- Respiratory: FiO2 exposure ---
fio2_data = organ_burden_24h[organ_burden_24h['itemid'] == 223835]
max_fio2 = fio2_data.groupby('subject_id')['valuenum'].max()
mean_fio2 = fio2_data.groupby('subject_id')['valuenum'].mean()

# --- Renal: urine output + oliguria ---
urine_data = organ_burden_24h[organ_burden_24h['itemid'].isin({226559, 226560})]
total_uo_24h = urine_data.groupby('subject_id')['valuenum'].sum()

# Attempt to reload weight for mL/kg/hr oliguria criterion
d_items = pd.read_csv(raw_path + "d_items.csv.gz", compression="gzip")
weight_items = d_items[d_items['label'].str.contains('weight', case=False, na=False)]
weight_itemids = set(weight_items['itemid'])

chunk_size = 1_000_000
weight_rows = []
cohort_subject_ids = set(all_subjects)
for chunk in pd.read_csv(
    raw_path + "chartevents.csv.gz", compression="gzip",
    usecols=['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum'],
    chunksize=chunk_size
):
    filtered = chunk[chunk['itemid'].isin(weight_itemids) & chunk['subject_id'].isin(cohort_subject_ids)]
    if len(filtered) > 0:
        weight_rows.append(filtered)
weight_events = pd.concat(weight_rows, ignore_index=True) if weight_rows else pd.DataFrame(columns=['subject_id', 'valuenum'])
patient_weight = weight_events.sort_values('charttime').groupby('subject_id')['valuenum'].first() if len(weight_events) > 0 else pd.Series(dtype=float)

print(f"Weight available for {patient_weight.notna().sum()} / {n_cohort} patients (for mL/kg/hr oliguria criterion)")

oliguria_flag = pd.Series(index=all_subjects, dtype=object)
for subj in all_subjects:
    uo = total_uo_24h.get(subj, np.nan)
    wt = patient_weight.get(subj, np.nan)
    if pd.isna(uo):
        oliguria_flag[subj] = np.nan
    elif not pd.isna(wt) and wt > 0:
        oliguria_flag[subj] = (uo / wt / 24) < 0.5  # mL/kg/hr
    else:
        oliguria_flag[subj] = uo < 500  # absolute fallback

n_weight_based = patient_weight.notna().sum()
n_absolute_fallback = total_uo_24h.notna().sum() - n_weight_based
print(f"Oliguria assessed via mL/kg/hr for {n_weight_based} patients, "
      f"absolute <500mL fallback for remaining {max(n_absolute_fallback,0)} with UO data")
print()

# ============================================================
# BUILD NORMALIZED FEATURES INTO ONE WORKING TABLE
# ============================================================

burden_raw = pd.DataFrame({'subject_id': all_subjects}).set_index('subject_id')

for organ, tpife_vars in organ_tpife_vars.items():
    for var in tpife_vars:
        for col in tpife_magnitude_cols(var):
            if col in tpife.columns:
                burden_raw[col] = tpife.set_index('subject_id')[col]

burden_raw['vaso_max_dose'] = max_vaso_dose
burden_raw['vaso_exposure_proxy'] = vaso_exposure_proxy
burden_raw['fio2_max'] = max_fio2
burden_raw['fio2_mean'] = mean_fio2
burden_raw['urine_output_24h'] = total_uo_24h

# ============================================================
# NORMALIZE all raw columns to [0,1] (min-max), invert urine output
# ============================================================

burden_norm = pd.DataFrame(index=burden_raw.index)
for col in burden_raw.columns:
    burden_norm[col] = minmax_normalize(burden_raw[col])

# Invert urine output: lower volume = higher renal burden
burden_norm['urine_output_burden'] = 1 - burden_norm['urine_output_24h']
burden_norm = burden_norm.drop(columns=['urine_output_24h'])

# Oliguria as a binary 0/1 burden contributor
burden_norm['oliguria_burden'] = oliguria_flag.map({True: 1.0, False: 0.0, np.nan: np.nan})

# ============================================================
# AGGREGATE INTO PER-ORGAN-SYSTEM BURDEN SCORES
# Row-wise mean with skipna=True: NaN only if ALL components are NaN.
# ============================================================

organ_score_components = {
    'Cardiovascular_score': [f'{c}' for var in organ_tpife_vars['Cardiovascular'] for c in tpife_magnitude_cols(var)]
                             + ['vaso_max_dose', 'vaso_exposure_proxy'],
    'Respiratory_score': [f'{c}' for var in organ_tpife_vars['Respiratory'] for c in tpife_magnitude_cols(var)]
                          + ['fio2_max', 'fio2_mean'],
    'Renal_score': [f'{c}' for var in organ_tpife_vars['Renal'] for c in tpife_magnitude_cols(var)]
                    + ['urine_output_burden', 'oliguria_burden'],
    'Hepatic_score': [f'{c}' for var in organ_tpife_vars['Hepatic'] for c in tpife_magnitude_cols(var)],
    'Hematologic_score': [f'{c}' for var in organ_tpife_vars['Hematologic'] for c in tpife_magnitude_cols(var)],
    'Metabolic_score': [f'{c}' for var in organ_tpife_vars['Metabolic'] for c in tpife_magnitude_cols(var)],
}

organ_index = pd.DataFrame(index=all_subjects)

for score_name, cols in organ_score_components.items():
    cols_present = [c for c in cols if c in burden_norm.columns]
    organ_index[score_name] = burden_norm[cols_present].mean(axis=1, skipna=True)

organ_index = organ_index.reset_index().rename(columns={'index': 'subject_id'})

# Confirm NaN-only-when-all-missing behavior explicitly
print("=" * 70)
print("CONFIRMING: organ score is NaN only when ALL its components are missing")
print("=" * 70)
for score_name, cols in organ_score_components.items():
    cols_present = [c for c in cols if c in burden_norm.columns]
    all_missing_mask = burden_norm[cols_present].isna().all(axis=1)
    score_is_nan_mask = organ_index[score_name].isna()
    matches = (all_missing_mask.values == score_is_nan_mask.values).all()
    print(f"{score_name}: NaN exactly when all components missing? {matches} "
          f"({all_missing_mask.sum()} patients with all components missing)")
print()

# ============================================================
# DISTRIBUTIONS + COVERAGE
# ============================================================

print("=" * 70)
print("PER-ORGAN-SYSTEM SCORE DISTRIBUTIONS AND COVERAGE")
print("=" * 70)

summary_rows = []
for score_name in organ_score_components.keys():
    s = organ_index[score_name]
    n_covered = s.notna().sum()
    pct = round(100 * n_covered / n_cohort, 2)
    summary_rows.append({
        'organ_system': score_name.replace('_score', ''),
        'coverage_n': n_covered, 'coverage_pct': pct,
        'mean': round(s.mean(), 3) if n_covered > 0 else np.nan,
        'median': round(s.median(), 3) if n_covered > 0 else np.nan,
        'min': round(s.min(), 3) if n_covered > 0 else np.nan,
        'max': round(s.max(), 3) if n_covered > 0 else np.nan,
    })

summary_df = pd.DataFrame(summary_rows).sort_values('coverage_pct', ascending=False)
print(summary_df.to_string(index=False))
print()

hepatic_row = summary_df[summary_df['organ_system'] == 'Hepatic']
print(f">>> Hepatic coverage: {hepatic_row['coverage_pct'].values[0]}% — "
      f"{'CONFIRMED weakest, as expected from earlier Bilirubin coverage findings' if hepatic_row['coverage_pct'].values[0] == summary_df['coverage_pct'].min() else 'NOT the weakest — worth investigating why'}")
print()

# ============================================================
# Save
# ============================================================

organ_index.to_csv(processed_path + "organ_burden_index.csv", index=False)
print(f"Saved organ_burden_index.csv. Shape: {organ_index.shape}")
print(f"Columns: {organ_index.columns.tolist()}")


In [ ]:
import pandas as pd
import numpy as np

processed_path = "../data/processed/"
raw_path = "../data/raw/"

tpife = pd.read_csv(processed_path + "tpife_features.csv")
vitals_labs_24h = pd.read_csv(processed_path + "vitals_labs_24h.csv")
vitals_labs_24h['charttime'] = pd.to_datetime(vitals_labs_24h['charttime'])

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])
n_cohort = len(cohort_step1)
all_subjects = cohort_step1['subject_id']

print(f"tpife_features.csv loaded: {tpife.shape}")
print(f"vitals_labs_24h.csv loaded: {vitals_labs_24h.shape}")
print()

# ============================================================
# Rebuild the exact same variable series used to build tpife_features.csv
# (itemid mapping + derived GCS_Total + derived MAP)
# ============================================================

d_labitems = pd.read_csv(raw_path + "d_labitems.csv.gz", compression="gzip")
label_lookup = d_labitems.set_index('itemid')['label'].to_dict()

vitals_map = {
    220045: 'HR', 220050: 'SBP', 220179: 'SBP', 220051: 'DBP', 220180: 'DBP',
    220210: 'RR', 223761: 'Temp', 223762: 'Temp', 220277: 'SpO2',
    223900: 'GCS_Verbal', 220739: 'GCS_Eye', 223901: 'GCS_Motor',
    51301: 'WBC', 51300: 'WBC', 50813: 'Lactate', 52442: 'Lactate', 53154: 'Lactate',
    50912: 'Creatinine', 52546: 'Creatinine', 51265: 'Platelets', 53189: 'Platelets',
    50885: 'Bilirubin', 53089: 'Bilirubin', 50820: 'pH',
}
vitals_labs_24h['variable'] = vitals_labs_24h['itemid'].map(vitals_map)
unmapped_mask = vitals_labs_24h['variable'].isna()

def infer_supplemental_var(itemid):
    label = str(label_lookup.get(itemid, '')).lower()
    if 'sodium' in label: return 'Sodium'
    if 'potassium' in label: return 'Potassium'
    if 'hematocrit' in label: return 'Hematocrit'
    return None

vitals_labs_24h.loc[unmapped_mask, 'variable'] = vitals_labs_24h.loc[unmapped_mask, 'itemid'].apply(infer_supplemental_var)

data = vitals_labs_24h.merge(cohort_step1[['subject_id', 'intime']], on='subject_id', how='inner')
data['hours_since_intime'] = (data['charttime'] - data['intime']).dt.total_seconds() / 3600
data_24h = data[(data['hours_since_intime'] >= 0) & (data['hours_since_intime'] <= 24)].copy()

gcs_components = data_24h[data_24h['variable'].isin(['GCS_Eye', 'GCS_Verbal', 'GCS_Motor'])]
gcs_pivot = gcs_components.pivot_table(
    index=['subject_id', 'charttime', 'hours_since_intime'], columns='variable', values='valuenum', aggfunc='first'
).reset_index()
for col in ['GCS_Eye', 'GCS_Verbal', 'GCS_Motor']:
    if col not in gcs_pivot.columns:
        gcs_pivot[col] = np.nan
gcs_pivot['valuenum'] = gcs_pivot['GCS_Eye'] + gcs_pivot['GCS_Verbal'] + gcs_pivot['GCS_Motor']
gcs_total_series = gcs_pivot.dropna(subset=['valuenum'])[['subject_id', 'charttime', 'hours_since_intime', 'valuenum']].copy()
gcs_total_series['variable'] = 'GCS_Total'

sbp_series = data_24h[data_24h['variable'] == 'SBP'][['subject_id', 'charttime', 'hours_since_intime', 'valuenum']].rename(columns={'valuenum': 'sbp'})
dbp_series = data_24h[data_24h['variable'] == 'DBP'][['subject_id', 'charttime', 'hours_since_intime', 'valuenum']].rename(columns={'valuenum': 'dbp'})
map_merged = sbp_series.merge(dbp_series, on=['subject_id', 'charttime', 'hours_since_intime'], how='inner')
map_merged['valuenum'] = (map_merged['sbp'] + 2 * map_merged['dbp']) / 3
map_series = map_merged[['subject_id', 'charttime', 'hours_since_intime', 'valuenum']].copy()
map_series['variable'] = 'MAP'

data_24h_full = pd.concat([data_24h, gcs_total_series, map_series], ignore_index=True)

THRESHOLDS = {
    'HR': (60, 100), 'RR': (12, 20), 'SBP': (90, 130), 'DBP': (60, 80), 'MAP': (70, 100),
    'Temp': (36.1, 37.8), 'SpO2': (95, 100),
    'WBC': (4.5, 11.0), 'Lactate': (0.5, 2.0), 'Creatinine': (0.6, 1.3),
    'Platelets': (150, 400), 'Bilirubin': (0.1, 1.2),
    'Sodium': (135, 145), 'Potassium': (3.5, 5.0), 'pH': (7.35, 7.45),
    'Hematocrit': (36, 50), 'GCS_Total': (13, 15),
}

# ============================================================
# NEW COMPONENT 1: max_deviation — largest |distance from nearest boundary|
# among all ABNORMAL readings (not interpolated — using actual charted points,
# since "deviation" is about how bad an actual measured value was, not a
# reconstructed interpolated extreme)
# ============================================================

def compute_max_deviation(values, low, high):
    if len(values) == 0:
        return np.nan
    deviations = []
    for v in values:
        if v < low:
            deviations.append(low - v)
        elif v > high:
            deviations.append(v - high)
    if len(deviations) == 0:
        return 0.0  # had readings, none abnormal -> zero deviation, not missing
    return max(deviations)

# ============================================================
# NEW COMPONENT 2: recovery duration — time from entering abnormal back to
# normal, per COMPLETED episode (episode still abnormal at end of 24h window
# is CENSORED and excluded from the average, tracked separately)
# Reuses the same linear-interpolation crossing logic as the original TPIFE build.
# ============================================================

def compute_recovery_durations(hours, values, low, high):
    n = len(hours)
    if n < 2:
        return {'recovery_durations': [], 'n_completed': 0, 'n_censored': 0}

    def classify(v):
        if v < low: return 'low'
        if v > high: return 'high'
        return 'normal'

    def frac_for_value(v0, v1, target):
        if v0 == v1:
            return None
        f = (target - v0) / (v1 - v0)
        return f if 0 < f < 1 else None

    episode_start_time = None
    recovery_durations = []
    n_censored = 0

    prev_t, prev_v = hours[0], values[0]
    prev_state = classify(prev_v)
    if prev_state != 'normal':
        episode_start_time = prev_t

    for i in range(1, n):
        t, v = hours[i], values[i]
        dt_total = t - prev_t
        if dt_total <= 0:
            prev_t, prev_v, prev_state = t, v, classify(v)
            continue

        breakpoints = sorted(filter(None.__ne__, [
            frac_for_value(prev_v, v, low), frac_for_value(prev_v, v, high)
        ]))
        fracs = [0.0] + breakpoints + [1.0]

        for j in range(len(fracs) - 1):
            f0, f1 = fracs[j], fracs[j + 1]
            fm = (f0 + f1) / 2
            vm = prev_v + fm * (v - prev_v)
            sub_state = classify(vm)
            t_mid_end = prev_t + f1 * dt_total

            if sub_state == 'normal' and episode_start_time is not None:
                # transitioning back to normal -> episode completes
                recovery_durations.append(t_mid_end - episode_start_time)
                episode_start_time = None
            elif sub_state != 'normal' and episode_start_time is None:
                # transitioning into abnormal -> episode starts
                episode_start_time = prev_t + f0 * dt_total

        prev_t, prev_v, prev_state = t, v, classify(v)

    if episode_start_time is not None:
        n_censored = 1  # still abnormal at end of window — not a completed recovery

    return {'recovery_durations': recovery_durations,
            'n_completed': len(recovery_durations),
            'n_censored': n_censored}

# ============================================================
# NEW COMPONENT 3 (needed for oscillation frequency): monitored duration per
# patient per variable = last reading hour - first reading hour
# ============================================================

variables = list(THRESHOLDS.keys())
new_component_results = {}

for var in variables:
    var_data = data_24h_full[data_24h_full['variable'] == var]
    per_patient = {}
    for subj_id, group in var_data.groupby('subject_id'):
        group_sorted = group.sort_values('hours_since_intime')
        hours = group_sorted['hours_since_intime'].values
        values = group_sorted['valuenum'].values

        low, high = THRESHOLDS[var]

        max_dev = compute_max_deviation(values, low, high)
        recov = compute_recovery_durations(hours, values, low, high)
        avg_recovery = np.mean(recov['recovery_durations']) if recov['n_completed'] > 0 else np.nan

        monitored_duration = (hours[-1] - hours[0]) if len(hours) >= 2 else np.nan

        per_patient[subj_id] = {
            'max_deviation': max_dev,
            'avg_recovery_duration_hrs': avg_recovery,
            'n_completed_recoveries': recov['n_completed'],
            'n_censored_episodes': recov['n_censored'],
            'monitored_duration_hrs': monitored_duration,
        }
    new_component_results[var] = pd.DataFrame.from_dict(per_patient, orient='index')

print("New components computed (max_deviation, recovery_duration, monitored_duration) "
      "for all 17 variables.")
print()

# ============================================================
# Coverage of the 2 NEW components, printed BEFORE combining
# ============================================================

print("=" * 70)
print("COVERAGE — max_deviation and avg_recovery_duration_hrs (NEW components)")
print("=" * 70)
new_coverage_rows = []
for var in variables:
    df = new_component_results[var]
    n_maxdev = df['max_deviation'].notna().sum()
    n_recovery = df['avg_recovery_duration_hrs'].notna().sum()
    n_censored_any = (df['n_censored_episodes'] > 0).sum()
    new_coverage_rows.append({
        'variable': var,
        'max_deviation_coverage_pct': round(100 * n_maxdev / n_cohort, 1),
        'avg_recovery_coverage_pct': round(100 * n_recovery / n_cohort, 1),
        'patients_with_censored_episode': n_censored_any,
    })
new_cov_df = pd.DataFrame(new_coverage_rows)
print(new_cov_df.to_string(index=False))
print()
print("NOTE: 'avg_recovery_coverage' will generally be LOWER than 'max_deviation_coverage' —")
print("max_deviation only needs >=1 abnormal reading, but recovery duration needs an abnormal")
print("episode that ACTUALLY RETURNS to normal within the 24h window. Patients who go abnormal")
print("and stay abnormal until end-of-window (censored) contribute 0 to their recovery average,")
print("not because nothing happened, but because recovery wasn't observed — this is a genuine")
print("right-censoring issue, not a data gap. 'patients_with_censored_episode' surfaces this.")
print()

# ============================================================
# COMBINE ALL 5 SUB-COMPONENTS INTO THE COMPOSITE INDEX, per variable
# ============================================================

print("=" * 70)
print("COMBINATION METHOD: min-max normalization per variable, then mean (skipna)")
print("=" * 70)
print("""
Same approach as the Organ Burden Index, for consistency across the pipeline.
Confirming it's appropriate here specifically:
  - All 5 sub-components are ALREADY oriented the same direction: higher value
    = more unstable/worse (crossings, frequency, duration abnormal, magnitude
    of deviation, AND recovery duration all increase with worse physiology).
    Unlike the Organ Burden Index's urine output, NOTHING needs inverting here
    — this is a cleaner case for simple averaging than organ burden was.
  - Min-max keeps the composite bounded [0,1] and cohort-relative, consistent
    with how organ_burden_index.csv should be interpreted alongside it.
  - Normalization is done PER VARIABLE (not pooled across variables), since
    e.g. HR's crossing counts and Bilirubin's crossing counts aren't on a
    comparable natural scale even after being called by the same feature name.
""")
print()

def minmax_normalize(series):
    valid = series.dropna()
    if len(valid) == 0 or valid.min() == valid.max():
        return pd.Series(np.nan, index=series.index)
    return (series - valid.min()) / (valid.max() - valid.min())

stability_index = pd.DataFrame({'subject_id': all_subjects}).set_index('subject_id')

for var in variables:
    instability_col = f'{var}_instability_score'
    time_abn_col = f'{var}_time_abnormal_hrs'

    if instability_col not in tpife.columns or time_abn_col not in tpife.columns:
        print(f"WARNING: {var} missing expected columns in tpife_features.csv — skipping.")
        continue

    tpife_indexed = tpife.set_index('subject_id')
    instability = tpife_indexed[instability_col]
    time_abnormal = tpife_indexed[time_abn_col]

    new_df = new_component_results[var]
    max_deviation = new_df['max_deviation']
    monitored_duration = new_df['monitored_duration_hrs']
    avg_recovery = new_df['avg_recovery_duration_hrs']

    oscillation_freq = instability / monitored_duration.replace(0, np.nan)

    components_raw = pd.DataFrame({
        'instability_score': instability,
        'oscillation_frequency': oscillation_freq,
        'time_abnormal_hrs': time_abnormal,
        'max_deviation': max_deviation,
        'avg_recovery_duration_hrs': avg_recovery,
    })

    components_norm = pd.DataFrame(index=components_raw.index)
    for col in components_raw.columns:
        components_norm[col] = minmax_normalize(components_raw[col])

    composite = components_norm.mean(axis=1, skipna=True)
    # NaN only if ALL 5 components are NaN for that patient
    all_missing = components_norm.isna().all(axis=1)
    composite[all_missing] = np.nan

    stability_index[f'{var}_stability_index'] = composite

stability_index = stability_index.reset_index()

print(f"tpife_stability_index.csv shape: {stability_index.shape}")
print()

# ============================================================
# Coverage + distribution summary per variable, same format as before
# ============================================================

print("=" * 70)
print("PER-VARIABLE COMPOSITE STABILITY INDEX — coverage and distribution")
print("=" * 70)
summary_rows = []
for var in variables:
    col = f'{var}_stability_index'
    if col not in stability_index.columns:
        continue
    s = stability_index[col]
    n_covered = s.notna().sum()
    summary_rows.append({
        'variable': var,
        'coverage_n': n_covered,
        'coverage_pct': round(100 * n_covered / n_cohort, 1),
        'mean': round(s.mean(), 3) if n_covered > 0 else np.nan,
        'median': round(s.median(), 3) if n_covered > 0 else np.nan,
        'min': round(s.min(), 3) if n_covered > 0 else np.nan,
        'max': round(s.max(), 3) if n_covered > 0 else np.nan,
    })
summary_df = pd.DataFrame(summary_rows).sort_values('coverage_pct', ascending=False)
print(summary_df.to_string(index=False))
print()

stability_index.to_csv(processed_path + "tpife_stability_index.csv", index=False)
print(f"Saved tpife_stability_index.csv. Shape: {stability_index.shape}")

In [ ]:
import pandas as pd
import numpy as np

processed_path = "../data/processed/"
raw_path = "../data/raw/"

tpife = pd.read_csv(processed_path + "tpife_features.csv")
vitals_labs_24h = pd.read_csv(processed_path + "vitals_labs_24h.csv")
vitals_labs_24h['charttime'] = pd.to_datetime(vitals_labs_24h['charttime'])

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])
n_cohort = len(cohort_step1)
all_subjects = cohort_step1['subject_id']

WINDOW_END_HOURS = 24.0

# ============================================================
# Rebuild the exact same variable series (itemid mapping + GCS_Total + MAP)
# ============================================================

d_labitems = pd.read_csv(raw_path + "d_labitems.csv.gz", compression="gzip")
label_lookup = d_labitems.set_index('itemid')['label'].to_dict()

vitals_map = {
    220045: 'HR', 220050: 'SBP', 220179: 'SBP', 220051: 'DBP', 220180: 'DBP',
    220210: 'RR', 223761: 'Temp', 223762: 'Temp', 220277: 'SpO2',
    223900: 'GCS_Verbal', 220739: 'GCS_Eye', 223901: 'GCS_Motor',
    51301: 'WBC', 51300: 'WBC', 50813: 'Lactate', 52442: 'Lactate', 53154: 'Lactate',
    50912: 'Creatinine', 52546: 'Creatinine', 51265: 'Platelets', 53189: 'Platelets',
    50885: 'Bilirubin', 53089: 'Bilirubin', 50820: 'pH',
}
vitals_labs_24h['variable'] = vitals_labs_24h['itemid'].map(vitals_map)
unmapped_mask = vitals_labs_24h['variable'].isna()

def infer_supplemental_var(itemid):
    label = str(label_lookup.get(itemid, '')).lower()
    if 'sodium' in label: return 'Sodium'
    if 'potassium' in label: return 'Potassium'
    if 'hematocrit' in label: return 'Hematocrit'
    return None

vitals_labs_24h.loc[unmapped_mask, 'variable'] = vitals_labs_24h.loc[unmapped_mask, 'itemid'].apply(infer_supplemental_var)

data = vitals_labs_24h.merge(cohort_step1[['subject_id', 'intime']], on='subject_id', how='inner')
data['hours_since_intime'] = (data['charttime'] - data['intime']).dt.total_seconds() / 3600
data_24h = data[(data['hours_since_intime'] >= 0) & (data['hours_since_intime'] <= 24)].copy()

gcs_components = data_24h[data_24h['variable'].isin(['GCS_Eye', 'GCS_Verbal', 'GCS_Motor'])]
gcs_pivot = gcs_components.pivot_table(
    index=['subject_id', 'charttime', 'hours_since_intime'], columns='variable', values='valuenum', aggfunc='first'
).reset_index()
for col in ['GCS_Eye', 'GCS_Verbal', 'GCS_Motor']:
    if col not in gcs_pivot.columns:
        gcs_pivot[col] = np.nan
gcs_pivot['valuenum'] = gcs_pivot['GCS_Eye'] + gcs_pivot['GCS_Verbal'] + gcs_pivot['GCS_Motor']
gcs_total_series = gcs_pivot.dropna(subset=['valuenum'])[['subject_id', 'charttime', 'hours_since_intime', 'valuenum']].copy()
gcs_total_series['variable'] = 'GCS_Total'

sbp_series = data_24h[data_24h['variable'] == 'SBP'][['subject_id', 'charttime', 'hours_since_intime', 'valuenum']].rename(columns={'valuenum': 'sbp'})
dbp_series = data_24h[data_24h['variable'] == 'DBP'][['subject_id', 'charttime', 'hours_since_intime', 'valuenum']].rename(columns={'valuenum': 'dbp'})
map_merged = sbp_series.merge(dbp_series, on=['subject_id', 'charttime', 'hours_since_intime'], how='inner')
map_merged['valuenum'] = (map_merged['sbp'] + 2 * map_merged['dbp']) / 3
map_series = map_merged[['subject_id', 'charttime', 'hours_since_intime', 'valuenum']].copy()
map_series['variable'] = 'MAP'

data_24h_full = pd.concat([data_24h, gcs_total_series, map_series], ignore_index=True)

THRESHOLDS = {
    'HR': (60, 100), 'RR': (12, 20), 'SBP': (90, 130), 'DBP': (60, 80), 'MAP': (70, 100),
    'Temp': (36.1, 37.8), 'SpO2': (95, 100),
    'WBC': (4.5, 11.0), 'Lactate': (0.5, 2.0), 'Creatinine': (0.6, 1.3),
    'Platelets': (150, 400), 'Bilirubin': (0.1, 1.2),
    'Sodium': (135, 145), 'Potassium': (3.5, 5.0), 'pH': (7.35, 7.45),
    'Hematocrit': (36, 50), 'GCS_Total': (13, 15),
}
variables = list(THRESHOLDS.keys())

# ============================================================
# CORRECTED recovery-duration function: censored episodes contribute
# (window_end - episode_start) as a lower-bound duration
# ============================================================

def compute_recovery_durations_censoring_aware(hours, values, low, high, window_end=WINDOW_END_HOURS):
    n = len(hours)
    if n < 2:
        return {'completed_durations': [], 'censored_durations': [],
                'n_completed': 0, 'n_censored': 0}

    def classify(v):
        if v < low: return 'low'
        if v > high: return 'high'
        return 'normal'

    def frac_for_value(v0, v1, target):
        if v0 == v1:
            return None
        f = (target - v0) / (v1 - v0)
        return f if 0 < f < 1 else None

    episode_start_time = None
    completed_durations = []

    prev_t, prev_v = hours[0], values[0]
    prev_state = classify(prev_v)
    if prev_state != 'normal':
        episode_start_time = prev_t

    for i in range(1, n):
        t, v = hours[i], values[i]
        dt_total = t - prev_t
        if dt_total <= 0:
            prev_t, prev_v, prev_state = t, v, classify(v)
            continue

        breakpoints = sorted(filter(None.__ne__, [
            frac_for_value(prev_v, v, low), frac_for_value(prev_v, v, high)
        ]))
        fracs = [0.0] + breakpoints + [1.0]

        for j in range(len(fracs) - 1):
            f0, f1 = fracs[j], fracs[j + 1]
            fm = (f0 + f1) / 2
            vm = prev_v + fm * (v - prev_v)
            sub_state = classify(vm)
            t_mid_end = prev_t + f1 * dt_total

            if sub_state == 'normal' and episode_start_time is not None:
                completed_durations.append(t_mid_end - episode_start_time)
                episode_start_time = None
            elif sub_state != 'normal' and episode_start_time is None:
                episode_start_time = prev_t + f0 * dt_total

        prev_t, prev_v, prev_state = t, v, classify(v)

    censored_durations = []
    if episode_start_time is not None:
        # Episode still abnormal at last reading — censored at window_end,
        # NOT at the last reading time, since the true recovery point (if any)
        # is unknown anywhere between last reading and window_end.
        censored_durations.append(window_end - episode_start_time)

    return {'completed_durations': completed_durations,
            'censored_durations': censored_durations,
            'n_completed': len(completed_durations),
            'n_censored': len(censored_durations)}

# ============================================================
# Recompute OLD (excluded-censored) and NEW (censoring-aware) recovery averages
# for all 17 variables
# ============================================================

recovery_results = {}

for var in variables:
    var_data = data_24h_full[data_24h_full['variable'] == var]
    low, high = THRESHOLDS[var]
    per_patient = {}

    for subj_id, group in var_data.groupby('subject_id'):
        group_sorted = group.sort_values('hours_since_intime')
        hours = group_sorted['hours_since_intime'].values
        values = group_sorted['valuenum'].values

        result = compute_recovery_durations_censoring_aware(hours, values, low, high)

        completed = result['completed_durations']
        censored = result['censored_durations']
        all_durations = completed + censored  # censored contributes as lower-bound

        old_avg = np.mean(completed) if len(completed) > 0 else np.nan
        new_avg = np.mean(all_durations) if len(all_durations) > 0 else np.nan

        per_patient[subj_id] = {
            'avg_recovery_OLD_excl_censored': old_avg,
            'avg_recovery_NEW_incl_censored': new_avg,
            'n_completed': result['n_completed'],
            'n_censored': result['n_censored'],
        }

    recovery_results[var] = pd.DataFrame.from_dict(per_patient, orient='index')

print("Recovery duration recomputed (OLD excl-censored vs NEW incl-censored) for all 17 variables.")
print()

# ============================================================
# Side-by-side OLD vs NEW distribution comparison per variable
# ============================================================

print("=" * 100)
print("avg_recovery_duration_hrs — OLD (excluded censored) vs NEW (censored included as lower bound)")
print("=" * 100)

comparison_rows = []
for var in variables:
    df = recovery_results[var]
    old_col, new_col = df['avg_recovery_OLD_excl_censored'], df['avg_recovery_NEW_incl_censored']
    n_censored_patients = (df['n_censored'] > 0).sum()

    comparison_rows.append({
        'variable': var,
        'n_censored_patients': n_censored_patients,
        'OLD_mean': round(old_col.mean(), 2) if old_col.notna().sum() > 0 else np.nan,
        'NEW_mean': round(new_col.mean(), 2) if new_col.notna().sum() > 0 else np.nan,
        'OLD_median': round(old_col.median(), 2) if old_col.notna().sum() > 0 else np.nan,
        'NEW_median': round(new_col.median(), 2) if new_col.notna().sum() > 0 else np.nan,
        'OLD_coverage_n': old_col.notna().sum(),
        'NEW_coverage_n': new_col.notna().sum(),
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values('n_censored_patients', ascending=False)
print(comparison_df.to_string(index=False))
print()
print("NOTE: NEW_coverage_n should be >= OLD_coverage_n for every variable — patients who had")
print("ONLY a censored episode (no completed one) previously got NaN under OLD logic, but now")
print("get a real (lower-bound) value under NEW logic. Confirm this holds.")
print()

coverage_check = (comparison_df['NEW_coverage_n'] >= comparison_df['OLD_coverage_n']).all()
print(f"Coverage monotonicity check (NEW >= OLD for all variables): {coverage_check}")
print()

# ============================================================
# Recompute the Composite TPIFE Stability Index using CORRECTED recovery values
# ============================================================

def minmax_normalize(series):
    valid = series.dropna()
    if len(valid) == 0 or valid.min() == valid.max():
        return pd.Series(np.nan, index=series.index)
    return (series - valid.min()) / (valid.max() - valid.min())

stability_index = pd.DataFrame({'subject_id': all_subjects}).set_index('subject_id')

for var in variables:
    instability_col = f'{var}_instability_score'
    time_abn_col = f'{var}_time_abnormal_hrs'
    if instability_col not in tpife.columns or time_abn_col not in tpife.columns:
        continue

    tpife_indexed = tpife.set_index('subject_id')
    instability = tpife_indexed[instability_col]
    time_abnormal = tpife_indexed[time_abn_col]

    var_data = data_24h_full[data_24h_full['variable'] == var]
    monitored_duration = var_data.groupby('subject_id')['hours_since_intime'].apply(
        lambda s: s.max() - s.min() if len(s) >= 2 else np.nan
    )

    low, high = THRESHOLDS[var]
    max_dev_dict = {}
    for subj_id, group in var_data.groupby('subject_id'):
        vals = group['valuenum'].values
        deviations = [max(low - v, 0) if v < low else max(v - high, 0) if v > high else 0 for v in vals]
        abnormal_devs = [d for d in deviations if d > 0]
        max_dev_dict[subj_id] = max(abnormal_devs) if abnormal_devs else (0.0 if len(vals) > 0 else np.nan)
    max_deviation = pd.Series(max_dev_dict)

    avg_recovery_new = recovery_results[var]['avg_recovery_NEW_incl_censored']

    oscillation_freq = instability / monitored_duration.replace(0, np.nan)

    components_raw = pd.DataFrame({
        'instability_score': instability,
        'oscillation_frequency': oscillation_freq,
        'time_abnormal_hrs': time_abnormal,
        'max_deviation': max_deviation,
        'avg_recovery_duration_hrs': avg_recovery_new,
    })

    components_norm = pd.DataFrame(index=components_raw.index)
    for col in components_raw.columns:
        components_norm[col] = minmax_normalize(components_raw[col])

    composite = components_norm.mean(axis=1, skipna=True)
    all_missing = components_norm.isna().all(axis=1)
    composite[all_missing] = np.nan

    stability_index[f'{var}_stability_index'] = composite

stability_index = stability_index.reset_index()

# ============================================================
# Updated coverage + distribution summary
# ============================================================

print("=" * 70)
print("UPDATED Composite TPIFE Stability Index — coverage and distribution")
print("(using censoring-aware recovery duration)")
print("=" * 70)
summary_rows = []
for var in variables:
    col = f'{var}_stability_index'
    if col not in stability_index.columns:
        continue
    s = stability_index[col]
    n_covered = s.notna().sum()
    summary_rows.append({
        'variable': var, 'coverage_n': n_covered,
        'coverage_pct': round(100 * n_covered / n_cohort, 1),
        'mean': round(s.mean(), 3) if n_covered > 0 else np.nan,
        'median': round(s.median(), 3) if n_covered > 0 else np.nan,
        'min': round(s.min(), 3) if n_covered > 0 else np.nan,
        'max': round(s.max(), 3) if n_covered > 0 else np.nan,
    })
summary_df = pd.DataFrame(summary_rows).sort_values('coverage_pct', ascending=False)
print(summary_df.to_string(index=False))
print()

stability_index.to_csv(processed_path + "tpife_stability_index.csv", index=False)
print(f"Saved tpife_stability_index.csv. Shape: {stability_index.shape}")

In [ ]:
import pandas as pd
import numpy as np

processed_path = "../data/processed/"

static_features = pd.read_csv(processed_path + "static_features.csv")
basic_features = pd.read_csv(processed_path + "basic_features.csv")
sofa_scores = pd.read_csv(processed_path + "sofa_scores.csv")
tpife = pd.read_csv(processed_path + "tpife_features.csv")
organ_burden_24h = pd.read_csv(processed_path + "organ_burden_inputs_24h.csv")
organ_burden_24h['charttime'] = pd.to_datetime(organ_burden_24h['charttime'])

cohort_step1 = pd.read_csv(processed_path + "cohort_step1.csv")
cohort_step1['intime'] = pd.to_datetime(cohort_step1['intime'])
n_cohort = len(cohort_step1)
all_subjects = cohort_step1['subject_id']

print(f"static_features.csv columns: {static_features.columns.tolist()}")
print()

sirs_exists = any('sirs' in c.lower() for c in static_features.columns)
print(f"SIRS already present in static_features.csv: {sirs_exists}")
print()

# ============================================================
# STEP 1: Compute SIRS score (0-4) if not already present
# ============================================================

print("=" * 70)
print("SIRS SCORE FORMULA (before computing)")
print("=" * 70)
print("""
SIRS criterion met per patient if their WORST first-24h value satisfies:
  1. Temperature:  Temp_max > 38  OR  Temp_min < 36
  2. Heart Rate:    HR_max > 90
  3. Resp Rate:     RR_max > 20
  4. WBC:           WBC_max > 12  OR  WBC_min < 4

SIRS_score = count of the 4 criteria met (0-4).
A criterion is NaN (not "not met") if its underlying basic_features value is
missing — SIRS_score for a patient is NaN only if ALL 4 criteria are NaN
(i.e., zero physiology data at all); otherwise it's the count of met criteria
among whatever was available (unmet-due-to-missing != unmet-due-to-normal,
flagged separately via n_sirs_criteria_evaluable).
""")

def sirs_temp(row):
    tmax, tmin = row.get('Temp_max'), row.get('Temp_min')
    if pd.isna(tmax) and pd.isna(tmin):
        return np.nan
    met = (not pd.isna(tmax) and tmax > 38) or (not pd.isna(tmin) and tmin < 36)
    return met

def sirs_hr(row):
    hmax = row.get('HR_max')
    if pd.isna(hmax):
        return np.nan
    return hmax > 90

def sirs_rr(row):
    rmax = row.get('RR_max')
    if pd.isna(rmax):
        return np.nan
    return rmax > 20

def sirs_wbc(row):
    wmax, wmin = row.get('WBC_max'), row.get('WBC_min')
    if pd.isna(wmax) and pd.isna(wmin):
        return np.nan
    met = (not pd.isna(wmax) and wmax > 12) or (not pd.isna(wmin) and wmin < 4)
    return met

basic_indexed = basic_features.set_index('subject_id')

sirs_components = pd.DataFrame(index=basic_indexed.index)
sirs_components['temp_crit'] = basic_indexed.apply(sirs_temp, axis=1)
sirs_components['hr_crit'] = basic_indexed.apply(sirs_hr, axis=1)
sirs_components['rr_crit'] = basic_indexed.apply(sirs_rr, axis=1)
sirs_components['wbc_crit'] = basic_indexed.apply(sirs_wbc, axis=1)

sirs_components['n_evaluable'] = sirs_components[['temp_crit', 'hr_crit', 'rr_crit', 'wbc_crit']].notna().sum(axis=1)
sirs_components['SIRS_score'] = sirs_components[['temp_crit', 'hr_crit', 'rr_crit', 'wbc_crit']].sum(axis=1, skipna=True)
sirs_components.loc[sirs_components['n_evaluable'] == 0, 'SIRS_score'] = np.nan

print(f"SIRS_score computed for {sirs_components['SIRS_score'].notna().sum()} / {n_cohort} patients")
print(f"SIRS_score value_counts():")
print(sirs_components['SIRS_score'].value_counts(dropna=False).sort_index())
print(f"\nPatients with all 4 criteria evaluable: {(sirs_components['n_evaluable']==4).sum()}")
print(f"Patients with partial criteria (1-3 evaluable): "
      f"{((sirs_components['n_evaluable']>0) & (sirs_components['n_evaluable']<4)).sum()}")
print()

# ============================================================
# STEP 2 formula preview: SIRS_score x sofa_total
# ============================================================

print("=" * 70)
print("INTERACTION FEATURE FORMULAS (review before computing)")
print("=" * 70)
print("""
1. SIRS_x_SOFA = SIRS_score * sofa_total
   (both already 0-N integer scores; simple product captures compounded
   inflammatory + organ-dysfunction burden)

2. Hypotension_x_RisingLactate = MAP_time_abnormal_hrs * Lactate_auc_over_threshold
   - Using MAP (not SBP) as the primary hypotension signal, since MAP is the
     more standard hemodynamic-adequacy metric (matches SOFA cardiovascular
     component's own use of MAP). SBP could be substituted if you prefer.
   - Using Lactate_auc_over_threshold (not early_late_trend) as the "rising
     lactate" signal: AUC captures cumulative severity of elevated lactate,
     which is a more standard "lactate burden" concept than early_late_trend
     (which only compares two 12h halves and can be small/misleading with
     sparse lactate draws, given lactate's low first-24h coverage). Flag if
     you'd rather use early_late_trend specifically for directional "rising".

3. Hypoxemia_x_Tachycardia = SpO2_time_abnormal_hrs * HR_time_abnormal_hrs
   (both duration-based, directly comparable units — hours abnormal x hours
   abnormal — capturing sustained co-occurrence of low O2 and high HR, a
   classic compensatory-physiology interaction)

4. RisingCreatinine_x_FallingUrineOutput = Creatinine_early_late_trend * UO_early_late_trend
   - Creatinine_early_late_trend: already in tpife_features.csv (mean first 12h
     minus mean last 12h). NOTE: since "rising" creatinine over time means
     LATER values are HIGHER, and early_late_trend = early_mean - late_mean,
     a RISING creatinine trend produces a NEGATIVE early_late_trend value.
   - UO_early_late_trend: NOT yet computed — building this now as
     (mean urine output rate in 0-12h) - (mean urine output rate in 12-24h),
     using organ_burden_inputs_24h.csv raw per-event volumes. A FALLING UO
     (worsening) also produces a POSITIVE UO_early_late_trend (early higher
     than late).
   - Because rising creatinine -> negative trend, and falling UO -> positive
     trend, their PRODUCT is NEGATIVE when both are worsening together. This
     is a real interpretability quirk: a MORE NEGATIVE value = WORSE renal
     trajectory here, not a more positive one, unlike the first 3 interaction
     terms where higher = worse. Flagging this explicitly so it isn't
     misread during modeling.
""")
print()

# ============================================================
# Compute urine output early_late_trend (per-event granularity, not TPIFE-style)
# ============================================================

urine_data = organ_burden_24h[organ_burden_24h['itemid'].isin({226559, 226560})].copy()
urine_data = urine_data.merge(cohort_step1[['subject_id', 'intime']], on='subject_id', how='inner')
urine_data['hours_since_intime'] = (urine_data['charttime'] - urine_data['intime']).dt.total_seconds() / 3600
urine_data_24h = urine_data[(urine_data['hours_since_intime'] >= 0) & (urine_data['hours_since_intime'] <= 24)]

def compute_uo_trend(group):
    early = group[group['hours_since_intime'] <= 12]['valuenum']
    late = group[group['hours_since_intime'] > 12]['valuenum']
    if len(early) == 0 or len(late) == 0:
        return np.nan
    return early.mean() - late.mean()

uo_trend = urine_data_24h.groupby('subject_id').apply(compute_uo_trend)
uo_trend.name = 'UO_early_late_trend'

print(f"UO_early_late_trend computed for {uo_trend.notna().sum()} / {n_cohort} patients "
      f"(requires >=1 urine reading in EACH of 0-12h and 12-24h windows)")
print()

# ============================================================
# Assemble all interaction features
# ============================================================

tpife_indexed = tpife.set_index('subject_id')
sofa_indexed = sofa_scores.set_index('subject_id')

interaction = pd.DataFrame({'subject_id': all_subjects}).set_index('subject_id')

# 1. SIRS x SOFA
interaction['SIRS_score'] = sirs_components['SIRS_score']
interaction['sofa_total'] = sofa_indexed['sofa_total']
interaction['SIRS_x_SOFA'] = interaction['SIRS_score'] * interaction['sofa_total']

# 2. Hypotension x rising lactate
interaction['MAP_time_abnormal_hrs'] = np.nan  # MAP wasn't in original tpife_features.csv build unless added
if 'MAP_time_abnormal_hrs' in tpife.columns:
    interaction['MAP_time_abnormal_hrs'] = tpife_indexed['MAP_time_abnormal_hrs']
else:
    print("NOTE: 'MAP_time_abnormal_hrs' not found in tpife_features.csv — check whether MAP")
    print("was included as a variable in that build. Falling back to SBP_time_abnormal_hrs.")
    interaction['MAP_time_abnormal_hrs'] = tpife_indexed.get('SBP_time_abnormal_hrs', np.nan)

interaction['Lactate_auc_over_threshold'] = tpife_indexed.get('Lactate_auc_over_threshold', np.nan)
interaction['Hypotension_x_RisingLactate'] = interaction['MAP_time_abnormal_hrs'] * interaction['Lactate_auc_over_threshold']

# 3. Hypoxemia x tachycardia
interaction['SpO2_time_abnormal_hrs'] = tpife_indexed.get('SpO2_time_abnormal_hrs', np.nan)
interaction['HR_time_abnormal_hrs'] = tpife_indexed.get('HR_time_abnormal_hrs', np.nan)
interaction['Hypoxemia_x_Tachycardia'] = interaction['SpO2_time_abnormal_hrs'] * interaction['HR_time_abnormal_hrs']

# 4. Rising creatinine x falling UO
interaction['Creatinine_early_late_trend'] = tpife_indexed.get('Creatinine_early_late_trend', np.nan)
interaction['UO_early_late_trend'] = uo_trend
interaction['RisingCreatinine_x_FallingUO'] = interaction['Creatinine_early_late_trend'] * interaction['UO_early_late_trend']

interaction = interaction.reset_index()

# ============================================================
# Coverage % and distribution for the 4 FINAL interaction features
# ============================================================

final_interaction_cols = ['SIRS_x_SOFA', 'Hypotension_x_RisingLactate',
                           'Hypoxemia_x_Tachycardia', 'RisingCreatinine_x_FallingUO']

print("=" * 70)
print("INTERACTION FEATURE COVERAGE AND DISTRIBUTION")
print("=" * 70)

summary_rows = []
for col in final_interaction_cols:
    s = interaction[col]
    n_covered = s.notna().sum()
    summary_rows.append({
        'feature': col,
        'coverage_n': n_covered,
        'coverage_pct': round(100 * n_covered / n_cohort, 1),
        'mean': round(s.mean(), 3) if n_covered > 0 else np.nan,
        'median': round(s.median(), 3) if n_covered > 0 else np.nan,
        'min': round(s.min(), 3) if n_covered > 0 else np.nan,
        'max': round(s.max(), 3) if n_covered > 0 else np.nan,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))
print()

print("NOTE: each interaction feature is NaN whenever EITHER of its two components is NaN")
print("(a strict AND-availability requirement, not imputed or defaulted) — this is why")
print("coverage here will generally be LOWER than either individual component's own coverage,")
print("especially for Hypotension_x_RisingLactate (Lactate has known low first-24h coverage)")
print("and RisingCreatinine_x_FallingUO (requires urine output readings spanning both halves")
print("of the 24h window, a stricter joint requirement).")
print()

# ============================================================
# Save — include SIRS_score and its component/evaluability columns too, for transparency
# ============================================================

output_df = interaction.merge(
    sirs_components[['n_evaluable']].rename(columns={'n_evaluable': 'SIRS_n_criteria_evaluable'}),
    left_on='subject_id', right_index=True, how='left'
)

output_df.to_csv(processed_path + "interaction_features.csv", index=False)
print(f"Saved interaction_features.csv. Shape: {output_df.shape}")
print(f"Columns: {output_df.columns.tolist()}")

In [ ]:
import pandas as pd
import numpy as np

processed_path = "../data/processed/"

interaction_features = pd.read_csv(processed_path + "interaction_features.csv")

print(f"interaction_features.csv loaded: {interaction_features.shape}")
print(f"Columns: {interaction_features.columns.tolist()}")
print()

required_cols = ['Creatinine_early_late_trend', 'UO_early_late_trend', 'RisingCreatinine_x_FallingUO']
missing = [c for c in required_cols if c not in interaction_features.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing} — cannot recompute without them.")

# ============================================================
# DECISION: keeping the SAME column name (RisingCreatinine_x_FallingUO)
# for simplicity, but overwriting its values with the corrected sign
# convention. Documenting explicitly here and in a comment column so
# the fix isn't silently invisible to future-you.
# ============================================================

old_values = interaction_features['RisingCreatinine_x_FallingUO'].copy()

interaction_features['RisingCreatinine_x_FallingUO'] = (
    -1 * interaction_features['Creatinine_early_late_trend']
) * interaction_features['UO_early_late_trend']

print("=" * 70)
print("SIGN CONVENTION FIX APPLIED")
print("=" * 70)
print("Column 'RisingCreatinine_x_FallingUO' now = (-1 * Creatinine_early_late_trend) * UO_early_late_trend")
print("Convention: HIGHER value = WORSE renal trajectory (rising creatinine AND falling UO together),")
print("consistent with SIRS_x_SOFA, Hypotension_x_RisingLactate, and Hypoxemia_x_Tachycardia.")
print("(Previously, this column had the OPPOSITE convention: more negative = worse.)")
print()

# ============================================================
# Verify: patients with clearly RISING creatinine AND clearly FALLING UO
# should now show a POSITIVE value. Spot-check 2-3 such patients manually.
# ============================================================

# "Clearly rising creatinine" = original Creatinine_early_late_trend is clearly negative
# (since early_late_trend = early_mean - late_mean, rising -> negative)
# "Clearly falling UO" = UO_early_late_trend is clearly positive
# (early_mean - late_mean, falling -> positive)

candidates = interaction_features[
    (interaction_features['Creatinine_early_late_trend'] < -0.3) &
    (interaction_features['UO_early_late_trend'] > 50)
].copy()

candidates = candidates.sort_values('RisingCreatinine_x_FallingUO', ascending=False)

print(f"Patients with clearly rising creatinine (trend < -0.3) AND clearly falling UO "
      f"(trend > 50): {len(candidates)}")
print()

if len(candidates) == 0:
    print("No patients meet BOTH strict criteria simultaneously — loosening thresholds to find "
          "spot-check examples instead.")
    candidates = interaction_features[
        (interaction_features['Creatinine_early_late_trend'] < 0) &
        (interaction_features['UO_early_late_trend'] > 0)
    ].sort_values('RisingCreatinine_x_FallingUO', ascending=False)
    print(f"Patients with ANY rising creatinine AND ANY falling UO: {len(candidates)}")
    print()

print("=" * 70)
print("SPOT-CHECK: 3 example patients (rising creatinine + falling UO)")
print("=" * 70)
spot_check_cols = ['subject_id', 'Creatinine_early_late_trend', 'UO_early_late_trend',
                    'RisingCreatinine_x_FallingUO']
print(candidates[spot_check_cols].head(3).to_string(index=False))
print()

for _, row in candidates.head(3).iterrows():
    creat_trend = row['Creatinine_early_late_trend']
    uo_trend = row['UO_early_late_trend']
    new_val = row['RisingCreatinine_x_FallingUO']
    is_positive = new_val > 0
    print(f"subject_id={row['subject_id']}: Creatinine_early_late_trend={creat_trend:.3f} "
          f"(negative = rising, as expected: {creat_trend < 0}), "
          f"UO_early_late_trend={uo_trend:.1f} (positive = falling, as expected: {uo_trend > 0}), "
          f"-> RisingCreatinine_x_FallingUO={new_val:.3f} "
          f"({'CORRECT: positive, higher=worse' if is_positive else 'UNEXPECTED: still negative — investigate'})")
print()

verification_passed = (candidates.head(3)['RisingCreatinine_x_FallingUO'] > 0).all() if len(candidates) > 0 else None
print(f"Verification (all 3 spot-checked patients show POSITIVE value): {verification_passed}")
print()

# ============================================================
# Updated distribution for the corrected feature
# ============================================================

s = interaction_features['RisingCreatinine_x_FallingUO']
n_covered = s.notna().sum()
n_cohort = len(interaction_features)

print("=" * 70)
print("UPDATED DISTRIBUTION — RisingCreatinine_x_FallingUO (corrected sign)")
print("=" * 70)
print(f"Coverage: {n_covered} / {n_cohort} ({round(100*n_covered/n_cohort, 1)}%) — unchanged from before,")
print("since only the SIGN changed, not which patients have a value.")
print(f"Mean:   {s.mean():.3f}")
print(f"Median: {s.median():.3f}")
print(f"Min:    {s.min():.3f}")
print(f"Max:    {s.max():.3f}")
print()

# Compare old vs new sign distribution to confirm the flip actually happened globally
n_sign_flipped = ((old_values > 0) & (s < 0)).sum() + ((old_values < 0) & (s > 0)).sum()
n_same_sign = ((old_values > 0) & (s > 0)).sum() + ((old_values < 0) & (s < 0)).sum()
n_both_nan = (old_values.isna() & s.isna()).sum()
print(f"Sanity check — rows where sign flipped vs old version: {n_sign_flipped}")
print(f"Rows where sign stayed the same (expected only where one factor was exactly 0): {n_same_sign}")
print(f"Rows both NaN (unaffected): {n_both_nan}")
print()

# ============================================================
# Re-save
# ============================================================

interaction_features.to_csv(processed_path + "interaction_features.csv", index=False)
print(f"Saved interaction_features.csv with corrected RisingCreatinine_x_FallingUO. "
      f"Shape: {interaction_features.shape}")

In [ ]:
import pandas as pd

processed_path = "../data/processed/"

tpife = pd.read_csv(processed_path + "tpife_features.csv")
interaction_features = pd.read_csv(processed_path + "interaction_features.csv")

print(f"tpife_features.csv loaded: {tpife.shape}")
print(f"interaction_features.csv loaded: {interaction_features.shape}")
print()

# ============================================================
# STEP 1: Definitively check whether MAP exists in tpife_features.csv
# ============================================================

map_cols = [c for c in tpife.columns if c.startswith('MAP_')]
sbp_cols = [c for c in tpife.columns if c.startswith('SBP_')]

print("=" * 70)
print("CHECKING: does 'MAP' exist as a variable in tpife_features.csv?")
print("=" * 70)
print(f"Columns starting with 'MAP_': {map_cols if map_cols else 'NONE FOUND'}")
print(f"Columns starting with 'SBP_': {sbp_cols if sbp_cols else 'NONE FOUND'}")
print()

map_exists = len(map_cols) > 0

if map_exists:
    print("RESULT: MAP columns EXIST in tpife_features.csv. The fallback to SBP should NOT")
    print("have triggered. Checking interaction_features.csv to confirm which one was actually used...")
else:
    print("RESULT: MAP columns DO NOT exist in tpife_features.csv (confirmed — MAP was never")
    print("computed as a standalone TPIFE variable, only used internally for SOFA/APACHE-II worst-value")
    print("scoring, not run through the full TPIFE abnormal-episode/instability pipeline). "
          "The fallback to SBP DEFINITELY triggered when interaction_features.csv was built.")
print()

# ============================================================
# STEP 2: Confirm what interaction_features.csv actually contains
# ============================================================

print("=" * 70)
print("interaction_features.csv column check")
print("=" * 70)
existing_cols = interaction_features.columns.tolist()
print(f"Full column list: {existing_cols}")
print()

has_mislabeled_map_col = 'MAP_time_abnormal_hrs' in existing_cols

if has_mislabeled_map_col:
    # Verify the actual VALUES in that column match SBP_time_abnormal_hrs from tpife_features.csv,
    # not some other data, before renaming — don't just trust the column name history
    if 'SBP_time_abnormal_hrs' in tpife.columns:
        tpife_sbp = tpife.set_index('subject_id')['SBP_time_abnormal_hrs']
        interaction_indexed = interaction_features.set_index('subject_id')
        current_col_vals = interaction_indexed['MAP_time_abnormal_hrs']

        # Compare, allowing for NaN-NaN matches
        comparison = pd.DataFrame({'current': current_col_vals, 'sbp_source': tpife_sbp}).dropna()
        matches = (comparison['current'] == comparison['sbp_source']).all() if len(comparison) > 0 else None

        print(f"Values in interaction_features.csv's 'MAP_time_abnormal_hrs' column match "
              f"tpife_features.csv's 'SBP_time_abnormal_hrs' values: {matches}")
        print(f"(Checked across {len(comparison)} non-NaN overlapping patients)")
        print()

        if matches:
            print("CONFIRMED: the fallback to SBP definitely triggered, and the column was")
            print("mislabeled as 'MAP_time_abnormal_hrs' when it actually holds SBP data. Renaming now.")
        else:
            print("WARNING: values do NOT match SBP data either — something else may have happened.")
            print("STOPPING before renaming — investigate manually rather than blindly relabeling.")
    else:
        print("WARNING: 'SBP_time_abnormal_hrs' also not found in tpife_features.csv — cannot verify")
        print("the fallback source. Stopping before renaming.")
        matches = False
else:
    print("'MAP_time_abnormal_hrs' column not found in interaction_features.csv at all — "
          "nothing to rename, or a different column name was used originally.")
    matches = False

print()

# ============================================================
# STEP 3: Rename if confirmed
# ============================================================

if has_mislabeled_map_col and matches:
    interaction_features = interaction_features.rename(columns={
        'MAP_time_abnormal_hrs': 'SBP_time_abnormal_hrs',
        'Hypotension_x_RisingLactate': 'Hypotension_SBP_x_RisingLactate'
    })

    print("Renamed columns:")
    print("  'MAP_time_abnormal_hrs' -> 'SBP_time_abnormal_hrs'")
    print("  'Hypotension_x_RisingLactate' -> 'Hypotension_SBP_x_RisingLactate'")
    print()

    interaction_features.to_csv(processed_path + "interaction_features.csv", index=False)
    print(f"Saved interaction_features.csv with corrected column names.")
    print(f"Updated columns: {interaction_features.columns.tolist()}")
else:
    print("NO RENAME PERFORMED. Either MAP was genuinely available and used correctly, or the")
    print("value-match verification failed/was inconclusive — review the output above before")
    print("taking any action manually.")

In [ ]:
import pandas as pd

processed_path = "../data/processed/"

tpife = pd.read_csv(processed_path + "tpife_features.csv")
interaction_features = pd.read_csv(processed_path + "interaction_features.csv")

print(f"tpife_features.csv loaded: {tpife.shape}")
print(f"interaction_features.csv loaded: {interaction_features.shape}")
print()

# ============================================================
# STEP 1: Does tpife_features.csv even HAVE a MAP_time_abnormal_hrs column?
# This must be checked first — comparing against a nonexistent column
# would either error or silently produce all-NaN, neither of which is
# a real answer.
# ============================================================

tpife_has_map_col = 'MAP_time_abnormal_hrs' in tpife.columns
interaction_has_map_col = 'MAP_time_abnormal_hrs' in interaction_features.columns

print(f"'MAP_time_abnormal_hrs' exists in tpife_features.csv: {tpife_has_map_col}")
print(f"'MAP_time_abnormal_hrs' exists in interaction_features.csv: {interaction_has_map_col}")
print()

if not tpife_has_map_col:
    print("STOPPING THE DIRECT COMPARISON HERE: tpife_features.csv has NO 'MAP_time_abnormal_hrs'")
    print("column at all — MAP was never computed as a full TPIFE variable in that file (confirmed")
    print("in the previous check too). There is nothing to merge/compare against on the MAP side,")
    print("so 'do the values match MAP' is not answerable — the honest answer is: MAP TPIFE stats")
    print("simply don't exist in tpife_features.csv, full stop.")
    print()
    print("Given this, the value currently sitting in interaction_features.csv's")
    print("'MAP_time_abnormal_hrs' column MUST have come from somewhere else. Showing the raw")
    print("side-by-side comparison against SBP (the only thing that COULD have produced it) below,")
    print("exactly as requested, so the actual source is traceable rather than assumed.")
    print()

    if not interaction_has_map_col:
        print("interaction_features.csv ALSO doesn't have this column (already renamed in the")
        print("previous step, most likely). Nothing further to trace — check the current column")
        print(f"list instead: {interaction_features.columns.tolist()}")
    else:
        merged = interaction_features[['subject_id', 'MAP_time_abnormal_hrs']].merge(
            tpife[['subject_id'] + ([c for c in ['SBP_time_abnormal_hrs'] if c in tpife.columns])],
            on='subject_id', how='left'
        )
        sample = merged.dropna(subset=['MAP_time_abnormal_hrs']).head(10)
        print("Sample comparison (interaction_features.csv's MAP_time_abnormal_hrs vs "
              "tpife_features.csv's SBP_time_abnormal_hrs — the only real candidate source):")
        print(sample.to_string(index=False))

else:
    # ============================================================
    # tpife_features.csv DOES have a real MAP_time_abnormal_hrs — do the direct merge/compare
    # ============================================================

    merged = interaction_features[['subject_id', 'MAP_time_abnormal_hrs']].rename(
        columns={'MAP_time_abnormal_hrs': 'interaction_MAP_value'}
    ).merge(
        tpife[['subject_id', 'MAP_time_abnormal_hrs']].rename(
            columns={'MAP_time_abnormal_hrs': 'tpife_MAP_value'}
        ),
        on='subject_id', how='left'
    )

    also_sbp = 'SBP_time_abnormal_hrs' in tpife.columns
    if also_sbp:
        merged = merged.merge(
            tpife[['subject_id', 'SBP_time_abnormal_hrs']].rename(
                columns={'SBP_time_abnormal_hrs': 'tpife_SBP_value'}
            ),
            on='subject_id', how='left'
        )

    overlap = merged.dropna(subset=['interaction_MAP_value', 'tpife_MAP_value'])
    print(f"Non-NaN overlap for direct MAP-vs-MAP comparison: {len(overlap)} patients")

    if len(overlap) > 0:
        matches = (overlap['interaction_MAP_value'] == overlap['tpife_MAP_value']).all()
        n_matching = (overlap['interaction_MAP_value'] == overlap['tpife_MAP_value']).sum()
        print(f"Values match exactly across the full overlap: {matches}")
        print(f"Matching rows: {n_matching} / {len(overlap)}")
        print()

        if matches:
            print("=" * 70)
            print("RESULT: CONFIRMED MATCH. MAP was used correctly all along.")
            print("=" * 70)
            print("The earlier SBP-comparison in the previous cell was checking the wrong column")
            print("entirely — it never actually disproved MAP usage, it just happened to ALSO match")
            print("SBP by coincidence (or the comparison logic there had a flaw). No rename needed.")
            print("interaction_features.csv is left UNCHANGED.")
        else:
            print("Values exist in both places but DON'T match exactly — showing raw side-by-side")
            print("sample below to trace the actual discrepancy:")
            print()
            sample_cols = ['subject_id', 'tpife_MAP_value', 'interaction_MAP_value']
            if also_sbp:
                sample_cols.insert(2, 'tpife_SBP_value')
            print(overlap[sample_cols].head(10).to_string(index=False))
    else:
        print("No overlapping non-NaN values found between the two MAP columns — cannot confirm")
        print("a match either way from this comparison alone. Printing raw values from both sources")
        print("for whatever rows exist, unfiltered:")
        print()
        sample_cols = ['subject_id', 'tpife_MAP_value', 'interaction_MAP_value']
        if also_sbp:
            sample_cols.insert(2, 'tpife_SBP_value')
        print(merged[sample_cols].head(10).to_string(index=False))

In [ ]:
import pandas as pd

processed_path = "../data/processed/"

# ============================================================
# STEP 1: Load every source file, print row counts BEFORE merging
# ============================================================

files = {
    'cohort_step1': "cohort_step1.csv",
    'static_features': "static_features.csv",
    'basic_features': "basic_features.csv",
    'tpife_features': "tpife_features.csv",
    'sofa_scores': "sofa_scores.csv",
    'apache_ii_scores': "apache_ii_scores.csv",
    'organ_burden_index': "organ_burden_index.csv",
    'tpife_stability_index': "tpife_stability_index.csv",
    'interaction_features': "interaction_features.csv",
}

dfs = {}
print("=" * 70)
print("SOURCE FILE ROW COUNTS (confirm all = 3122 before assuming outer==inner)")
print("=" * 70)
for name, fname in files.items():
    df = pd.read_csv(processed_path + fname)
    dfs[name] = df
    n_unique_subj = df['subject_id'].nunique() if 'subject_id' in df.columns else 'NO subject_id COLUMN'
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} cols, "
          f"{n_unique_subj} unique subject_id "
          f"{'(HAS DUPLICATES!)' if isinstance(n_unique_subj, int) and n_unique_subj != df.shape[0] else ''}")
print()

all_3122 = all(dfs[name].shape[0] == 3122 for name in dfs)
print(f"ALL files have exactly 3122 rows: {all_3122}")
if not all_3122:
    print("WARNING: not every file has 3122 rows — outer join will be REQUIRED (not just safe)")
    print("to avoid silently dropping patients. Proceeding with outer join regardless, per instruction.")
print()

# ============================================================
# STEP 2: Select only cohort_step1's specified subset of columns
# ============================================================

cohort_subset = dfs['cohort_step1'][
    ['subject_id', 'mortality_28d', 'immortal_time_flag', 'icu_admission_year',
     'cancer_type', 'delirium_subtype']
].copy()

dfs['cohort_step1'] = cohort_subset

# ============================================================
# STEP 3: Detect column-name collisions BEFORE merging
# ============================================================

print("=" * 70)
print("COLUMN COLLISION CHECK (excluding 'subject_id', which is the merge key)")
print("=" * 70)

all_columns_seen = {}
for name, df in dfs.items():
    for col in df.columns:
        if col == 'subject_id':
            continue
        all_columns_seen.setdefault(col, []).append(name)

collisions = {col: sources for col, sources in all_columns_seen.items() if len(sources) > 1}

if collisions:
    print(f"COLLISIONS FOUND ({len(collisions)} column names appear in multiple files):")
    for col, sources in collisions.items():
        print(f"  '{col}' appears in: {sources}")
    print()
else:
    print("No column-name collisions found across any of the 9 files. Safe to merge as-is.")
    print()

# ============================================================
# STEP 4: Resolve collisions explicitly (prefix with source file name)
# rather than letting pandas silently auto-suffix with _x/_y
# ============================================================

if collisions:
    print("RESOLVING: renaming each colliding column to '<source_file>__<original_name>'")
    print("so provenance stays explicit and unambiguous (not a generic _x/_y suffix).")
    print()
    for col, sources in collisions.items():
        for src in sources:
            new_name = f"{src}__{col}"
            dfs[src] = dfs[src].rename(columns={col: new_name})
            print(f"  {src}: '{col}' -> '{new_name}'")
    print()

# ============================================================
# STEP 5: Merge all 9 tables on subject_id, outer join
# ============================================================

merge_order = list(dfs.keys())
merged = dfs[merge_order[0]]

for name in merge_order[1:]:
    before_rows = len(merged)
    merged = merged.merge(dfs[name], on='subject_id', how='outer')
    after_rows = len(merged)
    if after_rows != before_rows:
        print(f"NOTE: row count changed after merging '{name}': {before_rows} -> {after_rows} "
              f"(outer join found subject_ids not previously seen)")

print()
print(f"Final merged shape: {merged.shape}")
print(f"Expected: (3122, N) — row count matches 3122: {merged.shape[0] == 3122}")
print()

if merged.shape[0] != 3122:
    extra_or_missing = set(merged['subject_id']) ^ set(dfs['cohort_step1']['subject_id'])
    print(f"WARNING: subject_id mismatch detected. Symmetric difference: {len(extra_or_missing)} IDs")
    print(f"Sample: {list(extra_or_missing)[:10]}")
print()

# ============================================================
# STEP 6: Save raw (un-imputed) feature table
# ============================================================

merged.to_csv(processed_path + "feature_table_raw.csv", index=False)
print(f"Saved feature_table_raw.csv. Shape: {merged.shape}")
print()

# ============================================================
# STEP 7: Missingness % per column, sorted highest to lowest
# ============================================================

n_total = len(merged)
missingness = merged.isna().sum().sort_values(ascending=False)
missingness_pct = (100 * missingness / n_total).round(2)

missingness_df = pd.DataFrame({
    'column': missingness.index,
    'n_missing': missingness.values,
    'pct_missing': missingness_pct.values
})

print("=" * 70)
print(f"MISSINGNESS % PER COLUMN (all {len(missingness_df)} columns, sorted highest to lowest)")
print("=" * 70)
pd.set_option('display.max_rows', None)
print(missingness_df.to_string(index=False))
pd.reset_option('display.max_rows')
print()

n_over_20pct = (missingness_df['pct_missing'] > 20).sum()
print(f"Columns with >20% missing (candidates for the 20% drop rule): {n_over_20pct} / {len(missingness_df)}")

In [ ]:
import pandas as pd
import numpy as np
import re

processed_path = "../data/processed/"

feature_table_raw = pd.read_csv(processed_path + "feature_table_raw.csv")
n_cohort = len(feature_table_raw)

print(f"feature_table_raw.csv loaded: {feature_table_raw.shape}")
print()

# ============================================================
# STEP 1: Compute missingness % per column
# ============================================================

missingness = feature_table_raw.isna().sum()
missingness_pct = (100 * missingness / n_cohort).round(2)

missingness_df = pd.DataFrame({
    'column': missingness.index,
    'n_missing': missingness.values,
    'pct_missing': missingness_pct.values
})

# ============================================================
# STEP 2: Group columns into "feature families" by source variable
# ============================================================

BASE_VARIABLES = ['HR', 'RR', 'SBP', 'DBP', 'MAP', 'Temp', 'SpO2',
                   'WBC', 'Lactate', 'Creatinine', 'Platelets', 'Bilirubin',
                   'Sodium', 'Potassium', 'pH', 'Hematocrit',
                   'GCS_Eye', 'GCS_Verbal', 'GCS_Motor', 'GCS_Total']

ORGAN_TO_VARS = {
    'Cardiovascular_score': ['MAP', 'Vasopressor'],
    'Respiratory_score': ['RR', 'SpO2', 'FiO2'],
    'Renal_score': ['Creatinine', 'UrineOutput'],
    'Hepatic_score': ['Bilirubin'],
    'Hematologic_score': ['Platelets'],
    'Metabolic_score': ['Lactate', 'pH', 'Sodium', 'Potassium'],
}

INTERACTION_VAR_MAP = {
    'SIRS_x_SOFA': ['SIRS_score', 'sofa_total'],
    'Hypotension_x_RisingLactate': ['MAP', 'SBP', 'Lactate'],
    'Hypotension_SBP_x_RisingLactate': ['SBP', 'Lactate'],
    'Hypoxemia_x_Tachycardia': ['SpO2', 'HR'],
    'RisingCreatinine_x_FallingUO': ['Creatinine', 'UrineOutput'],
}

def assign_family(colname):
    families = set()
    # Base variable prefix match (var + '_' at start of column name)
    for var in sorted(BASE_VARIABLES, key=len, reverse=True):  # longest first to avoid partial overlaps
        if colname.startswith(var + '_') or colname == f'{var}_stability_index':
            families.add(var)
    # Organ burden score columns
    if colname in ORGAN_TO_VARS:
        for v in ORGAN_TO_VARS[colname]:
            families.add(v)
    # Interaction feature columns
    if colname in INTERACTION_VAR_MAP:
        for v in INTERACTION_VAR_MAP[colname]:
            families.add(v)
    if not families:
        families.add('Other/Ungrouped')
    return families

missingness_df['families'] = missingness_df['column'].apply(assign_family)

# ============================================================
# STEP 3: Feature-family summary — for the heavy-missingness variables specifically
# ============================================================

print("=" * 80)
print("FEATURE FAMILY SUMMARY — Bilirubin, Lactate, pH (and any other heavy-missingness variable)")
print("=" * 80)

# Identify variables where >=1 of their derived columns is >20% missing
heavy_vars = set()
for _, row in missingness_df.iterrows():
    if row['pct_missing'] > 20:
        heavy_vars.update(row['families'])
heavy_vars.discard('Other/Ungrouped')

for var in sorted(heavy_vars):
    var_cols = missingness_df[missingness_df['families'].apply(lambda f: var in f)]
    n_total = len(var_cols)
    n_over20 = (var_cols['pct_missing'] > 20).sum()
    print(f"\n--- {var} ---")
    print(f"  Total derived columns for this variable: {n_total}")
    print(f"  Columns with >20% missing (would be DROPPED): {n_over20}")
    print(f"  Columns with <=20% missing (would SURVIVE): {n_total - n_over20}")
    print(var_cols[['column', 'pct_missing']].sort_values('pct_missing', ascending=False).to_string(index=False))

print()

# ============================================================
# STEP 4: Apply the 20% drop rule
# ============================================================

cols_to_drop = missingness_df.loc[missingness_df['pct_missing'] > 20, 'column'].tolist()
print(f"Total columns with >20% missing: {len(cols_to_drop)} (expected 82)")
print()

feature_table_mice_input = feature_table_raw.drop(columns=cols_to_drop)
print(f"feature_table_mice_input.csv shape after drop: {feature_table_mice_input.shape} "
      f"(expected (3122, 196))")
print()

# ============================================================
# STEP 5: Confirm whether any ENTIRE source variable is fully eliminated
# ============================================================

print("=" * 80)
print("FULL-ELIMINATION CHECK: did dropping >20%-missing columns wipe out an ENTIRE variable?")
print("=" * 80)

for var in sorted(heavy_vars):
    all_var_cols = missingness_df[missingness_df['families'].apply(lambda f: var in f)]['column'].tolist()
    remaining_var_cols = [c for c in all_var_cols if c in feature_table_mice_input.columns]
    fully_eliminated = len(remaining_var_cols) == 0
    print(f"{var}: {len(all_var_cols)} original columns -> {len(remaining_var_cols)} remaining "
          f"{'*** FULLY ELIMINATED ***' if fully_eliminated else '(partially thinned)'}")

print()

# ============================================================
# STEP 6: Complete-case table from the ORIGINAL 278-column table
# (before the 20%-column drop is applied)
# ============================================================

complete_case = feature_table_raw.dropna(axis=0, how='any')
n_complete_case = len(complete_case)
pct_complete_case = round(100 * n_complete_case / n_cohort, 3)

print("=" * 80)
print("COMPLETE-CASE FILTERING (from ORIGINAL 278-column table, before column drop)")
print("=" * 80)
print(f"Patients surviving complete-case filtering (zero missing values across ALL 278 columns): "
      f"{n_complete_case} / {n_cohort} ({pct_complete_case}%)")
if n_complete_case < 100:
    print("This is likely UNUSABLY SMALL for any meaningful analysis, as anticipated — reporting")
    print("plainly rather than downplaying it. With ~278 columns and multiple known-sparse")
    print("variables (Bilirubin, Lactate, pH, several TPIFE/organ-burden/interaction features",
          "all requiring joint non-missingness), complete-case analysis is not a viable path for")
    print("this dataset. This table is saved for documentation/comparison purposes only.")
print()

complete_case.to_csv(processed_path + "feature_table_complete_case.csv", index=False)
print(f"Saved feature_table_complete_case.csv. Shape: {complete_case.shape}")
print()

# ============================================================
# STEP 7: MICE imputation on feature_table_mice_input.csv
# ============================================================

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

# Identify column types: never impute identifiers or the outcome variable.
# Categorical text columns are NOT imputed by MICE here (MICE as implemented via
# IterativeImputer/BayesianRidge is for continuous numeric data) — they're
# carried through unchanged, and their own missingness (if any) is reported
# separately rather than silently imputed with an inappropriate method.

EXCLUDE_FROM_IMPUTATION = ['subject_id', 'mortality_28d']  # identifier + outcome, never imputed

categorical_cols = feature_table_mice_input.select_dtypes(include=['object']).columns.tolist()
numeric_cols = feature_table_mice_input.select_dtypes(include=[np.number]).columns.tolist()

cols_for_mice = [c for c in numeric_cols if c not in EXCLUDE_FROM_IMPUTATION]

print(f"Columns excluded from MICE (identifier/outcome): {EXCLUDE_FROM_IMPUTATION}")
print(f"Categorical columns carried through UNCHANGED (not MICE-imputed): {categorical_cols}")
if categorical_cols:
    for c in categorical_cols:
        n_miss = feature_table_mice_input[c].isna().sum()
        print(f"  '{c}': {n_miss} missing values ({round(100*n_miss/n_cohort,2)}%) — left as-is")
print(f"Numeric columns going into MICE: {len(cols_for_mice)}")
print()

print("Running IterativeImputer (BayesianRidge estimator, random_state=42, max_iter=10)...")
print("This may take a while given the column count.")

imputer = IterativeImputer(estimator=BayesianRidge(), random_state=42, max_iter=10, verbose=0)
imputed_array = imputer.fit_transform(feature_table_mice_input[cols_for_mice])

feature_table_mice = feature_table_mice_input.copy()
feature_table_mice[cols_for_mice] = imputed_array

remaining_missing_after_mice = feature_table_mice[cols_for_mice].isna().sum().sum()
print(f"Remaining missing values in MICE-imputed numeric columns (should be 0): "
      f"{remaining_missing_after_mice}")
print()

# ============================================================
# Save both final outputs, print final shapes
# ============================================================

feature_table_mice_input.to_csv(processed_path + "feature_table_mice_input.csv", index=False)
feature_table_mice.to_csv(processed_path + "feature_table_mice.csv", index=False)

print("=" * 80)
print("FINAL SHAPES")
print("=" * 80)
print(f"feature_table_mice_input.csv: {feature_table_mice_input.shape}")
print(f"feature_table_mice.csv: {feature_table_mice.shape}")
print(f"feature_table_complete_case.csv: {complete_case.shape}")

In [ ]:
import pandas as pd

processed_path = "../data/processed/"

feature_table_mice_input = pd.read_csv(processed_path + "feature_table_mice_input.csv")
n_cohort = len(feature_table_mice_input)

print(f"feature_table_mice_input.csv loaded: {feature_table_mice_input.shape} "
      f"(expected (3122, 196))")
print()

# ============================================================
# Rebuild complete-case using Option A: post-20%-drop 196-column table
# ============================================================

complete_case_optionA = feature_table_mice_input.dropna(axis=0, how='any')
n_complete_case = len(complete_case_optionA)
pct_retained = round(100 * n_complete_case / n_cohort, 2)

print("=" * 70)
print("OPTION A COMPLETE-CASE RESULT (post-20%-drop, 196-column basis)")
print("=" * 70)
print(f"Patients with ZERO missing values across all 196 columns: {n_complete_case} / {n_cohort} "
      f"({pct_retained}%)")
print()

# ============================================================
# Compare against the old (278-column) complete-case count for context
# ============================================================

old_complete_case = pd.read_csv(processed_path + "feature_table_complete_case.csv")
n_old = len(old_complete_case)
print(f"Previous complete-case count (278-column basis): {n_old} patients")
print(f"New complete-case count (196-column basis): {n_complete_case} patients")
print(f"Improvement: {n_complete_case - n_old} more patients retained "
      f"({round(n_complete_case/n_old, 2)}x the previous sample size)" if n_old > 0 else "")
print()

# ============================================================
# Honest assessment against the ~1000-1500 floor — no assumption either way
# ============================================================

FLOOR_LOW, FLOOR_HIGH = 1000, 1500

print("=" * 70)
print("ASSESSMENT AGAINST ~1000-1500 PATIENT SENSITIVITY-CHECK FLOOR")
print("=" * 70)

if n_complete_case >= FLOOR_HIGH:
    print(f"RESULT: {n_complete_case} patients clears the {FLOOR_HIGH} upper floor comfortably.")
    print("This is a usable sample size for a complete-case sensitivity check. Option A resolved it —")
    print("no need to fall back to Option C.")
elif n_complete_case >= FLOOR_LOW:
    print(f"RESULT: {n_complete_case} patients falls WITHIN the {FLOOR_LOW}-{FLOOR_HIGH} floor range —")
    print("workable but not generous, especially given the ~39% mortality rate means the smaller")
    print("outcome class could still be thin in absolute terms. Usable, but worth checking the")
    print("mortality_28d split within this subset (printed below) before fully trusting it.")
else:
    print(f"RESULT: {n_complete_case} patients is BELOW the {FLOOR_LOW} floor.")
    print("Option A alone did NOT fix this to a comfortable degree. This still looks underpowered")
    print("for a standalone sensitivity check. Recommend moving to Option C (core-feature-subset")
    print("complete-case: vitals + static + SOFA/APACHE, excluding known-sparse labs) rather than")
    print("accepting this number as-is.")
print()

# ============================================================
# Report outcome balance within the new complete-case subset (relevant either way)
# ============================================================

if 'mortality_28d' in complete_case_optionA.columns and n_complete_case > 0:
    mortality_counts = complete_case_optionA['mortality_28d'].value_counts()
    mortality_rate_subset = round(100 * complete_case_optionA['mortality_28d'].mean(), 2)
    print(f"mortality_28d distribution WITHIN this complete-case subset:")
    print(mortality_counts)
    print(f"Mortality rate in subset: {mortality_rate_subset}% "
          f"(compare against full-cohort rate for representativeness)")
    print()

# ============================================================
# Save
# ============================================================

complete_case_optionA.to_csv(processed_path + "feature_table_complete_case.csv", index=False)
print(f"Saved feature_table_complete_case.csv (Option A version, 196-column basis). "
      f"Shape: {complete_case_optionA.shape}")

In [ ]:
import pandas as pd, numpy as np, os, re, time

P = "../data/processed/"
RAW = "../data/raw/"

def find_raw(name):
    for r, _, fs in os.walk(RAW):
        if name in fs:
            return os.path.join(r, name)
    return None

print("=" * 70, "\n1. PROCESSED FILES (size, last modified)\n", "=" * 70)
for f in sorted(os.listdir(P)):
    fp = P + f
    if os.path.isfile(fp):
        print(f"{f:55s} {os.path.getsize(fp)/1e6:9.2f} MB  {time.strftime('%Y-%m-%d %H:%M', time.localtime(os.path.getmtime(fp)))}")

print("\n", "=" * 70, "\n2. VASOPRESSOR STATUS\n", "=" * 70)
pat = re.compile(r"vaso|pressor|norepi|epineph|dopamine|dobutamine|phenyl|vasopressin|mcg", re.I)
for f in ["organ_burden_inputs_24h.csv", "sofa_scores.csv", "organ_burden_index.csv", "feature_table_raw.csv"]:
    try:
        d = pd.read_csv(P + f, nrows=5000)
        cols = [c for c in d.columns if pat.search(c)]
        print(f"\n{f}: shape(first 5000 rows)={d.shape}")
        print("  vasopressor-like columns:", cols)
        for c in cols[:12]:
            if pd.api.types.is_numeric_dtype(d[c]):
                print(f"   {c}: non-null={d[c].notna().mean():.2%}, nonzero={(d[c].fillna(0)!=0).mean():.2%}, max={d[c].max():.3f}")
            else:
                print(f"   {c}: values={d[c].dropna().unique()[:8]}")
    except Exception as e:
        print(f, "ERROR:", e)

print("\n", "=" * 70, "\n3. LEAK / TIME / ID COLUMNS IN THE 132 SELECTED FEATURES\n", "=" * 70)
sel = pd.read_csv(P + "final_selected_features.csv")["feature"].tolist()
print("n selected:", len(sel))
leak_pat = re.compile(r"year|anchor|los|outtime|intime|dod|death|mort|disch|expire|hadm|stay_id|subject|immortal|_id$", re.I)
print("suspicious names:", [c for c in sel if leak_pat.search(c)])

print("\n", "=" * 70, "\n4. COVERAGE OF Na / K / Hct / pH / urine / FiO2 / MAP\n", "=" * 70)
ft = pd.read_csv(P + "feature_table_raw.csv", nrows=5)
allcols = ft.columns.tolist()
for key in ["sodium", "potassium", "hemat", "hct", "ph", "urine", "fio2", "map", "mean_bp", "lactate"]:
    hits = [c for c in allcols if key in c.lower()]
    insel = [c for c in hits if c in sel]
    print(f"{key:10s} in raw table: {len(hits):3d} cols | in selected 132: {len(insel)} -> {insel[:6]}")

print("\n", "=" * 70, "\n5. CALENDAR-ERA READINESS (anchor_year_group)\n", "=" * 70)
pp = find_raw("patients.csv.gz") or find_raw("patients.csv")
print("patients file:", pp)
pt = pd.read_csv(pp, usecols=["subject_id", "anchor_year_group"])
co = pd.read_csv(P + "cohort_step1.csv")
m = co.merge(pt, on="subject_id", how="left")
print("cohort rows:", len(co), "| missing anchor group:", m["anchor_year_group"].isna().sum())
print(pd.crosstab(m["anchor_year_group"], m["mortality_28d"], margins=True))
print("\nicu_admission_year range per group (shows the date-shift problem):")
print(m.groupby("anchor_year_group")["icu_admission_year"].agg(["min", "max", "count"]))
print("\nicu_admission_year in 132 selected features?:", "icu_admission_year" in sel)

In [ ]:
import pandas as pd, numpy as np, os
P = "../data/processed/"; RAW = "../data/raw/"
def find_raw(name):
    for r, _, fs in os.walk(RAW):
        if name in fs:
            return os.path.join(r, name)

print("=" * 70, "\nA. STRUCTURE OF SOFA / BURDEN FILES\n", "=" * 70)
for f in ["organ_burden_inputs_24h.csv", "sofa_scores.csv", "organ_burden_index.csv"]:
    d = pd.read_csv(P + f)
    print(f, d.shape); print(d.columns.tolist()); print(d.head(3).to_string()); print()

print("=" * 70, "\nB. SAVED VASOPRESSOR DOSES BY DRUG\n", "=" * 70)
ob = pd.read_csv(P + "organ_burden_inputs_24h.csv")
v = "valuenum_mcg_kg_min"
dc = next((c for c in ["label", "drug", "itemid", "variable"] if c in ob.columns), None)
print("drug column:", dc)
if dc:
    print(ob.groupby(dc)[v].describe(percentiles=[.5, .95, .99]).round(3).to_string())
    print("null share by drug:\n", ob.groupby(dc)[v].apply(lambda s: s.isna().mean()).round(3).to_string())

print("\n", "=" * 70, "\nC. RECOMPUTE FROM RAW inputevents\n", "=" * 70)
ITEMS = {221906: "norepinephrine", 221289: "epinephrine", 221662: "dopamine",
         221653: "dobutamine", 221749: "phenylephrine", 222315: "vasopressin"}
co = pd.read_csv(P + "cohort_step1.csv", parse_dates=["intime"])
stays = set(co.stay_id); intime = co.set_index("stay_id")["intime"]
fp = find_raw("inputevents.csv.gz")
hdr = pd.read_csv(fp, nrows=0).columns
use = [c for c in ["stay_id", "starttime", "itemid", "rate", "rateuom", "patientweight"] if c in hdr]
print("columns used:", use)
parts = []
for ch in pd.read_csv(fp, usecols=use, chunksize=1_000_000, parse_dates=["starttime"]):
    ch = ch[ch.itemid.isin(ITEMS) & ch.stay_id.isin(stays)]
    if len(ch): parts.append(ch)
ie = pd.concat(parts)
ie["t0"] = ie.stay_id.map(intime)
ie = ie[(ie.starttime >= ie.t0) & (ie.starttime < ie.t0 + pd.Timedelta(hours=24))].copy()
ie["drug"] = ie.itemid.map(ITEMS)
print("rows in first 24h:", len(ie))
print("\nrateuom by drug:"); print(ie.groupby(["drug", "rateuom"]).size().to_string())
print("\npatientweight missing share:", round(ie.patientweight.isna().mean(), 4))

u, r, w = ie.rateuom, ie.rate, ie.patientweight
ie["mcg_kg_min"] = np.select(
    [u == "mcg/kg/min", u == "mcg/min", u == "mcg/hour", u == "mg/kg/min", u == "mg/hour"],
    [r, r / w, r / 60 / w, r * 1000, r * 1000 / 60 / w], default=np.nan)
print("\nRecomputed dose by drug (mcg/kg/min; vasopressin is units/min so NaN by design):")
print(ie.groupby("drug").mcg_kg_min.describe(percentiles=[.5, .95, .99]).round(3).to_string())

if {"stay_id", "starttime"}.issubset(ob.columns):
    ob["starttime"] = pd.to_datetime(ob["starttime"])
    keys = ["stay_id", "starttime"] + (["itemid"] if "itemid" in ob.columns else [])
    mg = ie.merge(ob[keys + [v]], on=keys, how="inner")
    print(f"\nmatched {len(mg)} rows; max abs diff saved vs recomputed:",
          (mg[v] - mg.mcg_kg_min).abs().max())
    print("rows differing by >1e-6:", int(((mg[v] - mg.mcg_kg_min).abs() > 1e-6).sum()))
else:
    print("\nsaved file lacks stay_id/starttime - cannot row-match; compare the distributions above")

print("\n", "=" * 70, "\nD. FiO2 / URINE IN LOADED VITALS-LABS\n", "=" * 70)
vl = pd.read_csv(P + "vitals_labs_24h.csv")
print("columns:", vl.columns.tolist())
tc = next((c for c in ["label", "variable", "name", "feature"] if c in vl.columns), None)
if tc:
    names = vl[tc].astype(str).value_counts()
    print(names.to_string())
else:
    print(vl.itemid.value_counts().head(60).to_string())

print("\n", "=" * 70, "\nE. Na / K / Hct / pH COLUMNS AND DUPLICATES IN SELECTED 132\n", "=" * 70)
ft = pd.read_csv(P + "feature_table_raw.csv", nrows=3)
for k in ["sodium", "potassium", "hemat", "ph"]:
    print(k, "->", [c for c in ft.columns if k in c.lower()])
sel = pd.read_csv(P + "final_selected_features.csv")["feature"].tolist()
mice = pd.read_csv(P + "feature_table_mice.csv")
num = mice[[c for c in sel if c in mice.columns]].select_dtypes("number")
dup = num.T.duplicated(keep=False)
print("\nidentical-valued selected columns:", num.columns[dup.values].tolist())
cm = num.corr().abs().where(np.triu(np.ones((num.shape[1],) * 2), 1).astype(bool))
hi = cm.stack(); hi = hi[hi > 0.999]
print("pairs with |corr| > 0.999:", len(hi)); print(hi.head(30).to_string())

In [ ]:
import pandas as pd, numpy as np, os
P = "../data/processed/"; RAW = "../data/raw/"
def find_raw(name):
    for r, _, fs in os.walk(RAW):
        if name in fs:
            return os.path.join(r, name)

ob = pd.read_csv(P + "organ_burden_inputs_24h.csv")
sofa = pd.read_csv(P + "sofa_scores.csv")
obi = pd.read_csv(P + "organ_burden_index.csv")
co = pd.read_csv(P + "cohort_step1.csv", parse_dates=["intime"])
di = pd.read_csv(find_raw("d_items.csv.gz"), usecols=["itemid", "label", "unitname"])

print("=" * 70, "\n1. WHAT EACH ITEMID IN THE BURDEN INPUT FILE IS\n", "=" * 70)
ids = sorted(set(ob.itemid.unique()) | {221653, 221906, 221289, 221662, 221749, 222315})
print(di[di.itemid.isin(ids)].to_string(index=False))
print(ob.groupby("itemid").agg(rows=("subject_id", "size"), patients=("subject_id", "nunique")).to_string())
print(ob.flag_reason.value_counts(dropna=False).to_string())

print("\n", "=" * 70, "\n2. IMPLAUSIBLE DOSES (rows / patients)\n", "=" * 70)
lab = di.set_index("itemid").label.str.lower()
ob["label"] = ob.itemid.map(lab)
def cap(l):
    l = str(l)
    if "norepinephrine" in l: return 3.0
    if "epinephrine" in l: return 3.0
    if "phenylephrine" in l: return 10.0
    if "dopamine" in l: return 20.0
    if "dobutamine" in l: return 20.0
    return np.nan
ob["cap"] = ob.label.map(cap)
bad = ob[ob.valuenum_mcg_kg_min > ob.cap]
print(bad.groupby("label").agg(rows=("subject_id", "size"), patients=("subject_id", "nunique"),
                               max_dose=("valuenum_mcg_kg_min", "max")).to_string())

print("\n", "=" * 70, "\n3. SOFA CARDIOVASCULAR VS PRESSOR EXPOSURE\n", "=" * 70)
pres = ob[ob.valuenum_mcg_kg_min.notna()].groupby("subject_id").valuenum_mcg_kg_min.max().rename("max_dose_saved")
# raw check for dobutamine and vasopressin (missing from or unconverted in the saved file)
stays = set(co.stay_id); s2p = co.set_index("stay_id")["subject_id"]; t0 = co.set_index("stay_id")["intime"]
fp = find_raw("inputevents.csv.gz")
parts = []
for ch in pd.read_csv(fp, usecols=["stay_id", "starttime", "itemid"], chunksize=1_000_000, parse_dates=["starttime"]):
    ch = ch[ch.itemid.isin([221653, 222315]) & ch.stay_id.isin(stays)]
    if len(ch): parts.append(ch)
raw = pd.concat(parts)
raw = raw[(raw.starttime >= raw.stay_id.map(t0)) & (raw.starttime < raw.stay_id.map(t0) + pd.Timedelta(hours=24))]
dob = set(raw[raw.itemid == 221653].stay_id.map(s2p)); vaso = set(raw[raw.itemid == 222315].stay_id.map(s2p))
df = sofa[["subject_id", "cardio_score"]].merge(pres, on="subject_id", how="left")
df["any_saved_pressor"] = df.max_dose_saved.notna()
df["dobutamine_pt"] = df.subject_id.isin(dob); df["vasopressin_pt"] = df.subject_id.isin(vaso)
print("patients with raw dobutamine:", len(dob), "| with vasopressin:", len(vaso))
print("\ncardio_score by saved-pressor exposure:")
print(pd.crosstab(df.any_saved_pressor, df.cardio_score.fillna(-1)).to_string())
only_dob = df[df.dobutamine_pt & ~df.any_saved_pressor]
only_vas = df[df.vasopressin_pt & ~df.any_saved_pressor & ~df.dobutamine_pt]
print(f"\ndobutamine-only patients (n={len(only_dob)}) cardio_score:", only_dob.cardio_score.value_counts(dropna=False).to_dict())
print(f"vasopressin-only patients (n={len(only_vas)}) cardio_score:", only_vas.cardio_score.value_counts(dropna=False).to_dict())
ocv = obi.merge(df[["subject_id", "any_saved_pressor", "vasopressin_pt"]], on="subject_id")
print("\nmean Cardiovascular_score by group:")
print(ocv.groupby(["any_saved_pressor", "vasopressin_pt"]).Cardiovascular_score.agg(["count", "mean"]).round(3).to_string())

print("\n", "=" * 70, "\n4. FiO2 / URINE COVERAGE\n", "=" * 70)
n = len(co)
print("patients with FiO2 (223835) rows:", ob[ob.itemid == 223835].subject_id.nunique(), "/", n)
print("patients with urine rows:", ob[ob.itemid.isin([226559, 226560])].subject_id.nunique(), "/", n)
print("resp_score non-null:", round(sofa.resp_score.notna().mean(), 3),
      "| renal_score non-null:", round(sofa.renal_score.notna().mean(), 3))
print("Respiratory_score non-null:", round(obi.Respiratory_score.notna().mean(), 3),
      "| Renal_score non-null:", round(obi.Renal_score.notna().mean(), 3))
print("\nRenal_score distribution (saw 0.997 for some patients):")
print(obi.Renal_score.describe(percentiles=[.1, .5, .9, .99]).round(3).to_string())

In [ ]:
import pandas as pd, numpy as np, os, time

RAW = "../data/raw/"; OLD = "../data/processed/"; P2 = "../data/processed_v2/"
os.makedirs(P2, exist_ok=True)
def find_raw(name):
    for r, _, fs in os.walk(RAW):
        if name in fs:
            return os.path.join(r, name)
    raise FileNotFoundError(name)

# ------------------------------------------------------------
# 0. Final cohort -> new folder (the ONLY file taken from the old folder)
# ------------------------------------------------------------
co = pd.read_csv(OLD + "cohort_final.csv", parse_dates=["intime", "outtime"])
assert len(co) == 2777 and co.stay_id.is_unique and co.subject_id.is_unique
co.to_csv(P2 + "cohort_final.csv", index=False)
stays, subs = set(co.stay_id), set(co.subject_id)
t0_stay = co.set_index("stay_id")["intime"]
t0_subj = co.set_index("subject_id")["intime"]
with open(P2 + "MANIFEST.txt", "w") as f:
    f.write("processed_v2 inputs: raw MIMIC-IV v3.1 files + cohort_final.csv only. "
            "No file from data/processed/ other than cohort_final.csv is used.\n")

# ------------------------------------------------------------
# 1. Item IDs (every one verified against the dictionaries below)
# ------------------------------------------------------------
CHART = {220045: 'HR', 220050: 'SBP', 220179: 'SBP', 220051: 'DBP', 220180: 'DBP',
         220052: 'MAP', 220181: 'MAP', 225312: 'MAP', 220210: 'RR', 224690: 'RR',
         223761: 'Temp_F', 223762: 'Temp_C', 220277: 'SpO2',
         220739: 'GCS_Eye', 223900: 'GCS_Verbal', 223901: 'GCS_Motor',
         223835: 'FiO2', 226732: 'O2_device', 223849: 'Vent_mode',
         226512: 'Weight_kg', 224639: 'Weight_kg', 226531: 'Weight_lb'}
LAB = {51301: 'WBC', 51300: 'WBC', 50813: 'Lactate', 52442: 'Lactate', 53154: 'Lactate',
       50912: 'Creatinine', 52546: 'Creatinine', 51265: 'Platelets', 53189: 'Platelets',
       50885: 'Bilirubin', 53089: 'Bilirubin',
       50983: 'Sodium', 52623: 'Sodium', 50824: 'Sodium', 52455: 'Sodium',
       50971: 'Potassium', 52610: 'Potassium', 50822: 'Potassium', 52452: 'Potassium',
       50820: 'pH', 50821: 'PaO2', 50818: 'PaCO2', 52033: 'Specimen_type'}
INPUT = {221906: 'Norepinephrine', 221289: 'Epinephrine', 229617: 'Epinephrine',
         221662: 'Dopamine', 221653: 'Dobutamine', 221749: 'Phenylephrine',
         229630: 'Phenylephrine', 229631: 'Phenylephrine', 229632: 'Phenylephrine',
         222315: 'Vasopressin'}
UO = {226559: 1, 226560: 1, 226561: 1, 226584: 1, 226563: 1, 226564: 1, 226565: 1,
      226567: 1, 226557: 1, 226558: 1, 227489: 1, 227488: -1}   # 227488 = GU irrigant IN (subtracted)

d_items = pd.read_csv(find_raw("d_items.csv.gz"), usecols=["itemid", "label", "unitname"])
d_lab = pd.read_csv(find_raw("d_labitems.csv.gz"), usecols=["itemid", "label", "fluid", "category"])
hct = d_lab[d_lab.label.str.match(r"^Hematocrit", na=False) & (d_lab.fluid == "Blood")]
for iid in hct.itemid: LAB[iid] = 'Hematocrit'

def verify(mapping, dictionary, name):
    df = pd.DataFrame({"itemid": list(mapping), "our_name": list(mapping.values())}).merge(dictionary, on="itemid", how="left")
    print(f"\n--- {name} item IDs ---"); print(df.to_string(index=False))
    missing = df[df.label.isna()].itemid.tolist()
    assert not missing, f"{name}: item IDs not found in dictionary: {missing}"
verify(CHART, d_items, "CHARTEVENTS")
verify(LAB, d_lab, "LABEVENTS")
verify(INPUT, d_items, "INPUTEVENTS")
verify({k: 'Urine' for k in UO}, d_items, "OUTPUTEVENTS (urine)")

# ------------------------------------------------------------
# 2. chartevents: our stays, first 24h (weight kept for whole stay, earliest used later)
# ------------------------------------------------------------
t = time.time(); parts = []
for ch in pd.read_csv(find_raw("chartevents.csv.gz"), chunksize=2_000_000,
                      usecols=["stay_id", "charttime", "itemid", "value", "valuenum", "valueuom"],
                      dtype={"value": str, "valueuom": str}):
    ch = ch[ch.itemid.isin(CHART.keys()) & ch.stay_id.isin(stays)]
    if len(ch): parts.append(ch)
ce = pd.concat(parts, ignore_index=True)
ce["charttime"] = pd.to_datetime(ce.charttime)
ce["stay_id"] = ce.stay_id.astype(int)
ce["variable"] = ce.itemid.map(CHART)
ce["hours"] = (ce.charttime - ce.stay_id.map(t0_stay)).dt.total_seconds() / 3600
is_weight = ce.variable.isin(["Weight_kg", "Weight_lb"])
ce = ce[is_weight | ((ce.hours >= 0) & (ce.hours <= 24))]
ce.to_csv(P2 + "extract_chart.csv.gz", index=False)
print(f"\nchartevents: {len(ce):,} rows kept ({(time.time()-t)/60:.1f} min)")

# ------------------------------------------------------------
# 3. labevents: our patients, window [-6h, +24h] around ICU admission
# ------------------------------------------------------------
t = time.time(); parts = []
for ch in pd.read_csv(find_raw("labevents.csv.gz"), chunksize=2_000_000,
                      usecols=["subject_id", "hadm_id", "specimen_id", "itemid", "charttime", "value", "valuenum", "valueuom"],
                      dtype={"value": str, "valueuom": str}):
    ch = ch[ch.itemid.isin(LAB.keys()) & ch.subject_id.isin(subs)]
    if len(ch): parts.append(ch)
lb = pd.concat(parts, ignore_index=True)
lb["charttime"] = pd.to_datetime(lb.charttime)
lb["variable"] = lb.itemid.map(LAB)
lb["hours"] = (lb.charttime - lb.subject_id.map(t0_subj)).dt.total_seconds() / 3600
lb = lb[(lb.hours >= -6) & (lb.hours <= 24)]
lb["stay_id"] = lb.subject_id.map(co.set_index("subject_id")["stay_id"])
lb.to_csv(P2 + "extract_labs.csv.gz", index=False)
print(f"labevents: {len(lb):,} rows kept ({(time.time()-t)/60:.1f} min)")

# ------------------------------------------------------------
# 4. inputevents (vasoactive drugs): any infusion overlapping [0h, 24h]
# ------------------------------------------------------------
t = time.time(); parts = []
for ch in pd.read_csv(find_raw("inputevents.csv.gz"), chunksize=2_000_000,
                      usecols=["stay_id", "starttime", "endtime", "itemid", "rate", "rateuom",
                               "amount", "amountuom", "patientweight", "statusdescription"]):
    ch = ch[ch.itemid.isin(INPUT.keys()) & ch.stay_id.isin(stays)]
    if len(ch): parts.append(ch)
ie = pd.concat(parts, ignore_index=True)
ie["starttime"] = pd.to_datetime(ie.starttime); ie["endtime"] = pd.to_datetime(ie.endtime)
ie["drug"] = ie.itemid.map(INPUT)
ie["start_h"] = (ie.starttime - ie.stay_id.map(t0_stay)).dt.total_seconds() / 3600
ie["end_h"] = (ie.endtime - ie.stay_id.map(t0_stay)).dt.total_seconds() / 3600
ie = ie[(ie.start_h < 24) & (ie.end_h > 0)]
ie.to_csv(P2 + "extract_inputs.csv.gz", index=False)
print(f"inputevents: {len(ie):,} rows kept ({(time.time()-t)/60:.1f} min)")

# ------------------------------------------------------------
# 5. outputevents (urine): first 24h
# ------------------------------------------------------------
t = time.time(); parts = []
for ch in pd.read_csv(find_raw("outputevents.csv.gz"), chunksize=2_000_000,
                      usecols=["stay_id", "charttime", "itemid", "value"]):
    ch = ch[ch.itemid.isin(UO.keys()) & ch.stay_id.isin(stays)]
    if len(ch): parts.append(ch)
oe = pd.concat(parts, ignore_index=True)
oe["charttime"] = pd.to_datetime(oe.charttime)
oe["hours"] = (oe.charttime - oe.stay_id.map(t0_stay)).dt.total_seconds() / 3600
oe = oe[(oe.hours >= 0) & (oe.hours <= 24)]
oe["signed_ml"] = oe.value * oe.itemid.map(UO)
oe.to_csv(P2 + "extract_output.csv.gz", index=False)
print(f"outputevents: {len(oe):,} rows kept ({(time.time()-t)/60:.1f} min)")

# ------------------------------------------------------------
# 6. Coverage report (% of 2,777 patients with >=1 raw value) — BEFORE any cleaning
# ------------------------------------------------------------
N = len(co)
cov = pd.concat([
    ce[~ce.variable.isin(["Weight_kg", "Weight_lb"])].groupby("variable").stay_id.nunique(),
    ce[ce.variable.isin(["Weight_kg", "Weight_lb"])].groupby("variable").stay_id.nunique().rename(lambda x: x + " (any time)"),
    lb.groupby("variable").stay_id.nunique(),
    ie.groupby("drug").stay_id.nunique().rename(lambda x: "Drug: " + x),
    pd.Series({"Urine output (any item)": oe.stay_id.nunique()}),
]).rename("n_patients").to_frame()
cov["pct"] = (100 * cov.n_patients / N).round(1)
print("\n=== RAW COVERAGE (before cleaning), N =", N, "===")
print(cov.sort_values("pct", ascending=False).to_string())
print("\nSpecimen types seen (52033):", lb[lb.variable == "Specimen_type"].value.value_counts().head(10).to_dict())
print("O2 devices seen (top):", ce[ce.variable == "O2_device"].value.value_counts().head(12).to_dict())
print("\nSaved in processed_v2/: cohort_final.csv, extract_chart/labs/inputs/output.csv.gz, MANIFEST.txt")

In [ ]:
import pandas as pd, numpy as np

P2 = "../data/processed_v2/"
co = pd.read_csv(P2 + "cohort_final.csv", parse_dates=["intime"])
N = len(co)
ce = pd.read_csv(P2 + "extract_chart.csv.gz", parse_dates=["charttime"], dtype={"value": str})
lb = pd.read_csv(P2 + "extract_labs.csv.gz", parse_dates=["charttime"], dtype={"value": str})
ie = pd.read_csv(P2 + "extract_inputs.csv.gz", parse_dates=["starttime", "endtime"])
oe = pd.read_csv(P2 + "extract_output.csv.gz", parse_dates=["charttime"])
log = []   # cleaning log -> supplementary table
def note(var, step, before, after):
    log.append({"variable": var, "step": step, "rows_before": before, "rows_after": after, "rows_removed": before - after})

# ============================================================
# A. VITALS (chartevents), window 0-24 h
# ============================================================
v = ce[(ce.hours >= 0) & (ce.hours <= 24)].copy()
b = len(v); v = v.drop_duplicates(["stay_id", "charttime", "itemid", "value"]); note("all chart", "exact duplicates", b, len(v))

# Temperature: everything to Celsius FIRST, then plausibility (25-42 C)
t = v[v.variable.isin(["Temp_F", "Temp_C"])].copy()
f_in_c = (t.variable == "Temp_C") & (t.valuenum > 50)              # Fahrenheit typed into the Celsius item
t["value_c"] = np.where((t.variable == "Temp_F") | f_in_c, (t.valuenum - 32) * 5 / 9, t.valuenum)
b = len(t); t = t[t.value_c.between(25, 42)]; note("Temp", "range 25-42 C (after F->C)", b, len(t))
print(f"Temp: {int(f_in_c.sum())} Fahrenheit values found inside the Celsius item were converted")
temp = t.assign(variable="Temp", value=t.value_c)[["stay_id", "charttime", "hours", "variable", "value"]]

# GCS: eye/motor numeric; verbal from text (ETT / no response = 1, same convention as before)
VERBAL = {"Oriented": 5, "Confused": 4, "Inappropriate Words": 3, "Incomprehensible sounds": 2,
          "No Response": 1, "No Response-ETT": 1}
g = v[v.variable.isin(["GCS_Eye", "GCS_Verbal", "GCS_Motor"])].copy()
g["num"] = np.where(g.variable == "GCS_Verbal", g.value.str.strip().map(VERBAL), g.valuenum)
unmapped = g[(g.variable == "GCS_Verbal") & g.num.isna()].value.value_counts()
assert len(unmapped) == 0, f"Unmapped GCS verbal text: {unmapped.to_dict()}"
g["intubated_verbal"] = (g.variable == "GCS_Verbal") & (g.value.str.strip() == "No Response-ETT")
lim = {"GCS_Eye": (1, 4), "GCS_Verbal": (1, 5), "GCS_Motor": (1, 6)}
b = len(g); g = g[[lim[x][0] <= y <= lim[x][1] for x, y in zip(g.variable, g.num)]]; note("GCS components", "valid ranges", b, len(g))
gp = g.pivot_table(index=["stay_id", "charttime", "hours"], columns="variable", values="num", aggfunc="min").reset_index()
gp = gp.dropna(subset=["GCS_Eye", "GCS_Verbal", "GCS_Motor"])
gp["value"] = gp.GCS_Eye + gp.GCS_Verbal + gp.GCS_Motor
gcs = pd.concat([g.assign(value=g.num)[["stay_id", "charttime", "hours", "variable", "value"]],
                 gp.assign(variable="GCS_Total")[["stay_id", "charttime", "hours", "variable", "value"]]])

# FiO2 -> fraction 0.21-1.0
fi = v[v.variable == "FiO2"].copy()
fi["value"] = np.where(fi.valuenum > 1, fi.valuenum / 100, fi.valuenum)
b = len(fi); fi = fi[fi.value.between(0.21, 1.0)]; note("FiO2", "fraction 0.21-1.0", b, len(fi))
fio2 = fi[["stay_id", "charttime", "hours", "variable", "value"]]

# Other vitals: physiological ranges (same limits as the earlier validated cleaning; MAP added)
RANGES = {"HR": (20, 250), "SBP": (40, 300), "DBP": (20, 200), "MAP": (20, 200), "RR": (4, 60), "SpO2": (1, 100)}
o = v[v.variable.isin(RANGES)].copy()
o["value"] = o.valuenum
for var, (lo, hi) in RANGES.items():
    m = o.variable == var; b = m.sum()
    o = o[~m | o.value.between(lo, hi)]; note(var, f"range {lo}-{hi}", b, (o.variable == var).sum())

# SpO2 < 50: rule-based replacement of the earlier manual review.
# Keep only if another SpO2 within +/-60 min is also < 85 (a real desaturation trend);
# drop if it is isolated (no neighbour) or surrounded only by normal values (probe artefact).
sp = o[o.variable == "SpO2"].sort_values(["stay_id", "charttime"])
low = sp[sp.value < 50]
drop_idx = []
for idx, r in low.iterrows():
    nb = sp[(sp.stay_id == r.stay_id) & (sp.index != idx) & ((sp.charttime - r.charttime).abs() <= pd.Timedelta(minutes=60))]
    if len(nb) == 0 or (nb.value >= 85).all():
        drop_idx.append(idx)
b = (o.variable == "SpO2").sum(); o = o.drop(index=drop_idx); note("SpO2", "isolated <50% (artefact rule)", b, (o.variable == "SpO2").sum())
print(f"SpO2 <50%: {len(low)} readings, {len(drop_idx)} dropped as isolated artefacts, {len(low)-len(drop_idx)} kept as real desaturation")
vitals = pd.concat([o[["stay_id", "charttime", "hours", "variable", "value"]], temp, gcs, fio2])

# ============================================================
# B. LABS + BLOOD GAS (labevents), window -6 to +24 h
# ============================================================
l = lb.copy()
b = len(l); l = l.drop_duplicates(["stay_id", "charttime", "itemid", "specimen_id", "valuenum", "value"]); note("all labs", "exact duplicates", b, len(l))
spec = l[l.variable == "Specimen_type"].set_index("specimen_id").value.str.strip()
spec = spec[~spec.index.duplicated()]
l = l[l.variable != "Specimen_type"].copy()
bg = l.variable.isin(["pH", "PaO2", "PaCO2"])
l["specimen"] = l.specimen_id.map(spec)
b = bg.sum(); keep_bg = ~bg | (l.specimen == "ART.")
print("Blood-gas specimen of pH/PaO2/PaCO2 rows:", l.loc[bg, "specimen"].fillna("MISSING").value_counts().to_dict())
l = l[keep_bg]; note("pH/PaO2/PaCO2", "arterial specimen only", b, l.variable.isin(["pH", "PaO2", "PaCO2"]).sum())
l = l.dropna(subset=["valuenum"])
LAB_RANGES = {"WBC": (0, 500), "Lactate": (0.1, 30), "Creatinine": (0.1, 25), "Platelets": (1, 2000),
              "Bilirubin": (0.1, 70), "Sodium": (100, 180), "Potassium": (1.5, 10), "Hematocrit": (5, 75),
              "pH": (6.8, 7.8), "PaO2": (10, 700), "PaCO2": (5, 200)}
for var, (lo, hi) in LAB_RANGES.items():
    m = l.variable == var; b = m.sum()
    l = l[~m | l.valuenum.between(lo, hi)]; note(var, f"range {lo}-{hi}", b, (l.variable == var).sum())
l["variable"] = l.variable.replace({"pH": "pH_art", "PaO2": "PaO2_art", "PaCO2": "PaCO2_art"})
labs = l.drop(columns=["value"]).rename(columns={"valuenum": "value"})[["stay_id", "charttime", "hours", "variable", "value"]]

clean = pd.concat([vitals, labs], ignore_index=True)
clean["stay_id"] = clean.stay_id.astype(int)
clean = clean.sort_values(["stay_id", "variable", "charttime"]).reset_index(drop=True)
clean.to_csv(P2 + "clean_long.csv.gz", index=False)

# ============================================================
# C. PER-PATIENT CONTEXT: weight, ventilation, oxygen support
# ============================================================
w = ce[ce.variable.isin(["Weight_kg", "Weight_lb"])].copy()
w["kg"] = np.where(w.variable == "Weight_lb", w.valuenum * 0.453592, w.valuenum)
w = w[w.kg.between(25, 300)]
w["prio"] = w.itemid.map({226512: 0, 224639: 1, 226531: 2})
weight = w.sort_values(["stay_id", "prio", "charttime"]).groupby("stay_id").kg.first()
pw = ie[ie.patientweight.between(25, 300)].groupby("stay_id").patientweight.median()
weight = weight.combine_first(pw)

dev = v[v.variable == "O2_device"].copy(); dev["d"] = dev.value.str.strip().str.lower()
INVASIVE = {"endotracheal tube", "tracheostomy tube"}
NIV = {"bipap mask", "cpap mask"}
NO_O2 = {"none", "room air"}
ctx = pd.DataFrame({"stay_id": co.stay_id}).set_index("stay_id")
ctx["weight_kg"] = weight
ctx["invasive_vent_24h"] = dev[dev.d.isin(INVASIVE)].groupby("stay_id").size().reindex(ctx.index).notna()
ctx["niv_24h"] = dev[dev.d.isin(NIV)].groupby("stay_id").size().reindex(ctx.index).notna()
ctx["oxygen_support_24h"] = dev[~dev.d.isin(NO_O2)].groupby("stay_id").size().reindex(ctx.index).notna()
ctx["gcs_verbal_ETT_24h"] = g[g.intubated_verbal].groupby("stay_id").size().reindex(ctx.index).notna()
ctx["vent_mode_charted_24h"] = v[v.variable == "Vent_mode"].groupby("stay_id").size().reindex(ctx.index).notna()
print("\nAll O2-device text values (check nothing important is unmapped):")
print(dev.value.str.strip().value_counts().to_string())

# ============================================================
# D. VASOACTIVE DRUGS: standardised rates + norepinephrine-equivalent (NEE)
# ============================================================
d = ie.copy()
b = len(d); d = d[d.statusdescription.astype(str).str.lower() != "rewritten"]; note("drug rows", "drop 'Rewritten' (superseded) orders", b, len(d))
d["wt"] = d.stay_id.map(ctx.weight_kg).fillna(d.patientweight)
u = d.rateuom.astype(str).str.strip().str.lower()
print("\nRate units by drug:"); print(d.groupby(["drug", u]).size().to_string())
d["rate_std"] = np.select(
    [u.eq("mcg/kg/min"), u.eq("mcg/min"), u.eq("mg/kg/min"), u.eq("units/hour"), u.eq("units/min")],
    [d.rate, d.rate / d.wt, d.rate * 1000, d.rate / 60, d.rate], default=np.nan)   # catecholamines mcg/kg/min, vasopressin U/min
CAP = {"Norepinephrine": 5, "Epinephrine": 5, "Phenylephrine": 15, "Dopamine": 25, "Dobutamine": 25, "Vasopressin": 0.1}
implaus = d.rate_std > d.drug.map(CAP)
print("\nImplausible rates set to missing (charting errors):", d[implaus].groupby("drug").size().to_dict())
d.loc[implaus, "rate_std"] = np.nan
NEE_FACTOR = {"Norepinephrine": 1, "Epinephrine": 1, "Dopamine": 0.01, "Phenylephrine": 0.1, "Vasopressin": 2.5}
d["nee"] = d.rate_std * d.drug.map(NEE_FACTOR)                        # NaN for dobutamine (inotrope, not a vasopressor)
d["start_h"] = d.start_h.clip(lower=0); d["end_h"] = d.end_h.clip(upper=24)
d = d[d.end_h > d.start_h]
d[["stay_id", "drug", "start_h", "end_h", "rate_std", "nee"]].to_csv(P2 + "clean_vasoactive.csv.gz", index=False)

# ============================================================
# E. URINE OUTPUT: events + monitoring coverage
# ============================================================
uo = oe[(oe.hours >= 0) & (oe.hours <= 24)].copy()
b = len(uo); uo = uo.drop_duplicates(["stay_id", "charttime", "itemid", "value"]); note("urine", "exact duplicates", b, len(uo))
uo[["stay_id", "charttime", "hours", "itemid", "signed_ml"]].to_csv(P2 + "clean_urine.csv.gz", index=False)
uo_tot = uo.groupby("stay_id").signed_ml.sum().clip(lower=0)
span = uo.groupby("stay_id").hours.agg(lambda s: s.max() - s.min())
ctx["urine_24h_ml"] = uo_tot
ctx["urine_monitored_hours"] = span
ctx["urine_complete_24h"] = (span >= 20).reindex(ctx.index).fillna(False)   # >=20 h of charting -> usable for SOFA UO rule
ctx.reset_index().to_csv(P2 + "clean_patient_context.csv", index=False)
pd.DataFrame(log).to_csv(P2 + "cleaning_log.csv", index=False)

# ============================================================
# F. SUMMARY
# ============================================================
print("\n=== CLEANING LOG ===")
print(pd.DataFrame(log).to_string(index=False))
cov = clean.groupby("variable").agg(n_patients=("stay_id", "nunique"), n_rows=("value", "size"),
                                    median=("value", "median"), p1=("value", lambda s: s.quantile(.01)),
                                    p99=("value", lambda s: s.quantile(.99)))
cov["pct_patients"] = (100 * cov.n_patients / N).round(1)
print("\n=== CLEAN VARIABLE SUMMARY (check medians/percentiles look clinical) ===")
print(cov.round(3).sort_values("pct_patients", ascending=False).to_string())
print("\n=== PATIENT CONTEXT ===")
print(f"Weight available: {ctx.weight_kg.notna().sum()}/{N} | median {ctx.weight_kg.median():.1f} kg")
print(f"Invasive ventilation in first 24h: {ctx.invasive_vent_24h.sum()} ({ctx.invasive_vent_24h.mean():.1%})")
print(f"Non-invasive ventilation: {ctx.niv_24h.sum()} | any oxygen support: {ctx.oxygen_support_24h.sum()}")
print(f"Device-based vs vent-mode-based ventilation agreement:")
print(pd.crosstab(ctx.invasive_vent_24h, ctx.vent_mode_charted_24h).to_string())
print(f"Urine monitored >=20 h: {ctx.urine_complete_24h.sum()}/{N} | median 24h urine {ctx.urine_24h_ml.median():.0f} mL")
dd_ = d.groupby("drug").agg(patients=("stay_id", "nunique"), median_rate=("rate_std", "median"), max_rate=("rate_std", "max"))
print("\nVasoactive drugs after cleaning (catecholamines mcg/kg/min, vasopressin U/min):"); print(dd_.round(3).to_string())
print("\nSaved in processed_v2/: clean_long.csv.gz, clean_vasoactive.csv.gz, clean_urine.csv.gz, "
      "clean_patient_context.csv, cleaning_log.csv")

In [ ]:
%pip install comorbidipy

In [ ]:
import pandas as pd, numpy as np, os
import polars as pl
import comorbidipy
from comorbidipy import comorbidity
from comorbidipy.calculators.comorbidity import colnames
from importlib.metadata import version

RAW = "../data/raw/"; P2 = "../data/processed_v2/"
def find_raw(name):
    for r, _, fs in os.walk(RAW):
        if name in fs:
            return os.path.join(r, name)
    raise FileNotFoundError(name)
print("comorbidipy version:", version("comorbidipy"), "| polars version:", pl.__version__)

co = pd.read_csv(P2 + "cohort_final.csv")
N = len(co)
dx = pd.read_csv(find_raw("diagnoses_icd.csv.gz"), dtype={"icd_code": str})
dx = dx[dx.hadm_id.isin(co.hadm_id)].copy()
dx["icd_code"] = dx.icd_code.str.strip().str.replace(".", "", regex=False).str.upper()
print(f"Diagnosis rows for the {N} cohort admissions: {len(dx):,}")
print("Admissions by ICD version:", dx.groupby("hadm_id").icd_version.max().value_counts().to_dict())

ELIX = list(colnames["elixhauser"].keys())
parts = []
for ver, tag in [(9, "icd9"), (10, "icd10")]:
    sub = dx[dx.icd_version == ver]
    if len(sub) == 0:
        continue
    pl_in = pl.DataFrame({"hadm_id": sub.hadm_id.astype("int64").tolist(), "icd_code": sub.icd_code.astype(str).tolist()})
    res = comorbidity(pl_in,
                      id_col="hadm_id", code_col="icd_code", score="elixhauser", icd=tag,
                      variant="quan", weighting="van_walraven", assign0=True)
    res = pd.DataFrame(res.to_dict(as_series=False))
    parts.append(res)
elix = pd.concat(parts).groupby("hadm_id")[ELIX].max()          # an admission is coded in one ICD version
# van Walraven score recomputed on the combined flags (Quan-mapped categories, van Walraven 2009 weights)
VW = {"chf": 7, "carit": 5, "valv": -1, "pcd": 4, "pvd": 2, "hypunc": 0, "hypc": 0, "para": 7, "ond": 6,
      "cpd": 3, "diabunc": 0, "diabc": 0, "hypothy": 0, "rf": 5, "ld": 11, "pud": 0, "aids": 0, "lymph": 9,
      "metacanc": 12, "solidtum": 4, "rheumd": 0, "coag": 3, "obes": -4, "wloss": 6, "fed": 5, "blane": -2,
      "dane": -2, "alcohol": 0, "drug": -7, "psycho": 0, "depre": -3}
elix = elix.reindex(co.hadm_id).fillna(0).astype(int)
elix["elix_vw_score"] = sum(elix[k] * w for k, w in VW.items())
elix.columns = ["elix_" + c if not c.startswith("elix_") else c for c in elix.columns]
static = co[["subject_id", "hadm_id", "stay_id", "age_at_admission", "gender", "cancer_type"]].merge(
    elix.reset_index(), on="hadm_id", how="left", validate="1:1")
static["sex_male"] = (static.gender == "M").astype(int)
static = static.drop(columns=["gender"])
assert len(static) == N and static[[c for c in static.columns if c.startswith("elix_")]].notna().all().all()
static.to_csv(P2 + "static_comorbidities.csv", index=False)

# ---------------- checks ----------------
ver = dx.groupby("hadm_id").icd_version.max().reindex(co.hadm_id).values
flags = [c for c in static.columns if c.startswith("elix_") and c != "elix_vw_score"]
prev = pd.DataFrame({
    "overall_%": (100 * static[flags].mean()).round(1),
    "ICD-9 admissions_%": (100 * static.loc[ver == 9, flags].mean()).round(1),
    "ICD-10 admissions_%": (100 * static.loc[ver == 10, flags].mean()).round(1),
})
prev["label"] = [colnames["elixhauser"][c.replace("elix_", "")].strip() for c in prev.index]
print("\n=== ELIXHAUSER PREVALENCE (the old bug made ICD-10 near zero for range-coded categories) ===")
print(prev.sort_values("overall_%", ascending=False).to_string())
print(f"\nvan Walraven score: median {static.elix_vw_score.median():.0f}, IQR "
      f"{static.elix_vw_score.quantile(.25):.0f}-{static.elix_vw_score.quantile(.75):.0f}")
print("\nSaved processed_v2/static_comorbidities.csv", static.shape)

In [ ]:
import pandas as pd, numpy as np

P2 = "../data/processed_v2/"
co = pd.read_csv(P2 + "cohort_final.csv")
N = len(co); ids = co.stay_id
L = pd.read_csv(P2 + "clean_long.csv.gz", parse_dates=["charttime"])
VA = pd.read_csv(P2 + "clean_vasoactive.csv.gz")
CTX = pd.read_csv(P2 + "clean_patient_context.csv").set_index("stay_id")
ST = pd.read_csv(P2 + "static_comorbidities.csv").set_index("stay_id")

def var(name):
    return L[L.variable == name]
def worst(name, fn):
    return var(name).groupby("stay_id").value.agg(fn).reindex(ids)

# ------------------------------------------------------------
# Arterial blood gas + FiO2 pairing (FiO2 charted in the 4 h BEFORE the gas;
# if none charted and the patient had NO oxygen support at all in 24 h -> room air 0.21)
# ------------------------------------------------------------
abg = var("PaO2_art")[["stay_id", "charttime", "value"]].rename(columns={"value": "pao2"})
pc = var("PaCO2_art")[["stay_id", "charttime", "value"]].rename(columns={"value": "paco2"})
abg = abg.merge(pc, on=["stay_id", "charttime"], how="left")
fio2 = var("FiO2")[["stay_id", "charttime", "value"]].rename(columns={"value": "fio2", "charttime": "fio2_time"})
abg = abg.sort_values("charttime"); fio2 = fio2.sort_values("fio2_time")
abg = pd.merge_asof(abg, fio2, left_on="charttime", right_on="fio2_time", by="stay_id",
                    direction="backward", tolerance=pd.Timedelta(hours=4))
no_support = ~abg.stay_id.map(CTX.oxygen_support_24h).fillna(False).astype(bool)
abg.loc[abg.fio2.isna() & no_support, "fio2"] = 0.21
abg["pf"] = abg.pao2 / abg.fio2
support = (CTX.invasive_vent_24h | CTX.niv_24h).reindex(ids)

# SpO2/FiO2 fallback (Pandharipande 2009) for patients with no usable PaO2/FiO2
sp = var("SpO2")[["stay_id", "charttime", "value"]].rename(columns={"value": "spo2"})
sp = sp[sp.spo2 <= 97].sort_values("charttime")                 # S/F only valid when SpO2 <= 97%
sp = pd.merge_asof(sp, fio2, left_on="charttime", right_on="fio2_time", by="stay_id",
                   direction="backward", tolerance=pd.Timedelta(hours=1))
sp.loc[sp.fio2.isna() & ~sp.stay_id.map(CTX.oxygen_support_24h).fillna(False).astype(bool), "fio2"] = 0.21
sp["sf"] = sp.spo2 / sp.fio2

def resp_from_ratio(r, cuts, supp):
    if pd.isna(r): return np.nan
    if r < cuts[3] and supp: return 4
    if r < cuts[2] and supp: return 3
    if r < cuts[2]: return 2          # without respiratory support the score is capped at 2
    if r < cuts[1]: return 2
    if r < cuts[0]: return 1
    return 0
min_pf = abg.groupby("stay_id").pf.min().reindex(ids)
min_sf = sp.groupby("stay_id").sf.min().reindex(ids)
resp_pf = [resp_from_ratio(r, (400, 300, 200, 100), s) for r, s in zip(min_pf, support)]
resp_sf = [resp_from_ratio(r, (512, 357, 214, 89), s) for r, s in zip(min_sf, support)]
resp = pd.Series(np.where(pd.notna(resp_pf), resp_pf, resp_sf), index=ids)
resp_source = pd.Series(np.where(pd.notna(resp_pf), "PaO2/FiO2", np.where(pd.notna(resp_sf), "SpO2/FiO2", "missing")), index=ids)

# ------------------------------------------------------------
# SOFA (Vincent 1996), worst value in first 24 h
# ------------------------------------------------------------
plt_ = worst("Platelets", "min"); coag = plt_.apply(lambda x: np.nan if pd.isna(x) else 4 if x < 20 else 3 if x < 50 else 2 if x < 100 else 1 if x < 150 else 0)
bili = worst("Bilirubin", "max"); liver = bili.apply(lambda x: np.nan if pd.isna(x) else 4 if x >= 12 else 3 if x >= 6 else 2 if x >= 2 else 1 if x >= 1.2 else 0)
gcs_min = worst("GCS_Total", "min"); cns = gcs_min.apply(lambda x: np.nan if pd.isna(x) else 0 if x >= 15 else 1 if x >= 13 else 2 if x >= 10 else 3 if x >= 6 else 4)
cr = worst("Creatinine", "max")
cr_pts = cr.apply(lambda x: np.nan if pd.isna(x) else 4 if x >= 5 else 3 if x >= 3.5 else 2 if x >= 2 else 1 if x >= 1.2 else 0)
uo = CTX.urine_24h_ml.reindex(ids); uo_ok = CTX.urine_complete_24h.reindex(ids).fillna(False).astype(bool)
uo_pts = pd.Series(np.where(uo_ok & (uo < 200), 4, np.where(uo_ok & (uo < 500), 3, np.nan)), index=ids)
renal = pd.concat([cr_pts, uo_pts], axis=1).max(axis=1)

mx = VA.groupby(["stay_id", "drug"]).rate_std.max().unstack().reindex(ids)
for dname in ["Norepinephrine", "Epinephrine", "Dopamine", "Dobutamine"]:
    if dname not in mx: mx[dname] = np.nan
map_min = worst("MAP", "min")
def cardio(i):
    ne, ep, dp, db, m = mx.loc[i, "Norepinephrine"], mx.loc[i, "Epinephrine"], mx.loc[i, "Dopamine"], mx.loc[i, "Dobutamine"], map_min.loc[i]
    if (dp > 15) or (ep > 0.1) or (ne > 0.1): return 4
    if (dp > 5) or (0 < ep <= 0.1) or (0 < ne <= 0.1): return 3
    if (0 < dp <= 5) or (db > 0): return 2
    if pd.notna(m): return 1 if m < 70 else 0
    return np.nan
cv = pd.Series([cardio(i) for i in ids], index=ids)

sofa = pd.DataFrame({"sofa_resp": resp, "sofa_coag": coag, "sofa_liver": liver, "sofa_cardio": cv,
                     "sofa_cns": cns, "sofa_renal": renal})
comp = list(sofa.columns)
sofa["sofa_n_components"] = sofa[comp].notna().sum(axis=1)
sofa["sofa_total"] = sofa[comp].fillna(0).sum(axis=1)            # missing component = normal (0), standard convention
sofa["sofa_resp_source"] = resp_source

# ------------------------------------------------------------
# Modified APACHE-II (Knaus 1985), worst value in first 24 h
# ------------------------------------------------------------
def worst_points(name, fn):
    d = var(name).copy(); d["p"] = d.value.apply(fn)
    return d.groupby("stay_id").p.max().reindex(ids)
def p_temp(t): return 4 if t >= 41 or t < 30 else 3 if t >= 39 or t < 32 else 2 if t < 34 else 1 if t >= 38.5 or t < 36 else 0
def p_map(m): return 4 if m >= 160 or m < 50 else 3 if m >= 130 else 2 if m >= 110 or m < 70 else 0
def p_hr(h): return 4 if h >= 180 or h < 40 else 3 if h >= 140 or h < 55 else 2 if h >= 110 or h < 70 else 0
def p_rr(r): return 4 if r >= 50 or r < 6 else 3 if r >= 35 else 2 if r < 10 else 1 if r >= 25 or r < 12 else 0
def p_ph(x): return 4 if x >= 7.7 or x < 7.15 else 3 if x >= 7.6 or x < 7.25 else 2 if x < 7.33 else 1 if x >= 7.5 else 0
def p_na(x): return 4 if x >= 180 or x <= 110 else 3 if x >= 160 or x < 120 else 2 if x >= 155 or x < 130 else 1 if x >= 150 else 0
def p_k(x): return 4 if x >= 7 or x < 2.5 else 3 if x >= 6 else 2 if x < 3 else 1 if x >= 5.5 or x < 3.5 else 0
def p_cr(x): return 4 if x >= 3.5 else 3 if x >= 2 else 2 if x >= 1.5 or x < 0.6 else 0
def p_hct(x): return 4 if x >= 60 or x < 20 else 2 if x >= 50 or x < 30 else 1 if x >= 46 else 0
def p_wbc(x): return 4 if x >= 40 or x < 1 else 2 if x >= 20 or x < 3 else 1 if x >= 15 else 0
def p_oxy(r):
    if pd.isna(r.fio2) or pd.isna(r.pao2): return np.nan
    if r.fio2 >= 0.5:
        if pd.isna(r.paco2): return np.nan
        a = r.fio2 * 713 - r.paco2 / 0.8 - r.pao2
        return 4 if a >= 500 else 3 if a >= 350 else 2 if a >= 200 else 0
    return 0 if r.pao2 > 70 else 1 if r.pao2 >= 61 else 3 if r.pao2 >= 55 else 4
abg["oxy_p"] = abg.apply(p_oxy, axis=1)
ap = pd.DataFrame({
    "ap_temp": worst_points("Temp", p_temp), "ap_map": worst_points("MAP", p_map), "ap_hr": worst_points("HR", p_hr),
    "ap_rr": worst_points("RR", p_rr), "ap_oxygenation": abg.groupby("stay_id").oxy_p.max().reindex(ids),
    "ap_ph": worst_points("pH_art", p_ph), "ap_sodium": worst_points("Sodium", p_na), "ap_potassium": worst_points("Potassium", p_k),
    "ap_creatinine": worst_points("Creatinine", p_cr), "ap_hct": worst_points("Hematocrit", p_hct), "ap_wbc": worst_points("WBC", p_wbc),
    "ap_gcs": (15 - gcs_min)})
apc = list(ap.columns)
ap["apache_n_components"] = ap[apc].notna().sum(axis=1)
ap["apache_aps"] = ap[apc].fillna(0).sum(axis=1)
age = co.set_index("stay_id").age_at_admission.reindex(ids)
ap["apache_age_pts"] = age.apply(lambda a: 0 if a <= 44 else 2 if a <= 54 else 3 if a <= 64 else 5 if a <= 74 else 6)
immuno = (ST.cancer_type.eq("Hematologic") | ST.elix_lymph.eq(1) | ST.elix_aids.eq(1)).reindex(ids)
organ = (ST.elix_ld.eq(1) | ST.elix_chf.eq(1) | ST.elix_cpd.eq(1) | ST.elix_rf.eq(1)).reindex(ids)
ap["apache_chronic_pts"] = np.where(immuno | organ, 5, 0)       # non-operative admission assumed (5 points)
ap["apache_ii_total"] = ap.apache_aps + ap.apache_age_pts + ap.apache_chronic_pts

scores = pd.concat([sofa, ap], axis=1).reset_index()
assert len(scores) == N and scores.stay_id.is_unique
scores.to_csv(P2 + "severity_scores.csv", index=False)

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------
print("=== SOFA ===")
for c in comp:
    print(f"{c:12s} available {scores[c].notna().mean():6.1%} | distribution {scores[c].value_counts().sort_index().to_dict()}")
print("Respiratory source:", scores.sofa_resp_source.value_counts().to_dict())
print(f"SOFA total: median {scores.sofa_total.median():.0f} (IQR {scores.sofa_total.quantile(.25):.0f}-{scores.sofa_total.quantile(.75):.0f})")
print("\nCardiovascular SOFA vs vasoactive exposure (rows = any vasopressor/inotrope in 24h):")
exposed = scores.stay_id.isin(VA.stay_id)
print(pd.crosstab(exposed, scores.sofa_cardio.fillna(-1)).to_string())
print("\n=== MODIFIED APACHE-II ===")
for c in apc:
    print(f"{c:15s} available {scores[c].notna().mean():6.1%}")
print(f"APACHE-II total: median {scores.apache_ii_total.median():.0f} (IQR {scores.apache_ii_total.quantile(.25):.0f}-{scores.apache_ii_total.quantile(.75):.0f})")
print(f"Chronic-health points given to {(scores.apache_chronic_pts > 0).mean():.1%} of patients")
print("\nSaved processed_v2/severity_scores.csv", scores.shape)

In [ ]:
# ===================== STEP 7 (Notebook 2): basic features + TPIFE 4-stage framework =====================
# Reads ONLY processed_v2/clean_long.csv.gz and processed_v2/cohort_final.csv (both built in the redo).
# Writes ONLY to processed_v2/. No outcome column is read in this cell.
import pandas as pd, numpy as np, os, warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

P2 = "../data/processed_v2/"
co = pd.read_csv(P2 + "cohort_final.csv", usecols=["stay_id"])     # stay list only - NOT the outcome
N = len(co)
L = pd.read_csv(P2 + "clean_long.csv.gz")
assert L.hours.max() <= 24 + 1e-9, "data after 24h found - stop"

VITALS = ["HR", "MAP", "SBP", "RR", "SpO2", "Temp", "GCS_Total"]
LABS = ["Lactate", "Creatinine", "Platelets", "Bilirubin", "WBC", "Sodium", "Potassium",
        "Hematocrit", "pH_art", "PaO2_art", "PaCO2_art"]
# Window rule: vitals 0-24h, labs -6h to 24h (already applied in Step 3; re-checked here)
L = L[((L.variable.isin(VITALS)) & (L.hours >= 0)) | ((L.variable.isin(LABS)) & (L.hours >= -6))]

# ---------- Stage 2 thresholds (fixed in advance from standard clinical cut-offs, NOT tuned on outcomes) ----------
# (severe_low, mild_low, mild_high, severe_high); None = no limit on that side
TH = {"HR": (40, 60, 100, 130), "MAP": (55, 65, 110, 130), "SBP": (80, 90, 160, 180), "RR": (8, 12, 20, 30),
      "SpO2": (88, 92, None, None), "Temp": (35.0, 36.0, 38.0, 39.0), "GCS_Total": (9, 13, None, None),
      "Lactate": (None, None, 2.0, 4.0), "Creatinine": (None, None, 1.2, 2.0), "Platelets": (50, 150, None, None),
      "Bilirubin": (None, None, 1.2, 2.0), "WBC": (2, 4, 12, 20), "Sodium": (125, 135, 145, 155),
      "Potassium": (2.5, 3.5, 5.0, 6.0), "Hematocrit": (21, 30, None, None), "pH_art": (7.20, 7.35, 7.45, 7.55),
      "PaO2_art": (55, 80, None, None), "PaCO2_art": (25, 35, 45, 60)}
# How long one reading is assumed to stay valid (last value carried forward, capped). Temp/GCS are charted ~4-hourly.
CAP = {v: 3.0 for v in VITALS}; CAP.update({"Temp": 6.0, "GCS_Total": 6.0})
SEGS = [(0, 6, "early"), (6, 12, "mid"), (12, 24, "late")]          # Stage 1 segmentation

def state(x, th):
    sl, ml, mh, sh = th
    s = np.zeros(len(x), dtype=int)
    if ml is not None: s[x < ml] = 1
    if mh is not None: s[x > mh] = 1
    if sl is not None: s[x < sl] = 2
    if sh is not None: s[x > sh] = 2
    return s

def deviation(x, th):                     # how far beyond the normal band (0 inside the band)
    _, ml, mh, _ = th
    d = np.zeros(len(x))
    if ml is not None: d = np.maximum(d, ml - x)
    if mh is not None: d = np.maximum(d, x - mh)
    return d

def basic(t, x, p):
    f = {f"{p}_n": len(x), f"{p}_mean": x.mean(), f"{p}_min": x.min(), f"{p}_max": x.max(),
         f"{p}_std": x.std(ddof=1) if len(x) > 1 else np.nan, f"{p}_first": x[0], f"{p}_last": x[-1],
         f"{p}_delta": x[-1] - x[0] if len(x) > 1 else np.nan, f"{p}_slope": np.nan}
    if len(x) >= 3 and np.ptp(t) > 0:
        f[f"{p}_slope"] = np.polyfit(t, x, 1)[0]                      # change per hour
    return f

def tpife(t, x, v):
    th = TH[v]; s = state(x, th); dev = deviation(x, th)
    end = np.minimum(np.append(t[1:], 24.0), t + CAP[v]); end = np.minimum(end, 24.0)
    dur = np.clip(end - t, 0, None)                                    # time each reading covers
    obs = dur.sum()
    f = {f"{v}_obs_hours": obs}
    if obs <= 0:
        return f
    # Stage 3: extraction
    t_norm, t_mild, t_sev = dur[s == 0].sum(), dur[s == 1].sum(), dur[s == 2].sum()
    tot = t_norm + t_mild + t_sev                                      # share = part / total, so it cannot exceed 1
    f[f"{v}_frac_abn"] = (t_mild + t_sev) / tot                        # share of observed time abnormal
    f[f"{v}_frac_severe"] = t_sev / tot
    f[f"{v}_auc_dev"] = (dev * dur).sum() / obs                        # average depth beyond normal range
    gap = np.append(t[1:] > end[:-1] + 1e-9, False)                    # unobserved gap after reading i
    trans = int(((s[1:] != s[:-1]) & ~gap[:-1]).sum())
    f[f"{v}_transitions_per24h"] = trans * 24 / obs                    # instability
    ep, cur, eps = [], 0.0, 0                                          # abnormal episodes
    first_ep_dur, first_done = 0.0, False
    for i in range(len(x)):
        if s[i] >= 1:
            if cur == 0: eps += 1
            cur += dur[i]
        if cur > 0 and (s[i] == 0 or gap[i] or i == len(x) - 1):
            ep.append(cur)
            if not first_done: first_ep_dur, first_done = cur, True
            cur = 0.0
    f[f"{v}_episodes_per24h"] = eps * 24 / obs                         # frequency
    f[f"{v}_longest_episode_h"] = max(ep) if ep else 0.0               # persistence
    f[f"{v}_first_episode_h"] = first_ep_dur                           # recovery time (0 = never abnormal)
    f[f"{v}_unresolved"] = int(s[-1] >= 1)                             # still abnormal at last reading
    seg_frac = {}
    for a, b, name in SEGS:                                            # Stage 1 segments
        ov = np.clip(np.minimum(end, b) - np.maximum(t, a), 0, None)   # overlap of each reading with the segment
        if ov.sum() > 0:
            f[f"{v}_{name}_mean"] = (x * ov).sum() / ov.sum()          # time-weighted segment mean
            o_abn, o_norm = ov[s >= 1].sum(), ov[s == 0].sum()
            seg_frac[name] = o_abn / (o_abn + o_norm)                  # always within 0-1
    if "early" in seg_frac and "late" in seg_frac:
        f[f"{v}_late_minus_early_mean"] = f[f"{v}_late_mean"] - f[f"{v}_early_mean"]
        f[f"{v}_deterioration"] = seg_frac["late"] - seg_frac["early"]  # >0: more time abnormal late than early
    return f

rows = {sid: {} for sid in co.stay_id}
for (sid, v), g in L.sort_values(["stay_id", "variable", "hours"]).groupby(["stay_id", "variable"], sort=False):
    if sid not in rows or (v not in VITALS and v not in LABS):
        continue
    g = g.groupby("hours", as_index=False).value.mean()               # same-time duplicates -> one value
    t, x = g.hours.to_numpy(float), g.value.to_numpy(float)
    f = basic(t, x, v)
    if v in VITALS:
        f.update(tpife(t, x, v))
    else:                                                              # labs are too sparse for time-in-state
        s = state(x, TH[v])
        f.update({f"{v}_ever_abn": int((s >= 1).any()), f"{v}_ever_severe": int((s == 2).any()),
                  f"{v}_last_state": int(s[-1])})
    rows[sid].update(f)

F = pd.DataFrame.from_dict(rows, orient="index"); F.index.name = "stay_id"

# ---------- Stage 4: aggregation across vital signs ----------
fa = F[[f"{v}_frac_abn" for v in VITALS if f"{v}_frac_abn" in F]]
F["tpife_mean_frac_abn"] = fa.mean(axis=1)
F["tpife_n_vitals_ever_severe"] = (F[[f"{v}_frac_severe" for v in VITALS if f"{v}_frac_severe" in F]] > 0).sum(axis=1)
F["tpife_n_vitals_deteriorating"] = (F[[f"{v}_deterioration" for v in VITALS if f"{v}_deterioration" in F]] > 0).sum(axis=1)
F["tpife_n_vitals_unresolved"] = F[[f"{v}_unresolved" for v in VITALS if f"{v}_unresolved" in F]].sum(axis=1)
F["tpife_n_vitals_observed"] = fa.notna().sum(axis=1)
F = F.reset_index()

# ---------- checks ----------
assert len(F) == N and F.stay_id.is_unique and set(F.stay_id) == set(co.stay_id)
num = F.drop(columns="stay_id")
assert np.isfinite(num.to_numpy(float)[~np.isnan(num.to_numpy(float))]).all(), "infinite values"
TOL = 1e-9                                                             # floating-point rounding noise only
for v in VITALS:
    for c, lo, hi in [(f"{v}_frac_abn", 0, 1), (f"{v}_frac_severe", 0, 1), (f"{v}_deterioration", -1, 1)]:
        if c not in F:
            continue
        out = np.maximum(F[c] - hi, lo - F[c])                         # distance outside the allowed range
        if out.max() > 0:
            print(f"{c}: largest overshoot = {out.max():.3e} in {(out > 0).sum()} patients")
        if out.max() > TOL:                                            # bigger than rounding noise = real error
            print(F.loc[out > TOL, ["stay_id", c]].head())
            raise AssertionError(f"{c} truly out of range - send me this output")
        F[c] = F[c].clip(lo, hi)
print("Range check passed: all time-share features are within their limits.")

F.to_csv(P2 + "features_temporal.csv", index=False)
pd.DataFrame([{"variable": k, "severe_low": a, "mild_low": b, "mild_high": c, "severe_high": d,
               "carry_forward_cap_h": CAP.get(k, np.nan)} for k, (a, b, c, d) in TH.items()]
             ).to_csv(P2 + "tpife_thresholds.csv", index=False)

print(f"Saved processed_v2/features_temporal.csv  shape={F.shape}")
print("\nPatients with at least one reading (of", N, "):")
print(pd.Series({v: int(F[f"{v}_n"].notna().sum()) for v in VITALS + LABS if f"{v}_n" in F}).to_string())
show = ["HR_mean", "MAP_mean", "MAP_frac_abn", "MAP_frac_severe", "MAP_longest_episode_h", "HR_frac_abn",
        "SpO2_frac_abn", "RR_transitions_per24h", "GCS_Total_frac_abn", "Temp_deterioration",
        "MAP_obs_hours", "Lactate_max", "Lactate_ever_severe", "Creatinine_delta",
        "tpife_mean_frac_abn", "tpife_n_vitals_ever_severe", "tpife_n_vitals_deteriorating", "tpife_n_vitals_unresolved"]
d = F[[c for c in show if c in F]].describe(percentiles=[.25, .5, .75]).T[["count", "25%", "50%", "75%", "max"]]
print("\nKey feature summaries:\n", d.round(2).to_string())
miss = F.drop(columns="stay_id").isna().mean().sort_values(ascending=False)
n_feat = F.shape[1] - 1
n_high = int((miss > .2).sum())
print("\nFeatures:", n_feat)
print("Features with >20% missing:", n_high)
print("(the 20% rule is applied on TRAINING data only, in NB3)")
print("Most-missing features:")
print((100 * miss.head(8)).round(1).to_string())

In [ ]:
# ========= STEP 8 (Notebook 2): Organ-System Burden Index + stability components =========
# Reads ONLY processed_v2 files built in the redo. Writes ONLY to processed_v2/.
# No outcome column is read. All cut-offs are fixed clinical rules (nothing is fitted here).
import pandas as pd, numpy as np

P2 = "../data/processed_v2/"
co = pd.read_csv(P2 + "cohort_final.csv", usecols=["stay_id"])   # stay list only
ids = co.stay_id
N = len(co)
L = pd.read_csv(P2 + "clean_long.csv.gz", parse_dates=["charttime"])
VA = pd.read_csv(P2 + "clean_vasoactive.csv.gz")
CTX = pd.read_csv(P2 + "clean_patient_context.csv").set_index("stay_id").reindex(ids)
assert L.hours.max() <= 24 + 1e-9, "data after 24h found - stop"

def var(name):
    return L[L.variable == name]

# ---------------- A. Vasopressors (norepinephrine equivalents, NEE) ----------------
PRESSORS = ["Norepinephrine", "Epinephrine", "Dopamine", "Phenylephrine", "Vasopressin"]
V = VA.copy()
V["s"] = V.start_h.clip(0, 24)
V["e"] = V.end_h.clip(0, 24)
V = V[V.e > V.s]                                   # keep only time inside 0-24h

def pressor_summary(g):
    p = g[g.drug.isin(PRESSORS)]
    out = {"vaso_any": int(len(p) > 0),
           "dobutamine_any": int((g.drug == "Dobutamine").any()),
           "n_vasoactive_agents": g.drug.nunique()}
    if len(p) == 0:
        out.update({"nee_max": 0.0, "nee_mean_24h": 0.0, "vaso_hours": 0.0})
        return pd.Series(out)
    # hours on any vasopressor (union of intervals, overlaps counted once)
    iv = p[["s", "e"]].sort_values("s").to_numpy()
    hours, cs, ce = 0.0, iv[0, 0], iv[0, 1]
    for s, e in iv[1:]:
        if s > ce:
            hours += ce - cs
            cs, ce = s, e
        else:
            ce = max(ce, e)
    hours += ce - cs
    # highest TOTAL dose at any moment (drugs running together are added up)
    q = p[p.nee.notna()]
    if len(q) == 0:
        nee_max, nee_mean = np.nan, np.nan          # on a pressor but dose not usable
    else:
        cuts = np.unique(np.r_[q.s, q.e])
        mids = (cuts[:-1] + cuts[1:]) / 2
        tot = [q.nee[(q.s <= m) & (q.e > m)].sum() for m in mids]
        nee_max = float(max(tot)) if tot else 0.0
        nee_mean = float((q.nee * (q.e - q.s)).sum() / 24)
    out.update({"nee_max": nee_max, "nee_mean_24h": nee_mean, "vaso_hours": hours})
    return pd.Series(out)

vaso = V.groupby("stay_id").apply(pressor_summary).reindex(ids)
fill0 = {"vaso_any": 0, "dobutamine_any": 0, "n_vasoactive_agents": 0,
         "nee_max": 0.0, "nee_mean_24h": 0.0, "vaso_hours": 0.0}
none_rows = ~ids.isin(V.stay_id).to_numpy()
vaso.loc[none_rows] = vaso.loc[none_rows].fillna(fill0)   # no drug at all -> 0, not missing

# ---------------- B. Respiratory: FiO2, PaO2/FiO2, SpO2/FiO2, support level ----------------
o2 = CTX.oxygen_support_24h.fillna(False).astype(bool)
fio2 = var("FiO2")[["stay_id", "charttime", "value"]]
fio2 = fio2.rename(columns={"value": "fio2", "charttime": "fio2_time"})
fio2 = fio2.sort_values("fio2_time")
fio2_max = fio2.groupby("stay_id").fio2.max().reindex(ids)
fio2_max[fio2_max.isna() & ~o2] = 0.21              # no oxygen support at all -> room air

abg = var("PaO2_art")[["stay_id", "charttime", "value"]].rename(columns={"value": "pao2"})
abg = pd.merge_asof(abg.sort_values("charttime"), fio2, left_on="charttime",
                    right_on="fio2_time", by="stay_id", direction="backward",
                    tolerance=pd.Timedelta(hours=4))
abg.loc[abg.fio2.isna() & ~abg.stay_id.map(o2).fillna(False).astype(bool), "fio2"] = 0.21
pf_min = (abg.pao2 / abg.fio2).groupby(abg.stay_id).min().reindex(ids)

sp = var("SpO2")[["stay_id", "charttime", "value"]].rename(columns={"value": "spo2"})
sp = sp[sp.spo2 <= 97].sort_values("charttime")
sp = pd.merge_asof(sp, fio2, left_on="charttime", right_on="fio2_time", by="stay_id",
                   direction="backward", tolerance=pd.Timedelta(hours=1))
sp.loc[sp.fio2.isna() & ~sp.stay_id.map(o2).fillna(False).astype(bool), "fio2"] = 0.21
sf_min = (sp.spo2 / sp.fio2).groupby(sp.stay_id).min().reindex(ids)

inv = CTX.invasive_vent_24h.fillna(False).astype(bool)
niv = CTX.niv_24h.fillna(False).astype(bool)
support_level = np.select([inv, niv, o2], [3, 2, 1], 0)   # 0 none,1 oxygen,2 NIV,3 invasive

# ---------------- C. Renal: urine output rate (only when charting is complete) ----------------
uo_ok = CTX.urine_complete_24h.fillna(False).astype(bool)
uo_rate = CTX.urine_24h_ml / CTX.weight_kg / CTX.urine_monitored_hours
uo_rate = uo_rate.where(uo_ok)                     # incomplete charting -> missing, not low

# ---------------- D. Organ-System Burden Index (7 systems, each 0 to 1) ----------------
CAP = 3.0
def low_time_burden(name, mild, severe, cap=CAP):
    """Share of observed 0-24h time below cut-offs: mild counts 0.5, severe counts 1."""
    res = {}
    for sid, g in var(name).sort_values("hours").groupby("stay_id"):
        g = g.groupby("hours", as_index=False).value.mean()
        t, x = g.hours.to_numpy(float), g.value.to_numpy(float)
        end = np.minimum(np.minimum(np.append(t[1:], 24.0), t + cap), 24.0)
        dur = np.clip(end - t, 0, None)
        m = dur[(x < mild) & (x >= severe)].sum()
        s = dur[x < severe].sum()
        n = dur[x >= mild].sum()
        tot = m + s + n
        if tot > 0:
            res[sid] = ((0.5 * m + s) / tot, int((x < severe).any()), int((x < mild).any()))
    out = pd.DataFrame.from_dict(res, orient="index", columns=["b", "sev", "any"])
    return out.reindex(ids)

def level(x, mild, severe, high=True):
    """0 normal, 0.5 mild, 1 severe for a single worst value."""
    if high:
        return np.where(x.isna(), np.nan, np.where(x > severe, 1, np.where(x > mild, 0.5, 0)))
    return np.where(x.isna(), np.nan, np.where(x < severe, 1, np.where(x < mild, 0.5, 0)))

def worst(name, fn):
    return var(name).groupby("stay_id").value.agg(fn).reindex(ids)

mapb = low_time_burden("MAP", 65, 55)
spob = low_time_burden("SpO2", 92, 88)
gcsb = low_time_burden("GCS_Total", 13, 9, cap=6.0)

nee = vaso.nee_max.astype(float)
vaso_lvl = np.where(vaso.vaso_any == 0, 0.0,
                    np.where(nee > 0.1, 1.0, 0.5))          # any pressor 0.5, NEE > 0.1 = 1
sup_lvl = pd.Series(support_level, index=ids).map({0: 0.0, 1: 0.0, 2: 0.5, 3: 1.0})
uo_lvl = np.where(uo_rate.isna(), np.nan,
                  np.where(uo_rate < 0.3, 1.0, np.where(uo_rate < 0.5, 0.5, 0.0)))  # KDIGO

def combine(*parts):
    return pd.concat([pd.Series(np.asarray(p, float), index=ids) for p in parts],
                     axis=1).max(axis=1)              # worst signal in the system; NaN if none

SYS = {
    "cardiovascular": combine(mapb.b, vaso_lvl),
    "respiratory": combine(spob.b, sup_lvl, level(pf_min, 300, 200, high=False)),
    "renal": combine(level(worst("Creatinine", "max"), 1.2, 2.0), uo_lvl),
    "hepatic": combine(level(worst("Bilirubin", "max"), 1.2, 2.0)),
    "coagulation": combine(level(worst("Platelets", "min"), 150, 50, high=False)),
    "neurological": combine(gcsb.b),
    "metabolic": combine(level(worst("Lactate", "max"), 2.0, 4.0),
                         level(worst("pH_art", "min"), 7.35, 7.20, high=False)),
}
O = pd.DataFrame({f"osbi_{k}": v for k, v in SYS.items()})
sys_cols = list(O.columns)
O["osbi_total"] = O[sys_cols].fillna(0).sum(axis=1)       # missing system = normal (as in SOFA)
O["osbi_n_systems_observed"] = O[sys_cols].notna().sum(axis=1)
O["osbi_n_systems_involved"] = (O[sys_cols] > 0).sum(axis=1)
cr_max, bili_max = worst("Creatinine", "max"), worst("Bilirubin", "max")
plt_min = worst("Platelets", "min")
lac_max = worst("Lactate", "max")
ph_min = worst("pH_art", "min")
severe = pd.DataFrame({                                   # reached a severe level at any time
    "cardiovascular": (nee > 0.1) | (mapb.sev == 1),
    "respiratory": inv | (pf_min < 200) | (spob.sev == 1),
    "renal": (cr_max > 2.0) | (uo_rate < 0.3),
    "hepatic": bili_max > 2.0,
    "coagulation": plt_min < 50,
    "neurological": gcsb.sev == 1,
    "metabolic": (lac_max > 4.0) | (ph_min < 7.20)})
O["osbi_n_systems_severe"] = severe.sum(axis=1).astype(int)

# ---------------- E. Stability components (combined into the index in NB3) ----------------
# ARV = average change between consecutive hourly medians; CV = std / mean.
# Mixing different units needs scaling, and scaling is learned on TRAINING data only (NB3).
stab = {}
for v in ["HR", "MAP", "RR", "SpO2"]:
    d = var(v).copy()
    d["hr"] = np.floor(d.hours.clip(0, 23.999)).astype(int)
    h = d.groupby(["stay_id", "hr"]).value.median().reset_index()
    h["dv"] = h.groupby("stay_id").value.diff().abs()
    h["dh"] = h.groupby("stay_id").hr.diff()
    stab[f"stab_{v}_arv"] = h[h.dh == 1].groupby("stay_id").dv.mean().reindex(ids)
    g = var(v).groupby("stay_id").value
    stab[f"stab_{v}_cv"] = (g.std() / g.mean()).reindex(ids)
S = pd.DataFrame(stab)

# ---------------- assemble + checks + save ----------------
OUT = pd.concat([vaso.astype(float), pd.DataFrame({
    "fio2_max": fio2_max, "pf_min": pf_min, "sf_min": sf_min,
    "resp_support_level": support_level, "uo_ml_kg_h": uo_rate}, index=ids), O, S], axis=1)
OUT.index.name = "stay_id"
OUT = OUT.reset_index()
assert len(OUT) == N and OUT.stay_id.is_unique
num = OUT.drop(columns="stay_id").to_numpy(float)
assert np.isfinite(num[~np.isnan(num)]).all(), "infinite values"
assert OUT[sys_cols].stack().between(0, 1).all()
assert (OUT.nee_max.dropna() >= 0).all() and OUT.fio2_max.dropna().between(0.21, 1).all()
OUT.to_csv(P2 + "features_organ_stability.csv", index=False)

print("Saved processed_v2/features_organ_stability.csv", OUT.shape)
on_p = OUT.vaso_any == 1
print("\nVasopressor (any, 0-24h):", int(on_p.sum()), "patients")
print("On a pressor but dose not usable:", int(OUT.nee_max[on_p].isna().sum()))
print("Respiratory support level counts (0 none,1 O2,2 NIV,3 invasive):",
      OUT.resp_support_level.value_counts().sort_index().to_dict())
print("Urine rate available (complete charting + weight):", int(OUT.uo_ml_kg_h.notna().sum()))
show = ["nee_max", "nee_mean_24h", "vaso_hours", "fio2_max", "pf_min", "sf_min", "uo_ml_kg_h",
        *sys_cols, "osbi_total", "osbi_n_systems_involved", "osbi_n_systems_severe",
        *S.columns]
d = OUT[show].describe(percentiles=[.25, .5, .75]).T
print("\nSummaries:")
print(d[["count", "25%", "50%", "75%", "max"]].round(3).to_string())
n_obs = OUT.osbi_n_systems_observed.value_counts().sort_index().to_dict()
print("\nSystems observed per patient:", n_obs)

In [ ]:
# ========= STEP 9 (Notebook 2): SIRS count + interaction features =========
# Reads ONLY processed_v2 files built in the redo. Writes ONLY to processed_v2/.
# No outcome column is read. Products use fixed rules only (nothing is fitted here).
import pandas as pd, numpy as np

P2 = "../data/processed_v2/"
co = pd.read_csv(P2 + "cohort_final.csv", usecols=["stay_id"])   # stay list only
ids = co.stay_id
N = len(co)
L = pd.read_csv(P2 + "clean_long.csv.gz")
SC = pd.read_csv(P2 + "severity_scores.csv").set_index("stay_id").reindex(ids)
OS = pd.read_csv(P2 + "features_organ_stability.csv").set_index("stay_id").reindex(ids)
assert L.hours.max() <= 24 + 1e-9, "data after 24h found - stop"

def var(name):
    return L[L.variable == name]

def worst(name, fn):
    return var(name).groupby("stay_id").value.agg(fn).reindex(ids)

# ---------------- A. SIRS: whole-number count 0-4 (Bone 1992), any time in 0-24h ----------------
t_max, t_min = worst("Temp", "max"), worst("Temp", "min")
hr_max, rr_max = worst("HR", "max"), worst("RR", "max")
pco2_min = worst("PaCO2_art", "min")
wbc_max, wbc_min = worst("WBC", "max"), worst("WBC", "min")
crit = pd.DataFrame({
    "sirs_temp": (t_max > 38) | (t_min < 36),
    "sirs_hr": hr_max > 90,
    "sirs_resp": (rr_max > 20) | (pco2_min < 32),
    "sirs_wbc": (wbc_max > 12) | (wbc_min < 4)}).astype(int)
sirs = crit.sum(axis=1).astype(int)                 # NaN comparisons count as "not met"
assert sirs.between(0, 4).all()

# ---------------- B. Hourly grid (0-23 h) for time-matched pairs ----------------
# Hourly median; a reading is carried forward at most 2 more hours (same 3 h rule as Step 7).
def hourly(name):
    d = var(name)
    d = d.assign(hr=np.floor(d.hours.clip(0, 23.999)).astype(int))
    d = d[d.hours >= 0]
    g = d.groupby(["stay_id", "hr"]).value.median().unstack()
    g = g.reindex(index=ids, columns=range(24))
    return g.ffill(axis=1, limit=2)

H = {v: hourly(v) for v in ["MAP", "HR", "SpO2", "SBP"]}

def hours_where(mask, *need):
    known = np.logical_and.reduce([h.notna() for h in need])
    out = (mask & known).sum(axis=1).astype(float)
    out[~known.any(axis=1)] = np.nan                  # no usable hour at all -> missing
    return out

hypo_h = hours_where(H["MAP"] < 65, H["MAP"])
hypox_tachy_h = hours_where((H["SpO2"] < 92) & (H["HR"] > 100), H["SpO2"], H["HR"])
si = H["HR"] / H["SBP"]
shock_index_max = si.max(axis=1)

# ---------------- C. Directional interaction pairs ----------------
lac_max = worst("Lactate", "max")
# 1) hypotension x lactate: hours with MAP < 65 times the highest lactate
hypo_x_lactate = hypo_h * lac_max
# 2) hypoxaemia AND tachycardia at the same hour (counted directly, not multiplied)
# 3) creatinine rise x low urine output (urine only when fully charted)
cr = var("Creatinine").sort_values("hours").groupby("stay_id").value
cr_rise = (cr.last() - cr.first()).clip(lower=0).reindex(ids)
cr_rise[cr.count().reindex(ids) < 2] = np.nan
uo = OS.uo_ml_kg_h
oliguria = np.where(uo.isna(), np.nan, (uo < 0.5).astype(float))
cr_rise_x_oliguria = cr_rise * oliguria
# SIRS x SOFA (SOFA total from Step 6)
sirs_x_sofa = sirs * SC.sofa_total

OUT = pd.DataFrame({
    "sirs_count": sirs, **crit,
    "sirs_x_sofa": sirs_x_sofa,
    "hypotension_hours": hypo_h, "hypotension_x_lactate": hypo_x_lactate,
    "hypoxaemia_tachycardia_hours": hypox_tachy_h,
    "creat_rise": cr_rise, "creat_rise_x_oliguria": cr_rise_x_oliguria,
    "shock_index_max": shock_index_max}, index=ids)
OUT.index.name = "stay_id"
OUT = OUT.reset_index()

# ---------------- checks + save ----------------
assert len(OUT) == N and OUT.stay_id.is_unique
num = OUT.drop(columns="stay_id").to_numpy(float)
assert np.isfinite(num[~np.isnan(num)]).all(), "infinite values"
assert OUT.hypotension_hours.dropna().between(0, 24).all()
assert OUT.hypoxaemia_tachycardia_hours.dropna().between(0, 24).all()
OUT.to_csv(P2 + "features_interactions.csv", index=False)

print("Saved processed_v2/features_interactions.csv", OUT.shape)
print("\nSIRS count distribution (whole numbers only):")
print(OUT.sirs_count.value_counts().sort_index().to_dict())
print("Each SIRS criterion met (%):", (100 * OUT[list(crit)].mean()).round(1).to_dict())
show = ["sirs_x_sofa", "hypotension_hours", "hypotension_x_lactate",
        "hypoxaemia_tachycardia_hours", "creat_rise", "creat_rise_x_oliguria",
        "shock_index_max"]
d = OUT[show].describe(percentiles=[.25, .5, .75]).T
print("\nSummaries:")
print(d[["count", "25%", "50%", "75%", "max"]].round(3).to_string())
print("\nPatients with any hour MAP < 65:", int((OUT.hypotension_hours > 0).sum()))
print("Patients with any hour SpO2 < 92 AND HR > 100:",
      int((OUT.hypoxaemia_tachycardia_hours > 0).sum()))

In [ ]:
# ========= STEP 10 (Notebook 2): assemble the ONE final feature table (allow-list) =========
# Reads ONLY processed_v2 files built in the redo. Writes ONLY to processed_v2/.
# Nothing is imputed, scaled or selected here - all of that happens on TRAINING data in NB3.
import pandas as pd, numpy as np

P2 = "../data/processed_v2/"
co = pd.read_csv(P2 + "cohort_final.csv")
N = len(co)

def load(name):
    d = pd.read_csv(P2 + name)
    assert len(d) == N and d.stay_id.is_unique and set(d.stay_id) == set(co.stay_id), name
    return d.set_index("stay_id")

ST = load("static_comorbidities.csv")
SC = load("severity_scores.csv")
CX = load("clean_patient_context.csv")
TF = load("features_temporal.csv")
OS = load("features_organ_stability.csv")
IX = load("features_interactions.csv")

# ---------------- allow-list: every feature is named on purpose ----------------
static_cols = ["age_at_admission", "sex_male", "cancer_type"]
# Elixhauser: MIMIC has no present-on-admission (POA) flag, so codes that are usually
# ACUTE hospital-course problems are not used as predictors (they may arise after 24h).
# fed + carit were removed from the AHRQ refined measures; coag, ond, wloss, pcd are
# POA-dependent in AHRQ CMR v2025 and mostly acute in ICU care. Kept for Table 1 only.
ELIX_ACUTE = ["elix_fed", "elix_carit", "elix_coag", "elix_ond", "elix_wloss", "elix_pcd"]
VW = {"chf": 7, "carit": 5, "valv": -1, "pcd": 4, "pvd": 2, "para": 7, "ond": 6,
      "cpd": 3, "rf": 5, "ld": 11, "lymph": 9, "metacanc": 12, "solidtum": 4,
      "coag": 3, "obes": -4, "wloss": 6, "fed": 5, "blane": -2, "dane": -2,
      "drug": -7, "depre": -3}                      # van Walraven 2009 (others weight 0)
ST = ST.copy()
ST["elix_vw_chronic"] = sum(ST["elix_" + k] * w for k, w in VW.items()
                            if "elix_" + k not in ELIX_ACUTE)
# ICD-9 -> ICD-10 coding shifted diabetes and hypertension between the
# "complicated" and "uncomplicated" groups, so each pair is merged (weight 0 in VW).
ST["elix_diabetes"] = ST[["elix_diabc", "elix_diabunc"]].max(axis=1)
ST["elix_hypertension"] = ST[["elix_hypc", "elix_hypunc"]].max(axis=1)
MERGED = ["elix_diabc", "elix_diabunc", "elix_hypc", "elix_hypunc"]
elix_cols = [c for c in ST.columns if c.startswith("elix_")
             and c not in ELIX_ACUTE + MERGED + ["elix_vw_score"]]
sev_cols = ["sofa_resp", "sofa_coag", "sofa_liver", "sofa_cardio", "sofa_cns", "sofa_renal",
            "sofa_total", "ap_temp", "ap_map", "ap_hr", "ap_rr", "ap_oxygenation", "ap_ph",
            "ap_sodium", "ap_potassium", "ap_creatinine", "ap_hct", "ap_wbc", "ap_gcs",
            "apache_aps", "apache_age_pts", "apache_chronic_pts", "apache_ii_total"]
ctx_cols = ["weight_kg", "gcs_verbal_ETT_24h"]          # ventilation is in resp_support_level
tf_cols = list(TF.columns)
os_cols = [c for c in OS.columns if c != "osbi_n_systems_observed"]   # bookkeeping only
ix_cols = list(IX.columns)

X = pd.concat([ST[static_cols + elix_cols], SC[sev_cols], CX[ctx_cols],
               TF[tf_cols], OS[os_cols], IX[ix_cols]], axis=1)
X = X.reindex(co.stay_id)

# ---------------- fixed physiological rules (not fitted on data) ----------------
n_pf = int((X.pf_min > 700).sum())
X.loc[X.pf_min > 700, "pf_min"] = np.nan        # PaO2/FiO2 > 700 is impossible -> FiO2 mismatch
X["gcs_verbal_ETT_24h"] = X.gcs_verbal_ETT_24h.astype(float)

# ---------------- identical columns: REPORTED only, not dropped here ----------------
# Dropping after looking at ALL patients is a data-driven choice -> done in NB3 on train only.
num_cols = [c for c in X.columns if c != "cancer_type"]
dups, seen = [], {}
for c in num_cols:
    key = tuple(X[c].fillna(-99999).round(10))
    if key in seen:
        dups.append((c, seen[key]))
    else:
        seen[key] = c
X = X.copy()

# ---------------- leakage guard: banned words in feature names ----------------
BANNED_WORDS = ["mortality", "death", "expire", "discharge", "outtime", "subtype",
                "anchor", "temporal", "immortal", "hospice", "calendar", "year",
                "delirium", "cam_"]
BANNED_TOKENS = ["dod", "los", "died", "dead", "label", "target", "outcome"]
bad = [c for c in X.columns
       if any(w in c.lower() for w in BANNED_WORDS)
       or any(t in c.lower().split("_") for t in BANNED_TOKENS)]
assert not bad, f"banned columns found: {bad}"
assert X.index.is_unique and len(X) == N
num = X.drop(columns="cancer_type").to_numpy(float)
assert np.isfinite(num[~np.isnan(num)]).all(), "infinite values"

# ---------------- final table: IDs + label + split tags + features ----------------
keys = co.set_index("stay_id")[["subject_id", "hadm_id", "mortality_28d",
                                "est_calendar_year", "temporal_set"]]
assert keys.subject_id.is_unique, "one ICU stay per patient expected"
T = keys.join(X).reset_index()
T["mortality_28d"] = T.mortality_28d.astype(int)
T.to_csv(P2 + "model_table_v2.csv", index=False)

# ---------------- feature dictionary (for the paper's supplement) ----------------
BASIC = ("_n", "_mean", "_min", "_max", "_std", "_first", "_last", "_delta", "_slope")
def group(c):
    if c in static_cols: return "demographics/cancer"
    if c.startswith("elix_"): return "comorbidity (Elixhauser)"
    if c in sev_cols: return "severity score"
    if c in ctx_cols: return "context"
    if c in os_cols:
        return "organ burden (OSBI)" if c.startswith("osbi_") else \
               "stability component" if c.startswith("stab_") else "organ support"
    if c in ix_cols: return "interaction / SIRS"
    if c.startswith("tpife_"): return "TPIFE stage 4 (aggregation)"
    if c.endswith(BASIC): return "basic temporal"
    return "TPIFE stages 1-3"
feat = [c for c in T.columns if c not in ["stay_id"] + list(keys.columns)]
D = pd.DataFrame({"feature": feat,
                  "group": [group(c) for c in feat],
                  "n_unique": [T[c].nunique() for c in feat],
                  "missing_pct": [round(100 * T[c].isna().mean(), 1) for c in feat]})
D["type"] = np.where(D.feature == "cancer_type", "categorical",
                     np.where(D.n_unique <= 2, "binary", "numeric"))
D.to_csv(P2 + "feature_dictionary_v2.csv", index=False)

# ---------------- report ----------------
print("Saved processed_v2/model_table_v2.csv", T.shape)
print("Saved processed_v2/feature_dictionary_v2.csv", D.shape)
print("\nFeatures:", len(feat))
print(D.group.value_counts().to_string())
print("\nTypes:", D.type.value_counts().to_dict())
print("\nIdentical column pairs (info only, handled in NB3 on training data):", len(dups))
for a, b in dups:
    print(f"   {a}  (same as {b})")
print("\nElixhauser used as predictors:", len(elix_cols), "->", elix_cols)
print("Acute-type Elixhauser kept for Table 1 only:", ELIX_ACUTE)
print("\nPaO2/FiO2 > 700 set to missing:", n_pf)
print("Patients with nee_max > 1.0:", int((T.nee_max > 1).sum()),
      "| > 2.0:", int((T.nee_max > 2).sum()))
const = D.feature[D.n_unique <= 1].tolist()
print("Constant features (removed in NB3 on training data):", const)
print("Features > 20% missing (whole cohort, info only):", int((D.missing_pct > 20).sum()))
print("\nOutcome (28-day death) by era:")
print(T.groupby("temporal_set").mortality_28d.agg(["count", "sum", "mean"]).round(3).to_string())
print("Overall:", int(T.mortality_28d.sum()), "/", len(T), f"({T.mortality_28d.mean():.1%})")
print("\nBanned-name check: PASS | one row per patient: PASS | no infinite values: PASS")

In [ ]:
# ========= FINAL VERIFICATION of Notebook 1 + Notebook 2 (run at the END of Notebook 2) =========
# Read-only: it changes no file. Every check prints PASS / FAIL. Go to NB3 only if all PASS.
import os, pandas as pd, numpy as np
from sklearn.metrics import roc_auc_score

P1, P2, FIG2 = "../data/processed/", "../data/processed_v2/", "../figures_v2/"
R = []
def check(section, name, ok, detail=""):
    R.append((section, name, "PASS" if ok else "FAIL", detail))

def mt(f):
    return os.path.getmtime(P2 + f)

# ---------------- 1. files present + built in the right order ----------------
EXPECTED = ["cohort_final.csv", "extract_chart.csv.gz", "extract_labs.csv.gz",
            "extract_inputs.csv.gz", "extract_output.csv.gz", "clean_long.csv.gz",
            "clean_vasoactive.csv.gz", "clean_urine.csv.gz", "clean_patient_context.csv",
            "cleaning_log.csv", "static_comorbidities.csv", "severity_scores.csv",
            "features_temporal.csv", "tpife_thresholds.csv", "features_organ_stability.csv",
            "features_interactions.csv", "model_table_v2.csv", "feature_dictionary_v2.csv",
            "cohort_delirium_timing.csv"]
present = set(os.listdir(P2))
missing = [f for f in EXPECTED if f not in present]
check("1 files", "all expected processed_v2 files exist", not missing, str(missing))
SUPP = ["attrition_counts_final.csv", "icd_code_appendix.csv", "icd_code_exclusions.csv"]
check("1 files", "supplement tables copied to processed_v2", all(f in present for f in SUPP))
extra = sorted(present - set(EXPECTED) - set(SUPP) - {"MANIFEST.txt"})
print("Other files in processed_v2 (not used by the pipeline):", extra or "none")

DEPS = {
    "cohort_delirium_timing.csv": ["cohort_final.csv"],
    "extract_chart.csv.gz": ["cohort_final.csv"],
    "extract_labs.csv.gz": ["cohort_final.csv"],
    "extract_inputs.csv.gz": ["cohort_final.csv"],
    "extract_output.csv.gz": ["cohort_final.csv"],
    "clean_long.csv.gz": ["extract_chart.csv.gz", "extract_labs.csv.gz"],
    "clean_vasoactive.csv.gz": ["extract_inputs.csv.gz"],
    "clean_patient_context.csv": ["extract_chart.csv.gz", "extract_output.csv.gz"],
    "static_comorbidities.csv": ["cohort_final.csv"],
    "severity_scores.csv": ["clean_long.csv.gz", "clean_vasoactive.csv.gz",
                            "clean_patient_context.csv", "static_comorbidities.csv"],
    "features_temporal.csv": ["clean_long.csv.gz"],
    "features_organ_stability.csv": ["clean_long.csv.gz", "clean_vasoactive.csv.gz",
                                     "clean_patient_context.csv"],
    "features_interactions.csv": ["clean_long.csv.gz", "severity_scores.csv",
                                  "features_organ_stability.csv"],
    "model_table_v2.csv": ["static_comorbidities.csv", "severity_scores.csv",
                           "clean_patient_context.csv", "features_temporal.csv",
                           "features_organ_stability.csv", "features_interactions.csv"],
}
stale = [f"{o} older than {i}" for o, ins in DEPS.items() for i in ins
         if o in present and i in present and mt(o) < mt(i)]
check("1 files", "every file is newer than the files it was built from", not stale,
      "; ".join(stale))

# ---------------- 2. cohort ----------------
co = pd.read_csv(P2 + "cohort_final.csv", parse_dates=["intime", "outtime"])
N = len(co)
if os.path.exists(P1 + "cohort_final.csv"):
    c1 = pd.read_csv(P1 + "cohort_final.csv", parse_dates=["intime", "outtime"])
    same = c1.reset_index(drop=True).equals(co.reset_index(drop=True))
    check("2 cohort", "processed_v2 cohort == Step 1 cohort (exact copy)", same)
check("2 cohort", "N = 2777", N == 2777, f"N={N}")
check("2 cohort", "one ICU stay per patient", co.subject_id.is_unique and co.stay_id.is_unique)
check("2 cohort", "age >= 18", (co.age_at_admission >= 18).all())
check("2 cohort", "in ICU for >= 24h (24h eligibility)",
      ((co.outtime - co.intime) >= pd.Timedelta(hours=24)).all())
era_ok = ((co.est_calendar_year <= 2017) == (co.temporal_set == "development_era")).all()
check("2 cohort", "era tag matches estimated year (<= 2017 = development)", era_ok)
check("2 cohort", "outcome is 0/1 with no missing",
      co.mortality_28d.notna().all() and set(co.mortality_28d.astype(int)) <= {0, 1})
print("\nCancer type counts:", co.cancer_type.value_counts().to_dict())

# ---------------- 3. same 2777 patients in every file ----------------
ids = set(co.stay_id)
for f in ["static_comorbidities.csv", "severity_scores.csv", "clean_patient_context.csv",
          "features_temporal.csv", "features_organ_stability.csv",
          "features_interactions.csv", "model_table_v2.csv"]:
    d = pd.read_csv(P2 + f, usecols=["stay_id"])
    check("3 patients", f"{f}: 2777 unique rows, same patients",
          len(d) == N and d.stay_id.is_unique and set(d.stay_id) == ids)

# ---------------- 4. time windows (no data after 24h) ----------------
L = pd.read_csv(P2 + "clean_long.csv.gz")
VIT = ["HR", "SBP", "DBP", "MAP", "RR", "SpO2", "Temp", "GCS_Eye", "GCS_Verbal",
       "GCS_Motor", "GCS_Total", "FiO2"]
lv = L[L.variable.isin(VIT)]
lb = L[~L.variable.isin(VIT)]
check("4 windows", "vitals within 0-24h", lv.hours.between(0, 24).all(),
      f"{lv.hours.min():.2f} to {lv.hours.max():.2f}")
check("4 windows", "labs within -6h to 24h", lb.hours.between(-6, 24).all(),
      f"{lb.hours.min():.2f} to {lb.hours.max():.2f}")
VA = pd.read_csv(P2 + "clean_vasoactive.csv.gz")
check("4 windows", "drug infusions overlap 0-24h",
      ((VA.start_h < 24) & (VA.end_h > 0)).all())
U = pd.read_csv(P2 + "clean_urine.csv.gz")
check("4 windows", "urine within 0-24h", U.hours.between(0, 24).all())
check("4 windows", "blood gases arterial only",
      not L.variable.isin(["pH", "PaO2", "PaCO2"]).any())

# ---------------- 5. leakage guards ----------------
T = pd.read_csv(P2 + "model_table_v2.csv")
KEYS = ["stay_id", "subject_id", "hadm_id", "mortality_28d", "est_calendar_year", "temporal_set"]
feat = [c for c in T.columns if c not in KEYS]
WORDS = ["mortality", "death", "expire", "discharge", "outtime", "subtype", "anchor",
         "temporal", "immortal", "hospice", "calendar", "year", "delirium", "cam_"]
TOK = ["dod", "los", "died", "dead", "label", "target", "outcome"]
bad = [c for c in feat if any(w in c.lower() for w in WORDS)
       or any(t in c.lower().split("_") for t in TOK)]
check("5 leakage", "no outcome/date/subtype words in feature names", not bad, str(bad))
banned_any = []
for f in EXPECTED:
    if f.endswith(".csv") and f not in ["cohort_final.csv", "model_table_v2.csv",
                                        "cleaning_log.csv", "feature_dictionary_v2.csv",
                                        "tpife_thresholds.csv",
                                        "cohort_delirium_timing.csv"]:
        cols = pd.read_csv(P2 + f, nrows=0).columns
        banned_any += [f"{f}:{c}" for c in cols
                       if any(w in c.lower() for w in ["mortality", "subtype", "death"])]
check("5 leakage", "no outcome or delirium-subtype column in any feature file",
      not banned_any, str(banned_any))
# Leakage detector only (nothing is chosen from it): a single feature that almost
# perfectly predicts death would mean future information slipped in.
y = T.mortality_28d.astype(int)
flag = []
for c in feat:
    if c == "cancer_type":
        continue
    x = T[c]
    m = x.notna()
    if m.sum() > 100 and x[m].nunique() > 1 and y[m].nunique() == 2:
        a = roc_auc_score(y[m], x[m])
        if max(a, 1 - a) > 0.85:
            flag.append(f"{c}={max(a, 1 - a):.2f}")
check("5 leakage", "no single feature suspiciously predicts death (AUC > 0.85)",
      not flag, ", ".join(flag))

# ---------------- 6. value ranges ----------------
def rng(c, lo, hi):
    v = T[c].dropna()
    check("6 values", f"{c} within [{lo}, {hi}]", v.between(lo, hi).all(),
          f"{v.min():.3g} to {v.max():.3g}")
for c in [c for c in feat if c.endswith(("_frac_abn", "_frac_severe"))]:
    rng(c, 0, 1)
for c in [c for c in feat if c.endswith("_deterioration")]:
    rng(c, -1, 1)
for c in [c for c in feat if c.startswith("osbi_") and not c.startswith("osbi_n")]:
    if c != "osbi_total":
        rng(c, 0, 1)
rng("osbi_total", 0, 7)
for c in ["sofa_resp", "sofa_coag", "sofa_liver", "sofa_cardio", "sofa_cns", "sofa_renal"]:
    rng(c, 0, 4)
rng("sofa_total", 0, 24)
rng("apache_ii_total", 0, 71)
rng("sirs_count", 0, 4)
check("6 values", "sirs_count is whole numbers", (T.sirs_count % 1 == 0).all())
rng("fio2_max", 0.21, 1)
rng("pf_min", 0, 700)
rng("age_at_admission", 18, 120)
rng("nee_max", 0, 10)
elix = [c for c in feat if c.startswith("elix_") and not c.startswith("elix_vw")]
check("6 values", "Elixhauser flags are 0/1",
      all(set(T[c].dropna().unique()) <= {0, 1} for c in elix))
num = T[[c for c in feat if c != "cancer_type"]].to_numpy(float)
check("6 values", "no infinite values", np.isfinite(num[~np.isnan(num)]).all())

# ---------------- 7. roadmap items for NB1-NB2 are present ----------------
def has(*cols):
    return all(c in T.columns for c in cols)
ROAD = {
    "24h prediction-time eligibility (Option A)": True,
    "cancer type by primary site": "cancer_type" in T,
    "Elixhauser (Quan) + van Walraven": has("elix_metacanc", "elix_vw_chronic"),
    "SOFA (Vincent 1996)": has("sofa_total", "sofa_resp"),
    "modified APACHE-II": has("apache_ii_total", "ap_oxygenation"),
    "basic temporal features": has("HR_mean", "MAP_slope", "Lactate_max"),
    "TPIFE stage 1 segmentation": has("MAP_early_mean", "MAP_mid_mean", "MAP_late_mean"),
    "TPIFE stage 2 states": os.path.exists(P2 + "tpife_thresholds.csv"),
    "TPIFE stage 3 extraction": has("MAP_frac_abn", "MAP_episodes_per24h",
                                    "MAP_first_episode_h", "MAP_transitions_per24h",
                                    "MAP_longest_episode_h", "MAP_deterioration"),
    "TPIFE stage 4 aggregation": has("tpife_mean_frac_abn", "tpife_n_vitals_ever_severe"),
    "Organ-System Burden Index": has("osbi_total", "osbi_cardiovascular",
                                     "osbi_n_systems_severe"),
    "Composite Stability Index components": has("stab_HR_arv", "stab_MAP_cv"),
    "temporal interaction pairs (3)": has("hypotension_x_lactate",
                                          "hypoxaemia_tachycardia_hours",
                                          "creat_rise_x_oliguria"),
    "SIRS integer + SIRS x SOFA": has("sirs_count", "sirs_x_sofa"),
    "fairness variables (sex, age)": has("sex_male", "age_at_admission"),
    "feature dictionary": os.path.exists(P2 + "feature_dictionary_v2.csv"),
}
for k, v in ROAD.items():
    check("7 roadmap", k, bool(v))

# ---------------- 8. comorbidity rules + delirium timing (from the full audit) ----------------
ACUTE = ["elix_fed", "elix_carit", "elix_coag", "elix_ond", "elix_wloss", "elix_pcd"]
check("8 audit fixes", "acute-type Elixhauser NOT used as predictors",
      not [c for c in ACUTE if c in feat])
check("8 audit fixes", "diabetes / hypertension merged across ICD-9/10",
      has("elix_diabetes", "elix_hypertension")
      and not [c for c in ["elix_diabc", "elix_diabunc", "elix_hypc", "elix_hypunc"]
               if c in feat])
check("8 audit fixes", "van Walraven score = chronic version only",
      "elix_vw_chronic" in feat and "elix_vw_score" not in feat)
DT = pd.read_csv(P2 + "cohort_delirium_timing.csv")
check("8 audit fixes", "delirium timing: 2777 patients, every one meets inclusion rule",
      len(DT) == N and set(DT.stay_id) == ids and (DT.delirium_route != "NONE").all())
check("8 audit fixes", "delirium timing columns are NOT model features",
      not [c for c in DT.columns if c != "stay_id" and c in feat])
print("\nDelirium route:", DT.delirium_route.value_counts().to_dict(),
      "| CAM-ICU positive by 24h:", int(DT.cam_positive_by_24h.sum()))

# ---------------- 9. figures ----------------
for f in ["attrition_diagram_final.png", "attrition_diagram_final.pdf"]:
    check("9 figures", f"figures_v2/{f}", os.path.exists(FIG2 + f))

# ---------------- report ----------------
REP = pd.DataFrame(R, columns=["section", "check", "result", "detail"])
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)
print()
print(REP.to_string(index=False))
n_fail = int((REP.result == "FAIL").sum())
print(f"\nTOTAL: {len(REP)} checks | PASS {len(REP) - n_fail} | FAIL {n_fail}")
print("\nItems that belong to NB3/NB4 (not expected yet):")
for s in ["splits (random 80/20 + calendar era)", "20% missing rule, MICE, scaling on TRAIN",
          "Composite Stability Index (scaled on TRAIN)", "SHAP-RFE feature selection in folds",
          "models, calibration, uncertainty", "SHAP stability, risk tiers, decision table",
          "sex/age fairness analysis"]:
    print("  -", s)
print("\nALL CHECKS PASSED - ready for Notebook 3" if n_fail == 0
      else "\nSTOP - send me this output before Notebook 3")

In [ ]:
ap = pd.read_csv("../data/processed_v2/icd_code_appendix.csv")
print(ap.columns.tolist())
print(ap.iloc[:, -1].value_counts().head(10))